# fig4A_regulon_dotplot

Existing analysis and drawing code is collected in this one panel notebook. Original shared methods may produce the complete original group of figures. The final drawing section below retains this panel's existing layout.

Configure the existing ROOT placeholders for the inputs and outputs of each original section. The primary notebook cells retain their original order and numeric positions. Standalone programs and source helpers are complete independent source sections; their original command-line, file, namespace, and kernel assumptions remain in force. No new Run All workflow or runtime worker generator is added.

---

# Figure 4 upstream analysis methods

Accepted developmental regulon display and the complete three-regulon supplement scoring/model/append engines. The accepted primary Fig4A analysis is supplied separately from the source archive in Deve_regulon_dynamics.h5; the older script/Fig4a.py is not used as the final primary analysis.

Every actual storage address is replaced by `ROOT` or a string placeholder `ROOT`. Configure each input, output, runtime, program, and worker path independently before execution. HDF5 keys, public URLs, archived-source labels, numerical settings, seeds, statistical tests, and selection gates retain their original meaning.

This source-only release contains no expression matrices, fitted models, execution outputs, or hidden notebook attachments. The files were checked statically; the analysis was not rerun.

---

# Developmental regulon dynamics

Fig4A on the fixed E9.5-E18.5 CVG/SGN tree, including the early candidate arms.

**Normal use:** run all cells. Only the single formal H5 is read; no network inference,
tree fitting, GAM fitting, cell filtering or primary re-testing is performed.
The full 88-regulon significant result is preserved. The default main panel displays
20 Charles Fig3a TF-name references and evidence-supported early/downstream representatives.
`PANEL_MODE='all_significant'` exports all 88 under a separate stem without refitting.

Cell 4 documents and audits the display rules, including observed early library support,
native-axis TF RNA and sensitivity checks. E9.5 remains in the tree/plot; its single library
cannot establish cross-library robustness. Unknown downstream RNA features remain unknown.
No named candidate is promoted by changing an individual threshold. Reference inclusion
does not imply covariate robustness or identical networks to Charles.

Change appearance in cell 5, then rerun cells 5-7. Column order follows the exact fitted peak
bin, activation centroid and TF name, making the early-to-late sequence visible while retaining
shared/multimodal dynamics. No activity values or row order are changed to make a diagonal.
Figures are 600-DPI PNG and live-text, pure-vector Arial PDF.
`EXPORT_EARLY=False` omits the optional independently normalized observed early panel.

The H5 retains raw AUCell, native tree coordinates, all 217 statistics, BH217 primary and
BH868 fork results, the original 88-regulon values, all display evidence/sensitivity checks,
early RNA/library evidence, executed analytical sources and export provenance.
Primary screen: independent branch cubic Gaussian ML smooths versus a constant null,
F test, BH q<1e-4 and fitted range>0.025 across all 217 candidates. Fork localization is a
separate 868-test family. Residual/covariate checks do not change this primary screen.
Early arms remain partly age-confounded; AUCell is target-set enrichment, not TF RNA or causality.

The main figure uses the nine complete root/fork/tip branch labels supplied
for Fig3. `CVG to Uns.` joins native fitted pieces 2 and 4 (six existing rows)
across connector 3; the other eight branches retain three rows each. All 30
fitted bins, 58 columns, column order and original per-regulon normalization
are unchanged. The seven unmodeled connector cells are never filled with zeros.
These are complete-branch display groups, not a new model or cell annotation.
Diagnostic and RNA-availability evidence remains in the H5 without symbols on
the main figure. Negative Gaussian predictions are floored at zero only for
display; raw values, tests and genuine missing-data semantics are unchanged.


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#1. =========== Shared paths and runtime
from pathlib import Path
import os,sys,json,hashlib,subprocess,shutil

# Set this storage path independently before running.
ROOT=ROOT
if ROOT is None:
    # Set this storage path independently before running.
    ROOT=ROOT
# Set this storage path independently before running. Deve_regulon_dynamics.h5. Accepted primary result produced by fig4A_regulon_dynamics_analysis.py.
RESULT_FILE=ROOT
# Set this storage path independently before running.
FIGURE_DIR=ROOT
# Set this storage path independently before running. 05_Deve_regulon_dynamics.ipynb. Use the supplied fig4_regulon_supplement_analysis.ipynb source.
NOTEBOOK_FILE=ROOT
# Set this storage path independently before running.
CACHE_DIR=ROOT
DPI=600
BASE_OUTPUT_STEM='fig4A_Developmental_regulon_dynamics'
OUTPUT_STEM=BASE_OUTPUT_STEM
MAIN_TITLE='Developmental regulon dynamics'
SEED=42
Q_CUT=1e-4  # Frozen accepted threshold; not an appearance parameter.

CACHE_DIR.mkdir(parents=True,exist_ok=True)
FIGURE_DIR.mkdir(parents=True,exist_ok=True)
for key in ('TMP','TEMP','MPLCONFIGDIR'):os.environ[key]=str(CACHE_DIR)
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='1'
sys.dont_write_bytecode=True
import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
from matplotlib import font_manager
from PIL import Image

for font in ('arial.ttf','ariali.ttf','arialbd.ttf','arialbi.ttf'):
    # Set this storage path independently before running.
    path=ROOT/font
    if path.exists():font_manager.fontManager.addfont(str(path))
assert font_manager.findfont('Arial',fallback_to_default=False)
np.random.seed(SEED)


In [ ]:
#2. =========== Load the single frozen result file
def read_frame(group):
    return pd.DataFrame({c:(group[c].asstr()[()] if group[c].dtype.kind in 'OSU' else group[c][()])
        for c in group['columns'].asstr()[()]},index=group['index'].asstr()[()])

def column_frame(group):
    return pd.DataFrame({k:(v.asstr()[()] if v.dtype.kind in 'OSU' else v[()])
        for k,v in group.items() if not k.endswith('__missing')})

def content_hash(value):
    return hashlib.sha256(np.asarray(value,dtype='<f8').tobytes()).hexdigest()

assert RESULT_FILE.is_file(),f'Missing formal result: {RESULT_FILE}'
with h5py.File(RESULT_FILE,'r') as h:
    names=pd.Index(h['regulons'].asstr()[()])
    order=h['final/regulons'].asstr()[()].tolist()
    stats=read_frame(h['final/statistics']).loc[names]
    data=dict(names=names,segments=json.loads(h['segments_json'].asstr()[()]),
        bin_meta=column_frame(h['bin_metadata']),
        modes={'none':{'bins':pd.DataFrame(h['final/all_fitted_bin_auc'][()],columns=names)}},
        early_mean=pd.DataFrame(h['final/all_early_observed_auc'][()],columns=names),
        early_meta=column_frame(h['early_extension/group_metadata']))
    saved_relative=h['final/relative_activity'][()]
    saved_negative=h['final/negative_fitted_bin_mask'][()]
    result_manifest=json.loads(h['manifest_json'].asstr()[()])
    accepted_assessment=json.loads(h['final/assessment_json'].asstr()[()])
    assert len(h['cells'])==7355 and len(names)==217
    assert content_hash(h['auc'][()])==result_manifest['auc_content_sha256']


In [ ]:
#3. =========== Verify frozen scientific results
def bh(values):
    p=np.asarray(values,float);n=len(p);indices=np.argsort(p,kind='stable')
    answer=np.empty(n);answer[indices]=np.minimum(1,np.minimum.accumulate((p[indices]*n/np.arange(1,n+1))[::-1])[::-1])
    return answer

assert len(order)==88 and len(set(order))==88 and len(data['segments'])==10
assert data['bin_meta'].shape[0]==30 and data['bin_meta'].cells.min()>=31
np.testing.assert_allclose(bh(stats.p_value.where(stats.status.eq('ok'),1)),stats.q_global,atol=1e-15,rtol=1e-13)
selected=stats.status.eq('ok') & stats.q_global.lt(1e-4) & stats.A.gt(.025)
assert set(order)==set(stats.index[selected])
p=stats[['p_early','p_fork1','p_fork2','p_fork3']].to_numpy()
np.testing.assert_allclose(bh(p.ravel()).reshape(217,4),
    stats[['q_early','q_fork1','q_fork2','q_fork3']].to_numpy(),atol=1e-15,rtol=1e-13)
fitted=data['modes']['none']['bins'].loc[:,order].to_numpy()
relative=np.where(fitted < -1e-12,np.nan,np.maximum(fitted/fitted.max(axis=0),0))
np.testing.assert_array_equal(relative,saved_relative)
np.testing.assert_array_equal(fitted < -1e-12,saved_negative)
assert saved_negative.sum()==1 and stats.loc[order,'residual_flags'].gt(0).sum()==3
print('Loaded 88 accepted regulons, 10 segments and 30 fitted bins; no models were fitted.')


In [ ]:
#4. =========== Select representative columns without changing the primary screen
# These are transparent display rules, not additional primary significance tests.
PANEL_MODE='representatives'  # 'all_significant' shows all 88, using a separate output stem.
DISPLAY_RULES=dict(profile_rho_min=.50, early_auc_range_min=.025,
    high_activity_fraction=.80, rna_detection_min=.05,
    rna_positive_cells_min=5, rna_positive_libraries_min=2,
    redundancy_profile_rho_min=.90, redundancy_target_overlap_min=.50)
CHARLES_TFS=set('Chd1 Patz1 Pou2f1 Ybx1 E4f1 Klf7 Srebf2 Onecut2 Smad1 Sox4 '
    'Shox2 Bach2 Sox11 Etv5 Etv4 Pou4f1 Thrb Zfhx2 Bhlhe40 Mafb'.split())
SENSITIVITY_CHECKS=[('profile_rho_min',.40),('profile_rho_min',.60),
    ('rna_detection_min',.025),('rna_detection_min',.10),
    ('early_auc_range_min',.020),('early_auc_range_min',.030),
    ('high_activity_fraction',.75),('high_activity_fraction',.90)]

def display_evidence(result_file,stats,full_order,rules):
    from scipy.stats import spearmanr
    with h5py.File(result_file,'r') as h:
        names=pd.Index(h['regulons'].asstr()[()]);index={n:i for i,n in enumerate(names)}
        fitted=h['final/all_fitted_bin_auc'][()]
        bin_meta=column_frame(h['bin_metadata'])
        obs=column_frame(h['obs'])
        early_obs=column_frame(h['early_extension/obs'])
        auc=h['early_extension/summaries/common_auc/mean'][()]
        auc_lo=h['early_extension/summaries/common_auc/loo_min'][()]
        auc_hi=h['early_extension/summaries/common_auc/loo_max'][()]
        det_lo=h['early_extension/summaries/native_TF_detection_fraction/loo_min'][()]
        det=h['early_extension/cell_values/native_TF_detection_fraction'][()]
        late_rna=h['tf_rna_counts'][()]
        networks=json.loads(h['original_regulons_json'].asstr()[()])
        scoring_genes=set(h['genes'].asstr()[()])
    early_groups=np.arange(1,7)  # E9.5 has one library; retain in plot, not LOO validation.
    assert early_obs['group'].isin(range(7)).all()
    fit=fitted[:,[index[n] for n in full_order]]
    peak=fit.argmax(axis=0)
    weight=fit-fit.min(axis=0)
    centroid=(np.arange(30)[:,None]*weight).sum(axis=0)/weight.sum(axis=0)
    rows=[]
    for k,name in enumerate(full_order):
        j=index[name];s=stats.loc[name];band=int(peak[k]//3)
        is_early=band<3;reference=name.removesuffix('(+)') in CHARLES_TFS
        row=dict(regulon=name,charles_reference=reference,peak_bin=int(peak[k]),
            peak_band=band,activation_centroid=float(centroid[k]),
            library_profile_rho=float(s.library_profile_rho),
            residual_flag=bool(s.residual_flags>0),
            profile_support=bool(s.library_profile_rho>=rules['profile_rho_min']),
            domain='early' if is_early else 'downstream',
            early_observed_range=np.nan,early_loo_contrast=np.nan,
            early_high_groups='',observed_dynamics_support=False,
            rna_measured=True,rna_support=False,rna_detection=np.nan,
            rna_positive_cells=0,rna_positive_libraries=0)
        if is_early:
            observed=auc[early_groups,j]
            top=int(early_groups[np.argmax(observed)]);bottom=int(early_groups[np.argmin(observed)])
            amplitude=float(np.ptp(observed))
            contrast=float(auc_lo[top,j]-auc_hi[bottom,j])
            high=early_groups[observed>=rules['high_activity_fraction']*observed.max()]
            support_groups=[];best=None
            for g in high:
                positive=(early_obs['group'].to_numpy()==g)&(det[:,j]>0)
                cells=int(positive.sum());libraries=int(early_obs.loc[positive,'library'].nunique())
                metric=(float(det_lo[g,j]),cells,libraries)
                if best is None or metric>best:best=metric
                if metric[0]>=rules['rna_detection_min'] and cells>=rules['rna_positive_cells_min'] and libraries>=rules['rna_positive_libraries_min']:
                    support_groups.append(int(g))
            row.update(early_observed_range=amplitude,early_loo_contrast=contrast,
                early_high_groups=','.join(str(int(g)) for g in high),
                early_rna_support_groups=','.join(map(str,support_groups)),
                observed_dynamics_support=bool((amplitude>rules['early_auc_range_min'] or s.q_early<Q_CUT) and contrast>0),
                rna_measured=bool(np.isfinite(det[:,j]).all()),rna_support=bool(support_groups),
                rna_detection=best[0],rna_positive_cells=best[1],rna_positive_libraries=best[2])
        else:
            seg=int(bin_meta.iloc[int(peak[k])].seg)
            segment=obs.seg.to_numpy()==seg
            values=late_rna[segment,j];measured=bool(np.isfinite(values).any())
            if measured:
                assert np.isfinite(values).all(),(name,'partially missing peak-segment RNA')
                positive=segment & (late_rna[:,j]>0)
                detection=float((values>0).mean());cells=int(positive.sum())
                libraries=int(obs.loc[positive,'library'].nunique())
                rna_support=detection>=rules['rna_detection_min'] and cells>=rules['rna_positive_cells_min'] and libraries>=rules['rna_positive_libraries_min']
            else:
                detection=np.nan;cells=0;libraries=0;rna_support=False
            row.update(peak_segment=seg,observed_dynamics_support=bool(s[['q_fork1','q_fork2','q_fork3']].min()<Q_CUT),
                rna_measured=measured,rna_support=bool(rna_support),rna_detection=detection,
                rna_positive_cells=cells,rna_positive_libraries=libraries)
        # Missing downstream RNA features are unknown, never evidence of zero expression.
        row['automatic_representative']=bool(row['profile_support'] and row['observed_dynamics_support'] and
            (row['rna_support'] or (not is_early and not row['rna_measured'])))
        row['display_selected']=bool(reference or row['automatic_representative'])
        row['selection_reason']='literature reference' if reference else ('evidence-supported representative' if row['display_selected'] else 'full-results view only')
        rows.append(row)
    evidence=pd.DataFrame(rows).set_index('regulon')
    selected=evidence.index[evidence.display_selected].tolist()
    selected=sorted(selected,key=lambda n:(int(evidence.loc[n,'peak_bin']),float(evidence.loc[n,'activation_centroid']),n))
    assert set(evidence.index[evidence.charles_reference])=={n+'(+)' for n in CHARLES_TFS}
    # Diagnose redundancy, retaining distinct target networks despite similar curves.
    targets={n['name']:set(n['gene2weight']) & scoring_genes for n in networks}
    redundant=[];max_jaccard=0.
    for a,left in enumerate(selected):
        for right in selected[a+1:]:
            shared=len(targets[left]&targets[right]);union=len(targets[left]|targets[right])
            jaccard=shared/union if union else 0.;max_jaccard=max(max_jaccard,jaccard)
            overlap=shared/min(len(targets[left]),len(targets[right]))
            rho=float(spearmanr(fitted[:,index[left]],fitted[:,index[right]]).statistic)
            if rho>=rules['redundancy_profile_rho_min'] and overlap>=rules['redundancy_target_overlap_min']:
                redundant.append(dict(left=left,right=right,profile_rho=rho,target_overlap=overlap,target_jaccard=jaccard))
    return evidence,selected,dict(redundant_pairs=redundant,maximum_target_jaccard=max_jaccard)

with h5py.File(RESULT_FILE,'r') as h:
    full_order=h['final/regulons'].asstr()[()].tolist()
evidence,representative_order,redundancy=display_evidence(RESULT_FILE,stats,full_order,DISPLAY_RULES)
assert not redundancy['redundant_pairs'], 'Review redundant representatives before export.'
sensitivity=[];primary_set=set(representative_order)
all_checks=[primary_set]
for parameter,value in SENSITIVITY_CHECKS:
    trial=dict(DISPLAY_RULES);trial[parameter]=value
    _,trial_order,_=display_evidence(RESULT_FILE,stats,full_order,trial)
    trial_set=set(trial_order);all_checks.append(trial_set)
    sensitivity.append(dict(parameter=parameter,value=value,count=len(trial_set),
        added=sorted(trial_set-primary_set),removed=sorted(primary_set-trial_set)))
stable=set.intersection(*all_checks);union=set.union(*all_checks)
selection_audit=dict(rules=DISPLAY_RULES,sensitivity=sensitivity,
    stable_count=len(stable),union_count=len(union),stable_regulons=sorted(stable),
    reference_count=int(evidence.loc[representative_order,'charles_reference'].sum()),
    representative_count=int((~evidence.loc[representative_order,'charles_reference']).sum()),
    early_peak_count=int(evidence.loc[representative_order,'peak_band'].lt(3).sum()),
    downstream_peak_count=int(evidence.loc[representative_order,'peak_band'].ge(3).sum()),
    peak_bin_inversions=int(sum(evidence.loc[representative_order[i],'peak_bin']>evidence.loc[representative_order[j],'peak_bin']
        for i in range(len(representative_order)) for j in range(i+1,len(representative_order)))),
    redundancy=redundancy,notes=[
        'Display selection is separate from the frozen BH217 screen; all 88 remain available.',
        'Reference matches are TF names, not evidence of identical target sets or activity.',
        'Early support uses equal-library observed AUC, leave-one-library-out contrast and native-axis TF RNA.',
        'Downstream RNA unavailable means unknown. Residual flags do not exclude complex dynamics.',
        'Ordering follows the actual peak bin, then activation centroid and TF name; fitted values are unchanged.',
        'Stage/state association does not establish a causal fate determinant.',
        'The early versus downstream display emphasis is asymmetric; unshown late shared-stage programs remain valid.',
        'Leave-one-library-out bounds are descriptive sensitivity, not confidence intervals or elimination of confounding.',
        'Tree DFS row order is not a single chronological path; zero peak inversions only validates sorting.',
        'RNA evidence axis: early native counts, downstream common scoring-gene counts; unavailable does not mean absent.'])
assert selection_audit['reference_count']==20 and selection_audit['peak_bin_inversions']==0
assert PANEL_MODE in ('representatives','all_significant')
order=representative_order if PANEL_MODE=='representatives' else sorted(full_order,
    key=lambda n:(int(evidence.loc[n,'peak_bin']),float(evidence.loc[n,'activation_centroid']),n))
OUTPUT_STEM=BASE_OUTPUT_STEM+('_all' if PANEL_MODE=='all_significant' else '')
stats=stats.drop(columns=['charles_reference','rna_measured'],errors='ignore').join(evidence[['charles_reference','rna_measured']])
print(f"Display: {len(order)} / 88 significant regulons; {selection_audit['reference_count']} references + "
    f"{selection_audit['representative_count']} representatives. Primary BH217 statistics unchanged.")
print(f"Sensitivity: {len(stable)} stable; {len(union)} in union. Peak-bin inversions: 0.")


In [ ]:
#5. =========== Appearance parameters and editable vector renderer
# These settings change presentation only; all primary results stay frozen.
MAX_COLUMNS_PER_PANEL=72
EXPORT_EARLY=False       # Optional observed early panel; independent normalization, not the fitted-tree metric.
SEGMENT_LABELS={0:'Undiff. CVG',1:'CVG to VGN',2:'CVG to Uns.',5:'Ib/Ic shared',
    9:'Ib-related end',10:'Ic-related end',6:'Ia/II shared',7:'Ia-related end',8:'II-related end'}
# Preserve every original bin. Only join consecutive pieces on a complete path.
DISPLAY_BRANCHES=[
    dict(id=0,parent=None,source_segments=[0],connector_segments=[]),
    dict(id=1,parent=0,source_segments=[1],connector_segments=[]),
    dict(id=2,parent=0,source_segments=[2,4],connector_segments=[3]),
    dict(id=5,parent=2,source_segments=[5],connector_segments=[]),
    dict(id=9,parent=5,source_segments=[9],connector_segments=[]),
    dict(id=10,parent=5,source_segments=[10],connector_segments=[]),
    dict(id=6,parent=2,source_segments=[6],connector_segments=[]),
    dict(id=7,parent=6,source_segments=[7],connector_segments=[]),
    dict(id=8,parent=6,source_segments=[8],connector_segments=[])]
PLOT_STYLE=dict(
    column_pitch_in=.128, row_pitch_in=.100, segment_gap_in=.045,
    dot_diameter_ratio=.81, dot_edge_width=.32,
    gene_fontsize=6.0, row_fontsize=5.6, title_fontsize=11.0,
    subtitle_fontsize=5.7, footer_fontsize=4.8, legend_fontsize=5.5, legend_tick_fontsize=5.0,
    main_left_in=1.27, early_left_in=2.35, top_in=.62, bottom_in=1.24,
    tree_end_offset_in=.77, tree_primary_lane_in=.22, tree_secondary_lane_in=.375,
    gradient_steps=128)

def render(data,stats,order):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_rgb,LinearSegmentedColormap
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle,Circle
    plt.rcParams.update({'font.family':'Arial','font.size':6,'axes.linewidth':.45,
                         'savefig.dpi':DPI,'figure.dpi':120,'pdf.fonttype':42,
                         'pdf.use14corefonts':False,'pdf.compression':9})
    selected=pd.Index(order)
    fitted=data['modes']['none']['bins'].loc[:,selected].to_numpy()
    assert np.isfinite(fitted).all(), 'Missing fitted AUC must not be displayed as zero.'
    maxima=fitted.max(axis=0)
    assert np.all(maxima>0)
    raw_relative=fitted/maxima
    # Floor only the display of negative Gaussian predictions at the AUC lower bound.
    # Keep raw fitted values, tests, diagnostics and missing-data semantics unchanged.
    negative=fitted < -1e-12
    plotted=np.maximum(raw_relative,0)
    assert np.nanmax(plotted)<=1+1e-12
    early=data['early_mean'].loc[:,selected].to_numpy()
    assert np.isfinite(early).all(), 'Missing observed AUC must not be displayed as zero.'
    early_max=early.max(axis=0)
    early_relative=np.divide(early,early_max,out=np.zeros_like(early),where=early_max>0)
    red=stats.loc[selected,['q_early','q_fork1','q_fork2','q_fork3']].min(axis=1).lt(Q_CUT).to_numpy()
    references=stats.loc[selected,'charles_reference'].to_numpy(dtype=bool)
    cmap=LinearSegmentedColormap.from_list('white_red',['white','#ff0000'])
    dpi=DPI;col_pitch=PLOT_STYLE['column_pitch_in'];row_pitch=PLOT_STYLE['row_pitch_in'];gap=PLOT_STYLE['segment_gap_in']
    # A complete-branch display groups existing rows without resampling or refitting.
    branches=[];row_order=[];Y_main=np.empty(len(fitted),float);cursor=0.
    for specification in DISPLAY_BRANCHES:
        branch=dict(specification);rows=[]
        for sid in branch['source_segments']:
            rows.extend(np.flatnonzero(data['bin_meta'].seg.to_numpy()==sid).tolist())
        branch.update(label=SEGMENT_LABELS[branch['id']],rows=rows,
            row_count=len(rows),axis_top=float(cursor))
        Y_main[rows]=np.arange(len(rows))+.5+cursor
        row_order.extend(rows);branches.append(branch)
        cursor+=len(rows)+gap/row_pitch
    assert len(branches)==9 and [b['row_count'] for b in branches]==[3,3,6,3,3,3,3,3,3]
    np.testing.assert_array_equal(row_order,np.arange(len(fitted)))
    assert not data['bin_meta'].seg.eq(3).any(), 'Unmodeled connector cells must not acquire AUC points.'
    source_segments={s['id']:s for s in data['segments']}
    output=[];checks={}
    def draw(indices,path,early_panel=False,part=None):
        n=len(indices);left=PLOT_STYLE['early_left_in'] if early_panel else PLOT_STYLE['main_left_in']
        top=PLOT_STYLE['top_in'];bottom=PLOT_STYLE['bottom_in']
        nrows=7 if early_panel else 30
        body=nrows*row_pitch+(0 if early_panel else (len(branches)-1)*gap)
        width=left+n*col_pitch+.09;height=top+body+bottom
        fig=plt.figure(figsize=(width,height),facecolor='white')
        ax=fig.add_axes([left/width,bottom/height,n*col_pitch/width,body/height])
        ax.set_xlim(0,n);ax.set_ylim(body/row_pitch,0)
        ax.set_xticks(np.arange(n)+.5)
        labels=[selected[i] for i in indices]
        ax.set_xticklabels(labels,rotation=90,ha='center',va='top',fontsize=PLOT_STYLE['gene_fontsize'],fontstyle='italic')
        ax.tick_params(axis='x',length=0,pad=2)
        ax.set_yticks([])
        for spine in ax.spines.values():spine.set_visible(False)
        Y=np.arange(nrows)+.5
        if not early_panel:
            Y=Y_main.copy()
        vals=early_relative[:,indices] if early_panel else plotted[:,indices]
        x,y=np.meshgrid(np.arange(n)+.5,Y)
        diameter=PLOT_STYLE['dot_diameter_ratio']*row_pitch*72
        assert np.isfinite(vals).all()
        size=diameter**2*vals
        # Use independent closed paths in physical inches: no marker Forms.
        for r,c in np.ndindex(vals.shape):
            xpos=left+x[r,c]*col_pitch
            ypos=bottom+body-y[r,c]*row_pitch
            value=float(vals[r,c])
            radius=np.sqrt(float(size[r,c]))/(2*72)
            if value<=0:
                radius=np.sqrt(.18)/(2*72)
                face='#555555';edge='none';line=0
            else:
                face=cmap(value);edge='#4c4c4c';line=PLOT_STYLE['dot_edge_width']
            fig.add_artist(Circle((xpos,ypos),radius=radius,transform=fig.dpi_scale_trans,
                facecolor=face,edgecolor=edge,linewidth=line,clip_on=False,zorder=3))
        for tick,i in zip(ax.get_xticklabels(),indices):
            tick.set_color('#dc2638' if red[i] else '#111111')
            tick.set_fontweight('bold' if references[i] else 'normal')
        title='Observed early regulon activity' if early_panel else MAIN_TITLE
        suffix='' if part is None else f' ({part})'
        fig.text(.12/width,1-.23/height,('' if early_panel else 'a  ')+title+suffix,
                 fontsize=PLOT_STYLE['title_fontsize'],va='center')
        subtitle=('E9.5-E13.5; equal weight per library within each stage/state; observed AUC' if early_panel else
                  f'E9.5-E18.5; {len(selected)} regulons across 9 complete branches')
        fig.text(.12/width,1-.47/height,subtitle,fontsize=PLOT_STYLE['subtitle_fontsize'],va='center')
        if early_panel:
            label_lines=['E9.5  CVG neuroblast','E10.5  delaminating otic/CVG','E10.5  developing CVG',
                         'E11.5  proliferative CVG','E11.5  differentiating SGN',
                         'E13.5  late CVG','E13.5  differentiating SGN']
            for r,label in enumerate(label_lines):
                yc=(bottom+body-Y[r]*row_pitch)/height
                m=data['early_meta'].iloc[r]
                fig.text((left-.09)/width,yc,label+f' ({int(m.cells)}; {int(m.libraries)})',
                         ha='right',va='center',fontsize=PLOT_STYLE['row_fontsize'])
            ax.add_patch(Rectangle((0,0),n,nrows,fill=False,edgecolor='#333333',lw=.45,clip_on=False))
        else:
            centre={}
            for branch in branches:
                y0=branch['axis_top'];nr=branch['row_count']
                ax.add_patch(Rectangle((0,y0),n,nr,fill=False,edgecolor='#333333',lw=.45,clip_on=False))
                yb=(bottom+body-(y0+nr)*row_pitch)/height
                strip=fig.add_axes([(left-.15)/width,yb,.065/width,nr*row_pitch/height])
                strip.set_xlim(0,1);strip.set_ylim(0,1)
                steps=PLOT_STYLE['gradient_steps'];piece_start=0
                # Preserve each native color transition inside the joined branch strip.
                for sid in branch['source_segments']:
                    source=source_segments[sid]
                    count=int(data['bin_meta'].seg.eq(sid).sum())
                    colors=np.array([to_rgb(source['color_from']),to_rgb(source['color_to'])])
                    for step in range(steps):
                        weight=step/(steps-1)
                        face=colors[0]*(1-weight)+colors[1]*weight
                        top_fraction=(piece_start+count*(step+1)/steps)/nr
                        strip.add_patch(Rectangle((0,1-top_fraction),1,count/(steps*nr),
                            transform=strip.transAxes,facecolor=face,edgecolor='none',
                            linewidth=0,antialiased=False,clip_on=False))
                    piece_start+=count
                strip.set_xticks([]);strip.set_yticks([])
                for spine in strip.spines.values():spine.set_linewidth(.4)
                centre[branch['id']]=(bottom+body-(y0+nr/2)*row_pitch)/height
                fig.text((left-.205)/width,centre[branch['id']],branch['label'],
                         ha='right',va='center',fontsize=PLOT_STYLE['row_fontsize'])
            end=(left-PLOT_STYLE['tree_end_offset_in'])/width
            for parent in (0,2,5,6):
                children=[b['id'] for b in branches if b['parent']==parent]
                lane=PLOT_STYLE['tree_primary_lane_in' if parent in (0,2) else 'tree_secondary_lane_in']/width
                ys=[centre[parent]]+[centre[c] for c in children]
                fig.add_artist(Line2D([lane,lane],[min(ys),max(ys)],transform=fig.transFigure,color='#444444',lw=.5))
                for sid in [parent]+children:
                    fig.add_artist(Line2D([lane,end],[centre[sid],centre[sid]],transform=fig.transFigure,
                                         color='#444444',lw=.5))
        ly=.42
        legend_left=.14
        fig.text(legend_left/width,(ly+.17)/height,
                 'relative observed AUC' if early_panel else 'relative fitted AUC',fontsize=PLOT_STYLE['legend_fontsize'],va='center')
        base=1.44
        for j,value in enumerate((0,.25,.5,.75,1)):
            xpos=base+j*.29
            fig.add_artist(Circle((xpos,ly+.17),radius=(np.sqrt(.18) if value==0 else diameter*np.sqrt(value))/(2*72),
                facecolor='#555555' if value==0 else cmap(float(value)),
                edgecolor='none' if value==0 else '#4c4c4c',linewidth=0 if value==0 else PLOT_STYLE['dot_edge_width'],
                transform=fig.dpi_scale_trans,clip_on=False,zorder=3))
            fig.text(xpos/width,ly/height,f'{value:g}',fontsize=PLOT_STYLE['legend_tick_fontsize'],ha='center',va='top')
        note1=('Each regulon is max-normalized across seven observed groups; row brackets give cells; libraries.' if early_panel else
               'Max-normalized across 30 fitted bins; CVG to Uns. joins two fitted pieces (six rows).')
        note2='Bold: Charles Fig3a TF match; red: fork BH868 q < 1e-4.'
        for pos,note in zip((.18,.06),(note1,note2)):
            fig.text(.14/width,pos/height,note,fontsize=PLOT_STYLE['footer_fontsize'],va='bottom')
        # Clear default artist clipping; no clipping is needed for this layout.
        for axis in fig.axes:
            for axis_object in (axis.xaxis,axis.yaxis):
                for tick in axis_object.get_major_ticks()+axis_object.get_minor_ticks():
                    tick.tick1line.set_visible(False);tick.tick2line.set_visible(False)
        for artist in fig.findobj():
            if hasattr(artist,'set_clip_on'):artist.set_clip_on(False)
            if hasattr(artist,'set_clip_box'):artist.set_clip_box(None)
            if hasattr(artist,'set_clip_path'):artist.set_clip_path(None)
        fig.canvas.draw()
        renderer=fig.canvas.get_renderer()
        texts=list(fig.texts)+list(ax.get_xticklabels())
        boxes=[(t,t.get_window_extent(renderer)) for t in texts if t.get_text()]
        canvas=fig.bbox
        outside=[t.get_text() for t,b in boxes if b.x0<-.5 or b.y0<-.5 or b.x1>canvas.x1+.5 or b.y1>canvas.y1+.5]
        overlaps=[]
        for i,(t,b) in enumerate(boxes):
            for u,c in boxes[i+1:]:
                if min(b.x1,c.x1)-max(b.x0,c.x0)>.5 and min(b.y1,c.y1)-max(b.y0,c.y0)>.5:
                    overlaps.append([t.get_text(),u.get_text()])
        assert not outside,(path.name,'text outside canvas',outside)
        assert not overlaps,(path.name,'text overlap',overlaps)
        pdf_path=path.with_suffix('.pdf')
        fig.savefig(path,dpi=dpi,facecolor='white')
        fig.savefig(pdf_path,facecolor='white',metadata={'Title':MAIN_TITLE,'Creator':'05_Deve_regulon_dynamics.ipynb','CreationDate':None,'ModDate':None})
        checks[path.name]=dict(columns=n,rows=nrows,complete_branches=None if early_panel else len(branches),
            branch_row_counts=None if early_panel else [b['row_count'] for b in branches],
            dpi=dpi,width_in=width,height_in=height,
            text_outside=outside,text_overlap=overlaps,pdf=pdf_path.name,expected_clips=0,
            bytes=path.stat().st_size,pdf_bytes=pdf_path.stat().st_size)
        output.extend([path.name,pdf_path.name]);plt.close(fig)
    draw(np.arange(len(selected)),FIGURE_DIR/(OUTPUT_STEM+'.png'))
    if len(selected)>MAX_COLUMNS_PER_PANEL:
        for j,ids in enumerate(np.array_split(np.arange(len(selected)),int(np.ceil(len(selected)/MAX_COLUMNS_PER_PANEL)))):
            draw(ids,FIGURE_DIR/f'{OUTPUT_STEM}_{j+1}.png',part=f'{j+1}/{int(np.ceil(len(selected)/MAX_COLUMNS_PER_PANEL))}')
    if EXPORT_EARLY:
        draw(np.arange(len(selected)),FIGURE_DIR/(OUTPUT_STEM+'_early.png'),early_panel=True)
    return dict(files=output,figures=checks,negative_fitted_bin_count=int(negative.sum()),
                negative_bins=[dict(segment=int(data['bin_meta'].iloc[r].seg),bin=int(data['bin_meta'].iloc[r]['bin']),
                                    regulon=selected[c],raw_mean=float(fitted[r,c])) for r,c in np.argwhere(negative)],
                display_zero_tolerance=1e-12,normalization='per regulon maximum across fitted 30 bins; early observed independent maximum across 7 groups',
                label_red='any fork BH868 q<1e-4; not a primary gate',
                reference_mark='bold italic: Charles Fig3a TF-name match',
                segment_labels=SEGMENT_LABELS,display_branches=branches,
                display_row_positions=Y_main.tolist(),original_row_order=row_order,
                diagnostic_symbols_shown=False,
                negative_display_policy='max(raw fitted AUC,0), then per-regulon maximum normalization; ordinary zero-activity dot',
                display_floor_applied_bin_count=int((fitted<0).sum()),
                diagnostics_location='final/statistics and display_selection/evidence_json; raw values retained')

In [ ]:
#6. =========== Export PNG and editable PDF from the same canvas
figure_qa=render(data,stats,order)
assert figure_qa['negative_fitted_bin_count']==1
assert len(figure_qa['display_branches'])==9
assert figure_qa['original_row_order']==list(range(30))
print(f"Exported {len(figure_qa['figures'])} PNG/PDF pairs to {FIGURE_DIR}")


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

# Cleanup must never interpret the project/current directory as a temporary target.
def _release_cleanup_target(path):
    candidate = Path(path).resolve()
    current = Path.cwd().resolve()
    if candidate == current or candidate in current.parents or candidate == Path(candidate.anchor):
        raise RuntimeError("Configure the owned temporary cleanup path before running this section.")
    return path

#7. =========== Validate PDFs, save export provenance and clean runtime files
# Set this storage path independently before running.
SKILL_ROOT=ROOT
# Set this storage path independently before running. validate_figure_output.py. Use the supplied fig4_validate_figure_output.py.
VALIDATOR=ROOT
# Set this storage path independently before running.
RUNTIME=ROOT
# Set this storage path independently before running. python.exe.
QA_PYTHON=ROOT
# Set this storage path independently before running. pdftoppm.exe.
PDFTOPPM=ROOT
RENDER_DPI=120
KEEP_QA_RENDERS=False  # Keep Poppler pages only when manually inspecting a new appearance.

assert VALIDATOR.is_file() and QA_PYTHON.is_file() and PDFTOPPM.is_file(), 'PDF output is NOT CERTIFIED: missing validator or Poppler.'
validation_reports={}
for png_name,info in figure_qa['figures'].items():
    png=FIGURE_DIR/png_name;pdf=FIGURE_DIR/info['pdf']
    result=subprocess.run([str(QA_PYTHON),'-B',str(VALIDATOR),str(png),str(pdf),
        '--target-dpi',str(DPI),'--expected-size-in',f"{info['width_in']}x{info['height_in']}",
        '--expected-clips','0','--max-clips','0','--pdftoppm',str(PDFTOPPM),
        # Set this storage path independently before running.
        '--render-dpi',str(RENDER_DPI),'--render-dir',str(ROOT)],
        capture_output=True,text=True,encoding='utf-8',errors='replace',env=os.environ.copy())
    validation_reports[pdf.name]=result.stdout+result.stderr
    if result.returncode:
        print(validation_reports[pdf.name]);raise RuntimeError(f'Figure validation failed: {pdf}')
    with Image.open(png) as im:
        info['pixel_size']=list(im.size);info['actual_dpi']=list(im.info['dpi'])

def sha_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):digest.update(block)
    return digest.hexdigest()

notebook=json.loads(NOTEBOOK_FILE.read_text(encoding='utf-8'))
source_hash=hashlib.sha256('\n'.join(''.join(c.get('source',[])) for c in notebook['cells']).encode('utf-8')).hexdigest()
appearance_source=next(''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code' and 'def render(' in ''.join(c['source']))
exporter_source=appearance_source[appearance_source.index('def render('):]
exporter_hash=hashlib.sha256(exporter_source.encode('utf-8')).hexdigest()
publication=dict(title=MAIN_TITLE,formats=['png','pdf'],dpi=DPI,parameters=PLOT_STYLE,
    source_notebook=str(NOTEBOOK_FILE.relative_to(ROOT)),notebook_source_sha256=source_hash,
    exporter_source_sha256=exporter_hash,matplotlib_version=matplotlib.__version__,
    figure_qa=figure_qa,figure_sha256={name:sha_file(FIGURE_DIR/name) for name in figure_qa['files']},
    display_mode=PANEL_MODE,display_selection=selection_audit,display_order=order,
    segment_display_labels=SEGMENT_LABELS,complete_branch_specifications=DISPLAY_BRANCHES,
    pdf_profile=dict(clips_per_page=0,max_clip_depth=0,fonts=['Arial','Arial Italic','Arial Bold Italic'],raster_images=0),
    qa_renders_retained=KEEP_QA_RENDERS,
    validation_reports=validation_reports)
with h5py.File(RESULT_FILE,'a') as h:
    calibration=json.loads(h['publication/illustrator_smoke_json'].asstr()[()]) if 'publication/illustrator_smoke_json' in h else None
    publication['illustrator_calibration_matches_exporter']=bool(calibration and
        calibration['exporter_source_sha256']==exporter_hash and calibration['matplotlib_version']==matplotlib.__version__)
    if 'display_selection' in h:del h['display_selection']
    display_group=h.create_group('display_selection')
    display_group.create_dataset('regulons',data=representative_order,dtype=h5py.string_dtype('utf-8'))
    representative_fitted=data['modes']['none']['bins'].loc[:,representative_order].to_numpy()
    display_group.create_dataset('relative_activity',data=np.maximum(representative_fitted,0)/representative_fitted.max(axis=0))
    display_group.create_dataset('segment_labels_json',data=json.dumps(SEGMENT_LABELS),dtype=h5py.string_dtype('utf-8'))
    display_group.create_dataset('complete_branches_json',data=json.dumps(figure_qa['display_branches']),dtype=h5py.string_dtype('utf-8'))
    display_group.create_dataset('row_positions',data=figure_qa['display_row_positions'])
    display_group.create_dataset('audit_json',data=json.dumps(selection_audit,ensure_ascii=False),dtype=h5py.string_dtype('utf-8'))
    display_group.create_dataset('evidence_json',data=evidence.reset_index().to_json(orient='records'),dtype=h5py.string_dtype('utf-8'))
    display_group.create_dataset('source',data=next(''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code' and 'def display_evidence(' in ''.join(c['source'])),dtype=h5py.string_dtype('utf-8'))
    if 'publication' in h:del h['publication']
    publication_group=h.create_group('publication')
    publication_group.create_dataset('manifest_json',data=json.dumps(publication,ensure_ascii=False),dtype=h5py.string_dtype('utf-8'))
    if calibration:
        publication_group.create_dataset('illustrator_smoke_json',data=json.dumps(calibration,ensure_ascii=False),dtype=h5py.string_dtype('utf-8'))

# The renderer PNGs are inspected during delivery. Set True only when keeping QA renders for inspection.
if not KEEP_QA_RENDERS:
    # Set this storage path independently before running.
    cache=CACHE_DIR.resolve();assert cache.parent==ROOT
    removed=sum(p.stat().st_size for p in _release_cleanup_target(cache).rglob('*') if p.is_file())
    shutil.rmtree(_release_cleanup_target(cache))
    print(f'PASS: all figures are 600-DPI PNG and live-text vector PDF; removed {removed} bytes of runtime files.')
else:
    print(f'PASS: QA renders retained for visual inspection at {CACHE_DIR}')


## Additional prespecified regulons: Nhlh1, Neurod1 and Tbx3

This appended section leaves all eight original cells unchanged. Run **only this section**
to export the `-1` figure; it loads the single existing result H5 and preserves the original
58-column figure and its archived statistics. The default reuses the supplemental cache.
Set `RECOMPUTE_ADD3=True` only to repeat scoring/fitting from the recorded source files.

The three canonical signatures are fixed before testing: Nhlh1 is the B-selected late-UMI
62-target signature, Neurod1 is the 36-target positive motif-supported early signature
saved in Activity.h5 (its B score was unavailable under B's different coverage rule), and
Tbx3 is the 37-target native CVG/VGN signature. None replaces an old signature.
All three use A's original 11,451-gene ranking axis, weighted AUCell and >=20 measured
targets. Old217 raw P-values are combined with three new tests for BH220; the four
fork families jointly use old868 + new12 = BH880. Covariates and residual diagnostics
are retained without gene-specific tuning. The exact archived models/sample sets are reused.

The panel retains the original58 representatives and adds these three prespecified
follow-ups, ordered by the actual30-bin fitted peak and centroid. This is a targeted
extension, not a new unbiased screen of all early regulons. Primary-threshold status is
saved explicitly; requested inclusion is not itself evidence of statistical significance.
No actual VGN cells are added to the accepted Fig3 tree, so Tbx3 shows projection onto
the existing atlas. Early state/age/source confounding and causal limits remain.


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#8. =========== Supplemental configuration and frozen source bundle
from pathlib import Path
import os,sys,json,hashlib,subprocess,shutil
# Set this storage path independently before running.
ROOT=ROOT
# Set this storage path independently before running.
if ROOT is None:ROOT=ROOT
# Set this storage path independently before running. Deve_regulon_dynamics.h5. Accepted primary result produced by fig4A_regulon_dynamics_analysis.py.
RESULT_FILE=ROOT
# Set this storage path independently before running. 05_Deve_regulon_dynamics.ipynb. Use the supplied fig4_regulon_supplement_analysis.ipynb source.
NOTEBOOK_FILE=ROOT
# Set this storage path independently before running.
FIGURE_DIR=ROOT
# Set this storage path independently before running.
ADD3_WORK=ROOT
RECOMPUTE_ADD3=False  # False reuses the supplemental H5; True repeats the fixed analysis.
OUTPUT_STEM='fig4A_Developmental_regulon_dynamics-1'
DPI=600
Q_CUT=1e-4
MAIN_TITLE='Developmental regulon dynamics'
KEEP_ADD3_QA=False  # Set True only for the manual PDF-page inspection.
ADD3_WORK.mkdir(parents=True,exist_ok=True)
for key in ('TMP','TEMP','MPLCONFIGDIR'):os.environ[key]=str(ADD3_WORK)
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='1'
sys.dont_write_bytecode=True
import h5py,numpy as np,pandas as pd,matplotlib
matplotlib.use('Agg')
from matplotlib import font_manager
from PIL import Image
for font in ('arial.ttf','ariali.ttf','arialbd.ttf','arialbi.ttf'):
    # Set this storage path independently before running.
    candidate=ROOT/font
    if candidate.exists():font_manager.fontManager.addfont(str(candidate))
assert font_manager.findfont('Arial',fallback_to_default=False)

# Executed sources are retained here and in the existing H5 for reproducibility.
ADD3_SCORING_SOURCE='"""Score three approved canonical regulons on the unchanged Fig4A ranking axis.\n\nThis helper writes only its explicitly requested temporary NPZ.  It does not\nchange a formal result, infer a network, or import Fig4B\'s different scores.\n"""\n\n# Configure each file/directory placeholder independently; no storage addresses are shipped.\nfrom pathlib import Path\nROOT = Path(".")\n\n#1. =========== Shared configuration\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport sys\n\nfor _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n    os.environ[_name] = "1"\nsys.dont_write_bytecode = True\n\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\n\nSEED = 42\nAUC_THRESHOLD = 0.05\nMIN_MATCHED_TARGETS = 20\nRANKING_CHUNK_SIZE = 384\nEQUIVALENCE_TOLERANCE = 1e-12\nNEW_TFS = ("Nhlh1", "Neurod1", "Tbx3")\nSTUDIES = ("GSE178931", "GSE231513", "GSE165502", "GSE195500")\n\n\ndef _json_dataset(dataset):\n    value = dataset[()]\n    if isinstance(value, np.ndarray) and value.dtype == np.uint8:\n        value = value.tobytes().decode("utf-8")\n    elif isinstance(value, bytes):\n        value = value.decode("utf-8")\n    return json.loads(value)\n\n\ndef _file_sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\n#2. =========== Exact original Fig4A scoring arithmetic\ndef create_rankings(counts, genes):\n    """Preserve the archived seed, gene shuffle and zero-based first-tie ranks."""\n    expression = pd.DataFrame(counts.toarray(), columns=genes)\n    return (expression.sample(frac=1.0, replace=False, axis=1,\n                              random_state=SEED)\n            .rank(axis=1, ascending=False, method="first", na_option="bottom")\n            .astype("uint32") - 1)\n\n\ndef weighted_auc1d(ranking, weights, cutoff, max_auc):\n    """Arithmetic identical to the archived ctxcore weighted recovery integral."""\n    keep = ranking < cutoff\n    x, y = ranking[keep], weights[keep]\n    order = np.argsort(x)\n    x = np.concatenate((x[order], np.full((1,), cutoff, dtype=np.int_)))\n    y = y[order].cumsum()\n    return np.sum(np.diff(x) * y) / max_auc\n\n\ndef score_rankings(rankings, signatures):\n    """Use the original >=20 matched-target rule and matched-weight denominator.\n\n    Full canonical signatures are retained in provenance.  No 80% coverage gate\n    is introduced, and no absent feature is imputed into the ranking matrix.\n    """\n    cutoff = int(round(AUC_THRESHOLD * rankings.shape[1])) - 1\n    result = np.full((len(rankings), len(signatures)), np.nan)\n    genes = rankings.columns.to_numpy()\n    for j, signature in enumerate(signatures):\n        shared = np.array([gene in signature for gene in genes])\n        if int(shared.sum()) < MIN_MATCHED_TARGETS:\n            continue\n        weights = np.asarray([signature[gene] for gene in genes[shared]],\n                             dtype=np.float64)\n        values = rankings.iloc[:, shared].to_numpy()\n        maximum = float((cutoff + 1) * weights.sum())\n        result[:, j] = [weighted_auc1d(row, weights, cutoff, maximum)\n                        for row in values]\n    return result, cutoff\n\n\n#3. =========== Approved full canonical signatures and frozen Fig4A controls\ndef load_inputs(root):\n    root = Path(root).resolve()\n    # Set this storage path independently before running. Deve_regulon_dynamics.h5. Accepted primary result produced by fig4A_regulon_dynamics_analysis.py.\n    a_path = ROOT\n    # Set this storage path independently before running. Activity.h5. Produced by the SCENIC workers in fig4_SGN_program_analysis.ipynb.\n    b_path = ROOT\n    # Set this storage path independently before running. VGN-results.h5.\n    v_path = ROOT\n    with h5py.File(a_path, "r") as handle:\n        cells = handle["cells"].asstr()[()]\n        genes = handle["genes"].asstr()[()]\n        old_names = handle["regulons"].asstr()[()]\n        old_auc = handle["auc"][()]\n        rank_gate = handle["rank_gate"][()]\n        positive = handle["positive_genes"][()]\n        totals = handle["total_counts_scoring_axis"][()]\n        study = handle["obs/study"].asstr()[()]\n        original = {entry["name"]: entry for entry in\n                    _json_dataset(handle["original_regulons_json"])}\n        archived_source = _json_dataset(handle["analysis_code/legacy/early_source"])\n    assert len(cells) == 7355 and len(genes) == 11451\n    assert len(old_names) == 217 and old_auc.shape == (7355, 217)\n    assert len(np.unique(cells)) == len(cells)\n    assert len(np.unique(genes)) == len(genes)\n\n    with h5py.File(b_path, "r") as handle:\n        selected = _json_dataset(handle["activity/selected_regulons_json"])\n        nhlh1 = dict(selected["Nhlh1"])\n        assert nhlh1["source"] == "late_UMI"\n        assert len(nhlh1["gene2weight"]) == 62\n        cvg_signatures = _json_dataset(handle["CVG_VGN/rank5000/regulons_json"])\n        neurod1 = next(entry for entry in cvg_signatures\n                       if entry["transcription_factor"] == "Neurod1")\n        assert len(neurod1["gene2weight"]) == 36\n        assert neurod1[\'name\'] == \'Neurod1(+)\'\n        assert any(\'activating\' in str(value).lower() for value in neurod1.get(\'context\', []))\n    with h5py.File(v_path, "r") as handle:\n        signature_genes = handle["data/network_signature_genes"].asstr()[()]\n        signature_weights = handle["data/network_signature_weights"][()]\n        tbx3 = {"name": "Tbx3(+)", "transcription_factor": "Tbx3",\n                "gene2weight": dict(zip(signature_genes.tolist(),\n                                        signature_weights.tolist()))}\n        assert len(tbx3["gene2weight"]) == 37\n        motif_evidence = _json_dataset(handle["data/network_motif_support"])\n        assert any(row["TF"] == "Tbx3" and row["Enrichment|NES"] >= 3\n                   for row in motif_evidence)\n\n    definitions = []\n    gene_set = set(genes)\n    raw_definitions = [nhlh1, neurod1, tbx3]\n    sources = ["SGN-1/Activity.h5:activity/selected_regulons_json:Nhlh1:late_UMI",\n               "SGN-1/Activity.h5:CVG_VGN/rank5000/regulons_json:Neurod1",\n               "VGN/VGN-results.h5:data/network_signature_genes+weights"]\n    for tf, entry, source in zip(NEW_TFS, raw_definitions, sources):\n        weights = {str(gene): float(weight)\n                   for gene, weight in entry["gene2weight"].items()}\n        assert all(np.isfinite(weight) and weight > 0\n                   for weight in weights.values())\n        matched = [gene for gene in genes if gene in weights]\n        missing = sorted(set(weights) - gene_set)\n        definitions.append({"TF": tf, "regulon": tf + "(+)",\n                            "source": source, "gene2weight": weights,\n                            "full_targets": len(weights),\n                            "matched_targets": len(matched),\n                            "matched_genes_in_A_order": matched,\n                            "missing_genes": missing,\n                            "coverage": len(matched) / len(weights),\n                            "matched_weight_sum": sum(weights[g] for g in matched),\n                            "full_weight_sum": sum(weights.values()),\n                            "self_target_preserved": tf in weights,\n                            "scoreable": len(matched) >= MIN_MATCHED_TARGETS})\n    assert [entry["matched_targets"] for entry in definitions] == [58, 23, 27]\n    assert all(entry["scoreable"] for entry in definitions)\n    # Uniformly spaced controls cover the frozen tested family. Onecut1 checks\n    # that a <80% original-signature coverage is handled exactly as in Fig4A.\n    checks = sorted(set(np.linspace(0, len(old_names) - 1, 9, dtype=int).tolist()\n                        + [old_names.tolist().index("Onecut1(+)")]))\n    controls = [str(old_names[index]) for index in checks]\n    control_signatures = [original[name]["gene2weight"] for name in controls]\n    metadata = {"a_path": a_path, "b_path": b_path, "v_path": v_path,\n                "archived_source": archived_source,\n                "control_indices": checks, "control_names": controls,\n                "control_signatures": control_signatures}\n    return {"cells": cells, "genes": genes, "old_auc": old_auc,\n            "rank_gate": rank_gate, "positive": positive, "totals": totals,\n            "study": study, "definitions": definitions, "metadata": metadata}\n\n\n#4. =========== Score all original cells without changing the gene universe\ndef score_candidates(root, output=None):\n    root = Path(root).resolve()\n    inputs = load_inputs(root)\n    cells, genes = inputs["cells"], inputs["genes"]\n    cell_lookup = {cell: index for index, cell in enumerate(cells)}\n    definitions = inputs["definitions"]\n    signatures = [entry["gene2weight"] for entry in definitions]\n    controls = inputs["metadata"]["control_signatures"]\n    raw_auc = np.full((len(cells), len(signatures)), np.nan)\n    reproduced = np.full((len(cells), len(controls)), np.nan)\n    native_tf_counts = np.full_like(raw_auc, np.nan)\n    native_tf_log1p_cp10k = np.full_like(raw_auc, np.nan)\n    common_tf_counts = np.full_like(raw_auc, np.nan)\n    seen = np.zeros(len(cells), dtype=bool)\n    source_qa = []\n    input_paths = [inputs["metadata"][name] for name in\n                   ("a_path", "b_path", "v_path")]\n    for study in STUDIES:\n        # Set this storage path independently before running.\n        axes_path = ROOT / (study + "_axes.npz")\n        # Set this storage path independently before running.\n        counts_path = ROOT / (study + "_counts.npz")\n        input_paths.extend([axes_path, counts_path])\n        with np.load(axes_path, allow_pickle=False) as axes:\n            native_cells = axes["cells"].astype(str)\n            native_genes = axes["genes"].astype(str)\n        native_counts = sparse.load_npz(counts_path).tocsr()\n        assert native_counts.shape == (len(native_cells), len(native_genes))\n        native_gene_lookup = {gene: index for index, gene in enumerate(native_genes)}\n        positions = np.asarray([cell_lookup[cell] for cell in native_cells], int)\n        assert not seen[positions].any()\n        assert np.all(inputs["study"][positions] == study)\n        assert set(genes).issubset(native_gene_lookup)\n        common_counts = native_counts[:, [native_gene_lookup[gene]\n                                         for gene in genes]].tocsr()\n        positive = np.asarray((common_counts > 0).sum(axis=1)).ravel()\n        totals = np.asarray(common_counts.sum(axis=1)).ravel()\n        assert np.array_equal(positive, inputs["positive"][positions])\n        assert np.array_equal(totals, inputs["totals"][positions])\n        native_total = np.asarray(native_counts.sum(axis=1)).ravel()\n        for column, tf in enumerate(NEW_TFS):\n            assert tf in native_gene_lookup\n            values = native_counts[:, native_gene_lookup[tf]].toarray().ravel()\n            native_tf_counts[positions, column] = values\n            native_tf_log1p_cp10k[positions, column] = np.log1p(\n                values * 10000.0 / native_total)\n            common_column = genes.tolist().index(tf)\n            common_tf_counts[positions, column] = common_counts[:, common_column].toarray().ravel()\n        cutoff = int(round(AUC_THRESHOLD * len(genes))) - 1\n        expected_gate = positive > cutoff\n        assert np.array_equal(expected_gate, inputs["rank_gate"][positions])\n        for start in range(0, len(native_cells), RANKING_CHUNK_SIZE):\n            stop = min(start + RANKING_CHUNK_SIZE, len(native_cells))\n            ranked = create_rankings(common_counts[start:stop], genes)\n            values, actual_cutoff = score_rankings(ranked, signatures + controls)\n            assert actual_cutoff == cutoff == 572\n            selected_positions = positions[start:stop]\n            raw_auc[selected_positions] = values[:, :len(signatures)]\n            reproduced[selected_positions] = values[:, len(signatures):]\n        seen[positions] = True\n        source_qa.append({"study": study, "cells": len(positions),\n                          "native_measured_genes": len(native_genes),\n                          "common_counts_totals_exact": True,\n                          "common_counts_detection_exact": True,\n                          "rank_gate_exact": True})\n        print(f"Scored {study}: {len(positions)} original cells", flush=True)\n    assert seen.all()\n    assert np.isfinite(raw_auc).all()\n    assert np.isfinite(reproduced).all()\n    observed = inputs["old_auc"][:, inputs["metadata"]["control_indices"]]\n    difference = np.abs(observed - reproduced)\n    errors = difference.max(axis=0)\n    assert float(errors.max()) <= EQUIVALENCE_TOLERANCE, errors\n    assert np.array_equal(native_tf_counts, common_tf_counts)\n    validation = {name: float(error) for name, error in\n                  zip(inputs["metadata"]["control_names"], errors)}\n    metadata = {"schema": "Fig4A_add3_AUCell_v1", "cells": len(cells),\n                "gene_universe": len(genes), "seed": SEED,\n                "weighted": True, "normalize_to_sample_max": False,\n                "auc_threshold": AUC_THRESHOLD, "rank_cutoff_zero_based": 572,\n                "minimum_matched_targets": MIN_MATCHED_TARGETS,\n                "ranking_chunk_size": RANKING_CHUNK_SIZE,\n                "ranking": "Archived pandas seeded-column shuffle; descending first ties; zero-based uint32",\n                "recovery": "Archived ctxcore weighted_auc1d arithmetic; matched target weights in the shuffled ranking order",\n                "weight_denominator": "(cutoff+1) * sum of weights of targets measured on the fixed A axis",\n                "full_canonical_membership_preserved": True,\n                "coverage_80_percent_gate_applied": False,\n                "original_rank_gate_preserved": True,\n                "all_source_cells_scored_including_E10": True,\n                "input_hashes": {str(path.relative_to(root)): _file_sha256(path)\n                                 for path in input_paths},\n                "original_scoring_source_sha256": hashlib.sha256(\n                    inputs["metadata"]["archived_source"].encode()).hexdigest(),\n                "old_AUCell_equivalence_max_abs": float(errors.max()),\n                "old_AUCell_equivalence_by_regulon": validation,\n                "old_AUCell_validation_cells": len(cells),\n                "old_AUCell_validation_signatures": len(controls),\n                "source_qa": source_qa,\n                "runtime_versions": {"python": sys.version,\n                                     "numpy": np.__version__,\n                                     "pandas": pd.__version__},\n                "native_RNA": "Original native source counts; natural log1p CP10k using all supplied endogenous count features",\n                "limits": ["No regulatory-network inference was run.",\n                           "Native CVG/VGN Tbx3 targets are projected on the unchanged 7355-cell atlas; actual520 VGN are not added.",\n                           "Scoreability does not establish whole-tree association or primary-figure eligibility."]}\n    payload = {"cells": cells.astype(str), "genes": genes.astype(str),\n               "regulons": np.asarray([tf + "(+)" for tf in NEW_TFS]),\n               "auc": raw_auc, "rank_gate": inputs["rank_gate"],\n               "tf_rna_counts": common_tf_counts,\n               "native_tf_rna_counts": native_tf_counts,\n               "native_tf_log1p_CP10k": native_tf_log1p_cp10k,\n               "validation_regulons": np.asarray(inputs["metadata"]["control_names"]),\n               "validation_auc_reproduced": reproduced,\n               "validation_max_abs_error": errors,\n               "signatures_json": np.asarray(json.dumps(definitions)),\n               "metadata_json": np.asarray(json.dumps(metadata))}\n    if output is not None:\n        output = Path(output).resolve()\n        # Set this storage path independently before running.\n        allowed = ROOT\n        assert output.parent == allowed and output.suffix == ".npz"\n        output.parent.mkdir(parents=True, exist_ok=True)\n        np.savez_compressed(output, **payload)\n    print(f"AUCell equivalence: {len(controls)} frozen signatures x {len(cells)} cells; max abs error {errors.max():.3g}", flush=True)\n    return payload\n\n\nif __name__ == "__main__":\n    project_root = Path(__file__).resolve().parents[2]\n    # Set this storage path independently before running. score.npz.\n    score_candidates(project_root, ROOT)\n'
ADD3_MODEL_SOURCE='"""Append-only Fig4A GAM helpers: new signatures on frozen models and samples.\n\nNo formal artifact is mutated. The R engines below are copied from the formal\narchive with only family-size and fresh-validation index adapters.\n"""\n\n# Configure each file/directory placeholder independently; no storage addresses are shipped.\nfrom pathlib import Path\nROOT = Path(".")\n#1. =========== Shared configuration\nfrom pathlib import Path\nimport argparse,concurrent.futures,hashlib,json,os,subprocess,sys,time\nsys.dont_write_bytecode=True\nfor _k in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\'):os.environ[_k]=\'1\'\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import f as f_distribution\n# Set this storage path independently before running.\nROOT=ROOT\n# Set this storage path independently before running. Deve_regulon_dynamics.h5. Accepted primary result produced by fig4A_regulon_dynamics_analysis.py.\nBASELINE=ROOT\n# Set this storage path independently before running.\nWORK=ROOT\n# Set this storage path independently before running. Rscript.exe.\nRSCRIPT=ROOT\nNEW_NAMES=(\'Nhlh1(+)\',\'Neurod1(+)\',\'Tbx3(+)\')\nNEW_INDICES=np.arange(218,221)\nGLOBAL_MODES=(\'none\',\'depth\',\'study_only\',\'study\',\'library\')\nLOCAL_MODES=(\'none\',\'depth\',\'source\',\'source_depth\')\nEARLY_MODES=LOCAL_MODES+(\'stage\',\'stage_depth\')\nSEED=42\nGLOBAL_K=20\nFORK_K=10\nWORKERS=5\n\nGLOBAL_R = \'\\n#1. =========== Shared settings and command-line contract\\nsuppressPackageStartupMessages(library(mgcv))\\nargs <- commandArgs(trailingOnly=TRUE)\\nif (!length(args) %in% c(6L,7L)) {\\n  stop(\\\'Usage: Rscript associate_unified.R cells.csv auc.tsv.gz prefix mode K indices|all [validation_indices]\\\')\\n}\\nCELLS <- args[1]; AUC <- args[2]; PREFIX <- args[3]; MODE <- args[4]\\nK <- suppressWarnings(as.integer(args[5]))\\nSEED <- 42L\\nK_CHECK_REPS <- 500L\\nK_CHECK_SUBSAMPLE <- 2000L\\nFAMILY_SIZE <- 220L\\nCONSTANT_TOL <- 1e-12\\nVALIDATION_TOL <- 1e-7\\noptions(stringsAsFactors=FALSE, digits=17)\\nstopifnot(MODE %in% c(\\\'none\\\',\\\'depth\\\',\\\'study_only\\\',\\\'study\\\',\\\'library_only\\\',\\\'library\\\'), length(K)==1L,\\n          is.finite(K), K>=3L, dir.exists(dirname(PREFIX)))\\nparse_indices <- function(text) {\\n  if (identical(text,\\\'all\\\')) return(seq_len(FAMILY_SIZE))\\n  tokens <- trimws(strsplit(text,\\\',\\\',fixed=TRUE)[[1]])\\n  if (!length(tokens) || any(!grepl(\\\'^[0-9]+$\\\',tokens))) stop(\\\'Invalid regulon indices\\\')\\n  indices <- as.integer(tokens)\\n  stopifnot(!anyNA(indices),!anyDuplicated(indices),all(indices>=1L & indices<=FAMILY_SIZE))\\n  indices\\n}\\nINDICES <- parse_indices(args[6])\\nVALIDATION_INDICES <- if(length(args)==7L) parse_indices(args[7]) else c(218L,219L,220L)\\nstarted <- proc.time()[[\\\'elapsed\\\']]\\n\\n\\n#2. =========== Fixed cells, nuisance terms and independent segment smooths\\nd <- read.csv(CELLS,row.names=1,check.names=FALSE)\\nconnection <- gzfile(AUC,\\\'rt\\\')\\ny_all <- tryCatch(as.matrix(read.delim(connection,row.names=1,check.names=FALSE)),\\n                  finally=close(connection))\\nstorage.mode(y_all) <- \\\'double\\\'\\nrequired <- c(\\\'t\\\',\\\'seg\\\',\\\'study\\\',\\\'library\\\',\\\'log_detected\\\')\\nstopifnot(all(required %in% names(d)),identical(rownames(d),rownames(y_all)),\\n          !anyDuplicated(rownames(d)),!anyDuplicated(colnames(y_all)),\\n          ncol(y_all)==FAMILY_SIZE,all(is.finite(y_all)),all(y_all>=0 & y_all<=1),\\n          all(is.finite(d$t)),all(is.finite(d$log_detected)),\\n          !anyNA(d$seg),!anyNA(d$study),!anyNA(d$library))\\nd$seg <- factor(as.character(d$seg),levels=as.character(c(0,1,2,4,5,6,7,8,9,10)))\\nd$study <- factor(d$study); d$library <- factor(d$library)\\nstopifnot(!anyNA(d$seg),all(table(d$seg)>=20L))\\nsegment_rows <- lapply(levels(d$seg),function(s) which(d$seg==s))\\nnames(segment_rows) <- levels(d$seg)\\nfor (rows in segment_rows) stopifnot(length(unique(d$t[rows]))>=K,diff(range(d$t[rows]))>0)\\nd$u <- ave(d$t,d$seg,FUN=function(t) (t-min(t))/diff(range(t)))\\nstopifnot(all(is.finite(d$u)))\\nnull_rhs <- switch(MODE,none=\\\'1\\\',depth=\\\'log_detected\\\',\\n                   study_only=\\\'study\\\',study=\\\'study + log_detected\\\',library_only=\\\'library\\\',library=\\\'library + log_detected\\\')\\nf0 <- as.formula(paste(\\\'y ~\\\',null_rhs))\\nf1 <- as.formula(paste(\\\'y ~\\\',null_rhs,sprintf("+ seg + s(u,by=seg,bs=\\\'cr\\\',k=%d)",K)))\\nd$y <- y_all[,1L]\\nG0 <- gam(f0,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,\\n          fit=FALSE,na.action=na.fail)\\nG1 <- gam(f1,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,\\n          fit=FALSE,na.action=na.fail)\\nstopifnot(length(G1$smooth)==10L,length(G1$sp)==10L)\\nX0 <- G0$X; X1 <- G1$X\\nsmooth_segments <- vapply(G1$smooth,function(s) as.character(s$by.level),character(1))\\nstopifnot(setequal(smooth_segments,levels(d$seg)),!anyDuplicated(smooth_segments))\\nsmooth_columns <- lapply(G1$smooth,function(s) s$first.para:s$last.para)\\ntrajectory_columns <- sort(unique(c(which(startsWith(colnames(X1),\\\'seg\\\')),\\n                                   unlist(smooth_columns))))\\nstopifnot(length(trajectory_columns)==9L+sum(lengths(smooth_columns)))\\nX_trajectory <- X1[,trajectory_columns,drop=FALSE]\\n# Fixed-X thin QR is the exact sufficient-statistic reduction used by bam.fit for ML.\\ncache_qr <- function(G) {\\n  q <- qr(G$X,tol=0,LAPACK=TRUE)\\n  rp <- q$pivot; rp[rp] <- seq_len(ncol(G$X))\\n  G$fast_qr <- q\\n  G$fast_sp <- mgcv:::initial.spg(G$X,G$y,G$w,G$family,G$S,G$rank,G$off,\\n    offset=G$offset,L=G$L,lsp0=G$lsp0)\\n  G$fast_R <- qr.R(q)[,rp,drop=FALSE]\\n  stopifnot(max(abs(crossprod(G$fast_R)-crossprod(G$X)))<1e-7)\\n  G\\n}\\nG0 <- cache_qr(G0); G1 <- cache_qr(G1)\\n\\n\\n#3. =========== Cache validation and segment-specific residual diagnostics\\ncapture_conditions <- function(expr) {\\n  notices <- character()\\n  value <- withCallingHandlers(expr,warning=function(w) {\\n    notices <<- c(notices,conditionMessage(w)); invokeRestart(\\\'muffleWarning\\\')\\n  })\\n  list(value=value,warnings=unique(notices))\\n}\\nfit_response <- function(template,y) {\\n  y <- as.numeric(y); X <- template$X; N <- length(y); p <- ncol(X)\\n  initialization <- list(sp=template$fast_sp,scale=mean((y-mean(y))^2)/10)\\n  f <- qr.qty(template$fast_qr,y)[seq_len(p)]\\n  rss_extra <- sum(y*y)-sum(f*f)\\n  stopifnot(rss_extra>=-1e-10)\\n  template$X <- template$fast_R\\n  template$y <- f; template$w <- rep(1,p); template$n <- p\\n  template$offset <- rep(0,p); template$n.true <- N\\n  template$dev.extra <- template$pearson.extra <- max(0,rss_extra)\\n  template$mf$y <- y\\n  template$fast_qr <- template$fast_R <- template$fast_sp <- NULL\\n  model <- gam(G=template,method=\\\'ML\\\',gamma=1,in.out=initialization)\\n  mu <- as.numeric(X %*% coef(model)); r <- y-mu\\n  model$y <- y; model$model$y <- y\\n  model$fitted.values <- model$linear.predictors <- mu\\n  model$residuals <- r\\n  model$prior.weights <- model$weights <- model$working.weights <- rep(1,N)\\n  model$offset <- rep(0,N); model$z <- y\\n  model$deviance <- sum(r*r)\\n  model$null.deviance <- sum((y-mean(y))^2)\\n  model$df.residual <- N-sum(model$edf)\\n  model$df.null <- N-as.integer(template$intercept)\\n  model$aic <- model$family$aic(y,1,mu,rep(1,N),model$deviance)+2*sum(model$edf)\\n  model$hat <- rowSums((X %*% model$Vp)*X)/model$sig2\\n  stopifnot(identical(as.numeric(model$y),as.numeric(y)),\\n            identical(as.numeric(model$model$y),as.numeric(y)),\\n            all(model$prior.weights==1),length(model$hat)==N)\\n  model\\n}\\n\\nconverged <- function(model) {\\n  outer <- model$outer.info$conv\\n  (is.null(model$converged) || isTRUE(model$converged)) &&\\n    (is.null(outer) || identical(outer,0L) || identical(outer,0) ||\\n       grepl(\\\'full convergence\\\',paste(outer,collapse=\\\' \\\'),fixed=TRUE))\\n}\\nrho <- function(x,y) {\\n  if(length(x)<3L || diff(range(x))==0 || diff(range(y))==0) return(NA_real_)\\n  suppressWarnings(cor(x,y,method=\\\'spearman\\\'))\\n}\\nadjacent_index <- function(u,residual) {\\n  denominator <- mean(residual^2)\\n  if(length(u)<2L || !is.finite(denominator) || denominator<=0) return(NA_real_)\\n  mean(diff(residual[order(u)])^2)/2/denominator\\n}\\ntie_expected_sse <- function(values,groups,counts) {\\n  sums <- rowsum(values,groups,reorder=TRUE)\\n  squares <- rowsum(values*values,groups,reorder=TRUE)\\n  means <- sums/counts\\n  ss <- squares-sums*sums/counts\\n  ss[ss<0] <- 0\\n  variances <- ss/counts\\n  result <- 2*colSums(ss)\\n  m <- length(counts)\\n  if(m>1L)result <- result+colSums(variances[-m,,drop=FALSE]+variances[-1L,,drop=FALSE]+\\n    (means[-m,,drop=FALSE]-means[-1L,,drop=FALSE])^2)\\n  result\\n}\\n# Exactly average adjacent SSE across all within-t orderings; verify by enumeration.\\ntoy <- c(.1,-.3,.2,.6,-.1,.9)\\npermutations <- function(x)if(length(x)==1L)list(x)else unlist(lapply(seq_along(x),function(i)\\n  lapply(permutations(x[-i]),function(z)c(x[i],z))),recursive=FALSE)\\norders <- unlist(lapply(permutations(c(1L,2L)),function(a)lapply(permutations(c(3L,4L,5L)),function(b)c(a,b,6L))),recursive=FALSE)\\nstopifnot(abs(tie_expected_sse(matrix(toy,ncol=1),c(1,1,2,2,2,3),c(2L,3L,1L))-\\n  mean(vapply(orders,function(z)sum(diff(toy[z])^2),numeric(1))))<1e-12)\\nsegment_check <- function(u,residual,seed) {\\n  stopifnot(length(u)==length(residual),all(is.finite(u)),all(is.finite(residual)))\\n  set.seed(seed)\\n  if(length(u)>K_CHECK_SUBSAMPLE) {\\n    indices <- sample.int(length(u),K_CHECK_SUBSAMPLE)\\n    u <- u[indices]; residual <- residual[indices]\\n  }\\n  n <- length(residual); denominator <- mean(residual^2)\\n  if(n<2L || denominator<=0)return(list(k_index=NA_real_,p_value=NA_real_,\\n    checked_cells=n,status=\\\'zero_residual_variance\\\',tied_cells=NA_integer_,tied_groups=NA_integer_))\\n  groups <- match(u,sort(unique(u)));counts <- tabulate(groups)\\n  random <- vapply(seq_len(K_CHECK_REPS),function(i)residual[sample.int(n,n)],numeric(n))\\n  sse <- tie_expected_sse(cbind(residual,random),groups,counts)\\n  list(k_index=sse[1]/(2*(n-1L))/denominator,p_value=mean(sse[-1L]<sse[1]),\\n    checked_cells=n,status=\\\'ok\\\',tied_cells=sum(counts[counts>1L]),tied_groups=sum(counts>1L))\\n}\\nvalidate_segment_check <- function(rows,residual,seed) {\\n  z <- d[rows,,drop=FALSE]\\n  z$y <- residual\\n  single <- gam(y~s(u,bs=\\\'cr\\\',k=K),data=z,family=gaussian(),method=\\\'ML\\\',\\n                gamma=1,select=FALSE,na.action=na.fail)\\n  # A diagnostic-only fixture preserves a genuine single-segment smooth/model\\n  # frame while supplying exactly the joint model\\\'s residual vector to k.check.\\n  single$y <- as.numeric(residual)\\n  single$fitted.values <- rep(0,length(residual))\\n  single$linear.predictors <- rep(0,length(residual))\\n  fixture_residual <- as.numeric(residuals(single))\\n  injection_error <- max(abs(fixture_residual-residual))\\n  stopifnot(injection_error<1e-12)\\n  custom <- segment_check(z$u,fixture_residual,seed)\\n  set.seed(seed)\\n  official <- k.check(single,subsample=K_CHECK_SUBSAMPLE,n.rep=K_CHECK_REPS)\\n  k_error <- abs(custom$k_index-official[1,\\\'k-index\\\'])\\n  p_error <- abs(custom$p_value-official[1,\\\'p-value\\\'])\\n  stopifnot(k_error<1e-12,p_error==0)\\n  list(k_index_difference=k_error,permutation_p_difference=p_error,\\n       residual_injection_error=injection_error)\\n}\\nadd_frame <- function(rows,value) { rows[[length(rows)+1L]] <- value; rows }\\nsummary_rows <- list(); k_rows <- list(); segment_residual_rows <- list()\\nsource_residual_rows <- list(); validation_rows <- list()\\nmetadata_rows <- data.frame(metric=c(\\\'mode\\\',\\\'K\\\',\\\'family_size\\\',\\\'cells\\\',\\\'null_formula\\\',\\\'full_formula\\\',\\n    \\\'fit_method\\\',\\\'smooth_penalties\\\',\\\'k_check_method\\\',\\\'k_check_seed\\\',\\\'k_check_permutations\\\',\\n    \\\'k_check_subsample_per_segment\\\',\\\'k_check_denominator\\\',\\\'multiplicity\\\',\\\'mgcv_version\\\'),\\n  value=c(MODE,K,FAMILY_SIZE,nrow(d),deparse(f0),paste(deparse(f1),collapse=\\\' \\\'),\\n    \\\'Gaussian ML; gamma1; selectFALSE; unit weights\\\',\\\'Ten independent smoothing parameters; no id\\\',\\n    \\\'Tie-order-invariant adjacent-residual expected SSE separately within each biological native segment\\\',\\n    \\\'42 + original1basedRegulonIndex + segmentInteger\\\',K_CHECK_REPS,K_CHECK_SUBSAMPLE,\\n    \\\'mean(residual_segment^2); analytical average over within-t ties;500permutations; unique-t reduces to mgcv1D\\\',\\n    \\\'No shard-level correction; Python must correct the complete217 family\\\',\\n    as.character(packageVersion(\\\'mgcv\\\'))))\\n\\n\\n#4. =========== Fit responses and retain every failure and residual\\ntrajectory_values <- full_values <- residual_values <- matrix(NA_real_,nrow(d),length(INDICES),\\n  dimnames=list(rownames(d),colnames(y_all)[INDICES]))\\nfor(ii in seq_along(INDICES)) {\\n  j <- INDICES[ii]; name <- colnames(y_all)[j]; y <- as.numeric(y_all[,j])\\n  tick <- proc.time()[[\\\'elapsed\\\']]\\n  row <- data.frame(regulon=name,index=j,mode=MODE,K=K,status=\\\'failed\\\',p_value=NA_real_,\\n    A=NA_real_,RSS0=NA_real_,RSS1=NA_real_,df0=NA_real_,df1=NA_real_,test_df0=NA_real_,\\n    test_df1=NA_real_,test_delta_df=NA_real_,F_statistic=NA_real_,partial_R2=NA_real_,\\n    AIC_null=NA_real_,AIC_full=NA_real_,converged0=FALSE,converged1=FALSE,\\n    full_rank0=FALSE,full_rank1=FALSE,rank0=NA_integer_,rank1=NA_integer_,\\n    coefficients0=NA_integer_,coefficients1=NA_integer_,smooth_parameters=NA_integer_,\\n    residual_mean=NA_real_,residual_sd=NA_real_,residual_depth_rho=NA_real_,\\n    k_check_terms=0L,minimum_k_index=NA_real_,minimum_k_p=NA_real_,maximum_edf_ratio=NA_real_,\\n    cache_prediction_difference=NA_real_,cache_p_difference=NA_real_,\\n    segment_check_max_k_difference=NA_real_,segment_check_max_p_difference=NA_real_,\\n    warnings=\\\'\\\',error=\\\'\\\',fit_seconds=NA_real_)\\n  result <- tryCatch(capture_conditions({\\n    m0 <- fit_response(G0,y); m1 <- fit_response(G1,y)\\n    full <- as.numeric(X1 %*% coef(m1)); null <- as.numeric(X0 %*% coef(m0))\\n    stopifnot(max(abs(full-fitted(m1)))<1e-8,max(abs(null-fitted(m0)))<1e-8)\\n    residual <- y-full\\n    trajectory <- as.numeric(X_trajectory %*% coef(m1)[trajectory_columns])\\n    trajectory <- trajectory-mean(trajectory)+mean(y)\\n    rss0 <- sum((y-null)^2); rss1 <- sum(residual^2)\\n    stopifnot(abs(rss0-m0$deviance)<1e-7*max(1,rss0),\\n              abs(rss1-m1$deviance)<1e-7*max(1,rss1),\\n              length(m1$sp)==10L,all(is.finite(trajectory)),all(is.finite(full)))\\n    table <- anova(m0,m1,test=\\\'F\\\')\\n    p_column <- grep(\\\'^Pr\\\\\\\\(\\\',names(table),value=TRUE)\\n    stopifnot(nrow(table)==2L,length(p_column)==1L)\\n    p <- as.numeric(table[2,p_column]); f_statistic <- as.numeric(table[2,\\\'F\\\'])\\n    convergence <- c(converged(m0),converged(m1))\\n    ranks <- c(m0$rank==length(coef(m0)),m1$rank==length(coef(m1)))\\n    stopifnot(all(convergence),all(ranks),is.finite(p),p>=0,p<=1,\\n              is.finite(f_statistic),table[2,\\\'Df\\\']>0,table[2,\\\'Resid. Df\\\']>0)\\n    direct_prediction <- direct_p <- NA_real_\\n    if(j %in% VALIDATION_INDICES) {\\n      z <- d; z$y <- y\\n      fresh0 <- gam(f0,data=z,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      fresh1 <- gam(f1,data=z,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      direct_prediction <- max(abs(fitted(fresh0)-null),abs(fitted(fresh1)-full))\\n      direct_table <- anova(fresh0,fresh1,test=\\\'F\\\')\\n      direct_p <- abs(p-as.numeric(direct_table[2,p_column]))\\n      equivalence <- list(\\n        A=abs(diff(range(as.numeric(X_trajectory %*% coef(fresh1)[trajectory_columns])))-diff(range(trajectory))),\\n        RSS0=abs(rss0-fresh0$deviance),RSS1=abs(rss1-fresh1$deviance),\\n        EDF=max(abs(m1$edf-fresh1$edf)),EDF1=max(abs(m1$edf1-fresh1$edf1)),\\n        EDF2=if(!is.null(m1$edf2))max(abs(m1$edf2-fresh1$edf2))else 0,\\n        SP_relative=max(abs(m1$sp-fresh1$sp)/(1+abs(fresh1$sp))),\\n        F_relative=abs(f_statistic-as.numeric(direct_table[2,\\\'F\\\']))/(1+abs(as.numeric(direct_table[2,\\\'F\\\']))),\\n        DF0=abs(table[1,\\\'Resid. Df\\\']-direct_table[1,\\\'Resid. Df\\\']),\\n        DF1=abs(table[2,\\\'Resid. Df\\\']-direct_table[2,\\\'Resid. Df\\\']),\\n        AIC0=abs(AIC(m0)-AIC(fresh0)),AIC1=abs(AIC(m1)-AIC(fresh1)),\\n        ML_objective=abs(as.numeric(m1$gcv.ubre)-as.numeric(fresh1$gcv.ubre)))\\n      cat(paste(\\\'QR_EQUIVALENCE\\\',name,paste(names(equivalence),unlist(equivalence),sep=\\\'=\\\',collapse=\\\';\\\')), \\\'\\\\n\\\')\\n      stopifnot(equivalence$A<1e-7,equivalence$RSS0<1e-7,equivalence$RSS1<1e-7,\\n        equivalence$EDF<1e-4,equivalence$EDF1<1e-4,equivalence$EDF2<1e-4,\\n        equivalence$SP_relative<1e-3,equivalence$F_relative<1e-5,\\n        equivalence$DF0<1e-4,equivalence$DF1<1e-4,\\n        equivalence$AIC0<1e-4,equivalence$AIC1<1e-4,equivalence$ML_objective<1e-5)\\n      fresh_terms <- predict(fresh1,type=\\\'terms\\\')\\n      relevant <- colnames(fresh_terms)==\\\'seg\\\' | startsWith(colnames(fresh_terms),\\\'s(u\\\')\\n      fresh_trajectory <- rowSums(fresh_terms[,relevant,drop=FALSE])\\n      fresh_trajectory <- fresh_trajectory-mean(fresh_trajectory)+mean(y)\\n      direct_prediction <- max(direct_prediction,abs(fresh_trajectory-trajectory))\\n      stopifnot(direct_prediction<VALIDATION_TOL,direct_p<1e-8)\\n    }\\n    local_k_rows <- local_residual_rows <- local_validation_rows <- list()\\n    for(si in seq_along(G1$smooth)) {\\n      sid <- smooth_segments[si]; rows <- segment_rows[[sid]]\\n      edf <- sum(m1$edf[smooth_columns[[si]]]); k_prime <- length(smooth_columns[[si]])\\n      r <- residual[rows]; u <- d$u[rows]; seed <- SEED+j+as.integer(sid)\\n      diagnostic <- segment_check(u,r,seed)\\n      k_row <- data.frame(regulon=name,index=j,mode=MODE,K=K,seg=sid,\\n        term=G1$smooth[[si]]$label,n_segment=length(rows),n_checked=diagnostic$checked_cells,\\n        unique_u=length(unique(u)),k_prime=k_prime,edf=edf,edf_ratio=edf/k_prime,\\n        smoothing_parameter=as.numeric(m1$sp[si]),k_index=diagnostic$k_index,\\n        k_index_all_cells=adjacent_index(u,r),p_value=diagnostic$p_value,\\n        permutations=K_CHECK_REPS,seed=seed,status=diagnostic$status,\\n        tied_cells=diagnostic$tied_cells,tied_groups=diagnostic$tied_groups,\\n        diagnostic_method=\\\'Exact expected adjacent SSE over within-t ties; residual permutation\\\')\\n      local_k_rows <- add_frame(local_k_rows,k_row)\\n      local_residual_rows <- add_frame(local_residual_rows,data.frame(regulon=name,index=j,\\n        mode=MODE,K=K,seg=sid,cells=length(rows),residual_mean=mean(r),residual_sd=sd(r),\\n        residual_RMS=sqrt(mean(r^2)),residual_min=min(r),residual_max=max(r),\\n        residual_depth_rho=rho(r,d$log_detected[rows]),adjacent_k_index=adjacent_index(u,r)))\\n      if(j %in% VALIDATION_INDICES && diagnostic$status==\\\'ok\\\' && !anyDuplicated(u)) {\\n        check <- validate_segment_check(rows,r,seed)\\n        local_validation_rows <- add_frame(local_validation_rows,data.frame(regulon=name,index=j,\\n          mode=MODE,K=K,seg=sid,term=G1$smooth[[si]]$label,\\n          check=\\\'single_segment_mgcv_kcheck_with_identical_joint_residuals\\\',\\n          k_index_difference=check$k_index_difference,\\n          permutation_p_difference=check$permutation_p_difference,\\n          residual_injection_error=check$residual_injection_error))\\n      }\\n    }\\n    list(m0=m0,m1=m1,trajectory=trajectory,full=full,residual=residual,rss0=rss0,rss1=rss1,\\n      p=p,F=f_statistic,table=table,convergence=convergence,ranks=ranks,\\n      direct_prediction=direct_prediction,direct_p=direct_p,\\n      k=do.call(rbind,local_k_rows),segment_residual=do.call(rbind,local_residual_rows),\\n      validation=if(length(local_validation_rows))do.call(rbind,local_validation_rows)else NULL)\\n  }),error=function(e)e)\\n  if(inherits(result,\\\'error\\\')) {\\n    row$error <- conditionMessage(result)\\n  } else {\\n    value <- result$value\\n    row$status <- \\\'ok\\\'; row$p_value <- value$p; row$A <- diff(range(value$trajectory))\\n    row$RSS0 <- value$rss0; row$RSS1 <- value$rss1\\n    row$df0 <- value$m0$df.residual; row$df1 <- value$m1$df.residual\\n    row$test_df0 <- value$table[1,\\\'Resid. Df\\\']; row$test_df1 <- value$table[2,\\\'Resid. Df\\\']\\n    row$test_delta_df <- value$table[2,\\\'Df\\\']; row$F_statistic <- value$F\\n    row$partial_R2 <- (value$rss0-value$rss1)/value$rss0\\n    row$AIC_null <- AIC(value$m0); row$AIC_full <- AIC(value$m1)\\n    row$converged0 <- value$convergence[1]; row$converged1 <- value$convergence[2]\\n    row$full_rank0 <- value$ranks[1]; row$full_rank1 <- value$ranks[2]\\n    row$rank0 <- value$m0$rank; row$rank1 <- value$m1$rank\\n    row$coefficients0 <- length(coef(value$m0)); row$coefficients1 <- length(coef(value$m1))\\n    row$smooth_parameters <- length(value$m1$sp)\\n    row$residual_mean <- mean(value$residual); row$residual_sd <- sd(value$residual)\\n    row$residual_depth_rho <- rho(value$residual,d$log_detected)\\n    row$k_check_terms <- nrow(value$k)\\n    row$minimum_k_index <- min(value$k$k_index,na.rm=TRUE)\\n    row$minimum_k_p <- min(value$k$p_value,na.rm=TRUE)\\n    row$maximum_edf_ratio <- max(value$k$edf_ratio)\\n    row$cache_prediction_difference <- value$direct_prediction; row$cache_p_difference <- value$direct_p\\n    if(!is.null(value$validation)) {\\n      row$segment_check_max_k_difference <- max(value$validation$k_index_difference)\\n      row$segment_check_max_p_difference <- max(value$validation$permutation_p_difference)\\n      validation_rows <- add_frame(validation_rows,value$validation)\\n    }\\n    trajectory_values[,ii] <- value$trajectory; full_values[,ii] <- value$full\\n    residual_values[,ii] <- value$residual\\n    k_rows <- add_frame(k_rows,value$k)\\n    segment_residual_rows <- add_frame(segment_residual_rows,value$segment_residual)\\n    for(study in levels(d$study)) {\\n      hit <- which(d$study==study); r <- value$residual[hit]\\n      source_residual_rows <- add_frame(source_residual_rows,data.frame(regulon=name,index=j,\\n        mode=MODE,K=K,study=study,cells=length(hit),residual_mean=mean(r),residual_sd=sd(r),\\n        residual_RMS=sqrt(mean(r^2)),residual_depth_rho=rho(r,d$log_detected[hit])))\\n    }\\n    row$warnings <- paste(result$warnings,collapse=\\\' | \\\')\\n  }\\n  row$fit_seconds <- proc.time()[[\\\'elapsed\\\']]-tick\\n  summary_rows <- add_frame(summary_rows,row)\\n  if(ii%%10L==0L || ii==length(INDICES)) cat(sprintf(\\\'%s K%d: %d/%d; %s; %.1fs\\\\n\\\',\\n    MODE,K,ii,length(INDICES),row$status,proc.time()[[\\\'elapsed\\\']]-started))\\n}\\n\\n\\n#5. =========== Shard evidence; complete-family correction belongs to Python\\nwrite_rows <- function(rows,suffix) {\\n  if(!length(rows)) return(invisible(NULL))\\n  write.table(do.call(rbind,rows),paste0(PREFIX,suffix),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE,na=\\\'NA\\\')\\n}\\nsave_matrix <- function(matrix,suffix) {\\n  connection <- file(paste0(PREFIX,suffix),\\\'wb\\\')\\n  tryCatch(writeBin(as.numeric(matrix),connection,size=8,endian=\\\'little\\\'),finally=close(connection))\\n}\\nwrite_rows(summary_rows,\\\'.stats.tsv\\\')\\nwrite_rows(k_rows,\\\'.kcheck.tsv\\\')\\nwrite_rows(segment_residual_rows,\\\'.segment_residual.tsv\\\')\\nwrite_rows(source_residual_rows,\\\'.source_residual.tsv\\\')\\nwrite_rows(validation_rows,\\\'.validation.tsv\\\')\\nsave_matrix(trajectory_values,\\\'.trajectory.bin\\\')\\nsave_matrix(full_values,\\\'.fullpred.bin\\\')\\nsave_matrix(residual_values,\\\'.residuals.bin\\\')\\nmetadata_rows <- rbind(metadata_rows,data.frame(metric=c(\\\'elapsed_seconds\\\',\\\'indices\\\',\\\'diagnostic_scope\\\'),\\n  value=c(proc.time()[[\\\'elapsed\\\']]-started,paste(INDICES,collapse=\\\',\\\'),\\\'Every fitted regulon, all ten biological native segments; no candidate or diagnostic sampling\\\')))\\nwrite.table(metadata_rows,paste0(PREFIX,\\\'.metadata.tsv\\\'),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE,na=\\\'NA\\\')\\n\\n\'\n\nEARLY_R = \'\\n#1. =========== Fixed Gaussian ML models and diagnostic settings\\nsuppressPackageStartupMessages(library(mgcv))\\nargs <- commandArgs(trailingOnly=TRUE)\\nstopifnot(length(args)==8L)\\nCELLS<-args[1]; AUC<-args[2]; PREFIX<-args[3]; KIND<-args[4]\\nMODE<-args[5]; K<-as.integer(args[6]); INDICES<-as.integer(strsplit(args[7],\\\',\\\',fixed=TRUE)[[1]])\\nVALIDATE<-identical(args[8],\\\'1\\\')\\nSEED<-42L; REPS<-500L; SUBSAMPLE<-2000L; GRID_N<-51L\\noptions(stringsAsFactors=FALSE,digits=17)\\nd<-read.csv(CELLS,row.names=1,check.names=FALSE)\\ncon<-gzfile(AUC,\\\'rt\\\'); Y<-tryCatch(as.matrix(read.delim(con,row.names=1,check.names=FALSE)),finally=close(con))\\nstorage.mode(Y)<-\\\'double\\\'\\nstopifnot(identical(rownames(d),rownames(Y)),ncol(Y)==220L,all(is.finite(Y)),\\n          all(is.finite(d$v)),all(is.finite(d$log_detected)),all(Y>=0 & Y<=1))\\nd$study<-factor(d$study); d$stage<-factor(d$stage); d$branch<-factor(as.character(d$branch))\\nstopifnot(all(table(d$branch)>=20L),all(vapply(split(d$v,d$branch),function(v)length(unique(v))>=K,logical(1))))\\nsource_term<-if(nlevels(d$study)>1L)\\\'study\\\'else character()\\nterms<-c(if(MODE %in% c(\\\'source\\\',\\\'source_depth\\\'))source_term,\\n         if(MODE %in% c(\\\'stage\\\',\\\'stage_depth\\\'))\\\'stage\\\',\\n         if(MODE %in% c(\\\'depth\\\',\\\'source_depth\\\',\\\'stage_depth\\\'))\\\'log_detected\\\')\\nC<-if(length(terms))paste(terms,collapse=\\\' + \\\')else\\\'1\\\'\\nif(KIND==\\\'early\\\') {\\n  f0<-as.formula(paste(\\\'y ~\\\',C))\\n  f1<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ s(v,bs=\\\'cr\\\',k=%d)",K)))\\n} else {\\n  stopifnot(nlevels(d$branch)==2L)\\n  f0<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ s(v,bs=\\\'cr\\\',k=%d)",K)))\\n  f1<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ branch + s(v,by=branch,bs=\\\'cr\\\',k=%d)",K)))\\n}\\nd$y<-Y[,1L]\\ncache_qr<-function(G) {\\n  q<-qr(G$X,tol=0,LAPACK=TRUE); rp<-q$pivot; rp[rp]<-seq_len(ncol(G$X))\\n  G$fast_qr<-q; G$fast_R<-qr.R(q)[,rp,drop=FALSE]\\n  G$fast_sp<-mgcv:::initial.spg(G$X,G$y,G$w,G$family,G$S,G$rank,G$off,\\n                              offset=G$offset,L=G$L,lsp0=G$lsp0)\\n  stopifnot(max(abs(crossprod(G$fast_R)-crossprod(G$X)))<1e-7)\\n  G\\n}\\nG0<-cache_qr(gam(f0,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))\\nG1<-cache_qr(gam(f1,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))\\nX0<-G0$X; X1<-G1$X\\nsc<-lapply(G1$smooth,function(s)s$first.para:s$last.para)\\ntc<-sort(unique(c(which(startsWith(colnames(X1),\\\'branch\\\')),unlist(sc))))\\nXT<-X1[,tc,drop=FALSE]\\ngrid<-seq(min(d$v),max(d$v),length.out=GRID_N)\\n\\n#2. =========== Exact fixed-X sufficient-statistic fit and full-cell restoration\\nfit_response<-function(G,y) {\\n  X<-G$X; N<-length(y); p<-ncol(X)\\n  init<-list(sp=G$fast_sp,scale=mean((y-mean(y))^2)/10)\\n  qy<-qr.qty(G$fast_qr,y)[seq_len(p)]; extra<-sum(y*y)-sum(qy*qy)\\n  stopifnot(extra>=-1e-10)\\n  G$X<-G$fast_R;G$y<-qy;G$w<-rep(1,p);G$n<-p;G$offset<-rep(0,p)\\n  G$n.true<-N;G$dev.extra<-G$pearson.extra<-max(0,extra);G$mf$y<-y\\n  G$fast_qr<-G$fast_R<-G$fast_sp<-NULL\\n  m<-gam(G=G,method=\\\'ML\\\',gamma=1,in.out=init)\\n  mu<-as.numeric(X%*%coef(m));r<-y-mu\\n  m$y<-m$model$y<-y;m$fitted.values<-m$linear.predictors<-mu;m$residuals<-r\\n  m$prior.weights<-m$weights<-m$working.weights<-rep(1,N);m$offset<-rep(0,N);m$z<-y\\n  m$deviance<-sum(r*r);m$null.deviance<-sum((y-mean(y))^2)\\n  m$df.residual<-N-sum(m$edf);m$df.null<-N-as.integer(G$intercept)\\n  m$aic<-m$family$aic(y,1,mu,rep(1,N),m$deviance)+2*sum(m$edf)\\n  m$hat<-rowSums((X%*%m$Vp)*X)/m$sig2\\n  m\\n}\\nconverged<-function(m) {\\n  z<-m$outer.info$conv\\n  (is.null(m$converged)||isTRUE(m$converged)) &&\\n   (is.null(z)||identical(z,0L)||identical(z,0)||grepl(\\\'full convergence\\\',paste(z,collapse=\\\' \\\'),fixed=TRUE))\\n}\\ntie_expected_sse<-function(r,group,counts) {\\n  sums<-rowsum(cbind(r,r*r),group,reorder=TRUE)\\n  means<-sums[,1]/counts;ss<-pmax(0,sums[,2]-sums[,1]^2/counts)\\n  variances<-ss/counts\\n  total<-2*sum(ss)\\n  m<-length(counts)\\n  if(m>1L)total<-total+sum(variances[-m]+variances[-1L]+diff(means)^2)\\n  total\\n}\\n# The analytic expected SSE equals enumeration over every within-t tie order.\\ntoy_r<-c(.1,-.3,.2,.6,-.1,.9);toy_group<-c(1,1,2,2,2,3)\\ntoy_permutations<-function(x)if(length(x)==1L)list(x)else unlist(lapply(seq_along(x),function(i)\\n  lapply(toy_permutations(x[-i]),function(z)c(x[i],z))),recursive=FALSE)\\ntoy_orders<-lapply(toy_permutations(c(1L,2L)),function(a)lapply(toy_permutations(c(3L,4L,5L)),function(b)c(a,b,6L)))\\ntoy_orders<-unlist(toy_orders,recursive=FALSE)\\nstopifnot(abs(tie_expected_sse(toy_r,toy_group,c(2L,3L,1L))-\\n  mean(vapply(toy_orders,function(z)sum(diff(toy_r[z])^2),numeric(1))))<1e-12)\\nk_check<-function(v,r,seed) {\\n  set.seed(seed)\\n  if(length(r)>SUBSAMPLE) {idx<-sample.int(length(r),SUBSAMPLE);v<-v[idx];r<-r[idx]}\\n  n<-length(r);den<-mean(r*r)\\n  if(n<2L||den<=0)return(list(index=NA_real_,p=NA_real_,n=n,status=\\\'zero_residual_variance\\\'))\\n  group<-match(v,sort(unique(v)));counts<-tabulate(group)\\n  observed<-tie_expected_sse(r,group,counts)/(2*(n-1L))\\n  reference<-vapply(seq_len(REPS),function(i)tie_expected_sse(r[sample.int(n,n)],group,counts)/(2*(n-1L)),numeric(1))\\n  list(index=observed/den,p=mean(reference<observed),n=n,status=\\\'ok\\\',\\n    tied_cells=sum(counts[counts>1L]),tied_groups=sum(counts>1L),\\n    old_index=mean(diff(r[order(v)])^2)/2/den)\\n}\\ndiagnose<-function(m,G,r,j,role) {\\n  if(!length(G$smooth))return(NULL)\\n  rows<-list()\\n  for(si in seq_along(G$smooth)) {\\n    sm<-G$smooth[[si]]\\n    hit<-if(is.null(sm$by.level))seq_len(nrow(d))else which(as.character(d$branch)==as.character(sm$by.level))\\n    cols<-sm$first.para:sm$last.para;edf<-sum(m$edf[cols]);kp<-length(cols)\\n    seed<-SEED+j+si+if(role==\\\'null\\\')1000L else 0L\\n    z<-k_check(d$v[hit],r[hit],seed)\\n    check_k<-check_p<-NA_real_\\n    if(VALIDATE && j==1L && z$status==\\\'ok\\\' && !anyDuplicated(d$v[hit])) {\\n      fixture<-gam(rr~s(v,bs=\\\'cr\\\',k=K),data=data.frame(v=d$v[hit],rr=r[hit]),\\n                   family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE)\\n      fixture$y<-r[hit];fixture$fitted.values<-fixture$linear.predictors<-rep(0,length(hit))\\n      stopifnot(max(abs(residuals(fixture)-r[hit]))<1e-12)\\n      set.seed(seed);official<-k.check(fixture,subsample=SUBSAMPLE,n.rep=REPS)\\n      check_k<-abs(z$index-official[1,\\\'k-index\\\']);check_p<-abs(z$p-official[1,\\\'p-value\\\'])\\n      stopifnot(check_k<1e-12,check_p==0)\\n    }\\n    rows[[si]]<-data.frame(regulon=colnames(Y)[j],regulon_index=j,model_role=role,\\n      term=sm$label,branch=if(is.null(sm$by.level))\\\'all\\\'else as.character(sm$by.level),\\n      n_cells=length(hit),n_checked=z$n,unique_v=length(unique(d$v[hit])),k_prime=kp,\\n      edf=edf,edf_ratio=edf/kp,k_index=z$index,p_value=z$p,status=z$status,\\n      basis_flag=is.finite(z$p)&&z$p<.01&&edf/kp>.9,\\n      structure_flag=is.finite(z$p)&&z$p<.01&&z$index<.90,\\n      smoothing_parameter=as.numeric(m$sp[si]),seed=seed,permutations=REPS,\\n      official_k_difference=check_k,official_p_difference=check_p)\\n    rows[[si]]$tied_cells<-z$tied_cells\\n    rows[[si]]$tied_groups<-z$tied_groups\\n    rows[[si]]$old_ordered_k_index<-z$old_index\\n    rows[[si]]$diagnostic_method<-\\\'Tie-order-invariant expected adjacent SSE; fixed exact-t groups; permuted residuals\\\'\\n  }\\n  do.call(rbind,rows)\\n}\\n\\n#3. =========== Every response, local contrast and complete residual evidence\\nN<-nrow(d);J<-length(INDICES)\\nFULL<-RESID<-TRAJ<-matrix(NA_real_,N,J)\\nCONTRAST<-CONTRAST_SE<-matrix(NA_real_,GRID_N,J)\\nstatistics<-diagnostics<-validations<-list()\\nstarted<-proc.time()[[\\\'elapsed\\\']]\\nfor(ii in seq_along(INDICES)) {\\n  j<-INDICES[ii];y<-as.numeric(Y[,j]);tick<-proc.time()[[\\\'elapsed\\\']]\\n  row<-data.frame(regulon=colnames(Y)[j],regulon_index=j,status=\\\'failed\\\',test_status=\\\'not_run\\\',\\n    p_value=1,A=NA_real_,contrast_max_abs=NA_real_,contrast_mean=NA_real_,\\n    contrast_start=NA_real_,contrast_end=NA_real_,RSS0=NA_real_,RSS1=NA_real_,\\n    df0=NA_real_,df1=NA_real_,test_df0=NA_real_,test_df1=NA_real_,test_delta_df=NA_real_,\\n    F_statistic=NA_real_,partial_R2=NA_real_,converged0=FALSE,converged1=FALSE,\\n    full_rank0=FALSE,full_rank1=FALSE,full_basis_flags=NA_integer_,full_structure_flags=NA_integer_,\\n    null_basis_flags=NA_integer_,residual_mean=NA_real_,residual_sd=NA_real_,\\n    cache_prediction_difference=NA_real_,cache_F_relative_difference=NA_real_,\\n    warnings=\\\'\\\',error=\\\'\\\',fit_seconds=NA_real_)\\n  notices<-character()\\n  value<-tryCatch(withCallingHandlers({\\n    m0<-fit_response(G0,y);m1<-fit_response(G1,y)\\n    ok<-c(converged(m0),converged(m1),m0$rank==length(coef(m0)),m1$rank==length(coef(m1)))\\n    stopifnot(all(ok))\\n    full<-as.numeric(X1%*%coef(m1));null<-as.numeric(X0%*%coef(m0));r<-y-full\\n    tr<-as.numeric(XT%*%coef(m1)[tc]);tr<-tr-mean(tr)+mean(y)\\n    rss0<-sum((y-null)^2);rss1<-sum(r*r)\\n    stopifnot(max(abs(full-fitted(m1)))<1e-9,abs(rss1-m1$deviance)<1e-9)\\n    tab<-anova(m0,m1,test=\\\'F\\\');pcol<-grep(\\\'^Pr\\\\\\\\(\\\',names(tab),value=TRUE)\\n    p<-as.numeric(tab[2,pcol]);Fv<-as.numeric(tab[2,\\\'F\\\'])\\n    test_status<-if(is.finite(p)&&tab[2,\\\'Df\\\']>0&&is.finite(Fv)&&Fv>=0)\\\'ok\\\'else\\\'nonpositive_incremental_test\\\'\\n    if(test_status!=\\\'ok\\\')p<-1\\n    nd<-d[rep(1L,GRID_N),,drop=FALSE];nd$v<-grid\\n    if(KIND==\\\'early\\\') {\\n      L<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\');L<-sweep(L,2,L[1,],\\\'-\\\')\\n    } else {\\n      nd$branch<-factor(rep(levels(d$branch)[1],GRID_N),levels=levels(d$branch))\\n      LA<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\')\\n      nd$branch<-factor(rep(levels(d$branch)[2],GRID_N),levels=levels(d$branch))\\n      LB<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\');L<-LB-LA\\n    }\\n    nuisance<-setdiff(seq_len(ncol(X1)),tc)\\n    if(length(nuisance))stopifnot(max(abs(L[,nuisance,drop=FALSE]))<1e-12)\\n    delta<-as.numeric(L%*%coef(m1));se<-sqrt(pmax(0,rowSums((L%*%m1$Vp)*L)))\\n    dq<-rbind(diagnose(m1,G1,r,j,\\\'full\\\'),diagnose(m0,G0,y-null,j,\\\'null\\\'))\\n    pred_error<-F_error<-NA_real_;validation<-NULL\\n    if(VALIDATE && j %in% c(218L,219L,220L)) {\\n      dd<-d;dd$y<-y\\n      fresh0<-gam(f0,data=dd,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      fresh1<-gam(f1,data=dd,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      ft<-anova(fresh0,fresh1,test=\\\'F\\\')\\n      pred_error<-max(abs(fitted(fresh0)-null),abs(fitted(fresh1)-full))\\n      F_error<-if(test_status==\\\'ok\\\')abs(Fv-as.numeric(ft[2,\\\'F\\\']))/(1+abs(Fv))else 0\\n      edf_error<-max(abs(m1$edf-fresh1$edf),abs(m0$edf-fresh0$edf))\\n      ml_error<-max(abs(as.numeric(m0$gcv.ubre)-as.numeric(fresh0$gcv.ubre)),\\n                    abs(as.numeric(m1$gcv.ubre)-as.numeric(fresh1$gcv.ubre)))\\n      stopifnot(pred_error<1e-7,F_error<1e-5,edf_error<1e-4,ml_error<1e-5)\\n      validation<-data.frame(regulon=colnames(Y)[j],prediction_difference=pred_error,\\n        F_relative_difference=F_error,EDF_difference=edf_error,ML_objective_difference=ml_error)\\n    }\\n    list(m0=m0,m1=m1,ok=ok,full=full,r=r,tr=tr,rss0=rss0,rss1=rss1,tab=tab,p=p,F=Fv,\\n      test_status=test_status,delta=delta,se=se,dq=dq,pred_error=pred_error,F_error=F_error,validation=validation)\\n  },warning=function(w){notices<<-c(notices,conditionMessage(w));invokeRestart(\\\'muffleWarning\\\')}),error=function(e)e)\\n  if(inherits(value,\\\'error\\\'))row$error<-conditionMessage(value) else {\\n    row$status<-\\\'ok\\\';row$test_status<-value$test_status;row$p_value<-value$p\\n    row$A<-diff(range(value$tr));row$contrast_max_abs<-max(abs(value$delta))\\n    row$contrast_mean<-mean(value$delta);row$contrast_start<-value$delta[1L];row$contrast_end<-tail(value$delta,1)\\n    row$RSS0<-value$rss0;row$RSS1<-value$rss1;row$df0<-value$m0$df.residual;row$df1<-value$m1$df.residual\\n    row$test_df0<-value$tab[1,\\\'Resid. Df\\\'];row$test_df1<-value$tab[2,\\\'Resid. Df\\\'];row$test_delta_df<-value$tab[2,\\\'Df\\\']\\n    row$F_statistic<-value$F;row$partial_R2<-(value$rss0-value$rss1)/value$rss0\\n    row$converged0<-value$ok[1];row$converged1<-value$ok[2];row$full_rank0<-value$ok[3];row$full_rank1<-value$ok[4]\\n    row$full_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role==\\\'full\\\'])\\n    row$full_structure_flags<-sum(value$dq$structure_flag[value$dq$model_role==\\\'full\\\'])\\n    row$null_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role==\\\'null\\\'])\\n    row$residual_mean<-mean(value$r);row$residual_sd<-sd(value$r)\\n    row$cache_prediction_difference<-value$pred_error;row$cache_F_relative_difference<-value$F_error\\n    FULL[,ii]<-value$full;RESID[,ii]<-value$r;TRAJ[,ii]<-value$tr\\n    CONTRAST[,ii]<-value$delta;CONTRAST_SE[,ii]<-value$se\\n    diagnostics[[length(diagnostics)+1L]]<-value$dq\\n    if(!is.null(value$validation))validations[[length(validations)+1L]]<-value$validation\\n  }\\n  row$warnings<-paste(unique(notices),collapse=\\\' | \\\');row$fit_seconds<-proc.time()[[\\\'elapsed\\\']]-tick\\n  statistics[[ii]]<-row\\n  if(ii%%20L==0L||ii==J)cat(sprintf(\\\'%s %s K%d %d/%d %.1fs\\\\n\\\',KIND,MODE,K,ii,J,proc.time()[[\\\'elapsed\\\']]-started))\\n}\\nwrite_table<-function(rows,suffix)if(length(rows))write.table(do.call(rbind,rows),paste0(PREFIX,suffix),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE,na=\\\'NA\\\')\\nwrite_matrix<-function(x,suffix){con<-file(paste0(PREFIX,suffix),\\\'wb\\\');writeBin(as.numeric(x),con,size=8,endian=\\\'little\\\');close(con)}\\nwrite_table(statistics,\\\'.stats.tsv\\\');write_table(diagnostics,\\\'.diagnostics.tsv\\\');write_table(validations,\\\'.validation.tsv\\\')\\nwrite_matrix(FULL,\\\'.full.bin\\\');write_matrix(RESID,\\\'.residual.bin\\\');write_matrix(TRAJ,\\\'.trajectory.bin\\\')\\nwrite_matrix(CONTRAST,\\\'.contrast.bin\\\');write_matrix(CONTRAST_SE,\\\'.contrast_se.bin\\\')\\nwrite.table(data.frame(metric=c(\\\'R_version\\\',\\\'mgcv_version\\\',\\\'null_formula\\\',\\\'full_formula\\\',\\\'mode\\\',\\\'K\\\',\\\'cells\\\',\\\'permutations\\\',\\\'subsample\\\',\\\'seed\\\',\\\'computation\\\'),\\n value=c(R.version.string,as.character(packageVersion(\\\'mgcv\\\')),paste(deparse(f0),collapse=\\\' \\\'),paste(deparse(f1),collapse=\\\' \\\'),MODE,K,N,REPS,SUBSAMPLE,SEED,\\n \\\'Exact thin-QR Gaussian ML with original full-X smoothing and full-y scale initialization; full-cell attributes restored\\\')),\\n paste0(PREFIX,\\\'.metadata.tsv\\\'),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE)\\n\'\n\nLATE_R = \'\\n#1. =========== Fixed Gaussian ML models and diagnostic settings\\nsuppressPackageStartupMessages(library(mgcv))\\nargs <- commandArgs(trailingOnly=TRUE)\\nstopifnot(length(args)==8L)\\nCELLS<-args[1]; AUC<-args[2]; PREFIX<-args[3]; KIND<-args[4]\\nMODE<-args[5]; K<-as.integer(args[6]); INDICES<-as.integer(strsplit(args[7],\\\',\\\',fixed=TRUE)[[1]])\\nVALIDATE<-identical(args[8],\\\'1\\\')\\nSEED<-42L; REPS<-500L; SUBSAMPLE<-2000L; GRID_N<-51L\\noptions(stringsAsFactors=FALSE,digits=17)\\nd<-read.csv(CELLS,row.names=1,check.names=FALSE)\\ncon<-gzfile(AUC,\\\'rt\\\'); Y<-tryCatch(as.matrix(read.delim(con,row.names=1,check.names=FALSE)),finally=close(con))\\nstorage.mode(Y)<-\\\'double\\\'\\nstopifnot(identical(rownames(d),rownames(Y)),ncol(Y)==220L,all(is.finite(Y)),\\n          all(is.finite(d$v)),all(is.finite(d$log_detected)),all(Y>=0 & Y<=1))\\nd$study<-factor(d$study); d$branch<-factor(as.character(d$branch))\\nstopifnot(all(table(d$branch)>=20L),all(vapply(split(d$v,d$branch),function(v)length(unique(v))>=K,logical(1))))\\nsource_term<-if(nlevels(d$study)>1L)\\\'study\\\'else character()\\nterms<-c(if(MODE %in% c(\\\'source\\\',\\\'source_depth\\\'))source_term,\\n         if(MODE %in% c(\\\'depth\\\',\\\'source_depth\\\'))\\\'log_detected\\\')\\nC<-if(length(terms))paste(terms,collapse=\\\' + \\\')else\\\'1\\\'\\nif(KIND==\\\'early\\\') {\\n  f0<-as.formula(paste(\\\'y ~\\\',C))\\n  f1<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ s(v,bs=\\\'cr\\\',k=%d)",K)))\\n} else {\\n  stopifnot(nlevels(d$branch)==2L)\\n  f0<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ s(v,bs=\\\'cr\\\',k=%d)",K)))\\n  f1<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ branch + s(v,by=branch,bs=\\\'cr\\\',k=%d)",K)))\\n}\\nd$y<-Y[,1L]\\ncache_qr<-function(G) {\\n  q<-qr(G$X,tol=0,LAPACK=TRUE); rp<-q$pivot; rp[rp]<-seq_len(ncol(G$X))\\n  G$fast_qr<-q; G$fast_R<-qr.R(q)[,rp,drop=FALSE]\\n  G$fast_sp<-mgcv:::initial.spg(G$X,G$y,G$w,G$family,G$S,G$rank,G$off,\\n                              offset=G$offset,L=G$L,lsp0=G$lsp0)\\n  stopifnot(max(abs(crossprod(G$fast_R)-crossprod(G$X)))<1e-7)\\n  G\\n}\\nG0<-cache_qr(gam(f0,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))\\nG1<-cache_qr(gam(f1,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))\\nX0<-G0$X; X1<-G1$X\\nsc<-lapply(G1$smooth,function(s)s$first.para:s$last.para)\\ntc<-sort(unique(c(which(startsWith(colnames(X1),\\\'branch\\\')),unlist(sc))))\\nXT<-X1[,tc,drop=FALSE]\\ngrid<-seq(min(d$v),max(d$v),length.out=GRID_N)\\n\\n#2. =========== Exact fixed-X sufficient-statistic fit and full-cell restoration\\nfit_response<-function(G,y) {\\n  X<-G$X; N<-length(y); p<-ncol(X)\\n  init<-list(sp=G$fast_sp,scale=mean((y-mean(y))^2)/10)\\n  qy<-qr.qty(G$fast_qr,y)[seq_len(p)]; extra<-sum(y*y)-sum(qy*qy)\\n  stopifnot(extra>=-1e-10)\\n  G$X<-G$fast_R;G$y<-qy;G$w<-rep(1,p);G$n<-p;G$offset<-rep(0,p)\\n  G$n.true<-N;G$dev.extra<-G$pearson.extra<-max(0,extra);G$mf$y<-y\\n  G$fast_qr<-G$fast_R<-G$fast_sp<-NULL\\n  m<-gam(G=G,method=\\\'ML\\\',gamma=1,in.out=init)\\n  mu<-as.numeric(X%*%coef(m));r<-y-mu\\n  m$y<-m$model$y<-y;m$fitted.values<-m$linear.predictors<-mu;m$residuals<-r\\n  m$prior.weights<-m$weights<-m$working.weights<-rep(1,N);m$offset<-rep(0,N);m$z<-y\\n  m$deviance<-sum(r*r);m$null.deviance<-sum((y-mean(y))^2)\\n  m$df.residual<-N-sum(m$edf);m$df.null<-N-as.integer(G$intercept)\\n  m$aic<-m$family$aic(y,1,mu,rep(1,N),m$deviance)+2*sum(m$edf)\\n  m$hat<-rowSums((X%*%m$Vp)*X)/m$sig2\\n  m\\n}\\nconverged<-function(m) {\\n  z<-m$outer.info$conv\\n  (is.null(m$converged)||isTRUE(m$converged)) &&\\n   (is.null(z)||identical(z,0L)||identical(z,0)||grepl(\\\'full convergence\\\',paste(z,collapse=\\\' \\\'),fixed=TRUE))\\n}\\ntie_expected_sse<-function(r,group,counts) {\\n  sums<-rowsum(cbind(r,r*r),group,reorder=TRUE)\\n  means<-sums[,1]/counts;ss<-pmax(0,sums[,2]-sums[,1]^2/counts)\\n  variances<-ss/counts\\n  total<-2*sum(ss)\\n  m<-length(counts)\\n  if(m>1L)total<-total+sum(variances[-m]+variances[-1L]+diff(means)^2)\\n  total\\n}\\n# The analytic expected SSE equals enumeration over every within-t tie order.\\ntoy_r<-c(.1,-.3,.2,.6,-.1,.9);toy_group<-c(1,1,2,2,2,3)\\ntoy_permutations<-function(x)if(length(x)==1L)list(x)else unlist(lapply(seq_along(x),function(i)\\n  lapply(toy_permutations(x[-i]),function(z)c(x[i],z))),recursive=FALSE)\\ntoy_orders<-lapply(toy_permutations(c(1L,2L)),function(a)lapply(toy_permutations(c(3L,4L,5L)),function(b)c(a,b,6L)))\\ntoy_orders<-unlist(toy_orders,recursive=FALSE)\\nstopifnot(abs(tie_expected_sse(toy_r,toy_group,c(2L,3L,1L))-\\n  mean(vapply(toy_orders,function(z)sum(diff(toy_r[z])^2),numeric(1))))<1e-12)\\nk_check<-function(v,r,seed) {\\n  set.seed(seed)\\n  if(length(r)>SUBSAMPLE) {idx<-sample.int(length(r),SUBSAMPLE);v<-v[idx];r<-r[idx]}\\n  n<-length(r);den<-mean(r*r)\\n  if(n<2L||den<=0)return(list(index=NA_real_,p=NA_real_,n=n,status=\\\'zero_residual_variance\\\'))\\n  group<-match(v,sort(unique(v)));counts<-tabulate(group)\\n  observed<-tie_expected_sse(r,group,counts)/(2*(n-1L))\\n  reference<-vapply(seq_len(REPS),function(i)tie_expected_sse(r[sample.int(n,n)],group,counts)/(2*(n-1L)),numeric(1))\\n  list(index=observed/den,p=mean(reference<observed),n=n,status=\\\'ok\\\',\\n    tied_cells=sum(counts[counts>1L]),tied_groups=sum(counts>1L),\\n    old_index=mean(diff(r[order(v)])^2)/2/den)\\n}\\ndiagnose<-function(m,G,r,j,role) {\\n  if(!length(G$smooth))return(NULL)\\n  rows<-list()\\n  for(si in seq_along(G$smooth)) {\\n    sm<-G$smooth[[si]]\\n    hit<-if(is.null(sm$by.level))seq_len(nrow(d))else which(as.character(d$branch)==as.character(sm$by.level))\\n    cols<-sm$first.para:sm$last.para;edf<-sum(m$edf[cols]);kp<-length(cols)\\n    seed<-SEED+j+si+if(role==\\\'null\\\')1000L else 0L\\n    z<-k_check(d$v[hit],r[hit],seed)\\n    check_k<-check_p<-NA_real_\\n    if(VALIDATE && j==1L && z$status==\\\'ok\\\' && !anyDuplicated(d$v[hit])) {\\n      fixture<-gam(rr~s(v,bs=\\\'cr\\\',k=K),data=data.frame(v=d$v[hit],rr=r[hit]),\\n                   family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE)\\n      fixture$y<-r[hit];fixture$fitted.values<-fixture$linear.predictors<-rep(0,length(hit))\\n      stopifnot(max(abs(residuals(fixture)-r[hit]))<1e-12)\\n      set.seed(seed);official<-k.check(fixture,subsample=SUBSAMPLE,n.rep=REPS)\\n      check_k<-abs(z$index-official[1,\\\'k-index\\\']);check_p<-abs(z$p-official[1,\\\'p-value\\\'])\\n      stopifnot(check_k<1e-12,check_p==0)\\n    }\\n    rows[[si]]<-data.frame(regulon=colnames(Y)[j],regulon_index=j,model_role=role,\\n      term=sm$label,branch=if(is.null(sm$by.level))\\\'all\\\'else as.character(sm$by.level),\\n      n_cells=length(hit),n_checked=z$n,unique_v=length(unique(d$v[hit])),k_prime=kp,\\n      edf=edf,edf_ratio=edf/kp,k_index=z$index,p_value=z$p,status=z$status,\\n      basis_flag=is.finite(z$p)&&z$p<.01&&edf/kp>.9,\\n      structure_flag=is.finite(z$p)&&z$p<.01&&z$index<.90,\\n      smoothing_parameter=as.numeric(m$sp[si]),seed=seed,permutations=REPS,\\n      official_k_difference=check_k,official_p_difference=check_p)\\n    rows[[si]]$tied_cells<-z$tied_cells\\n    rows[[si]]$tied_groups<-z$tied_groups\\n    rows[[si]]$old_ordered_k_index<-z$old_index\\n    rows[[si]]$diagnostic_method<-\\\'Tie-order-invariant expected adjacent SSE; fixed exact-t groups; permuted residuals\\\'\\n  }\\n  do.call(rbind,rows)\\n}\\n\\n#3. =========== Every response, local contrast and complete residual evidence\\nN<-nrow(d);J<-length(INDICES)\\nFULL<-RESID<-TRAJ<-matrix(NA_real_,N,J)\\nCONTRAST<-CONTRAST_SE<-matrix(NA_real_,GRID_N,J)\\nstatistics<-diagnostics<-validations<-list()\\nstarted<-proc.time()[[\\\'elapsed\\\']]\\nfor(ii in seq_along(INDICES)) {\\n  j<-INDICES[ii];y<-as.numeric(Y[,j]);tick<-proc.time()[[\\\'elapsed\\\']]\\n  row<-data.frame(regulon=colnames(Y)[j],regulon_index=j,status=\\\'failed\\\',test_status=\\\'not_run\\\',\\n    p_value=1,A=NA_real_,contrast_max_abs=NA_real_,contrast_mean=NA_real_,\\n    contrast_start=NA_real_,contrast_end=NA_real_,RSS0=NA_real_,RSS1=NA_real_,\\n    df0=NA_real_,df1=NA_real_,test_df0=NA_real_,test_df1=NA_real_,test_delta_df=NA_real_,\\n    F_statistic=NA_real_,partial_R2=NA_real_,converged0=FALSE,converged1=FALSE,\\n    full_rank0=FALSE,full_rank1=FALSE,full_basis_flags=NA_integer_,full_structure_flags=NA_integer_,\\n    null_basis_flags=NA_integer_,residual_mean=NA_real_,residual_sd=NA_real_,\\n    cache_prediction_difference=NA_real_,cache_F_relative_difference=NA_real_,\\n    warnings=\\\'\\\',error=\\\'\\\',fit_seconds=NA_real_)\\n  notices<-character()\\n  value<-tryCatch(withCallingHandlers({\\n    m0<-fit_response(G0,y);m1<-fit_response(G1,y)\\n    ok<-c(converged(m0),converged(m1),m0$rank==length(coef(m0)),m1$rank==length(coef(m1)))\\n    stopifnot(all(ok))\\n    full<-as.numeric(X1%*%coef(m1));null<-as.numeric(X0%*%coef(m0));r<-y-full\\n    tr<-as.numeric(XT%*%coef(m1)[tc]);tr<-tr-mean(tr)+mean(y)\\n    rss0<-sum((y-null)^2);rss1<-sum(r*r)\\n    stopifnot(max(abs(full-fitted(m1)))<1e-9,abs(rss1-m1$deviance)<1e-9)\\n    tab<-anova(m0,m1,test=\\\'F\\\');pcol<-grep(\\\'^Pr\\\\\\\\(\\\',names(tab),value=TRUE)\\n    p<-as.numeric(tab[2,pcol]);Fv<-as.numeric(tab[2,\\\'F\\\'])\\n    test_status<-if(is.finite(p)&&tab[2,\\\'Df\\\']>0&&is.finite(Fv)&&Fv>=0)\\\'ok\\\'else\\\'nonpositive_incremental_test\\\'\\n    if(test_status!=\\\'ok\\\')p<-1\\n    nd<-d[rep(1L,GRID_N),,drop=FALSE];nd$v<-grid\\n    if(KIND==\\\'early\\\') {\\n      L<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\');L<-sweep(L,2,L[1,],\\\'-\\\')\\n    } else {\\n      nd$branch<-factor(rep(levels(d$branch)[1],GRID_N),levels=levels(d$branch))\\n      LA<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\')\\n      nd$branch<-factor(rep(levels(d$branch)[2],GRID_N),levels=levels(d$branch))\\n      LB<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\');L<-LB-LA\\n    }\\n    nuisance<-setdiff(seq_len(ncol(X1)),tc)\\n    if(length(nuisance))stopifnot(max(abs(L[,nuisance,drop=FALSE]))<1e-12)\\n    delta<-as.numeric(L%*%coef(m1));se<-sqrt(pmax(0,rowSums((L%*%m1$Vp)*L)))\\n    dq<-rbind(diagnose(m1,G1,r,j,\\\'full\\\'),diagnose(m0,G0,y-null,j,\\\'null\\\'))\\n    pred_error<-F_error<-NA_real_;validation<-NULL\\n    if(VALIDATE && j %in% c(218L,219L,220L)) {\\n      dd<-d;dd$y<-y\\n      fresh0<-gam(f0,data=dd,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      fresh1<-gam(f1,data=dd,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      ft<-anova(fresh0,fresh1,test=\\\'F\\\')\\n      pred_error<-max(abs(fitted(fresh0)-null),abs(fitted(fresh1)-full))\\n      F_error<-if(test_status==\\\'ok\\\')abs(Fv-as.numeric(ft[2,\\\'F\\\']))/(1+abs(Fv))else 0\\n      edf_error<-max(abs(m1$edf-fresh1$edf),abs(m0$edf-fresh0$edf))\\n      ml_error<-max(abs(as.numeric(m0$gcv.ubre)-as.numeric(fresh0$gcv.ubre)),\\n                    abs(as.numeric(m1$gcv.ubre)-as.numeric(fresh1$gcv.ubre)))\\n      stopifnot(pred_error<1e-7,F_error<1e-5,edf_error<1e-4,ml_error<1e-5)\\n      validation<-data.frame(regulon=colnames(Y)[j],prediction_difference=pred_error,\\n        F_relative_difference=F_error,EDF_difference=edf_error,ML_objective_difference=ml_error)\\n    }\\n    list(m0=m0,m1=m1,ok=ok,full=full,r=r,tr=tr,rss0=rss0,rss1=rss1,tab=tab,p=p,F=Fv,\\n      test_status=test_status,delta=delta,se=se,dq=dq,pred_error=pred_error,F_error=F_error,validation=validation)\\n  },warning=function(w){notices<<-c(notices,conditionMessage(w));invokeRestart(\\\'muffleWarning\\\')}),error=function(e)e)\\n  if(inherits(value,\\\'error\\\'))row$error<-conditionMessage(value) else {\\n    row$status<-\\\'ok\\\';row$test_status<-value$test_status;row$p_value<-value$p\\n    row$A<-diff(range(value$tr));row$contrast_max_abs<-max(abs(value$delta))\\n    row$contrast_mean<-mean(value$delta);row$contrast_start<-value$delta[1L];row$contrast_end<-tail(value$delta,1)\\n    row$RSS0<-value$rss0;row$RSS1<-value$rss1;row$df0<-value$m0$df.residual;row$df1<-value$m1$df.residual\\n    row$test_df0<-value$tab[1,\\\'Resid. Df\\\'];row$test_df1<-value$tab[2,\\\'Resid. Df\\\'];row$test_delta_df<-value$tab[2,\\\'Df\\\']\\n    row$F_statistic<-value$F;row$partial_R2<-(value$rss0-value$rss1)/value$rss0\\n    row$converged0<-value$ok[1];row$converged1<-value$ok[2];row$full_rank0<-value$ok[3];row$full_rank1<-value$ok[4]\\n    row$full_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role==\\\'full\\\'])\\n    row$full_structure_flags<-sum(value$dq$structure_flag[value$dq$model_role==\\\'full\\\'])\\n    row$null_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role==\\\'null\\\'])\\n    row$residual_mean<-mean(value$r);row$residual_sd<-sd(value$r)\\n    row$cache_prediction_difference<-value$pred_error;row$cache_F_relative_difference<-value$F_error\\n    FULL[,ii]<-value$full;RESID[,ii]<-value$r;TRAJ[,ii]<-value$tr\\n    CONTRAST[,ii]<-value$delta;CONTRAST_SE[,ii]<-value$se\\n    diagnostics[[length(diagnostics)+1L]]<-value$dq\\n    if(!is.null(value$validation))validations[[length(validations)+1L]]<-value$validation\\n  }\\n  row$warnings<-paste(unique(notices),collapse=\\\' | \\\');row$fit_seconds<-proc.time()[[\\\'elapsed\\\']]-tick\\n  statistics[[ii]]<-row\\n  if(ii%%20L==0L||ii==J)cat(sprintf(\\\'%s %s K%d %d/%d %.1fs\\\\n\\\',KIND,MODE,K,ii,J,proc.time()[[\\\'elapsed\\\']]-started))\\n}\\nwrite_table<-function(rows,suffix)if(length(rows))write.table(do.call(rbind,rows),paste0(PREFIX,suffix),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE,na=\\\'NA\\\')\\nwrite_matrix<-function(x,suffix){con<-file(paste0(PREFIX,suffix),\\\'wb\\\');writeBin(as.numeric(x),con,size=8,endian=\\\'little\\\');close(con)}\\nwrite_table(statistics,\\\'.stats.tsv\\\');write_table(diagnostics,\\\'.diagnostics.tsv\\\');write_table(validations,\\\'.validation.tsv\\\')\\nwrite_matrix(FULL,\\\'.full.bin\\\');write_matrix(RESID,\\\'.residual.bin\\\');write_matrix(TRAJ,\\\'.trajectory.bin\\\')\\nwrite_matrix(CONTRAST,\\\'.contrast.bin\\\');write_matrix(CONTRAST_SE,\\\'.contrast_se.bin\\\')\\nwrite.table(data.frame(metric=c(\\\'R_version\\\',\\\'mgcv_version\\\',\\\'null_formula\\\',\\\'full_formula\\\',\\\'mode\\\',\\\'K\\\',\\\'cells\\\',\\\'permutations\\\',\\\'subsample\\\',\\\'seed\\\',\\\'computation\\\'),\\n value=c(R.version.string,as.character(packageVersion(\\\'mgcv\\\')),paste(deparse(f0),collapse=\\\' \\\'),paste(deparse(f1),collapse=\\\' \\\'),MODE,K,N,REPS,SUBSAMPLE,SEED,\\n \\\'Exact thin-QR Gaussian ML with original full-X smoothing and full-y scale initialization; full-cell attributes restored\\\')),\\n paste0(PREFIX,\\\'.metadata.tsv\\\'),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE)\\n\'\n\nENGINE_PROVENANCE = {\'GLOBAL_R\': {\'archive_path\': \'analysis_code/global_new_source\', \'original_sha256\': \'caa2eb0d2cd9549e0d4457dc35ad79de17a353e57fc86b68b681ab581801350e\', \'adapted_sha256\': \'b4aaf0e9e6adf6d085856ae5bb56b2276358597ac1dcd8d5b4b4f6c500cf685e\'}, \'EARLY_R\': {\'archive_path\': \'analysis_code/early_fork_source\', \'original_sha256\': \'ef0e69e330042f8368db5cc4a34c06401b4c4f02512a6f5a84ee871889adf6fb\', \'adapted_sha256\': \'ec2d7e05c2e1fe8e7971e0aaaf5c0efa271184c10ec4dd9e0beef0c6cf44c311\'}, \'LATE_R\': {\'archive_path\': \'analysis_code/legacy/local_source\', \'original_sha256\': \'ef956e727c9996c4f19813bee17018105ad082559f826b19e56ac2bbaca139e4\', \'adapted_sha256\': \'38c1b3ca4954ebd5400b577b78c0f2046404fdae3ef17e5b6e4cae3edd9ff23c\'}}\n\n\n#2. =========== Frozen inputs, frame schema and task-local I/O\n\ndef sha(path):\n    value=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(2**20),b\'\'):value.update(block)\n    return value.hexdigest()\n\ndef read_frame(group,index=None):\n    columns=group[\'columns\'].asstr()[()].tolist() if \'columns\' in group else [k for k in group if k not in (\'index\',\'_index\')]\n    values={k:group[k].asstr()[()] if group[k].dtype.kind in \'OSU\' else group[k][()] for k in columns}\n    if \'index\' in group:index=group[\'index\'].asstr()[()]\n    elif \'_index\' in group:index=group[\'_index\'].asstr()[()]\n    return pd.DataFrame(values,index=index)\n\ndef strings(group,key,values):\n    group.create_dataset(key,data=np.asarray(values,object),dtype=h5py.string_dtype(),compression=\'gzip\')\n\ndef write_frame(group,frame):\n    frame=frame.copy()\n    if \'index\' in frame:frame=frame.rename(columns={\'index\':\'regulon_index\'})\n    strings(group,\'index\',frame.index.astype(str));strings(group,\'columns\',frame.columns.astype(str))\n    for k in frame:\n        value=frame[k].to_numpy()\n        if value.dtype.kind in \'OSU\':strings(group,k,frame[k].fillna(\'\').astype(str))\n        else:group.create_dataset(k,data=value,compression=\'gzip\')\n\ndef json_dataset(group,key,value):\n    group.create_dataset(key,data=json.dumps(value,ensure_ascii=False,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),dtype=h5py.string_dtype())\n\ndef prepare(score_npz,baseline,work):\n    with np.load(score_npz,allow_pickle=False) as z:\n        auc=z[\'auc\'].copy()\n        key=next((k for k in (\'cells\',\'cell_ids\') if k in z),None)\n        score_cells=None if key is None else z[key].astype(str)\n        key=next((k for k in (\'regulons\',\'names\',\'TFs\') if k in z),None)\n        names=np.asarray(NEW_NAMES) if key is None else z[key].astype(str)\n    assert tuple(names)==NEW_NAMES and auc.shape==(7355,3)\n    assert np.isfinite(auc).all(), \'Unsupported activity must remain missing; supply only estimable signatures for these fixed fits.\'\n    assert ((auc>=0)&(auc<=1)).all()\n    with h5py.File(baseline,\'r\') as h:\n        cells=pd.Index(h[\'cells\'].asstr()[()]);old_names=h[\'regulons\'].asstr()[()]\n        if score_cells is not None:np.testing.assert_array_equal(score_cells,cells)\n        obs=read_frame(h[\'obs\'],cells);old_auc=h[\'auc\'][()];mask=h[\'final/analysis_mask\'][()]\n        bins=read_frame(h[\'bin_metadata\'])\n        reference={mode:read_frame(h[f\'global_models/modes/{mode}/k20/stats\']) for mode in GLOBAL_MODES}\n        forks={}\n        for label in (\'early\',\'fork1\',\'fork2\',\'fork3\'):\n            path=\'early_fork\' if label==\'early\' else \'late_forks/\'+label\n            g=h[path];data=read_frame(g[\'obs\']);metadata=json.loads(g[\'manifest_json\'].asstr()[()])\n            if \'cells\' in g:data.index=pd.Index(g[\'cells\'].asstr()[()])\n            elif \'cell_id\' in data:data.index=pd.Index(data[\'cell_id\'].astype(str))\n            modes=EARLY_MODES if label==\'early\' else LOCAL_MODES\n            refs={mode:read_frame(g[f\'modes/{mode}/stats\'] if f\'modes/{mode}/stats\' in g else g[f\'modes/{mode}/k10/stats\']) for mode in modes}\n            forks[label]=dict(obs=data,grid=g[\'grid_v\'][()],manifest=metadata,reference=refs)\n        baseline_meta=json.loads(h[\'metadata_json\'].asstr()[()])\n    assert len(cells)==7355 and len(old_names)==217 and mask.sum()==7348\n    assert not (obs.loc[mask,\'seg\'].astype(int)==3).any()\n    joined=pd.DataFrame(np.c_[old_auc,auc],index=cells,columns=list(old_names)+list(NEW_NAMES))\n    design=dict(global_obs=obs.loc[mask].copy(),all_obs=obs,all_cells=cells,new_auc=auc,\n                bins=bins,reference=reference,forks=forks,mask=mask,anchor=str(old_names[0]))\n    exports={}\n    # Set this storage path independently before running.\n    global_dir=ROOT;global_dir.mkdir(parents=True,exist_ok=True)\n    for label,data in [(\'global\',design[\'global_obs\'])]+[(k,v[\'obs\']) for k,v in forks.items()]:\n        folder=work/label;folder.mkdir(parents=True,exist_ok=True)\n        assert data.index.is_unique and data.index.isin(cells).all()\n        matrix=joined.loc[data.index]\n        # Set this storage path independently before running. cells.csv.\n        data.to_csv(ROOT,index_label=\'cell_id\',float_format=\'%.17g\')\n        # Set this storage path independently before running. auc.tsv.gz.\n        matrix.to_csv(ROOT,sep=\'\\t\',index_label=\'cell_id\',float_format=\'%.17g\',compression={\'method\':\'gzip\',\'mtime\':0})\n        # Set this storage path independently before running. cells.csv.\n        exports[label]=dict(cells_sha256=sha(ROOT),auc_sha256=sha(ROOT),cells=len(data))\n    for name,source in [(\'global.R\',GLOBAL_R),(\'early.R\',EARLY_R),(\'late.R\',LATE_R)]:\n        (work/name).write_text(source,encoding=\'utf-8\',newline=\'\\n\')\n    design[\'exports\']=exports;design[\'baseline_metadata\']=baseline_meta\n    return design\n\n#3. =========== Reuse the exact archived engine; fit only new responses plus one anchor\n\ndef run_job(label,mode,work):\n    kind=\'global\' if label==\'global\' else (\'early\' if label==\'early\' else \'late\')\n    k=GLOBAL_K if kind==\'global\' else FORK_K\n    folder=work/label;prefix=folder/mode;engine=work/(kind+\'.R\')\n    # Set this storage path independently before running. cells.csv.\n    files={n:sha(p) for n,p in [(\'cells\',ROOT),(\'auc\',ROOT),(\'engine\',engine)]}\n    fingerprint=dict(label=label,mode=mode,K=k,indices=[1,218,219,220],files=files,seed=SEED)\n    marker=Path(str(prefix)+\'.complete.json\')\n    if marker.exists() and json.loads(marker.read_text())==fingerprint:return label,mode,prefix\n    indices=\'1,218,219,220\'\n    if kind==\'global\':\n        # Set this storage path independently before running. cells.csv.\n        args=[str(ROOT),str(ROOT),str(prefix),mode,str(k),indices,\'218,219,220\']\n    else:\n        # Set this storage path independently before running. cells.csv.\n        args=[str(ROOT),str(ROOT),str(prefix),\'fork\',mode,str(k),indices,\'1\']\n    env=dict(os.environ,TMP=str(work),TEMP=str(work),LANG=\'C\',LC_ALL=\'C\')\n    with open(str(prefix)+\'.log\',\'w\',encoding=\'utf-8\') as log:\n        result=subprocess.run([str(RSCRIPT),\'--vanilla\',str(engine),*args],env=env,cwd=ROOT,stdout=log,stderr=subprocess.STDOUT)\n    if result.returncode:raise RuntimeError(Path(str(prefix)+\'.log\').read_text(encoding=\'utf-8\',errors=\'replace\')[-4000:])\n    # Set this storage path independently before running. cells.csv.\n    assert files=={n:sha(p) for n,p in [(\'cells\',ROOT),(\'auc\',ROOT),(\'engine\',engine)]}\n    marker.write_text(json.dumps(fingerprint,sort_keys=True),encoding=\'utf-8\')\n    return label,mode,prefix\n\ndef tie_index(v,r):\n    _,g,n=np.unique(v,return_inverse=True,return_counts=True)\n    sums=np.bincount(g,weights=r);squares=np.bincount(g,weights=r*r)\n    mean=sums/n;ss=np.maximum(0,squares-sums*sums/n);var=ss/n\n    return (2*ss.sum()+(var[:-1]+var[1:]+np.diff(mean)**2).sum())/(2*(len(r)-1)*np.mean(r*r))\n\ndef collect(group,label,mode,prefix,design):\n    global_job=label==\'global\';data=design[\'global_obs\'] if global_job else design[\'forks\'][label][\'obs\']\n    stats_all=pd.read_csv(str(prefix)+\'.stats.tsv\',sep=\'\\t\',keep_default_na=False,na_values=[\'NA\']).set_index(\'regulon\')\n    stats=stats_all.loc[list(NEW_NAMES)].copy();stats=stats.rename(columns={\'index\':\'regulon_index\'})\n    stats[\'p_value\']=stats.p_value.where(stats.status.eq(\'ok\'),1).fillna(1)\n    reference=design[\'reference\'][mode] if global_job else design[\'forks\'][label][\'reference\'][mode]\n    anchor=design[\'anchor\'];new_anchor=stats_all.loc[anchor];old_anchor=reference.loc[anchor]\n    qa={}\n    for field in (\'p_value\',\'A\',\'RSS0\',\'RSS1\'):\n        qa[field+\'_difference\']=float(abs(new_anchor[field]-old_anchor[field]))\n        assert qa[field+\'_difference\']<(1e-7 if field!=\'p_value\' else 1e-8),(label,mode,field,qa)\n    qa[\'anchor_regulon\']=anchor\n    diagnostic_suffix=\'.kcheck.tsv\' if global_job else \'.diagnostics.tsv\'\n    d=pd.read_csv(str(prefix)+diagnostic_suffix,sep=\'\\t\');d=d.loc[d.regulon.isin(NEW_NAMES)].copy()\n    if global_job:\n        d[\'basis_flag\']=d.p_value.lt(.01)&d.edf_ratio.gt(.9)&d.status.eq(\'ok\')\n        d[\'structure_flag\']=d.p_value.lt(.01)&d.k_index.lt(.9)&d.status.eq(\'ok\')\n    arrays={}\n    specifications=[(\'.fullpred.bin\',\'full_prediction\',len(data)),(\'.residuals.bin\',\'residuals\',len(data)),(\'.trajectory.bin\',\'trajectory\',len(data))] if global_job else [(\'.full.bin\',\'full_prediction\',len(data)),(\'.residual.bin\',\'residuals\',len(data)),(\'.trajectory.bin\',\'trajectory\',len(data)),(\'.contrast.bin\',\'contrast\',51),(\'.contrast_se.bin\',\'contrast_se\',51)]\n    for suffix,key,nrows in specifications:\n        values=np.fromfile(str(prefix)+suffix,dtype=\'<f8\').reshape((nrows,4),order=\'F\')[:,1:].copy()\n        arrays[key]=values;group.create_dataset(key,data=values,compression=\'gzip\')\n    valid=stats.status.eq(\'ok\').to_numpy()\n    y=design[\'new_auc\'][design[\'all_cells\'].get_indexer(data.index)]\n    np.testing.assert_allclose(y[:,valid]-arrays[\'full_prediction\'][:,valid],arrays[\'residuals\'][:,valid],atol=1e-12)\n    np.testing.assert_allclose((arrays[\'residuals\'][:,valid]**2).sum(0),stats.RSS1.to_numpy()[valid],atol=1e-10,rtol=1e-8)\n    np.testing.assert_allclose(np.ptp(arrays[\'trajectory\'][:,valid],axis=0),stats.A.to_numpy()[valid],atol=1e-11,rtol=1e-10)\n    tested=valid & (stats.test_status.eq(\'ok\').to_numpy() if \'test_status\' in stats else True)\n    expected_F=((stats.RSS0-stats.RSS1)/stats.test_delta_df)/(stats.RSS1/stats.df1)\n    np.testing.assert_allclose(expected_F.to_numpy()[tested],stats.F_statistic.to_numpy()[tested],atol=1e-7,rtol=1e-7)\n    expected_p=f_distribution.sf(stats.F_statistic,stats.test_delta_df,stats.test_df1)\n    np.testing.assert_allclose(expected_p[tested],stats.p_value.to_numpy()[tested],atol=1e-10,rtol=1e-5)\n    if global_job:\n        for field in (\'bin_auc\',\'full_bin_auc\',\'observed_bin_auc\'):\n            out=[]\n            value=arrays[\'trajectory\'] if field==\'bin_auc\' else arrays[\'full_prediction\'] if field==\'full_bin_auc\' else y\n            for row in design[\'bins\'].itertuples():\n                hit=data.seg.eq(int(row.seg)).to_numpy() & data[\'bin\'].eq(int(row.bin)).to_numpy()\n                assert hit.sum()==int(row.cells),(row.seg,row.bin,hit.sum(),row.cells)\n                out.append(np.mean(value[hit],axis=0))\n            group.create_dataset(field,data=np.asarray(out),compression=\'gzip\')\n        rms=[]\n        for z in d.itertuples():\n            hit=data.seg.eq(int(z.seg)).to_numpy();r=arrays[\'residuals\'][hit,list(NEW_NAMES).index(z.regulon)]\n            rms.append(np.sqrt(np.mean(r*r)))\n            expected=tie_index(data.t.to_numpy()[hit],r)\n            assert abs(expected-z.k_index)<1e-10,(label,mode,z.regulon,z.seg,expected,z.k_index)\n        d[\'residual_RMS\']=rms;d[\'numerical_zero_residual\']=d.residual_RMS.le(1e-12)\n        d[\'diagnostic_informative\']=~d.numerical_zero_residual & d.status.eq(\'ok\')\n        d[\'informative_structure_flag\']=d.structure_flag & d.diagnostic_informative\n        for suffix,key in ((\'.segment_residual.tsv\',\'segment_residual_summary\'),(\'.source_residual.tsv\',\'source_residual_summary\')):\n            f=pd.read_csv(str(prefix)+suffix,sep=\'\\t\');f=f.loc[f.regulon.isin(NEW_NAMES)]\n            write_frame(group.create_group(key),f)\n        count=d.groupby(\'regulon\').agg(informative_structure_flags=(\'informative_structure_flag\',\'sum\'),basis_flags=(\'basis_flag\',\'sum\'))\n        stats=stats.join(count)\n        assert len(d)==int(valid.sum())*10\n    else:\n        group.create_dataset(\'contrast_lower_95\',data=arrays[\'contrast\']-1.96*arrays[\'contrast_se\'],compression=\'gzip\')\n        group.create_dataset(\'contrast_upper_95\',data=arrays[\'contrast\']+1.96*arrays[\'contrast_se\'],compression=\'gzip\')\n    write_frame(group.create_group(\'stats\'),stats);write_frame(group.create_group(\'diagnostics\'),d)\n    metadata=pd.read_csv(str(prefix)+\'.metadata.tsv\',sep=\'\\t\');write_frame(group.create_group(\'runtime_metadata\'),metadata)\n    if global_job:\n        validation=[]\n        for line in Path(str(prefix)+\'.log\').read_text(encoding=\'utf-8\').splitlines():\n            if line.startswith(\'QR_EQUIVALENCE \'):\n                _,name,values=line.split(\' \',2)\n                validation.append(dict(regulon=name,**{k:float(v) for k,v in (part.split(\'=\') for part in values.strip().split(\';\'))}))\n        write_frame(group.create_group(\'fresh_validation\'),pd.DataFrame(validation))\n    elif Path(str(prefix)+\'.validation.tsv\').exists():\n        v=pd.read_csv(str(prefix)+\'.validation.tsv\',sep=\'\\t\');write_frame(group.create_group(\'fresh_validation\'),v)\n    json_dataset(group,\'anchor_validation_json\',qa)\n    group.attrs[\'K\']=GLOBAL_K if global_job else FORK_K;group.attrs[\'mode\']=mode\n    group.attrs[\'multiplicity_correction\']=\'Not applied here; caller combines frozen raw P for BH220 global and BH880 forks.\'\n    return dict(analysis=label,mode=mode,models_ok=int(valid.sum()),basis_flags=int(d.basis_flag.sum()),\n                p_values=stats.p_value.tolist(),A=stats.A.tolist(),anchor=qa)\n\n#4. =========== Complete supplemental family; multiplicity belongs to the caller\n\ndef run_models(score_npz,baseline_h5=BASELINE,work=WORK):\n    baseline_h5=Path(baseline_h5);score_npz=Path(score_npz);work=Path(work).resolve()\n    # Set this storage path independently before running.\n    assert work.is_relative_to(ROOT)\n    work.mkdir(parents=True,exist_ok=True)\n    assert RSCRIPT.is_file()\n    inputs=dict(baseline_sha256=sha(baseline_h5),score_sha256=sha(score_npz))\n    design=prepare(score_npz,baseline_h5,work)\n    # Set this storage path independently before running. models.h5.\n    output=ROOT;temporary=ROOT;summary=[];started=time.monotonic()\n    with h5py.File(temporary,\'w\') as h:\n        h.attrs[\'schema\']=\'Fig4A_supplemental_add3_v1\';h.attrs[\'seed\']=SEED\n        h.attrs[\'code_sha256\']=sha(Path(__file__));h.attrs[\'primary_mode\']=\'none\'\n        strings(h,\'regulons\',NEW_NAMES);strings(h,\'all_cells\',design[\'all_cells\']);strings(h,\'cells\',design[\'global_obs\'].index)\n        h.create_dataset(\'observed_auc\',data=design[\'new_auc\'],compression=\'gzip\')\n        h.create_dataset(\'analysis_mask\',data=design[\'mask\']);write_frame(h.create_group(\'obs\'),design[\'global_obs\']);write_frame(h.create_group(\'bin_metadata\'),design[\'bins\'])\n        json_dataset(h,\'engine_provenance_json\',ENGINE_PROVENANCE)\n        json_dataset(h,\'input_metadata_json\',dict(**inputs,exports=design[\'exports\'],source_regulon_indices=[218,219,220],\n            primary_family=\'Old217 raw P plus these3; caller must BH220\',fork_family=\'Old868 raw P plus these12; caller must BH880\',\n            missing_policy=\'No missing AUCell is replaced with zero; failed fits have P1 and missing predictions\',\n            no_parameters_changed=\'Global fixedK20, local fixedK10, exact archived samples/covariates/formula/diagnostic procedure\'))\n        roots=h.create_group(\'modes\');froots=h.create_group(\'forks\')\n        jobs=[(\'global\',mode) for mode in GLOBAL_MODES]\n        jobs += [(\'early\',mode) for mode in EARLY_MODES]\n        jobs += [(label,mode) for label in (\'fork1\',\'fork2\',\'fork3\') for mode in LOCAL_MODES]\n        for label,v in design[\'forks\'].items():\n            g=froots.create_group(label);strings(g,\'cells\',v[\'obs\'].index);write_frame(g.create_group(\'obs\'),v[\'obs\']);g.create_dataset(\'grid_v\',data=v[\'grid\']);g.create_group(\'modes\');json_dataset(g,\'sampling_manifest_json\',v[\'manifest\'])\n        for phase in (jobs[:5],jobs[5:]):\n            with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as pool:\n                futures={pool.submit(run_job,label,mode,work):(label,mode) for label,mode in phase}\n                for future in concurrent.futures.as_completed(futures):\n                    label,mode,prefix=future.result()\n                    group=roots.create_group(mode) if label==\'global\' else froots[label][\'modes\'].create_group(mode)\n                    result=collect(group,label,mode,prefix,design);summary.append(result);h.flush()\n                    print(\'MODEL_COMPLETE \'+json.dumps({k:v for k,v in result.items() if k!=\'anchor\'}),flush=True)\n        for label in design[\'forks\']:\n            froots[label][\'stats\']=froots[label][\'modes/none/stats\']\n        h[\'primary\']=roots[\'none\'];json_dataset(h,\'summary_json\',summary);h.attrs[\'elapsed_seconds\']=time.monotonic()-started\n        assert sha(baseline_h5)==inputs[\'baseline_sha256\'],\'Baseline H5 changed during supplemental fit.\'\n    os.replace(temporary,output)\n    print(\'SUPPLEMENTAL_MODELS_COMPLETE \'+json.dumps(dict(output=str(output),seconds=time.monotonic()-started)),flush=True)\n    return output\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'score_npz\');parser.add_argument(\'--baseline\',default=str(BASELINE));parser.add_argument(\'--work\',default=str(WORK));args=parser.parse_args()\n    run_models(args.score_npz,args.baseline,args.work)\n'
ADD3_IO_SOURCE='"""Append-only persistence and plotting for the three prespecified follow-ups."""\nfrom pathlib import Path\nimport json, hashlib, copy\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import spearmanr\n\nADD3_GROUP = \'supplement_add3\'\nFOLLOWUP_NAMES = [\'Nhlh1(+)\', \'Neurod1(+)\', \'Tbx3(+)\']\nCHARLES_NAMES = set(\'Chd1 Patz1 Pou2f1 Ybx1 E4f1 Klf7 Srebf2 Onecut2 Smad1 Sox4 Shox2 Bach2 Sox11 Etv5 Etv4 Pou4f1 Thrb Zfhx2 Bhlhe40 Mafb\'.split())\n\ndef read_frame(group):\n    return pd.DataFrame({c:(group[c].asstr()[()] if group[c].dtype.kind in \'OSU\' else group[c][()])\n        for c in group[\'columns\'].asstr()[()]}, index=group[\'index\'].asstr()[()])\n\ndef column_frame(group):\n    return pd.DataFrame({k:(v.asstr()[()] if v.dtype.kind in \'OSU\' else v[()])\n        for k,v in group.items() if isinstance(v,h5py.Dataset) and not k.endswith(\'__missing\')})\n\ndef write_frame(group, frame):\n    group.create_dataset(\'index\',data=frame.index.astype(str).to_numpy(),dtype=h5py.string_dtype())\n    group.create_dataset(\'columns\',data=frame.columns.astype(str).to_numpy(),dtype=h5py.string_dtype())\n    for key in frame:\n        values=frame[key].to_numpy()\n        if values.dtype.kind in \'OSU\':\n            group.create_dataset(key,data=frame[key].fillna(\'\').astype(str).to_numpy(),dtype=h5py.string_dtype())\n        else: group.create_dataset(key,data=values)\n\ndef bh(values):\n    p=np.asarray(values,float);assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()\n    n=len(p);order=np.argsort(p,kind=\'stable\');q=np.empty(n)\n    q[order]=np.minimum(1,np.minimum.accumulate((p[order]*n/np.arange(1,n+1))[::-1])[::-1])\n    return q\n\ndef early_summaries(baseline, cells, auc, counts):\n    e=column_frame(baseline[\'early_extension/obs\'])\n    pos=pd.Index(cells).get_indexer(e.cell_id.astype(str));assert (pos>=0).all()\n    means=[];low=[];high=[];detection=[];units=[]\n    for number in range(7):\n        indices=np.flatnonzero(e.group.to_numpy()==number)\n        libraries=e.library.iloc[indices].astype(str).to_numpy()\n        levels=np.unique(libraries)\n        matrix=np.array([auc[pos[indices[libraries==lib]]].mean(axis=0) for lib in levels])\n        detected=np.array([(counts[pos[indices[libraries==lib]]]>0).mean(axis=0) for lib in levels])\n        means.append(matrix.mean(axis=0));detection.append(detected.mean(axis=0))\n        if len(levels)>1:\n            loo=(matrix.sum(axis=0)-matrix)/(len(levels)-1)\n            low.append(loo.min(axis=0));high.append(loo.max(axis=0))\n        else:\n            low.append(np.full(3,np.nan));high.append(np.full(3,np.nan))\n        units.append({\'group\':number,\'cells\':len(indices),\'libraries\':len(levels)})\n    return dict(mean=np.array(means),loo_min=np.array(low),loo_max=np.array(high),\n        tf_detection=np.array(detection),units=units)\n\ndef assemble_add3(result_file, score_file, model_file, source_bundle, original_manifest):\n    with np.load(score_file,allow_pickle=False) as z:\n        score={k:z[k].copy() for k in z.files}\n    assert score[\'regulons\'].tolist()==FOLLOWUP_NAMES\n    with h5py.File(result_file,\'r\') as h,h5py.File(model_file,\'r\') as m:\n        old_names=h[\'regulons\'].asstr()[()].tolist();all_names=old_names+FOLLOWUP_NAMES\n        assert len(all_names)==220 and len(set(all_names))==220\n        np.testing.assert_array_equal(h[\'cells\'].asstr()[()],score[\'cells\'])\n        np.testing.assert_array_equal(m[\'all_cells\'].asstr()[()],score[\'cells\'])\n        np.testing.assert_array_equal(m[\'observed_auc\'][()],score[\'auc\'])\n        frozen=read_frame(h[\'final/statistics\']).loc[old_names]\n        added=read_frame(m[\'modes/none/stats\']).loc[FOLLOWUP_NAMES]\n        assert added.status.eq(\'ok\').all()\n        added[\'residual_flags\']=added.informative_structure_flags\n        fork_names=[\'early\',\'fork1\',\'fork2\',\'fork3\']\n        for fork in fork_names:\n            f=read_frame(m[\'forks/\'+fork+\'/stats\']).loc[FOLLOWUP_NAMES]\n            assert f.status.eq(\'ok\').all()\n            added[\'p_\'+fork]=f.p_value\n        combined=pd.concat([frozen,added],axis=0,sort=False).loc[all_names]\n        combined[\'q_global\']=bh(combined.p_value.where(combined.status.eq(\'ok\'),1))\n        fork_p=combined[[\'p_\'+x for x in fork_names]].to_numpy()\n        combined[[\'q_\'+x for x in fork_names]]=bh(fork_p.ravel()).reshape(220,4)\n        combined[\'primary_pass\']=combined.status.eq(\'ok\')&combined.q_global.lt(1e-4)&combined.A.gt(.025)\n        combined[\'prespecified_followup\']=combined.index.isin(FOLLOWUP_NAMES)\n        combined[\'charles_reference\']=[x.removesuffix(\'(+)\') in CHARLES_NAMES for x in all_names]\n        curves=m[\'modes/none/bin_auc\'][()]\n        assert curves.shape==(30,3) and np.isfinite(curves).all()\n        mode_stats={}\n        for mode in [\'none\',\'depth\',\'study_only\',\'study\',\'library\']:\n            old=read_frame(h[f\'global_models/modes/{mode}/k20/stats\']).loc[old_names]\n            new=read_frame(m[f\'modes/{mode}/stats\']).loc[FOLLOWUP_NAMES]\n            assert new.status.eq(\'ok\').all()\n            joined=pd.concat([old,new],axis=0,sort=False).loc[all_names]\n            joined[\'q_BH220\']=bh(joined.p_value.where(joined.status.eq(\'ok\'),1))\n            joined[\'primary_threshold_pass_BH220\']=joined.status.eq(\'ok\')&joined.q_BH220.lt(1e-4)&joined.A.gt(.025)\n            mode_stats[mode]=joined\n            if mode!=\'none\':\n                combined.loc[FOLLOWUP_NAMES,mode+\'_A\']=new.A\n                combined.loc[FOLLOWUP_NAMES,mode+\'_q_global\']=joined.loc[FOLLOWUP_NAMES,\'q_BH220\']\n                adjusted=m[f\'modes/{mode}/bin_auc\'][()]\n                combined.loc[FOLLOWUP_NAMES,mode+\'_profile_rho\']=[spearmanr(curves[:,j],adjusted[:,j]).statistic for j in range(3)]\n        early=early_summaries(h,score[\'cells\'],score[\'auc\'],score[\'tf_rna_counts\'])\n        observed_summary=[]\n        obs=column_frame(h[\'obs\'])\n        for j,name in enumerate(FOLLOWUP_NAMES):\n            positive=score[\'tf_rna_counts\'][:,j]>0\n            observed_summary.append(dict(regulon=name,positive_cells=int(positive.sum()),\n                positive_libraries=int(obs.library[positive].nunique()),TF_detected_fraction=float(positive.mean())))\n        old_fitted=h[\'final/all_fitted_bin_auc\'][()]\n        old_order=h[\'display_selection/regulons\'].asstr()[()].tolist()\n        metadata=dict(schema=\'Fig4A_add3_v1\',primary_family=220,fork_family=880,\n            original_tested=217,original_displayed=58,added=FOLLOWUP_NAMES,displayed=61,\n            primary_method=\'Archived branch-specific cubic Gaussian ML GAM K20 versus constant; F test; BH220 q<1e-4 and fitted range>.025\',\n            fork_method=\'Archived samples and K10 models; combined original868 plus new12 BH880\',\n            selection=\'58 frozen display representatives plus3 user-prespecified follow-ups; no gene-specific parameter changes\',\n            inference_scope=\'Conditional exploratory follow-up on the fixed Fig3 tree, not a new unbiased early-network screen\',\n            primary_pass_new={name:bool(combined.loc[name,\'primary_pass\']) for name in FOLLOWUP_NAMES},\n            original_display_still_primary=int(combined.loc[old_order,\'primary_pass\'].sum()),\n            actual_VGN_cells_added=0,TF_RNA_support=observed_summary,\n            negative_new_bins=int((curves<0).sum()),\n            limitations=[\'Three signatures have distinct recorded inference sources; the same A scoring axis/model is used.\',\n                \'The actual520 VGN cohort is absent from this fixed tree; Tbx3 shows projection on the existing atlas.\',\n                \'Early state, age and source remain partly confounded; association does not establish causal differentiation.\',\n                \'Covariate and residual checks diagnose robustness; they are not gene-specific retention gates.\',\n                \'Requested follow-up columns are distinguished in metadata from the automatic representative selection.\'])\n    # Append only this owned group. Existing datasets and publication remain untouched.\n    with h5py.File(result_file,\'a\') as h,h5py.File(model_file,\'r\') as m:\n        if ADD3_GROUP in h:del h[ADD3_GROUP]\n        g=h.create_group(ADD3_GROUP);g.attrs[\'complete\']=False\n        for key in [\'regulons\',\'cells\',\'genes\',\'auc\',\'rank_gate\',\'tf_rna_counts\',\'native_tf_rna_counts\',\'native_tf_log1p_CP10k\',\n                    \'validation_regulons\',\'validation_auc_reproduced\',\'validation_max_abs_error\']:\n            value=score[key]\n            if value.dtype.kind in \'OSU\':g.create_dataset(key,data=value.astype(object),dtype=h5py.string_dtype(),compression=\'gzip\')\n            else:g.create_dataset(key,data=value,compression=\'gzip\')\n        for key in [\'metadata_json\',\'signatures_json\']:\n            g.create_dataset(\'scoring_\'+key,data=str(score[key]),dtype=h5py.string_dtype())\n        write_frame(g.create_group(\'combined_statistics\'),combined)\n        for mode,frame in mode_stats.items():write_frame(g.require_group(\'covariate_BH220\').create_group(mode),frame)\n        g.create_dataset(\'fitted_bin_auc\',data=curves)\n        g.create_dataset(\'early_group_mean_auc\',data=early[\'mean\'])\n        for key in [\'loo_min\',\'loo_max\',\'tf_detection\']:g.create_dataset(\'early_group_\'+key,data=early[key])\n        for key,value in [(\'metadata_json\',metadata),(\'original_preservation_json\',original_manifest),(\'early_group_units_json\',early[\'units\'])]:\n            g.create_dataset(key,data=json.dumps(value,ensure_ascii=False),dtype=h5py.string_dtype())\n        sources=g.create_group(\'source_code\')\n        for key,value in source_bundle.items():sources.create_dataset(key,data=value,dtype=h5py.string_dtype())\n        fit=g.create_group(\'model_fit\')\n        for key in m:m.copy(key,fit,name=key)\n        fit.attrs.update(dict(m.attrs));g.attrs[\'complete\']=True\n        h.flush()\n    return metadata\n\ndef load_plot_inputs(result_file):\n    with h5py.File(result_file,\'r\') as h:\n        g=h[ADD3_GROUP];assert g.attrs[\'complete\']\n        original_names=h[\'regulons\'].asstr()[()].tolist()\n        names=original_names+g[\'regulons\'].asstr()[()].tolist()\n        stats=read_frame(g[\'combined_statistics\']).loc[names]\n        data=dict(names=pd.Index(names),segments=json.loads(h[\'segments_json\'].asstr()[()]),\n            bin_meta=column_frame(h[\'bin_metadata\']),\n            modes={\'none\':{\'bins\':pd.DataFrame(np.column_stack([h[\'final/all_fitted_bin_auc\'][()],g[\'fitted_bin_auc\'][()]]),columns=names)}},\n            early_mean=pd.DataFrame(np.column_stack([h[\'final/all_early_observed_auc\'][()],g[\'early_group_mean_auc\'][()]]),columns=names),\n            early_meta=column_frame(h[\'early_extension/group_metadata\']))\n        original_order=h[\'display_selection/regulons\'].asstr()[()].tolist()\n        selected=original_order+FOLLOWUP_NAMES\n        fitted=data[\'modes\'][\'none\'][\'bins\'].loc[:,selected].to_numpy()\n        peaks=np.argmax(fitted,axis=0)\n        weights=np.maximum(fitted-fitted.min(axis=0),0)\n        centres=(weights*np.arange(30)[:,None]).sum(axis=0)/weights.sum(axis=0)\n        selected=[selected[k] for k in sorted(range(len(selected)),key=lambda k:(peaks[k],centres[k],selected[k]))]\n        assert len(selected)==61 and len(set(selected))==61\n        assert set(selected)==set(original_order)|set(FOLLOWUP_NAMES)\n        assert np.all(np.diff(np.argmax(data[\'modes\'][\'none\'][\'bins\'].loc[:,selected],axis=0))>=0)\n        np.testing.assert_array_equal(data[\'modes\'][\'none\'][\'bins\'].loc[:,original_order],\n            h[\'final/all_fitted_bin_auc\'][()][:,pd.Index(original_names).get_indexer(original_order)])\n    return data,stats,selected\n\ndef derive_renderer(notebook, namespace):\n    cell=next(\'\'.join(c[\'source\']) for c in notebook[\'cells\']\n        if c[\'cell_type\']==\'code\' and \'def render(data,stats,order):\' in \'\'.join(c[\'source\']))\n    source=cell[cell.index(\'def render(data,stats,order):\'):]\n    source=source.replace(\'def render(data,stats,order):\',\'def render_add3(data,stats,order):\',1)\n    source=source.replace(\'BH868\',\'BH880\')\n    source=source.replace("f\'E9.5-E18.5; {len(selected)} regulons across 9 complete branches\'",\n        "f\'E9.5-E18.5; 58 original + 3 follow-up regulons across 9 complete branches\'")\n    source=source.replace("note2=\'Bold: Charles Fig3a TF match; red: fork BH880 q < 1e-4.\'",\n        "note2=\'Bold: Charles Fig3a TF match; red: fork BH880 q < 1e-4. Three preselected follow-up regulons; primary BH220.\'")\n    # Preserve the calibrated vector geometry and semantic text objects.\n    scope=dict(namespace)\n    exec(compile(source,\'<Fig4A supplement renderer>\',\'exec\'),scope)\n    return scope[\'render_add3\'], source\n\ndef dataset_fingerprints(handle, skip=ADD3_GROUP):\n    output={}\n    def visit(name,dataset):\n        if name==skip or name.startswith(skip+\'/\'):return\n        if not isinstance(dataset,h5py.Dataset):return\n        if dataset.dtype.kind in \'OSU\':\n            raw=json.dumps(np.asarray(dataset.asstr()[()]).tolist(),ensure_ascii=False,sort_keys=True).encode()\n        else:raw=np.asarray(dataset[()]).tobytes()\n        output[name]=[str(dataset.dtype),list(dataset.shape),hashlib.sha256(raw).hexdigest()]\n    handle.visititems(visit)\n    return output\n'

add3_io={'__name__':'fig4a_add3_io'}
exec(compile(ADD3_IO_SOURCE,'<Fig4A add3 archive>','exec'),add3_io)


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#9. =========== Score, test and append only the supplemental result group
source_bundle=dict(scoring=ADD3_SCORING_SOURCE,modeling=ADD3_MODEL_SOURCE,archive=ADD3_IO_SOURCE)
with h5py.File(RESULT_FILE,'r') as handle:
    add3_cached='supplement_add3' in handle and bool(handle['supplement_add3'].attrs.get('complete',False))
    original_fingerprints=add3_io['dataset_fingerprints'](handle)
notebook_before=json.loads(NOTEBOOK_FILE.read_text(encoding='utf-8'))
original_manifest=dict(original_cell_count=8,
    original_cells_sha256=hashlib.sha256(json.dumps(notebook_before['cells'][:8],ensure_ascii=False,sort_keys=True).encode()).hexdigest(),
    original_dataset_count=len(original_fingerprints),
    original_dataset_manifest_sha256=hashlib.sha256(json.dumps(original_fingerprints,sort_keys=True).encode()).hexdigest(),
    original_figures_sha256={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in FIGURE_DIR.glob('*')
        if p.name in ['fig4A_Developmental_regulon_dynamics.png','fig4A_Developmental_regulon_dynamics.pdf']})
if RECOMPUTE_ADD3 or not add3_cached:
    # Set this storage path independently before running. score_helpers.py.
    score_script=ROOT;model_script=ROOT
    score_script.write_text(ADD3_SCORING_SOURCE,encoding='utf-8')
    model_script.write_text(ADD3_MODEL_SOURCE,encoding='utf-8')
    # Set this storage path independently before running. python.exe.
    scoring_python=ROOT
    assert scoring_python.is_file(),'The archived SCENIC runtime is required only for recomputation.'
    # Set this storage path independently before running. score.npz.
    if RECOMPUTE_ADD3 or not (ROOT).is_file():
        subprocess.run([str(scoring_python),'-B',str(score_script)],check=True,cwd=ROOT)
    # Set this storage path independently before running. models.h5.
    model_result=ROOT
    if RECOMPUTE_ADD3 or not model_result.is_file():
        # Set this storage path independently before running. score.npz.
        subprocess.run([sys.executable,'-B',str(model_script),str(ROOT)],check=True,cwd=ROOT)
    # Set this storage path independently before running. score.npz.
    add3_io['assemble_add3'](RESULT_FILE,ROOT,model_result,source_bundle,original_manifest)
with h5py.File(RESULT_FILE,'r') as handle:
    assert original_fingerprints==add3_io['dataset_fingerprints'](handle),'A pre-existing dataset changed.'
    add3_metadata=json.loads(handle['supplement_add3/metadata_json'].asstr()[()])
    combined=add3_io['read_frame'](handle['supplement_add3/combined_statistics'])
    print(combined.loc[add3_io['FOLLOWUP_NAMES'],['A','q_global','primary_pass','residual_flags']].to_string())
print('Supplement ready: BH220 primary / BH880 forks; old datasets preserved.')


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

# Cleanup must never interpret the project/current directory as a temporary target.
def _release_cleanup_target(path):
    candidate = Path(path).resolve()
    current = Path.cwd().resolve()
    if candidate == current or candidate in current.parents or candidate == Path(candidate.anchor):
        raise RuntimeError("Configure the owned temporary cleanup path before running this section.")
    return path

#10. =========== Redraw the 61-column panel and validate the new PNG/PDF
add3_data,add3_stats,add3_order=add3_io['load_plot_inputs'](RESULT_FILE)
notebook_current=json.loads(NOTEBOOK_FILE.read_text(encoding='utf-8'))
appearance_cell=next(''.join(c['source']) for c in notebook_current['cells']
    if c['cell_type']=='code' and 'def render(data,stats,order):' in ''.join(c['source']))
# Reuse the accepted appearance parameters; the original renderer/code is not edited.
exec(appearance_cell[:appearance_cell.index('def render(data,stats,order):')],globals())
OUTPUT_STEM='fig4A_Developmental_regulon_dynamics-1'
MAX_COLUMNS_PER_PANEL=72
EXPORT_EARLY=False
add3_renderer,add3_renderer_source=add3_io['derive_renderer'](notebook_current,globals())
add3_qa=add3_renderer(add3_data,add3_stats,add3_order)
assert len(add3_qa['display_branches'])==9 and add3_qa['original_row_order']==list(range(30))
assert len(add3_qa['figures'])==1
# Set this storage path independently before running.
SKILL_ROOT=ROOT
# Set this storage path independently before running. validate_figure_output.py. Use the supplied fig4_validate_figure_output.py.
VALIDATOR=ROOT
# Set this storage path independently before running.
RUNTIME=ROOT
# Set this storage path independently before running. python.exe.
QA_PYTHON=ROOT
# Set this storage path independently before running. pdftoppm.exe.
PDFTOPPM=ROOT
assert all(p.is_file() for p in (VALIDATOR,QA_PYTHON,PDFTOPPM))
add3_reports={}
for png_name,info in add3_qa['figures'].items():
    png=FIGURE_DIR/png_name;pdf=FIGURE_DIR/info['pdf']
    validation=subprocess.run([str(QA_PYTHON),'-B',str(VALIDATOR),str(png),str(pdf),
        '--target-dpi',str(DPI),'--expected-size-in',f"{info['width_in']}x{info['height_in']}",
        '--expected-clips','0','--max-clips','0','--pdftoppm',str(PDFTOPPM),
        # Set this storage path independently before running.
        '--render-dpi','120','--render-dir',str(ROOT)],capture_output=True,text=True,encoding='utf-8',errors='replace')
    add3_reports[pdf.name]=validation.stdout+validation.stderr
    if validation.returncode:print(add3_reports[pdf.name]);raise RuntimeError('Supplement figure validation failed.')
    with Image.open(png) as image:info.update(pixel_size=list(image.size),actual_dpi=list(image.info['dpi']))
add3_publication=dict(output_stem=OUTPUT_STEM,display_order=add3_order,figure_qa=add3_qa,
    figure_sha256={name:hashlib.sha256((FIGURE_DIR/name).read_bytes()).hexdigest() for name in add3_qa['files']},
    validation_reports=add3_reports,primary_family=220,fork_family=880,
    primary_threshold_not_met=[name for name in add3_io['FOLLOWUP_NAMES'] if not bool(add3_stats.loc[name,'primary_pass'])],
    profile=dict(clip_groups=0,max_clip_depth=0,fonts=['Arial','Arial Italic','Arial Bold Italic'],raster_images=0),
    calibrated_backend_reused=True,matplotlib_version=matplotlib.__version__,
    renderer_source_sha256=hashlib.sha256(add3_renderer_source.encode()).hexdigest(),
    notebook_source_sha256=hashlib.sha256('\n'.join(''.join(c.get('source',[])) for c in notebook_current['cells']).encode()).hexdigest())
with h5py.File(RESULT_FILE,'a') as handle:
    supplement=handle['supplement_add3']
    for key in ('publication_json','display_regulons','display_relative_activity'):
        if key in supplement:del supplement[key]
    supplement.create_dataset('publication_json',data=json.dumps(add3_publication,ensure_ascii=False),dtype=h5py.string_dtype())
    supplement.create_dataset('display_regulons',data=add3_order,dtype=h5py.string_dtype())
    values=add3_data['modes']['none']['bins'].loc[:,add3_order].to_numpy()
    supplement.create_dataset('display_relative_activity',data=np.maximum(values,0)/values.max(axis=0))
    assert original_fingerprints==add3_io['dataset_fingerprints'](handle)
for name,digest in original_manifest['original_figures_sha256'].items():
    assert hashlib.sha256((FIGURE_DIR/name).read_bytes()).hexdigest()==digest
if not KEEP_ADD3_QA:
    # Set this storage path independently before running.
    cache=ADD3_WORK.resolve();assert cache.parent==ROOT
    removed=sum(p.stat().st_size for p in _release_cleanup_target(cache).rglob('*') if p.is_file())
    shutil.rmtree(_release_cleanup_target(cache))
    print(f'PASS: new61-column PNG/PDF; removed {removed} bytes of supplemental runtime files.')
else:print(f'PASS: supplemental QA pages retained at {ADD3_WORK}')


## Original SCENIC network and scoring notebook

The following existing notebook cells are copied unchanged, including original shared figure outputs. Use this section in its original context; similarly named globals in other sections are not a new shared execution API.


# Figure 4 upstream analysis methods

Complete GRNBoost2, cisTarget motif pruning, fixed-axis AUCell, network stability, and accepted transcription-factor dynamics. The accepted integration RNA and Slingshot coordinates are inputs. The embedded SCENIC worker and Slingshot R worker are also supplied as plain source files.

Every actual storage address is replaced by `ROOT` or a string placeholder `ROOT`. Configure each input, output, runtime, program, and worker path independently before execution. HDF5 keys, public URLs, archived-source labels, numerical settings, seeds, statistical tests, and selection gates retain their original meaning.

This source-only release contains no expression matrices, fitted models, execution outputs, or hidden notebook attachments. The files were checked statically; the analysis was not rerun.

---

# SCENIC: accepted transcription-factor dynamics

Start from the saved embryonic integration RNA, restore full native source counts,
rebuild the three GRNBoost2/motif networks when requested, and score all networks on
the same source-local rankings. The accepted figure and its editable PDF are the
only figure outputs. No previous SCENIC script or temporary analysis result is read.

The integration object has 11,467 common genes. Full source RNA therefore comes
from its corresponding QC count files. The saved accepted Slingshot model is reused;
the identical PCA/Slingshot fit is included as a fallback when that file is absent.
The figure uses GSE178931, GSE165502 and GSE195500. The single-library E10.5 source
does not enter this accepted figure or its multi-library stability test.

The nine Genealogy candidate positions are a frozen, previously accepted curation
(RNA dynamics, UniProt/MGI function and research scope). This notebook verifies
their network stability; it does not repeat the literature review or RNA GAM tests.
Binary Slingshot weights mean path membership, not fate probabilities.

Use a Python 3.11 kernel with AnnData 0.12, NumPy, pandas, SciPy, h5py, scikit-learn,
Matplotlib and IPython. SCENIC inference runs in a compatible separate interpreter:
pySCENIC 0.12.1, arboreto 0.1.6, NumPy 1.23.5, pandas 1.5.3, scikit-learn 1.3.2,
dask 2024.4.2 and joblib. Set SCENIC_PYTHON when that interpreter is elsewhere.
The four official mm10 motif/TF reference files must be in RESOURCE_DIR.

REBUILD_RESULTS=False reuses the two final result files for direct figure editing.
True restores source RNA and rebuilds all three complete networks; this is the
expensive analysis path. Plot parameters are kept in the final plotting cell.

In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#1. =========== Shared settings
from pathlib import Path
import io
import itertools
import json
import hashlib
import os
import pickle
import shutil
import subprocess
import sys
import tempfile
from urllib.parse import quote, unquote

# Set this storage path independently before running.
ROOT = ROOT
if ROOT.name == "Code_def":
    # Set this storage path independently before running.
    ROOT = ROOT
# Set this storage path independently before running. CSS_3000_Embryonic_Inte_adata.h5ad.
INPUT = ROOT
# Set this storage path independently before running. Slingshot.h5ad.
TRAJECTORY = ROOT
# Set this storage path independently before running.
DATA_DIR = ROOT
# Set this storage path independently before running.
FIG_DIR = ROOT
# Set this storage path independently before running. SCENIC.h5. Produced by fig4_SCENIC_analysis.ipynb.
RESULT_FILE = ROOT
# Set this storage path independently before running. SCENIC_TFs.tsv.
CANDIDATE_FILE = ROOT
# Set this storage path independently before running.
RESOURCE_DIR = ROOT  # Official reference inputs, not fitted results.
SCENIC_PYTHON = Path(os.environ.get("SCENIC_PYTHON",
    # Set this storage path independently before running. python.exe.
    str(ROOT)))
R_SCRIPT = os.environ.get("RSCRIPT", shutil.which("Rscript") or "Rscript")
REBUILD_RESULTS = False
SEED, N_THREADS, PRUNE_WORKERS = 42, 8, 4
sys.dont_write_bytecode = True
# Set this storage path independently before running.
TMP_DIR = ROOT
TMP_DIR.mkdir(parents=True, exist_ok=True)
_runtime = tempfile.TemporaryDirectory(prefix="run-", dir=TMP_DIR)
WORK = Path(_runtime.name)
# Set this storage path independently before running.
os.environ["MPLCONFIGDIR"] = str(ROOT)
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ[key] = "1"

import h5py
import numpy as np
import pandas as pd
import anndata as ad
from anndata.io import read_elem
from scipy import sparse
from scipy.stats import rankdata
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.ticker import FormatStrFormatter
from matplotlib.backend_bases import RendererBase
from matplotlib.backends.backend_pdf import PdfFile, RendererPdf, GraphicsContextPdf, Op
from matplotlib.transforms import Bbox
from IPython.display import display, Image
plt.rcdefaults()

def file_hash(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


In [ ]:
#2. =========== Accepted panel and network scope
TF_POOL = 'Foxp1 Foxp2 Tshz2 Tshz3 Zbtb20 Hic1 Bhlhe22 Tcf4 Pbx3 Pbx1 Meis1 Meis2 Nfia Nfib Nfix Sox4 Sox11 Ebf1 Ebf2 Ebf3 Neurod2 Nhlh1 Nhlh2 Myt1 Myt1l Zfhx3 Casz1 Etv4 Maf Mafb Onecut1 Onecut2 Onecut3 Klf7 Klf6 Klf9 Zbtb18 Rora Rorb Nr2f1 Nr2f2 Gata2 Gata3 Isl1 Isl2 Neurod1 Neurod4 Neurog1 Neurog2 Ascl1 Hes1 Hes5 Hey1 Hey2 Tcf12 Tcf7l2 Npas3 Arnt2 Pou3f1 Pou3f2 Pou3f3 Pou4f1 Pou4f2 Satb1 Satb2 Bcl11a Bcl11b Insm1 Shox2 Prox1 Runx1 Zeb1 Zeb2 Sox2 Sox9 Sox5 Sox6 Rfx3 Bcl6 Bach2 Esrrg'.split()
FINAL_PANELS = {"Ia": ["Klf6", "Sox11"], "II": ["Klf6", "Bcl11a", "Sox11"],
                "Ib": ["Klf6"], "Ic": ["Klf6", "Sox11"], "CVG": ["Nr2f1"]}
LINEAGES = {"Ia": 3, "Ib": 1, "Ic": 2, "II": 4, "CVG": 5}
UNIPROT = {"Klf6": "O08584", "Sox11": "Q7M6Y2", "Bcl11a": "Q9QYE3", "Nr2f1": "Q60632"}
MGI = {"Klf6": "MGI:1346318", "Sox11": "MGI:98359",
       "Bcl11a": "MGI:106190", "Nr2f1": "MGI:1352451"}
CURATION = {
    "Klf6": "Neuronal axon-growth function; SGN lineage mechanism remains unvalidated.",
    "Sox11": "Neural and otic-progenitor differentiation studied, including a 2025 CRISPRi preprint; endogenous SGN subtype mechanism remains unvalidated.",
    "Bcl11a": "Neural differentiation and sensory-circuit wiring studied outside the ear; SGN lineage mechanism remains unvalidated.",
    "Nr2f1": "Inner-ear epithelial/vestibular function and targets already studied; CVG-specific neuronal mechanism remains unvalidated."}
PRIMARY_REFERENCES = {
    "Klf6": "https://pubmed.ncbi.nlm.nih.gov/19815778/;https://pubmed.ncbi.nlm.nih.gov/30135567/",
    "Sox11": "https://pubmed.ncbi.nlm.nih.gov/22378879/;https://pubmed.ncbi.nlm.nih.gov/40236205/",
    "Bcl11a": "https://pmc.ncbi.nlm.nih.gov/articles/PMC4067532/;https://pubmed.ncbi.nlm.nih.gov/19616629/",
    "Nr2f1": "https://pubmed.ncbi.nlm.nih.gov/17021036/;https://pmc.ncbi.nlm.nih.gov/articles/PMC6214362/"}
PALETTE_CELL = {"Differentiating SGN": "#006FA6", "Late CVG": "#6B7900",
    "Immature Type Ia SGN": "#00846F", "Immature Type Ib SGN": "#00489C",
    "Immature Type Ic SGN": "#CC0744", "Immature Type II SGN": "#B79762"}


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#3. =========== Integration RNA and accepted Slingshot coordinates
SLINGSHOT_PARAMS = dict(root="CVG neuroblast",
    endpoints=["Immature Type Ia SGN", "Immature Type II SGN", "Immature Type Ib SGN",
               "Immature Type Ic SGN", "Late CVG"],
    df=5, view_df=6, approx_points=200, maxit=40, thresh=.001, shrink=1,
    reweight=False, reassign=False, seed=0, css_pcs=30, trajectory_pcs=10)
SLINGSHOT_R = "suppressPackageStartupMessages(library(slingshot))\nsuppressPackageStartupMessages(library(jsonlite))\nwork <- normalizePath(commandArgs(trailingOnly=TRUE)[1],winslash='/',mustWork=TRUE)\ncfg <- fromJSON(file.path(work,'parameters.json'))\nobs <- read.csv(file.path(work,'obs.csv'),row.names=1,check.names=FALSE)\nrd <- as.matrix(read.csv(file.path(work,'white10.csv'),row.names=1))\nset.seed(cfg$seed)\npto <- getLineages(rd,as.character(obs$cell_type_plot),start.clus=cfg$root,\n                   end.clus=cfg$endpoints,dist.method='simple',omega=FALSE,use.median=FALSE)\nfit <- getCurves(pto,shrink=cfg$shrink,extend='n',stretch=0,\n                 approx_points=cfg$approx_points,maxit=cfg$maxit,thresh=cfg$thresh,\n                 smoother='smooth.spline',df=cfg$df,reweight=cfg$reweight,\n                 reassign=cfg$reassign,allow.breaks=TRUE,shrink.method='cosine')\nwrite.csv(slingPseudotime(fit),file.path(work,'pseudotime.csv'),quote=FALSE)\nwrite.csv(slingCurveWeights(fit),file.path(work,'weights.csv'),quote=FALSE)\nwrite_json(list(lineages=unname(slingLineages(fit)),lineage_names=colnames(slingPseudotime(fit))),\n           file.path(work,'lineages.json'),auto_unbox=TRUE)\n"

def trajectory_from_integration(a):
    if TRAJECTORY.exists():
        saved = ad.read_h5ad(TRAJECTORY)
        info = saved.uns["slingshot"]
        assert saved.obs_names.equals(a.obs_names) and info["input_sha256"] == file_hash(INPUT)
        assert info["parameters_json"] == json.dumps(SLINGSHOT_PARAMS, sort_keys=True)
        pt = np.asarray(saved.obsm["slingshot_pseudotime"], float)
        weights = np.asarray(saved.obsm["slingshot_weights"], float)
        paths = [list(info["lineages"][key]) for key in info["lineage_names"]]
        palettes = json.loads(info["palettes_json"])
        provenance = dict(method="Accepted Slingshot result",
            input_file=str(TRAJECTORY.relative_to(ROOT)), result_sha256=file_hash(TRAJECTORY))
    else:
        # Exact upstream PCA and native-curve fit; no age-based trajectory substitution.
        with threadpool_limits(limits=1):
            pcs = PCA(n_components=30, svd_solver="full", random_state=0).fit_transform(a.obsm["X_css"])[:, :10]
        white10 = pcs / pcs.std(axis=0, ddof=1)
        # Set this storage path independently before running. obs.csv.
        a.obs[["cell_type_plot"]].to_csv(ROOT)
        # Set this storage path independently before running. white10.csv.
        pd.DataFrame(white10, index=a.obs_names).to_csv(ROOT)
        # Set this storage path independently before running. parameters.json.
        (ROOT).write_text(json.dumps(SLINGSHOT_PARAMS), encoding="utf8")
        # Set this storage path independently before running. fit.R.
        (ROOT).write_text(SLINGSHOT_R, encoding="utf8")
        env = dict(os.environ, LC_ALL="C", TMP=str(WORK), TEMP=str(WORK))
        # Set this storage path independently before running. fit.R.
        run = subprocess.run([R_SCRIPT, str(ROOT), str(WORK)],
                             capture_output=True, text=True, env=env)
        if run.returncode:
            raise RuntimeError(run.stderr)
        # Set this storage path independently before running. lineages.json.
        info = json.loads((ROOT).read_text())
        # Set this storage path independently before running. pseudotime.csv.
        pt = pd.read_csv(ROOT, index_col=0).loc[a.obs_names].to_numpy(float)
        # Set this storage path independently before running. weights.csv.
        weights = pd.read_csv(ROOT, index_col=0).loc[a.obs_names].to_numpy(float)
        paths, palettes = info["lineages"], {"cell": PALETTE_CELL}
        provenance = dict(method="Exact upstream Slingshot fit from saved integration",
                          parameters=SLINGSHOT_PARAMS)
    assert [path[-1] for path in paths] == [
        "Immature Type Ib SGN", "Immature Type Ic SGN", "Immature Type Ia SGN",
        "Immature Type II SGN", "Late CVG"]
    assert pt.shape == weights.shape == (a.n_obs, 5)
    assert set(np.unique(weights)) == {0, 1} and np.array_equal(np.isfinite(pt), weights > 0)
    return pt, weights, palettes, provenance

input_sha = file_hash(INPUT)
REBUILD = REBUILD_RESULTS or not (RESULT_FILE.exists() and CANDIDATE_FILE.exists())
if REBUILD:
    a = ad.read_h5ad(INPUT)
    obs = a.obs.copy()
    for field in ("library_id", "source_object_key", "stage_standardized", "study_id"):
        obs[field] = obs[field].astype(object)
    pt, weights, palettes, trajectory_info = trajectory_from_integration(a)


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#4. =========== Restore complete native RNA axes

def read_source_counts(path, cell_ids):
    with h5py.File(path, "r") as handle:
        source_obs = read_elem(handle["obs"])
        var = read_elem(handle["var"])
        contract = read_elem(handle["uns/matrix_contract"])
        assert contract["matrix_value_type"] == "raw_count"
        assert not contract["current_pipeline_normalization_applied"]
        positions = source_obs.index.get_indexer(cell_ids)
        assert (positions >= 0).all() and source_obs.index.is_unique and var.index.is_unique
        raw = handle["X"]
        assert raw.attrs["encoding-type"] == "csr_matrix"
        data, indices, indptr = [], [], [0]
        for position in positions:
            lo, hi = int(raw["indptr"][position]), int(raw["indptr"][position + 1])
            data.append(raw["data"][lo:hi])
            indices.append(raw["indices"][lo:hi])
            indptr.append(indptr[-1] + hi - lo)
        counts = sparse.csr_matrix((np.concatenate(data), np.concatenate(indices),
                                   np.asarray(indptr)), shape=(len(cell_ids), len(var)))
    assert np.isfinite(counts.data).all() and (counts.data >= 0).all()
    assert np.equal(counts.data, np.rint(counts.data)).all()
    return counts.astype(np.int32), var, contract


def prepare_sources(a, obs):
    # Normalize on each complete eligible native axis before inference filtering.
    cycle_genes = ["Mki67", "Top2a", "Pcna", "Tk1", "Ube2c", "Ccnb1", "Ccnb2",
                   "Cdk1", "Cenpf", "Tyms", "Clspn"]
    assert obs.index.equals(a.obs_names) and a.obs_names.is_unique
    obs = obs.copy()
    sources = {}
    for study in ("GSE178931", "GSE165502", "GSE195500"):
        positions = np.flatnonzero(a.obs.study_id.astype(str).eq(study).to_numpy())
        cells = a.obs_names[positions]
        local = a.obs.iloc[positions]
        if study == "GSE178931":
            matrices, order, var = [], [], None
            for source_key in local.source_object_key.astype(str).unique():
                subset = np.flatnonzero(local.source_object_key.astype(str).eq(source_key).to_numpy())
                stage = local.iloc[subset].stage_standardized.iloc[0].split(".")[0]
                # Set this storage path independently before running.
                path = next((ROOT).glob(f"{stage}_{study}__{source_key}.h5ad"))
                matrix, current_var, contract = read_source_counts(path, cells[subset])
                assert var is None or var.index.equals(current_var.index)
                var = current_var
                matrices.append(matrix)
                order.extend(subset)
            counts = sparse.vstack(matrices, format="csr")[np.argsort(order)]
        else:
            name = ("E14_E15_E16_E17_E18_P3_unresolved_GSE165502__all_4992_raw_cells.h5ad"
                    if study == "GSE165502" else "E14_E16_E18_P1_GSE195500__all_5441_local_cells.h5ad")
            # Set this storage path independently before running.
            counts, var, contract = read_source_counts(ROOT / name, cells)
        symbols = (var["symbol"].astype(str) if "symbol" in var
                   else pd.Series(var.index.astype(str), index=var.index))
        # Check the unfiltered overlapping counts against the integration output.
        overlap = pd.Index(symbols).get_indexer(a.var.gene_symbol.astype(str))
        assert (overlap >= 0).all()
        assert (counts[:, overlap] != a.layers["counts"][positions]).nnz == 0
        excluded = symbols.str.match(r"(?i)^mt-|^ERCC[-_]|^Hb[ab]-").to_numpy()
        for field in ("mt_feature", "ERCC_feature", "reporter_or_vector_feature", "technical_feature"):
            if field in var:
                excluded |= var[field].to_numpy(bool)
        if "chromosome" in var:
            excluded |= var.chromosome.isin(["chrM", "MT", "M"]).to_numpy()
        unique = ~symbols.duplicated(keep=False).to_numpy()
        eligible = ~excluded & unique & symbols.ne("").to_numpy()
        counts = counts[:, eligible].tocsr()
        genes = symbols.iloc[np.flatnonzero(eligible)].to_numpy(dtype=str)
        normalization_total = np.asarray(counts.sum(axis=1)).ravel()
        assert (normalization_total > 0).all()
        expression = counts.astype(np.float32).copy()
        expression.data *= np.repeat((10000.0 / normalization_total).astype(np.float32),
                                     np.diff(expression.indptr))
        np.log1p(expression.data, out=expression.data)
        training_mask = (np.ones(len(cells), dtype=bool) if study == "GSE178931" else
                         (local.source_state_evaluation_allowed.eq(True).to_numpy() &
                          local.source_rna_mixture_review_flag.eq(False).to_numpy()))
        cc_columns = pd.Index(genes).get_indexer(cycle_genes)
        assert (cc_columns >= 0).all()
        cc = expression[:, cc_columns].toarray()
        cc_z = (cc - cc.mean(axis=0)) / np.maximum(cc.std(axis=0), 1e-8)
        obs.loc[cells, "cycle_score"] = cc_z.mean(axis=1)
        obs.loc[cells, "log_detected_genes"] = np.log1p(
            np.asarray((expression > 0).sum(axis=1)).ravel())
        sources[study] = dict(expression=expression, genes=genes,
            cells=cells.to_numpy(dtype=str), training_mask=training_mask,
            normalization_total=normalization_total, matrix_unit=str(contract["matrix_unit"]),
            source_genes=len(var), excluded_technical_genes=int(excluded.sum()),
            duplicate_symbols_excluded=int((~unique).sum()), exact_common_counts_match=True)
    obs["library_block"] = obs.library_id.fillna(obs.source_object_key).fillna("unresolved").astype(str)
    obs["stage_numeric"] = obs.stage_standardized.str.extract(r"E(\d+\.\d+)")[0].astype(float)
    return sources, obs

if REBUILD:
    sources, obs = prepare_sources(a, obs)
    for study, data in sources.items():
        sparse.save_npz(WORK / (study + "_expression.npz"), data["expression"])
        np.savez_compressed(WORK / (study + "_axes.npz"),
            genes=data["genes"], cells=data["cells"], training_mask=data["training_mask"])
    print("Native RNA restored:", {study: len(data["cells"]) for study, data in sources.items()})


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#5. =========== Complete GRNBoost2, motif pruning and fixed-axis AUCell
WORKER_SOURCE = '\n# Configure each file/directory placeholder independently; no storage addresses are shipped.\nfrom pathlib import Path\nROOT = Path(".")\n\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport pickle\nimport warnings\n\nfor key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"):\n    os.environ[key] = "1"\nwarnings.filterwarnings("ignore", message="Not prepending group keys", category=FutureWarning, module="pyscenic.utils")\n\nimport dask\ndask.config.set({"dataframe.query-planning": False})\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom joblib import Parallel, delayed\nfrom arboreto.core import EARLY_STOP_WINDOW_LENGTH, SGBM_KWARGS, infer_partial_network\nfrom ctxcore.genesig import Regulon\nfrom ctxcore.rnkdb import FeatherRankingDatabase\nfrom ctxcore.recovery import derive_rank_cutoff\nfrom pyscenic.utils import modules_from_adjacencies\nfrom pyscenic.prune import prune2df, df2regulons\nfrom pyscenic.aucell import create_rankings, aucell4r\n\nSEED = 42\nMIN_DETECTED_CELLS = 3\nMIN_TARGETS = 20\nMIN_COVERAGE = 0.80\nAUC_THRESHOLD = 0.05\nNES_THRESHOLD = 3.0\nRANK_THRESHOLD = 1500\nRHO_THRESHOLD = 0.03\nDATABASES = ("mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather",\n             "mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather")\nANNOTATIONS = "motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl"\n\n\ndef infer_target(index, expression, genes, tf_expression, tf_genes, seed):\n    return infer_partial_network(regressor_type="GBM", regressor_kwargs=SGBM_KWARGS,\n        tf_matrix=tf_expression, tf_matrix_gene_names=tf_genes,\n        target_gene_name=genes[index], target_gene_expression=expression[:, index],\n        include_meta=False, early_stop_window_length=EARLY_STOP_WINDOW_LENGTH, seed=seed)\n\n\ndef encode_regulon(regulon):\n    return dict(name=regulon.name, transcription_factor=regulon.transcription_factor,\n        gene2weight={str(g): float(w) for g, w in regulon.gene2weight.items()},\n        gene2occurrence=({str(g): float(w) for g, w in regulon.gene2occurrence.items()}\n                         if regulon.gene2occurrence is not None else None),\n        context=list(regulon.context), score=float(regulon.score), nes=float(regulon.nes),\n        orthologous_identity=float(regulon.orthologous_identity),\n        similarity_qvalue=float(regulon.similarity_qvalue), annotation=str(regulon.annotation))\n\n\ndef decode_regulon(plain):\n    values = plain.copy()\n    values["context"] = frozenset(values["context"])\n    return Regulon(**values)\n\n\ndef infer_network(source, resources, workers, prune_workers, workdir):\n    matrix, mask = source["expression"], source["training_mask"]\n    keep = np.asarray((matrix[mask] > 0).sum(axis=0)).ravel() >= MIN_DETECTED_CELLS\n    genes = source["genes"][keep].tolist()\n    expression = np.ascontiguousarray(matrix[mask][:, keep].toarray(), dtype=np.float32)\n    # Set this storage path independently before running. allTFs_mm.txt.\n    official_tfs = set((ROOT).read_text().splitlines())\n    tf_columns = [i for i, gene in enumerate(genes) if gene in official_tfs]\n    tf_genes = [genes[i] for i in tf_columns]\n    tf_expression = np.ascontiguousarray(expression[:, tf_columns])\n    assert np.isfinite(expression).all() and tf_genes\n    frames = Parallel(n_jobs=workers, max_nbytes="1M", temp_folder=str(workdir),\n                      batch_size=1, verbose=0)(\n        delayed(infer_target)(i, expression, genes, tf_expression, tf_genes, SEED)\n        for i in range(len(genes)))\n    adjacency = pd.concat(frames, ignore_index=True).sort_values("importance", ascending=False)\n    assert not adjacency.duplicated(["TF", "target"]).any()\n    assert not adjacency.TF.eq(adjacency.target).any()\n    assert np.isfinite(adjacency.importance).all() and adjacency.importance.gt(0).all()\n    frame = pd.DataFrame(expression, index=source["cells"][mask], columns=genes)\n    modules = modules_from_adjacencies(adjacency, frame, min_genes=MIN_TARGETS,\n        rho_dichotomize=True, keep_only_activating=True, rho_threshold=RHO_THRESHOLD,\n        rho_mask_dropouts=False)\n    databases = [FeatherRankingDatabase(str(resources / name), name=name) for name in DATABASES]\n    motif = prune2df(databases, modules, str(resources / ANNOTATIONS),\n        rank_threshold=RANK_THRESHOLD, auc_threshold=AUC_THRESHOLD, nes_threshold=NES_THRESHOLD,\n        client_or_address="dask_multiprocessing", num_workers=prune_workers)\n    assert not motif.empty\n    regulons = df2regulons(motif, save_columns=["NES", "Annotation", "MotifSimilarityQvalue", "OrthologousIdentity"])\n    assert regulons and len({reg.name for reg in regulons}) == len(regulons)\n    assert all(set(reg.genes).issubset(set(genes)) for reg in regulons)\n    metadata = dict(training_cells=int(mask.sum()), cells_scored=len(mask), inference_genes=len(genes),\n        inference_TFs=len(tf_genes), targets_fitted=len(genes), edges=len(adjacency),\n        modules=len(modules), motif_rows=len(motif), regulons=len(regulons),\n        database_coverage=[dict(database=db.name, genes=db.total_genes,\n            inference_gene_coverage=len(set(genes).intersection(db.genes))/len(genes)) for db in databases])\n    return regulons, metadata\n\n\ndef score_networks(data, regulons, pool, workers):\n    # A single native gene axis and single tie-breaking ranking per source.\n    networks = {origin: {reg.transcription_factor: reg for reg in regs\n                        if reg.transcription_factor in pool} for origin, regs in regulons.items()}\n    activity, coverage = {}, []\n    for study, source in data.items():\n        matrix = source["expression"]\n        expressed = np.asarray((matrix > 0).sum(axis=0)).ravel() > 0\n        genes, cells = source["genes"][expressed], source["cells"]\n        gene_set, signatures = set(genes), []\n        for origin, regs in networks.items():\n            for tf in pool:\n                reg = regs.get(tf)\n                shared = gene_set.intersection(reg.genes) if reg is not None else set()\n                cov = len(shared)/len(reg) if reg is not None else np.nan\n                accepted = reg is not None and len(shared) >= MIN_TARGETS and cov >= MIN_COVERAGE\n                coverage.append(dict(study=study, network=origin, gene=tf,\n                    original_targets=len(reg) if reg is not None else 0, matched_targets=len(shared),\n                    coverage=cov, score_accepted=accepted, TF_feature_measured=tf in set(source["genes"]),\n                    reason="accepted" if accepted else ("no_regulon" if reg is None else "target_coverage_or_size")))\n                if accepted:\n                    signatures.append(reg.copy(name=origin + "::" + tf,\n                        gene2weight={g: w for g, w in reg.gene2weight.items() if g in shared}))\n        frame = pd.DataFrame(matrix[:, expressed].toarray(), index=cells, columns=genes)\n        ranking_window = derive_rank_cutoff(AUC_THRESHOLD, len(genes))\n        positive = np.asarray((matrix[:, expressed] > 0).sum(axis=1)).ravel()\n        rankings = create_rankings(frame, seed=SEED)\n        scores = aucell4r(rankings, signatures, auc_threshold=AUC_THRESHOLD, noweights=False,\n                          normalize=False, num_workers=workers)\n        assert scores.index.equals(frame.index) and np.isfinite(scores.to_numpy()).all()\n        assert ((scores.to_numpy() >= 0) & (scores.to_numpy() <= 1)).all()\n        activity[study] = dict(values=scores.to_numpy(), columns=scores.columns.to_numpy(dtype=str),\n            cells=cells, positive_genes=positive, ranking_window=int(ranking_window),\n            rank_gate=positive > ranking_window, ranking_genes=genes,\n            ranking_gene_sha256=hashlib.sha256("\\n".join(genes).encode()).hexdigest())\n    return activity, coverage\n\n\ndef main():\n    global SEED\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--workdir", type=Path, required=True)\n    parser.add_argument("--resources", type=Path, required=True)\n    parser.add_argument("--pool", type=json.loads, required=True)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--workers", type=int, default=8)\n    parser.add_argument("--prune-workers", type=int, default=4)\n    args = parser.parse_args()\n    SEED = args.seed\n    for key in ("TMP", "TEMP", "NUMBA_CACHE_DIR"):\n        os.environ[key] = str(args.workdir)\n    data = {}\n    for study in ("GSE178931", "GSE165502", "GSE195500"):\n        with np.load(args.workdir / f"{study}_axes.npz") as axes:\n            data[study] = {key: axes[key] for key in ("genes", "cells", "training_mask")}\n        data[study]["expression"] = sparse.load_npz(args.workdir / f"{study}_expression.npz")\n    regulons, training = {}, {}\n    for study, source in data.items():\n        regulons[study], training[study] = infer_network(\n            source, args.resources, args.workers, args.prune_workers, args.workdir)\n        print(f"{study}: {training[study][\'training_cells\']:,} training cells; "\n              f"{len(regulons[study]):,} motif-supported regulons", flush=True)\n    activity, coverage = score_networks(data, regulons, args.pool, args.prune_workers)\n    resources = {}\n    for name in (*DATABASES, ANNOTATIONS, "allTFs_mm.txt"):\n        digest = hashlib.sha256()\n        with (args.resources / name).open("rb") as stream:\n            for block in iter(lambda: stream.read(1024 * 1024), b""):\n                digest.update(block)\n        resources[name] = digest.hexdigest()\n    metadata = dict(method="GRNBoost2", seed=SEED, workers=args.workers, prune_workers=args.prune_workers,\n        training=training, min_detected_cells=MIN_DETECTED_CELLS, min_module_genes=MIN_TARGETS,\n        min_matched_targets=MIN_TARGETS, minimum_target_coverage=MIN_COVERAGE,\n        motif_auc_threshold=AUC_THRESHOLD, activity_auc_threshold=AUC_THRESHOLD,\n        NES_threshold=NES_THRESHOLD, rank_threshold=RANK_THRESHOLD, rho_threshold=RHO_THRESHOLD,\n        activating_only=True, dropout_masking=False, weighted_activity=True, activity_normalized=False,\n        ranking_created_once_for_all_networks=True, missing_genes_zero_imputed=False,\n        training_rule={"GSE178931": "All accepted source cells", "late_sources":\n            "source_state_evaluation_allowed == True and source_rna_mixture_review_flag == False"},\n        resource_sha256=resources,\n        runtime_versions={package: importlib.metadata.version(package) for package in\n            ("pyscenic", "arboreto", "numpy", "pandas", "scikit-learn", "dask", "joblib")})\n    result = dict(regulons={study: [encode_regulon(reg) for reg in regs]\n                           for study, regs in regulons.items()},\n                  activity=activity, coverage=coverage, metadata=metadata)\n    # Set this storage path independently before running. computed.pkl.\n    with (ROOT).open("wb") as stream:\n        pickle.dump(result, stream, protocol=pickle.HIGHEST_PROTOCOL)\n\n\nif __name__ == "__main__":\n    main()\n'

if REBUILD:
    # Set this storage path independently before running. SCENIC_worker.py. Use fig4_scenic_worker_source.py, or the identical embedded worker.
    (ROOT).write_text(WORKER_SOURCE, encoding="utf8")
    # Set this storage path independently before running.
    env = dict(os.environ, TMP=str(WORK), TEMP=str(WORK), NUMBA_CACHE_DIR=str(ROOT))
    # Set this storage path independently before running. SCENIC_worker.py. Use fig4_scenic_worker_source.py, or the identical embedded worker.
    run = subprocess.run([str(SCENIC_PYTHON), str(ROOT),
        "--workdir", str(WORK), "--resources", str(RESOURCE_DIR),
        "--pool", json.dumps(TF_POOL), "--seed", str(SEED),
        "--workers", str(N_THREADS), "--prune-workers", str(PRUNE_WORKERS)],
        capture_output=True, text=True, env=env)
    if run.returncode:
        raise RuntimeError(run.stderr)
    print(run.stdout.strip())
    # Set this storage path independently before running. computed.pkl.
    with (ROOT).open("rb") as stream:
        computed = pickle.load(stream)


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#6. =========== Multiple-network stability and accepted candidate positions
def weighted_mean(y, w):
    return np.average(y, axis=0, weights=w)

def weighted_sd(y, w):
    mu = weighted_mean(y, w)
    return np.sqrt(weighted_mean((y - mu) ** 2, w))

def weighted_corr(x, y, w, rank=False):
    if rank:
        x = rankdata(x)
        y = np.apply_along_axis(rankdata, 0, y)
    xc = x - np.average(x, weights=w)
    yc = y - weighted_mean(y, w)
    numerator = weighted_mean(xc[:, None] * yc, w)
    denominator = np.sqrt(np.average(xc ** 2, weights=w)) * weighted_sd(y, w)
    return np.divide(numerator, denominator, out=np.full(y.shape[1], np.nan), where=denominator > 1e-12)

def network_stability(obs, pt, weights, computed):
    """Conservative candidate filter on fixed-cell, fixed-ranking comparisons."""
    # These thresholds define the new exploratory stability requirement.
    MIN_NETWORKS = 2
    MIN_CELLS = 50
    MIN_LIBRARIES = 3
    MIN_PARTIAL_RHO = .20
    MIN_LOO_DIRECTION = .80
    MIN_BIN_COR = .50
    N_QUANTILE_BINS = 8
    coverage = pd.DataFrame(computed['coverage'])
    pool = list(dict.fromkeys(coverage.gene))
    mapping={'Ia':3,'II':4,'Ib':1,'Ic':2,'CVG':5}
    detail, pairs, bins_records, source_summary = [], [], [], []
    def nuisance(local):
        extra=local[['log_detected_genes','cycle_score']].to_numpy(float)
        extra=(extra-extra.mean(0))/np.maximum(extra.std(0),1e-8)
        blocks=pd.get_dummies(local.library_block,drop_first=True,dtype=float).to_numpy()
        return np.column_stack([np.ones(len(local)),blocks,extra])
    def trend(local,x,y,w):
        design=nuisance(local)
        rootw=np.sqrt(w[:,None])
        rank_x=rankdata(x)
        rx=rank_x-design@np.linalg.lstsq(design*rootw,rank_x*np.sqrt(w),rcond=None)[0]
        ranks=np.column_stack([rankdata(y[:,k]) for k in range(y.shape[1])])
        ry=ranks-design@np.linalg.lstsq(design*rootw,ranks*rootw,rcond=None)[0]
        rho=weighted_corr(rx,ry,w)
        full=np.column_stack([design,x])
        coef=np.linalg.lstsq(full*rootw,y*rootw,rcond=None)[0]
        sd=weighted_sd(y,w)
        slope=np.divide(coef[-1],sd,out=np.full(y.shape[1],np.nan),where=sd>1e-12)
        adjusted=y-design[:,1:]@coef[1:-1]
        return rho,slope,adjusted
    for study in ['GSE178931','GSE165502','GSE195500']:
        saved = computed['activity'][study]
        activity = pd.DataFrame(saved['values'],index=saved['cells'],columns=saved['columns'])
        gate = pd.Series(saved['rank_gate'],index=saved['cells'])
        for lineage,j in mapping.items():
            if lineage=='CVG' and study!='GSE178931': continue
            hit=obs.study_id.eq(study).to_numpy() & obs.index.isin(activity.index)
            hit &= obs.source_state_evaluation_allowed.to_numpy()
            hit &= ~obs.source_rna_mixture_review_flag.to_numpy()
            hit &= np.isfinite(pt[:,j-1]) & (weights[:,j-1]>0)
            hit &= gate.reindex(obs.index,fill_value=False).to_numpy()
            q=np.flatnonzero(hit)
            local=obs.iloc[q]
            libraries=local.library_block.nunique()
            if len(q)<MIN_CELLS or libraries<MIN_LIBRARIES: continue
            x=pt[q,j-1]
            x=(x-x.min())/(x.max()-x.min())
            w=1/local.groupby('library_block').library_block.transform('size').to_numpy()
            a=activity.reindex(local.index)
            y=a.to_numpy(float)
            if not y.shape[1]: continue
            rho,slope,adjusted=trend(local,x,y,w)
            folds=[]
            for library in local.library_block.unique():
                keep=local.library_block.ne(library).to_numpy()
                if keep.sum()<MIN_CELLS: continue
                folds.append(trend(local.loc[keep],x[keep],y[keep],w[keep])[1])
            folds=np.asarray(folds)
            loo=np.mean(np.sign(folds)==np.sign(slope),axis=0) if len(folds) else np.full(len(slope),np.nan)
            cuts=np.unique(np.quantile(x,np.linspace(0,1,N_QUANTILE_BINS+1)))
            bi=np.clip(np.searchsorted(cuts,x,side='right')-1,0,len(cuts)-2)
            bin_y=np.asarray([weighted_mean(adjusted[bi==b],w[bi==b]) for b in np.unique(bi)])
            indexed={}
            for k,column in enumerate(a):
                network,gene=column.split('::')
                tested=np.isfinite(rho[k]) and np.isfinite(slope[k])
                reliable=bool(tested and abs(rho[k])>=MIN_PARTIAL_RHO
                              and loo[k]>=MIN_LOO_DIRECTION
                              and np.sign(rho[k])==np.sign(slope[k]))
                indexed.setdefault(gene,[]).append((network,k,reliable))
                detail.append(dict(lineage=lineage,gene=gene,study=study,network=network,
                    cells=len(q),libraries=libraries,partial_rho=float(rho[k]),
                    slope_std=float(slope[k]),loo_direction=float(loo[k]),network_signal_pass=reliable,
                    adjusted_for_library_complexity_cycle=True))
                for b in range(len(bin_y)):
                    bins_records.append(dict(lineage=lineage,gene=gene,study=study,network=network,
                        bin=b,adjusted_AUC=float(bin_y[b,k])))
            for gene,entries in indexed.items():
                cors=[]
                for first,second in itertools.combinations(entries,2):
                    cor=float(np.corrcoef(bin_y[:,first[1]],bin_y[:,second[1]])[0,1])
                    cors.append(cor)
                    pairs.append(dict(lineage=lineage,gene=gene,study=study,
                        network_a=first[0],network_b=second[0],bin_curve_cor=cor))
                signs=[int(np.sign(slope[k])) if np.isfinite(slope[k]) else 0 for _,k,_ in entries]
                n=len(entries)
                stable=bool(n>=MIN_NETWORKS and all(v for _,_,v in entries)
                    and len(set(signs))==1 and len(cors)>0 and np.isfinite(cors).all()
                    and min(cors)>=MIN_BIN_COR)
                source_summary.append(dict(lineage=lineage,gene=gene,study=study,
                    n_networks=n,source_stable=stable,direction=signs[0] if len(set(signs))==1 else 0,
                    rho_min=min(abs(rho[k]) for _,k,_ in entries),
                    loo_direction_min=min(loo[k] for _,k,_ in entries),
                    curve_cor_min=min(cors) if cors else np.nan))
    d=pd.DataFrame(detail); s=pd.DataFrame(source_summary); results=[]
    for lineage in mapping:
        relevant=['GSE178931'] if lineage=='CVG' else ['GSE165502','GSE195500']
        for gene in pool:
            local=s.loc[s.lineage.eq(lineage)&s.gene.eq(gene)&s.study.isin(relevant)]
            good=local.loc[local.source_stable]
            signal=d.loc[d.lineage.eq(lineage)&d.gene.eq(gene)&d.study.isin(relevant)&d.network_signal_pass]
            direction=int(good.direction.iloc[0]) if len(good) else 0
            opposite=bool(len(signal) and direction and (np.sign(signal.slope_std)!=direction).any())
            eligible=bool(len(good) and good.direction.nunique()==1 and not opposite)
            results.append(dict(lineage=lineage,gene=gene,network_eligible=eligible,
                stable_sources=';'.join(good.study) if eligible else '',
                n_networks_available_max=int(local.n_networks.max()) if len(local) else 0,
                n_networks_min=int(good.n_networks.min()) if len(good) else 0,
                rho_min=float(good.rho_min.min()) if len(good) else np.nan,
                loo_direction_min=float(good.loo_direction_min.min()) if len(good) else np.nan,
                curve_cor_min=float(good.curve_cor_min.min()) if len(good) else np.nan,
                network_direction=direction,opposite_strong_source_signal=opposite,
                network_status='pass' if eligible else ('opposite_source_signal' if opposite
                    else ('insufficient_qualified_networks' if not len(local) or local.n_networks.max()<MIN_NETWORKS
                    else 'failed_signal_direction_or_curve_stability'))))
    summary=pd.DataFrame(results)
    metadata=dict(minimum_networks=MIN_NETWORKS,minimum_cells=MIN_CELLS,
        minimum_libraries=MIN_LIBRARIES,minimum_partial_rho=MIN_PARTIAL_RHO,
        minimum_leave_library_out_direction=MIN_LOO_DIRECTION,
        minimum_adjusted_bin_curve_correlation=MIN_BIN_COR,
        all_evaluable_networks_must_agree=True,ranking_gene_axis_fixed_within_source=True,
        source_raw_AUC_pooled=False,independent_biological_validation=False,
        causal_TF_validation=False,SGN_support_requires_late_source=True,
        CVG_support_requires_early_source=True,shared_early_SGN_not_repeated_as_subtype_evidence=True,
        seed=SEED,SGN_scope='Late-source subtype support',CVG_scope='One early source')
    return dict(network_summary=summary,network_details=d,source_stability=s,
        network_pairs=pd.DataFrame(pairs),network_bins=pd.DataFrame(bins_records),
        stability_parameters=metadata)


def accepted_candidates(summary):
    panel = pd.DataFrame([(lineage, gene) for lineage, genes in FINAL_PANELS.items()
                          for gene in genes], columns=["lineage", "gene"])
    selected = panel.merge(summary, on=["lineage", "gene"], validate="one_to_one")
    assert len(selected) == 9 and selected.network_eligible.eq(True).all()
    selected["selected"] = True
    selected["panel_role"] = "potential_TF"
    selected["UniProt"] = selected.gene.map(UNIPROT)
    selected["MGI"] = selected.gene.map(MGI)
    selected["curation_scope"] = selected.gene.map(CURATION)
    selected["primary_references"] = selected.gene.map(PRIMARY_REFERENCES)
    selected["RNA_literature_curation"] = "Frozen accepted Genealogy panel; not refitted here"
    selected["caution"] = ""
    selected.loc[selected.lineage.eq("II") & selected.gene.eq("Klf6"), "caution"] = (
        "Partial rho is just above 0.20; boundary strength. Display-k shape sensitivity noted in Genealogy.")
    selected.loc[selected.gene.eq("Nr2f1"), "caution"] = "CVG evidence is from one early study."
    return selected

if REBUILD:
    checked = network_stability(obs, pt, weights, computed)
    candidates = accepted_candidates(checked["network_summary"])
    obs_fields = ["study_id", "library_id", "source_object_key", "stage_standardized",
        "stage_numeric", "cell_type_plot", "source_state_evaluation_allowed",
        "source_rna_mixture_review_flag", "cycle_score", "log_detected_genes", "library_block"]
    if "biological_replicate_id" in obs:
        obs_fields.append("biological_replicate_id")
    state = dict(obs=obs[obs_fields], pseudotime=pt, weights=weights, palettes=palettes,
        activity=computed["activity"], regulons=computed["regulons"],
        coverage=pd.DataFrame(computed["coverage"]), **checked,
        metadata=dict(input_file=str(INPUT.relative_to(ROOT)), input_sha256=input_sha,
            trajectory=trajectory_info, network=computed["metadata"],
            curated_panels=FINAL_PANELS, TF_pool=TF_POOL,
            scope="Three source cohorts underlying the accepted figure; all atlas identities retained",
            source_raw_AUC_pooled=False, independent_biological_validation=False,
            causal_TF_validation=False))


In [ ]:
#7. =========== Two reusable key result files
def _json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, frozenset)):
        return list(value)
    if value is pd.NA:
        return None
    raise TypeError(type(value).__name__)

def _put(group, name, value):
    if isinstance(value, pd.DataFrame):
        target = group.create_group(name)
        target.attrs["kind"] = "frame"
        payload = value.to_dict(orient="split")
        payload["index_name"] = value.index.name
        payload["dtypes"] = [str(dtype) for dtype in value.dtypes]
        _put(target, "payload", payload)
    elif isinstance(value, dict):
        target = group.create_group(name)
        target.attrs["kind"] = "dict"
        for key, item in value.items():
            _put(target, quote(str(key), safe=""), item)
    elif isinstance(value, np.ndarray) and value.dtype.kind not in "OU":
        target = group.create_dataset(name, data=value, compression="gzip", shuffle=True)
        target.attrs["kind"] = "array"
    elif isinstance(value, np.ndarray) and value.dtype.kind in "OU":
        target = group.create_dataset(name, data=value.astype(object),
                                     dtype=h5py.string_dtype("utf-8"), compression="gzip")
        target.attrs["kind"] = "strings"
    else:
        data = json.dumps(value, ensure_ascii=False, default=_json_default).encode("utf8")
        target = group.create_dataset(name, data=np.frombuffer(data, dtype=np.uint8), compression="gzip")
        target.attrs["kind"] = "json"

def _get(target):
    kind = target.attrs["kind"]
    if kind == "dict":
        return {unquote(key): _get(item) for key, item in target.items()}
    if kind == "frame":
        payload = _get(target["payload"])
        frame = pd.DataFrame(payload["data"], index=payload["index"], columns=payload["columns"])
        frame.index.name = payload["index_name"]
        for column, dtype in zip(frame.columns, payload["dtypes"]):
            frame[column] = frame[column].astype(dtype)
        return frame
    if kind == "strings":
        return target.asstr()[()]
    if kind == "array":
        return target[()]
    return json.loads(target[()].tobytes().decode("utf8"))

def save_results(state, candidates):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with h5py.File(WORK / RESULT_FILE.name, "w") as handle:
        handle.attrs["schema"] = "SCENIC_reusable_v1"
        _put(handle, "state", state)
    candidates.to_csv(WORK / CANDIDATE_FILE.name, sep="\t", index=False)
    shutil.copyfile(WORK / RESULT_FILE.name, RESULT_FILE)
    shutil.copyfile(WORK / CANDIDATE_FILE.name, CANDIDATE_FILE)

def load_results():
    with h5py.File(RESULT_FILE, "r") as handle:
        state = _get(handle["state"])
    candidates = pd.read_csv(CANDIDATE_FILE, sep="\t")
    assert state["metadata"]["input_sha256"] == file_hash(INPUT)
    assert candidates.network_eligible.eq(True).all()
    return state, candidates

if REBUILD:
    save_results(state, candidates)
state, candidates = load_results()
print(f"Ready: {len(candidates)} candidate positions; {candidates.gene.nunique()} TFs; two key result files.")


In [ ]:
#8. =========== Accepted figure layout
FIGURE_SIZE = (7.2, 3.33)
PNG_DPI = 600
ACTIVITY_Z_LIMIT = 2.0
BASENAME = "SCENIC_TF_dynamics"
def plot_tf_dynamics(state, candidates):
    # Use the accepted candidate table; appearance changes do not refit models.
    chosen = candidates.loc[candidates.panel_role.eq('potential_TF') & candidates.selected.eq(True)]
    assert len(chosen) and chosen.network_eligible.eq(True).all()
    preferred = ['Sox11','Klf6','Bcl11a','Nr2f1']
    selected_genes = set(chosen.gene)
    TFS = [g for g in preferred if g in selected_genes]
    TFS += sorted(selected_genes.difference(TFS))
    LATE_TFS = TFS[:4]
    EARLY_PATHS = [(3, 'SGN'), (5, 'CVG')]
    LATE_PATHS = [(3, 'Ia'), (1, 'Ib'), (2, 'Ic'), (4, 'II')]
    FIGSIZE = FIGURE_SIZE
    RASTER_DPI = PNG_DPI
    Z_LIMIT = ACTIVITY_Z_LIMIT


    obs, pt, weights = state["obs"], state["pseudotime"], state["weights"]
    coverage, details = state["coverage"], state["network_details"]
    activities, gates = {}, {}
    for study, saved in state["activity"].items():
        activities[study] = pd.DataFrame(saved["values"],index=saved["cells"],columns=saved["columns"])
        gates[study] = pd.Series(saved["rank_gate"],index=saved["cells"]).reindex(obs.index,fill_value=False).to_numpy(bool)
    allowed = obs.source_state_evaluation_allowed.to_numpy(bool).copy()
    allowed &= ~obs.source_rna_mixture_review_flag.to_numpy(bool)
    def source_mask(study):
        return allowed & obs.study_id.eq(study).to_numpy() & gates[study]
    def check_signature(study,network,gene):
        row = coverage.loc[coverage.study.eq(study) & coverage.network.eq(network) & coverage.gene.eq(gene)]
        assert len(row)==1 and bool(row.score_accepted.iloc[0])
        assert row.coverage.iloc[0]>=.8 and row.matched_targets.iloc[0]>=20
        column=network+"::"+gene
        assert column in activities[study] and np.isfinite(activities[study][column]).all()
        return column
    palettes = state["palettes"]

    matrices, path_counts = {}, {}
    for study, paths in [('GSE178931', EARLY_PATHS), ('GSE165502', LATE_PATHS)]:
        columns = [check_signature(study,'GSE165502',tf) for tf in TFS]
        blocks, path_counts[study] = [], {}
        for lineage,_ in paths:
            hit = source_mask(study) & np.isfinite(pt[:,lineage-1]) & (weights[:,lineage-1]>0)
            local = obs.loc[hit]
            x = pt[hit,lineage-1]
            cuts = np.unique(np.quantile(x,np.linspace(0,1,9)))
            assert len(cuts) == 9
            groups = np.clip(np.searchsorted(cuts,x,side='right')-1,0,7)
            w = 1/local.groupby('library_block').library_block.transform('size').to_numpy()
            values = activities[study].loc[local.index,columns].to_numpy(float)
            blocks.append(np.asarray([weighted_mean(values[groups==b],w[groups==b]) for b in range(8)]).T)
            path_counts[study][lineage] = len(local)
        raw = pd.DataFrame(np.concatenate(blocks,axis=1),index=TFS)
        assert np.isfinite(raw.to_numpy()).all() and raw.ge(0).all().all()
        sd = raw.std(axis=1, ddof=0)
        assert sd.gt(0).all()
        matrices[study] = raw.sub(raw.mean(axis=1), axis=0).div(sd, axis=0).to_numpy()
    assert [path_counts['GSE178931'][path] for path, _ in EARLY_PATHS] == [930, 490]

    mapping = {'Ia':3,'Ib':1,'Ic':2,'II':4,'CVG':5}
    stage_data, plot_sources = {}, {}
    for tf in LATE_TFS:
        selected = chosen.loc[chosen.gene.eq(tf)]
        sources = set(';'.join(selected.stable_sources.fillna('')).split(';'))
        study = next(s for s in ['GSE165502','GSE195500','GSE178931'] if s in sources)
        paths = selected.loc[selected.stable_sources.fillna('').str.split(';').apply(lambda v: study in v),'lineage']
        selected_paths = [mapping[path]-1 for path in paths]
        assert selected_paths
        column = check_signature(study,study,tf)
        evidence = details.loc[details.study.eq(study) & details.network.eq(study) & details.gene.eq(tf) & details.lineage.isin(paths)]
        assert len(evidence) == len(paths) and evidence.network_signal_pass.eq(True).all()
        hit = source_mask(study) & np.any((weights[:,selected_paths]>0) & np.isfinite(pt[:,selected_paths]),axis=1)
        local = obs.loc[hit,['library_block','stage_standardized','stage_numeric']].copy()
        local['AUC'] = activities[study].loc[local.index,column].to_numpy(float)
        means = local.groupby(['stage_numeric','stage_standardized','library_block'],observed=True).agg(AUC=('AUC','mean'),cells=('AUC','size')).reset_index()
        assert not means.duplicated(['stage_numeric','library_block']).any()
        stage_data[tf] = means
        plot_sources[tf] = dict(study=study,network=study,lineages=list(paths),cells=len(local),libraries=local.library_block.nunique())

    plt.rcParams.update({'font.family': 'Arial', 'font.size': 8,
        'axes.labelsize': 8, 'axes.titlesize': 8.5, 'xtick.labelsize': 7,
        'ytick.labelsize': 7, 'axes.linewidth': .65, 'savefig.facecolor': 'white'})
    fig = plt.figure(figsize=FIGSIZE, facecolor='white')
    fig.text(.027, .983, 'a', fontsize=11, weight='bold', va='top')

    # a. Relative activity by path; source blocks remain separated.
    heat_specs = [(.135, .238, 'GSE178931', EARLY_PATHS, 'E9.5–E13.5'),
                  (.391, .476, 'GSE165502', LATE_PATHS, 'E14.5–E18.5')]
    cmap = plt.get_cmap('RdBu_r').copy()
    for x, width, study, paths, label in heat_specs:
        ax = fig.add_axes([x, .550, width, .315])
        im = ax.imshow(matrices[study], cmap=cmap, vmin=-Z_LIMIT, vmax=Z_LIMIT,
                       aspect='auto', interpolation='nearest')
        ax.set_yticks(range(len(TFS)), TFS if study == 'GSE178931' else [])
        for tick in ax.get_yticklabels(): tick.set_fontstyle('italic')
        ax.tick_params(axis='y', length=0, labelsize=8.5, pad=4)
        ax.set_xticks([])
        for boundary in range(1, len(paths)):
            ax.axvline(8 * boundary - .5, color='white', lw=1.5)
        for spine in ax.spines.values(): spine.set_visible(False)
        fig.text(x + width / 2, .977, label, fontsize=8.5, ha='center', va='top')
        branch_colors = []
        for lineage, name in paths:
            key = ('Differentiating SGN' if lineage == 3 else 'Late CVG') if study == 'GSE178931' else (
                  'Immature Type ' + name + ' SGN')
            branch_colors.append(palettes['cell'][key])
        strip = fig.add_axes([x, .881, width, .010])
        strip.imshow(np.repeat(np.arange(len(paths)), 8)[None, :],
                     cmap=ListedColormap(branch_colors), aspect='auto', interpolation='nearest',
                     vmin=-.5, vmax=len(paths)-.5)
        strip.set_axis_off()
        for i, (_, name) in enumerate(paths):
            fig.text(x + width * (i + .5) / len(paths), .900, name,
                     fontsize=8, ha='center', va='bottom')
    cax = fig.add_axes([.902, .600, .012, .205])
    cb = fig.colorbar(im, cax=cax, ticks=[-2, 0, 2], extend='both')
    cb.set_label('Activity (z)', fontsize=8, labelpad=4)
    cb.ax.tick_params(labelsize=7, length=2)
    fig.text(.5, .513, 'Pseudotime →', ha='center', va='top', fontsize=8)

    # b. Actual stage means: grey library points and equal-library averages.
    fig.text(.027, .426, 'b', fontsize=11, weight='bold', va='top')
    facet_width = (.732 - .044*(len(LATE_TFS)-1))/len(LATE_TFS)
    for i, tf in enumerate(LATE_TFS):
        ax = fig.add_axes([.135 + i * (facet_width+.044), .140, facet_width, .206])
        data = stage_data[tf]
        values = data.groupby('stage_numeric').AUC.mean().sort_index()
        ages = values.index.to_numpy(float)
        ymax = max(.05, np.ceil(data.AUC.max()*1.10/.05)*.05)
        ax.plot(ages, values, color='#3A6474', lw=1.3, marker='o', ms=2.6, zorder=3)
        for age in ages:
            individual = data.loc[data.stage_numeric.eq(age)].sort_values('library_block').AUC.to_numpy()
            jitter = np.linspace(-.09, .09, len(individual)) if len(individual) > 1 else [0]
            ax.scatter(age + np.asarray(jitter), individual, s=8, c='.55', alpha=.75,
                       edgecolors='none', zorder=2)
        ax.set_title(tf, loc='center', pad=5, fontsize=8.5, fontstyle='italic')
        ax.set_xlim(ages.min()-.25, ages.max()+.25)
        ax.set_ylim(0, ymax)
        ax.set_yticks([0, ymax/2, ymax])
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.3g'))
        ticks = [ages[0],ages[len(ages)//2],ages[-1]]
        ax.set_xticks(ticks, ['E'+str(age) for age in ticks])
        ax.tick_params(length=2.5, pad=2, labelsize=7)
        if i == 0:
            ax.set_ylabel('Activity (AUC)', fontsize=8, labelpad=4)
        ax.spines[['top', 'right']].set_visible(False)
    fig.text(.5, .040, 'Embryonic day', ha='center', fontsize=8)

    return fig


In [ ]:
#9. =========== Illustrator-editable vector PDF
class ThreeGroupPdfRenderer(RendererPdf):
    draw_path_collection = RendererBase.draw_path_collection

    def draw_markers(self, gc, marker_path, marker_trans, path, trans, rgbFace=None):
        if len(marker_path.vertices) == 2:
            rgbFace = None
        RendererBase.draw_markers(self, gc, marker_path, marker_trans, path, trans, rgbFace)

    def check_gc(self, gc, fillcolor=None):
        clean = self.new_gc()
        clean.copy_properties(gc)
        clean.set_clip_rectangle(None)
        clean.set_clip_path(None)
        super().check_gc(clean, fillcolor)

def editable_pdf(fig, path):
    # Heatmaps and colour strips become individual flat vector cells.
    for ax in fig.axes:
        for image in list(ax.images):
            values = np.asarray(image.get_array())
            rows, columns = values.shape[:2]
            xlim, ylim = ax.get_xlim(), ax.get_ylim()
            ax.pcolormesh(np.arange(columns+1)-.5, np.arange(rows+1)-.5, values,
                          cmap=image.get_cmap(), norm=image.norm, shading="flat",
                          edgecolors="face", linewidth=.15, antialiased=False, rasterized=False)
            image.remove()
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
        for collection in ax.collections:
            collection.set_rasterized(False)
            if isinstance(collection, matplotlib.collections.QuadMesh):
                collection.set_edgecolor("face")
                collection.set_linewidth(.15)
                collection.set_antialiased(False)
    dpi = fig.dpi
    buffer = io.BytesIO()
    with matplotlib.rc_context({"pdf.fonttype": 42, "pdf.use14corefonts": False}):
        try:
            fig.set_dpi(72)
            fig.canvas.draw()
            width, height = fig.get_size_inches()
            pdf = PdfFile(buffer, metadata={"Title": "SCENIC transcription-factor dynamics"})
            pdf.newPage(width, height)
            renderer = ThreeGroupPdfRenderer(pdf, dpi, height, width)
            graphics, guides, texts = [fig.patch], [], list(fig.texts)
            for ax in fig.axes:
                if not ax.get_visible():
                    continue
                graphics += [ax.patch] + list(ax.collections) + list(ax.patches) + list(ax.lines)
                if ax.axison:
                    guides += list(ax.spines.values())
                    ticks = ax.xaxis._update_ticks() + ax.yaxis._update_ticks()
                    guides += [part for tick in ticks for part in (tick.tick1line, tick.tick2line)]
                    texts += [part for tick in ticks for part in (tick.label1, tick.label2)]
                    texts += [ax.xaxis.label,ax.yaxis.label,ax.xaxis.get_offset_text(),ax.yaxis.get_offset_text()]
                texts += list(ax.texts) + [ax.title,ax._left_title,ax._right_title]
            for name, artists in (("Graphics",graphics),("Axes/Guides",guides),("Text",texts)):
                pdf.write(("% "+name+"\n").encode("ascii"))
                pdf.output(Op.gsave,Bbox.from_bounds(0,0,width*72,height*72),Op.rectangle,Op.clip,Op.endpath)
                renderer.gc = GraphicsContextPdf(pdf)
                seen = set()
                for artist in artists:
                    if artist.get_visible() and id(artist) not in seen:
                        artist.draw(renderer)
                        seen.add(id(artist))
                renderer.finalize()
                pdf.output(Op.grestore)
            pdf.finalize()
            pdf.close()
        finally:
            fig.set_dpi(dpi)
    path.write_bytes(buffer.getvalue())


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

# Cleanup must never interpret the project/current directory as a temporary target.
def _release_cleanup_target(path):
    candidate = Path(path).resolve()
    current = Path.cwd().resolve()
    if candidate == current or candidate in current.parents or candidate == Path(candidate.anchor):
        raise RuntimeError("Configure the owned temporary cleanup path before running this section.")
    return path

#10. =========== Export from reusable results
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig = plot_tf_dynamics(state, candidates)
png = FIG_DIR / (BASENAME + ".png")
pdf = FIG_DIR / (BASENAME + ".pdf")
fig.savefig(png, dpi=PNG_DPI, facecolor="white")
editable_pdf(fig, pdf)
plt.close(fig)
display(Image(filename=str(png), width=950))
print("Saved: 600-DPI PNG and editable vector PDF.")
_runtime.cleanup()
if TMP_DIR.exists() and not any(_release_cleanup_target(TMP_DIR).iterdir()):
    _release_cleanup_target(TMP_DIR).rmdir()


The HDF5 result retains the complete motif-supported network definitions, all scored
source activities, gene-ranking checks, aligned Slingshot membership/coordinates,
library/stage metadata, multi-network evidence and analysis parameters. The TSV
contains the nine accepted positions and their support/curation notes.

For figure editing, run the notebook with REBUILD_RESULTS=False, or run cells
1, 2, 7-10 after defining the plotting settings. No network refit is needed.
Panel a is the GSE165502 programme, standardized separately within early and late
sources. Panel b uses each candidate's qualifying source and selected-path cell
union, with equal-library observed stage averages. Absolute AUC is not pooled across
sources; stage averages and adjusted pseudotime trends are distinct summaries.
PDF text remains Arial text; heatmap cells, markers, curves and guides are vectors,
organized under three peer page-level clipping groups.

## Original standalone source: fig4A_regulon_dynamics_analysis.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# Accepted primary Fig4A whole-tree and early-fork analysis on tree_results-1.pkl; excludes seven connector cells; K=20.
# Original source label: analysis_code:master_source
# Original source SHA256: e9b449f1afd82d50e05d38a44018ef4031cdf63fb1c6a2adad1531ce68aa3a13
# Only storage addresses and release comments were changed; analysis was not executed.

"""Fig4A: Charles-style regulon dynamics on the accepted early-CVG/SGN tree.

Primary: constant-null whole-tree Gaussian spline association, BH217 q<1e-4,
fitted amplitude>0.025. Independent branch-wise cubic regression splines, ML.
Four fork comparisons localize dynamics; BH868 is a separate evidence family.
Source/depth/library and early-age models are sensitivities, not all-pass gates.
The seven computational connector cells are retained in measurements/context
but do not define a biological spline state (7348 modeled,7355 measured).
Native projection.t is used for models; log1p display lengths are not predictors.
Default redraws the frozen final H5. --recompute refits global/early-fork models;
late-fork fits are reused only after exact cohort/relative-t equivalence checks.
All outputs and temporary runtime files stay under Tmp/Fig4a.
"""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

# Cleanup must never interpret the project/current directory as a temporary target.
def _release_cleanup_target(path):
    candidate = Path(path).resolve()
    current = Path.cwd().resolve()
    if candidate == current or candidate in current.parents or candidate == Path(candidate.anchor):
        raise RuntimeError("Configure the owned temporary cleanup path before running this section.")
    return path

#1. =========== Shared settings
from pathlib import Path
import os,sys,json,hashlib,pickle,subprocess,shutil,argparse,time
sys.dont_write_bytecode=True
# Set this storage path independently before running.
ROOT=ROOT
# Set this storage path independently before running.
DESTINATION=ROOT
# Set this storage path independently before running.
WORK=ROOT
# Set this storage path independently before running. tree_results-1.pkl. Accepted tree input produced by the released Figure 3 analysis.
TREE_PATH=ROOT
# Set this storage path independently before running. tree_validation-1.json.
TREE_CHECK=ROOT
SEED=42
Q_CUT=1e-4
A_CUT=.025
BIO_SEGMENTS=[0,1,2,4,5,9,10,6,7,8]
PARENTS={0:None,1:0,2:0,4:2,5:4,9:5,10:5,6:4,7:6,8:6}
LABELS={0:'CVG common',1:'VGN cand.',2:'SGN cand.',4:'SGN common',5:'Ib/Ic',9:'Ib',10:'Ic',6:'Ia/II',7:'Ia',8:'II'}
MODULE_SOURCES={'global_new': '"""New-tree whole-tree SCENIC association on ten biological native segments.\n\nPrimary: constant-null versus segment-specific cubic-regression GAM, Gaussian\nML, unit weights. Five nuisance modes use identical spline structures. Every\n217-signature family gets BH and Bonferroni companion correction; primary\nselection uses BH q<1e-4 and trajectory A>.025. K10 is upgraded to K20 for all\nsignatures/all modes only if any tie-aware basis-capacity diagnostic triggers.\nThe seven connector cells remain archived with all7355 inputs;7348 cells enter\nassociation. The registered early Y is a conditional candidate-state topology;\nsource/age and library/age confounding preclude animal-level causal inference.\n"""\n\n# Configure each file/directory placeholder independently; no storage addresses are shipped.\nfrom pathlib import Path\nROOT = Path(".")\n#1. =========== Shared settings and frozen inputs\nfrom pathlib import Path\nimport argparse\nimport concurrent.futures\nimport hashlib\nimport json\nimport os\nimport pickle\nimport subprocess\nimport sys\nimport time\nsys.dont_write_bytecode=True\n# Set this storage path independently before running.\nROOT=ROOT\n# Set this storage path independently before running.\nDIRECTORY=ROOT\n# Set this storage path independently before running.\nWORK=ROOT\n# Set this storage path independently before running. Fig4A.h5.\nINPUT=ROOT\n# Set this storage path independently before running. tree_results-1.pkl. Accepted tree input produced by the released Figure 3 analysis.\nTREE=ROOT\n# Set this storage path independently before running. tree_validation-1.json.\nVALIDATION=ROOT\n# Set this storage path independently before running. Fig4A_global_new.h5.\nOUTPUT=ROOT\n# Set this storage path independently before running. Rscript.exe.\nRSCRIPT=ROOT\nMODES=(\'none\',\'depth\',\'study_only\',\'study\',\'library\')\nK_INITIAL,K_EXTENSION=10,20\nWORKERS=5\nSEED=42\nSEGMENT_IDS=(0,1,2,4,5,6,7,8,9,10)\nfor key in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\'):os.environ[key]=\'1\'\nfor key in (\'TMP\',\'TEMP\'):os.environ[key]=str(WORK)\nos.environ[\'LANG\']=os.environ[\'LC_ALL\']=\'C\'\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import f as f_distribution,false_discovery_control\n\n\nR_SOURCE = r\'\'\'\n#1. =========== Shared settings and command-line contract\nsuppressPackageStartupMessages(library(mgcv))\nargs <- commandArgs(trailingOnly=TRUE)\nif (!length(args) %in% c(6L,7L)) {\n  stop(\'Usage: Rscript associate_unified.R cells.csv auc.tsv.gz prefix mode K indices|all [validation_indices]\')\n}\nCELLS <- args[1]; AUC <- args[2]; PREFIX <- args[3]; MODE <- args[4]\nK <- suppressWarnings(as.integer(args[5]))\nSEED <- 42L\nK_CHECK_REPS <- 500L\nK_CHECK_SUBSAMPLE <- 2000L\nFAMILY_SIZE <- 217L\nCONSTANT_TOL <- 1e-12\nVALIDATION_TOL <- 1e-7\noptions(stringsAsFactors=FALSE, digits=17)\nstopifnot(MODE %in% c(\'none\',\'depth\',\'study_only\',\'study\',\'library_only\',\'library\'), length(K)==1L,\n          is.finite(K), K>=3L, dir.exists(dirname(PREFIX)))\nparse_indices <- function(text) {\n  if (identical(text,\'all\')) return(seq_len(FAMILY_SIZE))\n  tokens <- trimws(strsplit(text,\',\',fixed=TRUE)[[1]])\n  if (!length(tokens) || any(!grepl(\'^[0-9]+$\',tokens))) stop(\'Invalid regulon indices\')\n  indices <- as.integer(tokens)\n  stopifnot(!anyNA(indices),!anyDuplicated(indices),all(indices>=1L & indices<=FAMILY_SIZE))\n  indices\n}\nINDICES <- parse_indices(args[6])\nVALIDATION_INDICES <- if(length(args)==7L) parse_indices(args[7]) else c(1L,109L,217L)\nstarted <- proc.time()[[\'elapsed\']]\n\n\n#2. =========== Fixed cells, nuisance terms and independent segment smooths\nd <- read.csv(CELLS,row.names=1,check.names=FALSE)\nconnection <- gzfile(AUC,\'rt\')\ny_all <- tryCatch(as.matrix(read.delim(connection,row.names=1,check.names=FALSE)),\n                  finally=close(connection))\nstorage.mode(y_all) <- \'double\'\nrequired <- c(\'t\',\'seg\',\'study\',\'library\',\'log_detected\')\nstopifnot(all(required %in% names(d)),identical(rownames(d),rownames(y_all)),\n          !anyDuplicated(rownames(d)),!anyDuplicated(colnames(y_all)),\n          ncol(y_all)==FAMILY_SIZE,all(is.finite(y_all)),all(y_all>=0 & y_all<=1),\n          all(is.finite(d$t)),all(is.finite(d$log_detected)),\n          !anyNA(d$seg),!anyNA(d$study),!anyNA(d$library))\nd$seg <- factor(as.character(d$seg),levels=as.character(c(0,1,2,4,5,6,7,8,9,10)))\nd$study <- factor(d$study); d$library <- factor(d$library)\nstopifnot(!anyNA(d$seg),all(table(d$seg)>=20L))\nsegment_rows <- lapply(levels(d$seg),function(s) which(d$seg==s))\nnames(segment_rows) <- levels(d$seg)\nfor (rows in segment_rows) stopifnot(length(unique(d$t[rows]))>=K,diff(range(d$t[rows]))>0)\nd$u <- ave(d$t,d$seg,FUN=function(t) (t-min(t))/diff(range(t)))\nstopifnot(all(is.finite(d$u)))\nnull_rhs <- switch(MODE,none=\'1\',depth=\'log_detected\',\n                   study_only=\'study\',study=\'study + log_detected\',library_only=\'library\',library=\'library + log_detected\')\nf0 <- as.formula(paste(\'y ~\',null_rhs))\nf1 <- as.formula(paste(\'y ~\',null_rhs,sprintf("+ seg + s(u,by=seg,bs=\'cr\',k=%d)",K)))\nd$y <- y_all[,1L]\nG0 <- gam(f0,data=d,family=gaussian(),method=\'ML\',gamma=1,select=FALSE,\n          fit=FALSE,na.action=na.fail)\nG1 <- gam(f1,data=d,family=gaussian(),method=\'ML\',gamma=1,select=FALSE,\n          fit=FALSE,na.action=na.fail)\nstopifnot(length(G1$smooth)==10L,length(G1$sp)==10L)\nX0 <- G0$X; X1 <- G1$X\nsmooth_segments <- vapply(G1$smooth,function(s) as.character(s$by.level),character(1))\nstopifnot(setequal(smooth_segments,levels(d$seg)),!anyDuplicated(smooth_segments))\nsmooth_columns <- lapply(G1$smooth,function(s) s$first.para:s$last.para)\ntrajectory_columns <- sort(unique(c(which(startsWith(colnames(X1),\'seg\')),\n                                   unlist(smooth_columns))))\nstopifnot(length(trajectory_columns)==9L+sum(lengths(smooth_columns)))\nX_trajectory <- X1[,trajectory_columns,drop=FALSE]\n# Fixed-X thin QR is the exact sufficient-statistic reduction used by bam.fit for ML.\ncache_qr <- function(G) {\n  q <- qr(G$X,tol=0,LAPACK=TRUE)\n  rp <- q$pivot; rp[rp] <- seq_len(ncol(G$X))\n  G$fast_qr <- q\n  G$fast_sp <- mgcv:::initial.spg(G$X,G$y,G$w,G$family,G$S,G$rank,G$off,\n    offset=G$offset,L=G$L,lsp0=G$lsp0)\n  G$fast_R <- qr.R(q)[,rp,drop=FALSE]\n  stopifnot(max(abs(crossprod(G$fast_R)-crossprod(G$X)))<1e-7)\n  G\n}\nG0 <- cache_qr(G0); G1 <- cache_qr(G1)\n\n\n#3. =========== Cache validation and segment-specific residual diagnostics\ncapture_conditions <- function(expr) {\n  notices <- character()\n  value <- withCallingHandlers(expr,warning=function(w) {\n    notices <<- c(notices,conditionMessage(w)); invokeRestart(\'muffleWarning\')\n  })\n  list(value=value,warnings=unique(notices))\n}\nfit_response <- function(template,y) {\n  y <- as.numeric(y); X <- template$X; N <- length(y); p <- ncol(X)\n  initialization <- list(sp=template$fast_sp,scale=mean((y-mean(y))^2)/10)\n  f <- qr.qty(template$fast_qr,y)[seq_len(p)]\n  rss_extra <- sum(y*y)-sum(f*f)\n  stopifnot(rss_extra>=-1e-10)\n  template$X <- template$fast_R\n  template$y <- f; template$w <- rep(1,p); template$n <- p\n  template$offset <- rep(0,p); template$n.true <- N\n  template$dev.extra <- template$pearson.extra <- max(0,rss_extra)\n  template$mf$y <- y\n  template$fast_qr <- template$fast_R <- template$fast_sp <- NULL\n  model <- gam(G=template,method=\'ML\',gamma=1,in.out=initialization)\n  mu <- as.numeric(X %*% coef(model)); r <- y-mu\n  model$y <- y; model$model$y <- y\n  model$fitted.values <- model$linear.predictors <- mu\n  model$residuals <- r\n  model$prior.weights <- model$weights <- model$working.weights <- rep(1,N)\n  model$offset <- rep(0,N); model$z <- y\n  model$deviance <- sum(r*r)\n  model$null.deviance <- sum((y-mean(y))^2)\n  model$df.residual <- N-sum(model$edf)\n  model$df.null <- N-as.integer(template$intercept)\n  model$aic <- model$family$aic(y,1,mu,rep(1,N),model$deviance)+2*sum(model$edf)\n  model$hat <- rowSums((X %*% model$Vp)*X)/model$sig2\n  stopifnot(identical(as.numeric(model$y),as.numeric(y)),\n            identical(as.numeric(model$model$y),as.numeric(y)),\n            all(model$prior.weights==1),length(model$hat)==N)\n  model\n}\n\nconverged <- function(model) {\n  outer <- model$outer.info$conv\n  (is.null(model$converged) || isTRUE(model$converged)) &&\n    (is.null(outer) || identical(outer,0L) || identical(outer,0) ||\n       grepl(\'full convergence\',paste(outer,collapse=\' \'),fixed=TRUE))\n}\nrho <- function(x,y) {\n  if(length(x)<3L || diff(range(x))==0 || diff(range(y))==0) return(NA_real_)\n  suppressWarnings(cor(x,y,method=\'spearman\'))\n}\nadjacent_index <- function(u,residual) {\n  denominator <- mean(residual^2)\n  if(length(u)<2L || !is.finite(denominator) || denominator<=0) return(NA_real_)\n  mean(diff(residual[order(u)])^2)/2/denominator\n}\ntie_expected_sse <- function(values,groups,counts) {\n  sums <- rowsum(values,groups,reorder=TRUE)\n  squares <- rowsum(values*values,groups,reorder=TRUE)\n  means <- sums/counts\n  ss <- squares-sums*sums/counts\n  ss[ss<0] <- 0\n  variances <- ss/counts\n  result <- 2*colSums(ss)\n  m <- length(counts)\n  if(m>1L)result <- result+colSums(variances[-m,,drop=FALSE]+variances[-1L,,drop=FALSE]+\n    (means[-m,,drop=FALSE]-means[-1L,,drop=FALSE])^2)\n  result\n}\n# Exactly average adjacent SSE across all within-t orderings; verify by enumeration.\ntoy <- c(.1,-.3,.2,.6,-.1,.9)\npermutations <- function(x)if(length(x)==1L)list(x)else unlist(lapply(seq_along(x),function(i)\n  lapply(permutations(x[-i]),function(z)c(x[i],z))),recursive=FALSE)\norders <- unlist(lapply(permutations(c(1L,2L)),function(a)lapply(permutations(c(3L,4L,5L)),function(b)c(a,b,6L))),recursive=FALSE)\nstopifnot(abs(tie_expected_sse(matrix(toy,ncol=1),c(1,1,2,2,2,3),c(2L,3L,1L))-\n  mean(vapply(orders,function(z)sum(diff(toy[z])^2),numeric(1))))<1e-12)\nsegment_check <- function(u,residual,seed) {\n  stopifnot(length(u)==length(residual),all(is.finite(u)),all(is.finite(residual)))\n  set.seed(seed)\n  if(length(u)>K_CHECK_SUBSAMPLE) {\n    indices <- sample.int(length(u),K_CHECK_SUBSAMPLE)\n    u <- u[indices]; residual <- residual[indices]\n  }\n  n <- length(residual); denominator <- mean(residual^2)\n  if(n<2L || denominator<=0)return(list(k_index=NA_real_,p_value=NA_real_,\n    checked_cells=n,status=\'zero_residual_variance\',tied_cells=NA_integer_,tied_groups=NA_integer_))\n  groups <- match(u,sort(unique(u)));counts <- tabulate(groups)\n  random <- vapply(seq_len(K_CHECK_REPS),function(i)residual[sample.int(n,n)],numeric(n))\n  sse <- tie_expected_sse(cbind(residual,random),groups,counts)\n  list(k_index=sse[1]/(2*(n-1L))/denominator,p_value=mean(sse[-1L]<sse[1]),\n    checked_cells=n,status=\'ok\',tied_cells=sum(counts[counts>1L]),tied_groups=sum(counts>1L))\n}\nvalidate_segment_check <- function(rows,residual,seed) {\n  z <- d[rows,,drop=FALSE]\n  z$y <- residual\n  single <- gam(y~s(u,bs=\'cr\',k=K),data=z,family=gaussian(),method=\'ML\',\n                gamma=1,select=FALSE,na.action=na.fail)\n  # A diagnostic-only fixture preserves a genuine single-segment smooth/model\n  # frame while supplying exactly the joint model\'s residual vector to k.check.\n  single$y <- as.numeric(residual)\n  single$fitted.values <- rep(0,length(residual))\n  single$linear.predictors <- rep(0,length(residual))\n  fixture_residual <- as.numeric(residuals(single))\n  injection_error <- max(abs(fixture_residual-residual))\n  stopifnot(injection_error<1e-12)\n  custom <- segment_check(z$u,fixture_residual,seed)\n  set.seed(seed)\n  official <- k.check(single,subsample=K_CHECK_SUBSAMPLE,n.rep=K_CHECK_REPS)\n  k_error <- abs(custom$k_index-official[1,\'k-index\'])\n  p_error <- abs(custom$p_value-official[1,\'p-value\'])\n  stopifnot(k_error<1e-12,p_error==0)\n  list(k_index_difference=k_error,permutation_p_difference=p_error,\n       residual_injection_error=injection_error)\n}\nadd_frame <- function(rows,value) { rows[[length(rows)+1L]] <- value; rows }\nsummary_rows <- list(); k_rows <- list(); segment_residual_rows <- list()\nsource_residual_rows <- list(); validation_rows <- list()\nmetadata_rows <- data.frame(metric=c(\'mode\',\'K\',\'family_size\',\'cells\',\'null_formula\',\'full_formula\',\n    \'fit_method\',\'smooth_penalties\',\'k_check_method\',\'k_check_seed\',\'k_check_permutations\',\n    \'k_check_subsample_per_segment\',\'k_check_denominator\',\'multiplicity\',\'mgcv_version\'),\n  value=c(MODE,K,FAMILY_SIZE,nrow(d),deparse(f0),paste(deparse(f1),collapse=\' \'),\n    \'Gaussian ML; gamma1; selectFALSE; unit weights\',\'Ten independent smoothing parameters; no id\',\n    \'Tie-order-invariant adjacent-residual expected SSE separately within each biological native segment\',\n    \'42 + original1basedRegulonIndex + segmentInteger\',K_CHECK_REPS,K_CHECK_SUBSAMPLE,\n    \'mean(residual_segment^2); analytical average over within-t ties;500permutations; unique-t reduces to mgcv1D\',\n    \'No shard-level correction; Python must correct the complete217 family\',\n    as.character(packageVersion(\'mgcv\'))))\n\n\n#4. =========== Fit responses and retain every failure and residual\ntrajectory_values <- full_values <- residual_values <- matrix(NA_real_,nrow(d),length(INDICES),\n  dimnames=list(rownames(d),colnames(y_all)[INDICES]))\nfor(ii in seq_along(INDICES)) {\n  j <- INDICES[ii]; name <- colnames(y_all)[j]; y <- as.numeric(y_all[,j])\n  tick <- proc.time()[[\'elapsed\']]\n  row <- data.frame(regulon=name,index=j,mode=MODE,K=K,status=\'failed\',p_value=NA_real_,\n    A=NA_real_,RSS0=NA_real_,RSS1=NA_real_,df0=NA_real_,df1=NA_real_,test_df0=NA_real_,\n    test_df1=NA_real_,test_delta_df=NA_real_,F_statistic=NA_real_,partial_R2=NA_real_,\n    AIC_null=NA_real_,AIC_full=NA_real_,converged0=FALSE,converged1=FALSE,\n    full_rank0=FALSE,full_rank1=FALSE,rank0=NA_integer_,rank1=NA_integer_,\n    coefficients0=NA_integer_,coefficients1=NA_integer_,smooth_parameters=NA_integer_,\n    residual_mean=NA_real_,residual_sd=NA_real_,residual_depth_rho=NA_real_,\n    k_check_terms=0L,minimum_k_index=NA_real_,minimum_k_p=NA_real_,maximum_edf_ratio=NA_real_,\n    cache_prediction_difference=NA_real_,cache_p_difference=NA_real_,\n    segment_check_max_k_difference=NA_real_,segment_check_max_p_difference=NA_real_,\n    warnings=\'\',error=\'\',fit_seconds=NA_real_)\n  result <- tryCatch(capture_conditions({\n    m0 <- fit_response(G0,y); m1 <- fit_response(G1,y)\n    full <- as.numeric(X1 %*% coef(m1)); null <- as.numeric(X0 %*% coef(m0))\n    stopifnot(max(abs(full-fitted(m1)))<1e-8,max(abs(null-fitted(m0)))<1e-8)\n    residual <- y-full\n    trajectory <- as.numeric(X_trajectory %*% coef(m1)[trajectory_columns])\n    trajectory <- trajectory-mean(trajectory)+mean(y)\n    rss0 <- sum((y-null)^2); rss1 <- sum(residual^2)\n    stopifnot(abs(rss0-m0$deviance)<1e-7*max(1,rss0),\n              abs(rss1-m1$deviance)<1e-7*max(1,rss1),\n              length(m1$sp)==10L,all(is.finite(trajectory)),all(is.finite(full)))\n    table <- anova(m0,m1,test=\'F\')\n    p_column <- grep(\'^Pr\\\\(\',names(table),value=TRUE)\n    stopifnot(nrow(table)==2L,length(p_column)==1L)\n    p <- as.numeric(table[2,p_column]); f_statistic <- as.numeric(table[2,\'F\'])\n    convergence <- c(converged(m0),converged(m1))\n    ranks <- c(m0$rank==length(coef(m0)),m1$rank==length(coef(m1)))\n    stopifnot(all(convergence),all(ranks),is.finite(p),p>=0,p<=1,\n              is.finite(f_statistic),table[2,\'Df\']>0,table[2,\'Resid. Df\']>0)\n    direct_prediction <- direct_p <- NA_real_\n    if(j %in% VALIDATION_INDICES) {\n      z <- d; z$y <- y\n      fresh0 <- gam(f0,data=z,family=gaussian(),method=\'ML\',gamma=1,select=FALSE,na.action=na.fail)\n      fresh1 <- gam(f1,data=z,family=gaussian(),method=\'ML\',gamma=1,select=FALSE,na.action=na.fail)\n      direct_prediction <- max(abs(fitted(fresh0)-null),abs(fitted(fresh1)-full))\n      direct_table <- anova(fresh0,fresh1,test=\'F\')\n      direct_p <- abs(p-as.numeric(direct_table[2,p_column]))\n      equivalence <- list(\n        A=abs(diff(range(as.numeric(X_trajectory %*% coef(fresh1)[trajectory_columns])))-diff(range(trajectory))),\n        RSS0=abs(rss0-fresh0$deviance),RSS1=abs(rss1-fresh1$deviance),\n        EDF=max(abs(m1$edf-fresh1$edf)),EDF1=max(abs(m1$edf1-fresh1$edf1)),\n        EDF2=if(!is.null(m1$edf2))max(abs(m1$edf2-fresh1$edf2))else 0,\n        SP_relative=max(abs(m1$sp-fresh1$sp)/(1+abs(fresh1$sp))),\n        F_relative=abs(f_statistic-as.numeric(direct_table[2,\'F\']))/(1+abs(as.numeric(direct_table[2,\'F\']))),\n        DF0=abs(table[1,\'Resid. Df\']-direct_table[1,\'Resid. Df\']),\n        DF1=abs(table[2,\'Resid. Df\']-direct_table[2,\'Resid. Df\']),\n        AIC0=abs(AIC(m0)-AIC(fresh0)),AIC1=abs(AIC(m1)-AIC(fresh1)),\n        ML_objective=abs(as.numeric(m1$gcv.ubre)-as.numeric(fresh1$gcv.ubre)))\n      cat(paste(\'QR_EQUIVALENCE\',name,paste(names(equivalence),unlist(equivalence),sep=\'=\',collapse=\';\')), \'\\n\')\n      stopifnot(equivalence$A<1e-7,equivalence$RSS0<1e-7,equivalence$RSS1<1e-7,\n        equivalence$EDF<1e-4,equivalence$EDF1<1e-4,equivalence$EDF2<1e-4,\n        equivalence$SP_relative<1e-3,equivalence$F_relative<1e-5,\n        equivalence$DF0<1e-4,equivalence$DF1<1e-4,\n        equivalence$AIC0<1e-4,equivalence$AIC1<1e-4,equivalence$ML_objective<1e-5)\n      fresh_terms <- predict(fresh1,type=\'terms\')\n      relevant <- colnames(fresh_terms)==\'seg\' | startsWith(colnames(fresh_terms),\'s(u\')\n      fresh_trajectory <- rowSums(fresh_terms[,relevant,drop=FALSE])\n      fresh_trajectory <- fresh_trajectory-mean(fresh_trajectory)+mean(y)\n      direct_prediction <- max(direct_prediction,abs(fresh_trajectory-trajectory))\n      stopifnot(direct_prediction<VALIDATION_TOL,direct_p<1e-8)\n    }\n    local_k_rows <- local_residual_rows <- local_validation_rows <- list()\n    for(si in seq_along(G1$smooth)) {\n      sid <- smooth_segments[si]; rows <- segment_rows[[sid]]\n      edf <- sum(m1$edf[smooth_columns[[si]]]); k_prime <- length(smooth_columns[[si]])\n      r <- residual[rows]; u <- d$u[rows]; seed <- SEED+j+as.integer(sid)\n      diagnostic <- segment_check(u,r,seed)\n      k_row <- data.frame(regulon=name,index=j,mode=MODE,K=K,seg=sid,\n        term=G1$smooth[[si]]$label,n_segment=length(rows),n_checked=diagnostic$checked_cells,\n        unique_u=length(unique(u)),k_prime=k_prime,edf=edf,edf_ratio=edf/k_prime,\n        smoothing_parameter=as.numeric(m1$sp[si]),k_index=diagnostic$k_index,\n        k_index_all_cells=adjacent_index(u,r),p_value=diagnostic$p_value,\n        permutations=K_CHECK_REPS,seed=seed,status=diagnostic$status,\n        tied_cells=diagnostic$tied_cells,tied_groups=diagnostic$tied_groups,\n        diagnostic_method=\'Exact expected adjacent SSE over within-t ties; residual permutation\')\n      local_k_rows <- add_frame(local_k_rows,k_row)\n      local_residual_rows <- add_frame(local_residual_rows,data.frame(regulon=name,index=j,\n        mode=MODE,K=K,seg=sid,cells=length(rows),residual_mean=mean(r),residual_sd=sd(r),\n        residual_RMS=sqrt(mean(r^2)),residual_min=min(r),residual_max=max(r),\n        residual_depth_rho=rho(r,d$log_detected[rows]),adjacent_k_index=adjacent_index(u,r)))\n      if(j %in% VALIDATION_INDICES && diagnostic$status==\'ok\' && !anyDuplicated(u)) {\n        check <- validate_segment_check(rows,r,seed)\n        local_validation_rows <- add_frame(local_validation_rows,data.frame(regulon=name,index=j,\n          mode=MODE,K=K,seg=sid,term=G1$smooth[[si]]$label,\n          check=\'single_segment_mgcv_kcheck_with_identical_joint_residuals\',\n          k_index_difference=check$k_index_difference,\n          permutation_p_difference=check$permutation_p_difference,\n          residual_injection_error=check$residual_injection_error))\n      }\n    }\n    list(m0=m0,m1=m1,trajectory=trajectory,full=full,residual=residual,rss0=rss0,rss1=rss1,\n      p=p,F=f_statistic,table=table,convergence=convergence,ranks=ranks,\n      direct_prediction=direct_prediction,direct_p=direct_p,\n      k=do.call(rbind,local_k_rows),segment_residual=do.call(rbind,local_residual_rows),\n      validation=if(length(local_validation_rows))do.call(rbind,local_validation_rows)else NULL)\n  }),error=function(e)e)\n  if(inherits(result,\'error\')) {\n    row$error <- conditionMessage(result)\n  } else {\n    value <- result$value\n    row$status <- \'ok\'; row$p_value <- value$p; row$A <- diff(range(value$trajectory))\n    row$RSS0 <- value$rss0; row$RSS1 <- value$rss1\n    row$df0 <- value$m0$df.residual; row$df1 <- value$m1$df.residual\n    row$test_df0 <- value$table[1,\'Resid. Df\']; row$test_df1 <- value$table[2,\'Resid. Df\']\n    row$test_delta_df <- value$table[2,\'Df\']; row$F_statistic <- value$F\n    row$partial_R2 <- (value$rss0-value$rss1)/value$rss0\n    row$AIC_null <- AIC(value$m0); row$AIC_full <- AIC(value$m1)\n    row$converged0 <- value$convergence[1]; row$converged1 <- value$convergence[2]\n    row$full_rank0 <- value$ranks[1]; row$full_rank1 <- value$ranks[2]\n    row$rank0 <- value$m0$rank; row$rank1 <- value$m1$rank\n    row$coefficients0 <- length(coef(value$m0)); row$coefficients1 <- length(coef(value$m1))\n    row$smooth_parameters <- length(value$m1$sp)\n    row$residual_mean <- mean(value$residual); row$residual_sd <- sd(value$residual)\n    row$residual_depth_rho <- rho(value$residual,d$log_detected)\n    row$k_check_terms <- nrow(value$k)\n    row$minimum_k_index <- min(value$k$k_index,na.rm=TRUE)\n    row$minimum_k_p <- min(value$k$p_value,na.rm=TRUE)\n    row$maximum_edf_ratio <- max(value$k$edf_ratio)\n    row$cache_prediction_difference <- value$direct_prediction; row$cache_p_difference <- value$direct_p\n    if(!is.null(value$validation)) {\n      row$segment_check_max_k_difference <- max(value$validation$k_index_difference)\n      row$segment_check_max_p_difference <- max(value$validation$permutation_p_difference)\n      validation_rows <- add_frame(validation_rows,value$validation)\n    }\n    trajectory_values[,ii] <- value$trajectory; full_values[,ii] <- value$full\n    residual_values[,ii] <- value$residual\n    k_rows <- add_frame(k_rows,value$k)\n    segment_residual_rows <- add_frame(segment_residual_rows,value$segment_residual)\n    for(study in levels(d$study)) {\n      hit <- which(d$study==study); r <- value$residual[hit]\n      source_residual_rows <- add_frame(source_residual_rows,data.frame(regulon=name,index=j,\n        mode=MODE,K=K,study=study,cells=length(hit),residual_mean=mean(r),residual_sd=sd(r),\n        residual_RMS=sqrt(mean(r^2)),residual_depth_rho=rho(r,d$log_detected[hit])))\n    }\n    row$warnings <- paste(result$warnings,collapse=\' | \')\n  }\n  row$fit_seconds <- proc.time()[[\'elapsed\']]-tick\n  summary_rows <- add_frame(summary_rows,row)\n  if(ii%%10L==0L || ii==length(INDICES)) cat(sprintf(\'%s K%d: %d/%d; %s; %.1fs\\n\',\n    MODE,K,ii,length(INDICES),row$status,proc.time()[[\'elapsed\']]-started))\n}\n\n\n#5. =========== Shard evidence; complete-family correction belongs to Python\nwrite_rows <- function(rows,suffix) {\n  if(!length(rows)) return(invisible(NULL))\n  write.table(do.call(rbind,rows),paste0(PREFIX,suffix),sep=\'\\t\',quote=FALSE,row.names=FALSE,na=\'NA\')\n}\nsave_matrix <- function(matrix,suffix) {\n  connection <- file(paste0(PREFIX,suffix),\'wb\')\n  tryCatch(writeBin(as.numeric(matrix),connection,size=8,endian=\'little\'),finally=close(connection))\n}\nwrite_rows(summary_rows,\'.stats.tsv\')\nwrite_rows(k_rows,\'.kcheck.tsv\')\nwrite_rows(segment_residual_rows,\'.segment_residual.tsv\')\nwrite_rows(source_residual_rows,\'.source_residual.tsv\')\nwrite_rows(validation_rows,\'.validation.tsv\')\nsave_matrix(trajectory_values,\'.trajectory.bin\')\nsave_matrix(full_values,\'.fullpred.bin\')\nsave_matrix(residual_values,\'.residuals.bin\')\nmetadata_rows <- rbind(metadata_rows,data.frame(metric=c(\'elapsed_seconds\',\'indices\',\'diagnostic_scope\'),\n  value=c(proc.time()[[\'elapsed\']]-started,paste(INDICES,collapse=\',\'),\'Every fitted regulon, all ten biological native segments; no candidate or diagnostic sampling\')))\nwrite.table(metadata_rows,paste0(PREFIX,\'.metadata.tsv\'),sep=\'\\t\',quote=FALSE,row.names=FALSE,na=\'NA\')\n\n\'\'\'\n\n\n#2. =========== HDF5 frames, multiplicity and native-projection preparation\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as stream:\n        for block in iter(lambda:stream.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef strings(group,key,values):\n    group.create_dataset(key,data=np.asarray(values,object),dtype=h5py.string_dtype(),compression=\'gzip\')\n\n\ndef write_frame(group,frame):\n    frame=frame.rename(columns={\'index\':\'regulon_index\'})\n    strings(group,\'index\',frame.index.astype(str));strings(group,\'columns\',frame.columns)\n    for col in frame:\n        values=frame[col].to_numpy()\n        if values.dtype.kind in \'OSU\':strings(group,col,frame[col].fillna(\'\').astype(str))\n        else:group.create_dataset(col,data=values,compression=\'gzip\')\n\n\ndef read_frame(group):\n    return pd.DataFrame({c:group[c].asstr()[()] if group[c].dtype.kind in \'OSU\' else group[c][()]\n        for c in group[\'columns\'].asstr()[()]},index=group[\'index\'].asstr()[()])\n\n\ndef write_json(group,key,value):\n    def finite(x):\n        if isinstance(x,dict):return {str(k):finite(v) for k,v in x.items()}\n        if isinstance(x,(list,tuple,np.ndarray)):return [finite(v) for v in x]\n        if isinstance(x,np.generic):return finite(x.item())\n        if isinstance(x,float) and not np.isfinite(x):return None\n        return x\n    group.create_dataset(key,data=json.dumps(finite(value),ensure_ascii=False,allow_nan=False),dtype=h5py.string_dtype())\n\n\ndef adjusted_pvalues(values):\n    p=np.clip(np.nan_to_num(np.asarray(values,float),nan=1,posinf=1,neginf=1),0,1)\n    order=np.argsort(p,kind=\'stable\');q=np.empty(len(p))\n    q[order]=np.minimum(1,np.minimum.accumulate((p[order]*len(p)/np.arange(1,len(p)+1))[::-1])[::-1])\n    return np.minimum(1,p*len(p)),q\n\n\ndef prepare():\n    with h5py.File(INPUT,\'r\') as h:\n        cells=pd.Index(h[\'cells\'].asstr()[()],name=\'cell_id\')\n        names=pd.Index(h[\'regulons\'].asstr()[()],name=\'regulon\')\n        values=h[\'auc\'][()]\n        old={k:h[\'obs\'][k].asstr()[()] if h[\'obs\'][k].dtype.kind in \'OSU\' else h[\'obs\'][k][()]\n             for k in (\'study\',\'library\',\'log_detected\',\'detected_genes\')}\n    assert len(cells)==7355 and len(names)==217 and values.shape==(7355,217)\n    assert cells.is_unique and names.is_unique and np.isfinite(values).all()\n    with open(TREE,\'rb\') as stream:tree=pickle.load(stream)\n    native_obs=tree[\'obs\'];take=native_obs.index.get_indexer(cells)\n    assert len(native_obs)==len(cells) and np.all(take>=0) and len(np.unique(take))==7355\n    native=native_obs.reindex(cells)\n    np.testing.assert_array_equal(native.study_id.astype(str),np.asarray(old[\'study\']).astype(str))\n    qualified_library=native.study_id.astype(str)+\'__\'+native.library_id.astype(str)\n    np.testing.assert_array_equal(qualified_library,np.asarray(old[\'library\']).astype(str))\n    projection=tree[\'projection\']\n    edge_id=np.asarray(projection[\'edge_id\'])[take]\n    t=np.asarray(projection[\'t\'],float)[take]\n    segment_id=np.full(7355,-1,dtype=int)\n    for seg in tree[\'segments\']:\n        hit=np.isin(edge_id,np.asarray(seg[\'edge_ids\']))\n        assert np.all(segment_id[hit]==-1)\n        segment_id[hit]=int(seg[\'id\'])\n    assert np.all(segment_id>=0)\n    connector=np.array([next(s for s in tree[\'segments\'] if s[\'id\']==sid)[\'role\']==\'connector\' for sid in segment_id])\n    assert connector.sum()==7 and set(np.unique(segment_id[~connector]))==set(SEGMENT_IDS)\n    all_obs=pd.DataFrame(dict(t=t,seg=segment_id,stage=native.stage_standardized.astype(str),\n        type=native.state.astype(str),study=native.study_id.astype(str),library=qualified_library,\n        native_library_id=native.library_id.astype(str),\n        log_detected=np.asarray(old[\'log_detected\'],float),detected_genes=np.asarray(old[\'detected_genes\']),\n        projection_component=np.asarray(tree[\'projection_component\'])[take],\n        native_edge_id=edge_id,native_alpha=np.asarray(projection[\'alpha\'])[take],\n        included_in_association=~connector),index=cells)\n    obs=all_obs.loc[~connector].copy();auc=pd.DataFrame(values[~connector],index=obs.index,columns=names)\n    assert len(obs)==7348\n    segments=[]\n    for raw in tree[\'segments\']:\n        if int(raw[\'id\']) not in SEGMENT_IDS:continue\n        rows=obs[obs.seg.eq(raw[\'id\'])]\n        assert len(rows)>=20 and rows.t.nunique()>=K_EXTENSION\n        segments.append(dict(id=int(raw[\'id\']),start=int(raw[\'start\']),end=int(raw[\'end\']),\n            role=raw[\'role\'],label=raw[\'label\'],n_cells=len(rows),n_unique_t=rows.t.nunique(),\n            native_t_start=float(projection[\'root_distance\'][raw[\'start\']]),\n            native_t_end=float(projection[\'root_distance\'][raw[\'end\']]),\n            observed_t_lower=float(rows.t.min()),observed_t_upper=float(rows.t.max()),\n            source_counts=rows.study.value_counts().to_dict(),stage_counts=rows.stage.value_counts().to_dict(),\n            edge_ids=list(map(int,raw[\'edge_ids\']))))\n    # Set this storage path independently before running. cells.csv.\n    obs.to_csv(ROOT,index_label=\'cell_id\',float_format=\'%.17g\')\n    # Set this storage path independently before running. auc.tsv.gz.\n    auc.to_csv(ROOT,sep=\'\\t\',index_label=\'cell_id\',float_format=\'%.17g\',compression={\'method\':\'gzip\',\'mtime\':0})\n    metadata=dict(tree_file=str(TREE),tree_sha256=sha(TREE),validation_sha256=sha(VALIDATION),\n        input_auc_content_sha256=hashlib.sha256(np.asarray(values,dtype=\'<f8\').tobytes()).hexdigest(),\n        all_cells=7355,association_cells=7348,connector_cells=7,\n        connector_policy=\'Archive every original cell and native projection; seven component-connection cells have no biological-segment smooth and are excluded only from association\',\n        input_annotation_policy=\'Exact original cell IDs and labels; new segmentation from native edge membership\',\n        library_identifier_policy=\'Study-qualified original library IDs (study_id + __ + native library_id), independently checked against the original AUCell metadata\',\n        tree_provenance=tree[\'provenance\'],tree_build_validation=tree[\'build_validation\'],\n        library_scope=\'Libraries/plates are not assumed to be independent animals; every library contains one stage\',\n        early_scope=\'Candidate early Y from registered components; age-source-library confounding remains; association is conditional on selected tree, not lineage tracing\',\n        pseudotime=\'Native tree projection t; affine within-native-segment conditioning only; publication display coordinates/log1p are never used for models\')\n    return all_obs,obs,auc,values,names,segments,tree,take,metadata\n\n\n#3. =========== Fingerprinted parallel model families\n\ndef run_shard(mode,k,part,indices):\n    folder=WORK/f\'{mode}_k{k}\';folder.mkdir(parents=True,exist_ok=True)\n    prefix=folder/f\'part{part}\'\n    sources={n:sha(WORK/n) for n in (\'cells.csv\',\'auc.tsv.gz\',\'associate_global_new.R\')}\n    fingerprint=dict(sources=sources,mode=mode,K=k,indices=indices.tolist(),seed=SEED)\n    marker=Path(str(prefix)+\'.complete.json\')\n    if marker.exists() and json.loads(marker.read_text(encoding=\'utf-8\'))==fingerprint:return prefix\n    # Set this storage path independently before running. associate_global_new.R.\n    cmd=[str(RSCRIPT),\'--vanilla\',str(ROOT),str(ROOT),str(ROOT),\n         str(prefix),mode,str(k),\',\'.join(map(str,indices))]\n    with open(str(prefix)+\'.log\',\'w\',encoding=\'utf-8\') as log:\n        result=subprocess.run(cmd,cwd=ROOT,env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)\n    if result.returncode:raise RuntimeError(Path(str(prefix)+\'.log\').read_text(encoding=\'utf-8\',errors=\'replace\')[-5000:])\n    assert {n:sha(WORK/n) for n in sources}==sources\n    for suffix in (\'.stats.tsv\',\'.kcheck.tsv\',\'.trajectory.bin\',\'.fullpred.bin\',\'.residuals.bin\',\'.metadata.tsv\'):\n        assert Path(str(prefix)+suffix).is_file(),str(prefix)+suffix\n    marker.write_text(json.dumps(fingerprint,sort_keys=True),encoding=\'utf-8\')\n    return prefix\n\n\ndef independent_tie_indices(obs,residuals,names,diagnostics):\n    maximum=0.\n    for sid,frame in obs.groupby(\'seg\',sort=True):\n        indices=obs.index.get_indexer(frame.index);r=residuals[indices]\n        _,inverse,counts=np.unique(frame.t.to_numpy(),return_inverse=True,return_counts=True)\n        sums=np.zeros((len(counts),len(names)));squares=sums.copy()\n        np.add.at(sums,inverse,r);np.add.at(squares,inverse,r*r)\n        mu=sums/counts[:,None];ss=np.maximum(0,squares-sums*sums/counts[:,None]);vv=ss/counts[:,None]\n        sse=2*ss.sum(axis=0)+(vv[:-1]+vv[1:]+np.diff(mu,axis=0)**2).sum(axis=0)\n        expected=sse/(2*(len(frame)-1)*np.mean(r*r,axis=0))\n        actual=diagnostics[diagnostics.seg.eq(sid)].set_index(\'regulon\').loc[names,\'k_index\'].to_numpy()\n        good=np.isfinite(expected)&np.isfinite(actual)\n        difference=float(np.max(np.abs(expected[good]-actual[good])))\n        assert difference<1e-10,(sid,difference)\n        maximum=max(maximum,difference)\n    return maximum\n\n\ndef collect(group,prefixes,shards,obs,auc,names):\n    stats=pd.concat([pd.read_csv(str(p)+\'.stats.tsv\',sep=\'\\t\',keep_default_na=False,na_values=[\'NA\']) for p in prefixes]).set_index(\'regulon\').loc[names]\n    assert len(stats)==217 and stats.index.is_unique\n    stats[\'p_value\']=stats.p_value.where(stats.status.eq(\'ok\'),1).fillna(1)\n    stats[\'p_Bonferroni\'],stats[\'q_BH\']=adjusted_pvalues(stats.p_value)\n    np.testing.assert_allclose(stats.q_BH,false_discovery_control(stats.p_value.to_numpy(),method=\'bh\'),atol=1e-15,rtol=1e-14)\n    stats[\'primary_threshold_pass\']=stats.status.eq(\'ok\')&stats.q_BH.lt(1e-4)&stats.A.gt(.025)\n    diagnostics=pd.concat([pd.read_csv(str(p)+\'.kcheck.tsv\',sep=\'\\t\') for p in prefixes],ignore_index=True)\n    valid=stats.status.eq(\'ok\')\n    assert len(diagnostics)==int(valid.sum())*10\n    assert diagnostics.groupby(\'regulon\').seg.nunique().eq(10).all()\n    diagnostics[\'basis_flag\']=diagnostics.status.eq(\'ok\')&diagnostics.p_value.lt(.01)&diagnostics.edf_ratio.gt(.9)\n    diagnostics[\'structure_flag\']=diagnostics.status.eq(\'ok\')&diagnostics.p_value.lt(.01)&diagnostics.k_index.lt(.90)\n    diagnostics[\'diagnostic_q_BH\']=adjusted_pvalues(diagnostics.p_value)[1]\n    counts=diagnostics.groupby(\'regulon\').agg(basis_flags=(\'basis_flag\',\'sum\'),structure_flags=(\'structure_flag\',\'sum\'),diagnostic_terms=(\'seg\',\'nunique\'))\n    stats=stats.join(counts)\n    arrays={}\n    for suffix,key in ((\'.trajectory.bin\',\'trajectory\'),(\'.fullpred.bin\',\'full_prediction\'),(\'.residuals.bin\',\'residuals\')):\n        array=np.full((len(obs),217),np.nan)\n        for prefix,indices in zip(prefixes,shards):\n            values=np.fromfile(str(prefix)+suffix,dtype=\'<f8\').reshape((len(obs),len(indices)),order=\'F\')\n            array[:,indices-1]=values\n        assert np.isfinite(array[:,valid]).all()\n        group.create_dataset(key,data=array,compression=\'gzip\',compression_opts=4)\n        arrays[key]=array\n    np.testing.assert_allclose(auc.to_numpy()[:,valid]-arrays[\'full_prediction\'][:,valid],arrays[\'residuals\'][:,valid],atol=1e-13)\n    np.testing.assert_allclose((arrays[\'residuals\'][:,valid]**2).sum(0),stats.loc[valid,\'RSS1\'],atol=1e-11,rtol=1e-9)\n    np.testing.assert_allclose(np.ptp(arrays[\'trajectory\'][:,valid],axis=0),stats.loc[valid,\'A\'],atol=1e-12,rtol=1e-12)\n    expected_F=((stats.RSS0-stats.RSS1)/stats.test_delta_df)/(stats.RSS1/stats.df1)\n    np.testing.assert_allclose(expected_F[valid],stats.loc[valid,\'F_statistic\'],atol=1e-8,rtol=1e-9)\n    expected_p=f_distribution.sf(stats.F_statistic,stats.test_delta_df,stats.test_df1)\n    np.testing.assert_allclose(expected_p[valid],stats.loc[valid,\'p_value\'],atol=1e-10,rtol=1e-6)\n    tie_difference=independent_tie_indices(obs,arrays[\'residuals\'],names,diagnostics)\n    direct=stats.cache_prediction_difference.dropna()\n    assert len(direct)==3 and direct.max()<1e-7\n    equivalence=[];runtime=[]\n    for i,prefix in enumerate(prefixes):\n        meta=pd.read_csv(str(prefix)+\'.metadata.tsv\',sep=\'\\t\');meta[\'shard\']=i;runtime.append(meta)\n        for line in Path(str(prefix)+\'.log\').read_text(encoding=\'utf-8\',errors=\'replace\').splitlines():\n            if line.startswith(\'QR_EQUIVALENCE \'):\n                _,name,values=line.split(\' \',2)\n                equivalence.append(dict(regulon=name,**{k:float(v) for k,v in (item.split(\'=\') for item in values.strip().split(\';\'))}))\n    assert len(equivalence)==3\n    write_frame(group.create_group(\'stats\'),stats)\n    write_frame(group.create_group(\'diagnostics\'),diagnostics)\n    write_frame(group.create_group(\'runtime_metadata\'),pd.concat(runtime,ignore_index=True))\n    write_frame(group.create_group(\'fresh_fit_validation\'),pd.DataFrame(equivalence))\n    official=[pd.read_csv(str(p)+\'.validation.tsv\',sep=\'\\t\') for p in prefixes if Path(str(p)+\'.validation.tsv\').exists()]\n    if official:write_frame(group.create_group(\'unique_t_official_diagnostic_validation\'),pd.concat(official,ignore_index=True))\n    group.attrs[\'independent_numpy_tie_index_max_difference\']=tie_difference\n    group.attrs[\'primary_selected_count\']=int(stats.primary_threshold_pass.sum())\n    group.attrs[\'diagnostic_terms\']=len(diagnostics)\n    return stats,diagnostics\n\n\n#4. =========== New-tree evidence, uniform basis selection and final artifact\n\ndef run():\n    WORK.mkdir(parents=True,exist_ok=True)\n    # Set this storage path independently before running. associate_global_new.R.\n    (ROOT).write_text(R_SOURCE,encoding=\'utf-8\',newline=\'\\n\')\n    all_obs,obs,auc,all_values,names,segments,tree,take,metadata=prepare()\n    shards=[np.arange(1,218)[i::WORKERS] for i in range(WORKERS)]\n    # Set this storage path independently before running. Fig4A_global_new.h5.\n    started=time.monotonic();temporary=ROOT\n    results={};trigger=[];chosen=K_INITIAL\n    with h5py.File(temporary,\'w\') as h:\n        h.attrs[\'schema\']=\'Fig4A_global_new_v1\';h.attrs[\'seed\']=SEED\n        h.attrs[\'code_sha256\']=sha(Path(__file__))\n        # Set this storage path independently before running. associate_global_new.R.\n        h.attrs[\'R_engine_sha256\']=sha(ROOT)\n        h.attrs[\'input_auc_content_sha256\']=metadata[\'input_auc_content_sha256\']\n        strings(h,\'regulons\',names);strings(h,\'cells\',obs.index);strings(h,\'all_cells\',all_obs.index)\n        h.create_dataset(\'analysis_mask\',data=all_obs.included_in_association.to_numpy())\n        h.create_dataset(\'observed_auc\',data=auc.to_numpy(),compression=\'gzip\')\n        h.create_dataset(\'all_observed_auc\',data=all_values,compression=\'gzip\')\n        write_frame(h.create_group(\'obs\'),obs);write_frame(h.create_group(\'all_obs\'),all_obs)\n        write_json(h,\'segments_json\',segments);write_json(h,\'metadata_json\',metadata)\n        write_json(h,\'methods_json\',dict(primary=\'none constant-null GaussianML whole-tree association; BH217 q<1e-4 and A>.025\',\n            model=\'y~nuisance+seg+s(u,by=seg,bs=cr,k=K);10independent lambdas; unit weights; gamma1;ML\',\n            nuisance_modes=list(MODES),initial_K=10,extension_K=20,\n            basis_rule=\'Any tie-aware diagnostic p<.01 and EDF/kprime>.9 in any217/mode triggers uniformK20 for all217/all modes; no TF/P-specific K\',\n            residuals=\'Every217x10 biological segment; expected adjacent SSE averaged over all within-native-t orders,500 residual permutations; ordinary unique-t case reduces to1D mgcv k-index\',\n            amplitude=\'Range of trajectory-only seg+smooth fitted component centered to observed mean; primarynone equals full prediction\',\n            selection_exclusions=\'Residual structure, source/depth sensitivity and biological marker expectations are not additional primary significance gates\',\n            replication=\'Conditional cell-level exploratory association on selected expression tree; no animal-level causal inference\'))\n        native=h.create_group(\'native_tree\')\n        for key in (\'F\',\'B\'):native.create_dataset(key,data=tree[\'model\'][key],compression=\'gzip\')\n        for key,value in tree[\'projection\'].items():\n            if isinstance(value,np.ndarray):native.create_dataset(key,data=value,compression=\'gzip\' if value.ndim else None)\n            elif np.isscalar(value):native.attrs[key]=value\n        native.create_dataset(\'all_cells_take_index\',data=take)\n        write_json(native,\'projection_order_json\',dict(native_source_order=\'tree.obs.index\',all_cells_order=\'originalfrozenAUCell cell order; take_index maps source arrays to all_cells\'))\n        strings(h,\'R_source\',R_SOURCE.splitlines())\n        modes=h.create_group(\'modes\')\n        groups={mode:modes.create_group(mode) for mode in MODES}\n        for k in (K_INITIAL,K_EXTENSION):\n            any_basis=False;round_summary={}\n            for mode in MODES:\n                print(f\'START_GLOBAL_NEW {mode} K{k} all217 ten biological native segments N7348\',flush=True)\n                with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as pool:\n                    futures=[pool.submit(run_shard,mode,k,part,indices) for part,indices in enumerate(shards)]\n                    prefixes=[f.result() for f in futures]\n                group=groups[mode].create_group(f\'k{k}\');group.attrs[\'K\']=k;group.attrs[\'mode\']=mode\n                stats,diagnostics=collect(group,prefixes,shards,obs,auc,names)\n                flags=int(diagnostics.basis_flag.sum());any_basis|=flags>0\n                round_summary[mode]=dict(models_ok=int(stats.status.eq(\'ok\').sum()),primary_threshold_pass=int(stats.primary_threshold_pass.sum()),\n                    basis_flags=flags,structure_flags=int(diagnostics.structure_flag.sum()),max_fresh_prediction_difference=float(stats.cache_prediction_difference.max()))\n                print(json.dumps(dict(K=k,mode=mode,**round_summary[mode])),flush=True);h.flush()\n            results[f\'k{k}\']=round_summary\n            if k==K_INITIAL:\n                trigger=round_summary\n                if not any_basis:break\n                chosen=K_EXTENSION\n        h[\'primary\']=modes[\'none\'][f\'k{chosen}\']\n        h.attrs[\'chosen_K\']=chosen;h.attrs[\'primary_mode\']=\'none\'\n        h.attrs[\'elapsed_seconds\']=time.monotonic()-started\n        write_json(h,\'summary_json\',dict(chosen_K=chosen,rounds=results,initial_K_triggers=trigger))\n        assert sha(TREE)==metadata[\'tree_sha256\'] and sha(VALIDATION)==metadata[\'validation_sha256\']\n    os.replace(temporary,OUTPUT)\n    finalize_numerical_qa()\n    print(\'GLOBAL_NEW_COMPLETE \'+json.dumps(dict(output=str(OUTPUT),chosen_K=chosen,summary=results,seconds=time.monotonic()-started)),flush=True)\n\n\ndef finalize_numerical_qa():\n    """Archive independent null checks and interpretability beside raw diagnostics.\n\n    This adds no selection gate and never changes fitted values, association P,\n    raw k-index/P/flags, K, or candidate membership. Numerical residuals on\n    locally all-zero signatures cannot be interpreted as biological lack of fit.\n    """\n    numerical_floor=1e-12\n    summaries=[]\n    def replace_frame(group,key,frame):\n        if key in group:del group[key]\n        write_frame(group.create_group(key),frame)\n    with h5py.File(OUTPUT,\'r+\') as h:\n        obs=read_frame(h[\'obs\']);names=pd.Index(h[\'regulons\'].asstr()[()])\n        Y=h[\'observed_auc\'][()]\n        intercept=np.ones((len(obs),1))\n        depth=obs.log_detected.to_numpy(float)[:,None]\n        study=pd.get_dummies(obs.study.astype(str),drop_first=True).to_numpy(float)\n        library=pd.get_dummies(obs.library.astype(str),drop_first=True).to_numpy(float)\n        designs={\'none\':intercept,\'depth\':np.c_[intercept,depth],\n            \'study_only\':np.c_[intercept,study],\'study\':np.c_[intercept,study,depth],\n            \'library\':np.c_[intercept,library,depth]}\n        for mode in MODES:\n            X=designs[mode];rank=np.linalg.matrix_rank(X)\n            assert rank==X.shape[1]\n            Q,_=np.linalg.qr(X,mode=\'reduced\')\n            null_residuals=Y-Q@(Q.T@Y)\n            null_rss=np.sum(null_residuals*null_residuals,axis=0)\n            for label,group in h[\'modes\'][mode].items():\n                if not label.startswith(\'k\'):continue\n                stats=read_frame(group[\'stats\']);diagnostics=read_frame(group[\'diagnostics\'])\n                np.testing.assert_array_equal(stats.index,names)\n                residuals=group[\'residuals\'][()]\n                rms={sid:np.sqrt(np.mean(residuals[np.where(obs.seg.eq(sid))[0]]**2,axis=0))\n                     for sid in SEGMENT_IDS}\n                diagnostics[\'residual_RMS\']=[rms[int(row.seg)][names.get_loc(row.regulon)]\n                    for row in diagnostics.itertuples(index=False)]\n                diagnostics[\'numerical_zero_residual\']=diagnostics.residual_RMS.le(numerical_floor)\n                diagnostics[\'diagnostic_informative\']=diagnostics.status.eq(\'ok\')&~diagnostics.numerical_zero_residual\n                diagnostics[\'informative_structure_flag\']=diagnostics.structure_flag&diagnostics.diagnostic_informative\n                diagnostics[\'informative_basis_flag\']=diagnostics.basis_flag&diagnostics.diagnostic_informative\n                # mgcv\'s raw mean(null<observed) is kept; report finite-permutation resolution separately.\n                exceedances=np.rint(diagnostics.p_value.to_numpy(float)*diagnostics.permutations.to_numpy(float))\n                diagnostics[\'permutation_p_plus1\']=(exceedances+1)/(diagnostics.permutations.to_numpy(float)+1)\n                diagnostics[\'diagnostic_q_BH_plus1\']=adjusted_pvalues(diagnostics.permutation_p_plus1)[1]\n                counts=diagnostics.groupby(\'regulon\').agg(\n                    numerical_zero_residual_terms=(\'numerical_zero_residual\',\'sum\'),\n                    informative_structure_flags=(\'informative_structure_flag\',\'sum\'),\n                    informative_basis_flags=(\'informative_basis_flag\',\'sum\'))\n                stats=stats.drop(columns=list(counts.columns),errors=\'ignore\').join(counts)\n                valid=stats.status.eq(\'ok\').to_numpy()\n                rss_difference=float(np.max(np.abs(stats.RSS0.to_numpy()[valid]-null_rss[valid])))\n                df_difference=float(np.max(np.abs(stats.df0.to_numpy()[valid]-(len(obs)-rank))))\n                assert rss_difference<1e-8 and df_difference<1e-9\n                stats[\'independent_OLS_null_RSS_difference\']=np.abs(stats.RSS0.to_numpy()-null_rss)\n                replace_frame(group,\'stats\',stats);replace_frame(group,\'diagnostics\',diagnostics)\n                folder=WORK/f\'{mode}_{label}\'\n                for suffix,key in ((\'.segment_residual.tsv\',\'segment_residual_summary\'),(\'.source_residual.tsv\',\'source_residual_summary\')):\n                    pieces=[pd.read_csv(path,sep=\'\\t\') for path in sorted(folder.glob(\'part*\'+suffix))]\n                    assert pieces\n                    frame=pd.concat(pieces,ignore_index=True)\n                    if key==\'segment_residual_summary\':\n                        assert len(frame)==int(valid.sum())*10\n                        check=diagnostics.merge(frame[[\'regulon\',\'seg\',\'residual_RMS\']],on=[\'regulon\',\'seg\'],\n                            validate=\'one_to_one\',suffixes=(\'_numpy\',\'_R\'))\n                        np.testing.assert_allclose(check.residual_RMS_numpy,check.residual_RMS_R,atol=1e-14,rtol=1e-10)\n                    replace_frame(group,key,frame)\n                summary=dict(mode=mode,K=int(label[1:]),null_rank=int(rank),\n                    independent_null_RSS_max_difference=rss_difference,independent_null_df_max_difference=df_difference,\n                    raw_structure_flags=int(diagnostics.structure_flag.sum()),\n                    informative_structure_flags=int(diagnostics.informative_structure_flag.sum()),\n                    numerical_zero_residual_terms=int(diagnostics.numerical_zero_residual.sum()),\n                    raw_basis_flags=int(diagnostics.basis_flag.sum()),\n                    informative_basis_flags=int(diagnostics.informative_basis_flag.sum()))\n                summaries.append(summary)\n                for key,value in summary.items():group.attrs[key]=value\n        for key in (\'numerical_diagnostic_qa_json\',\'python_source\'):\n            if key in h:del h[key]\n        write_json(h,\'numerical_diagnostic_qa_json\',dict(numerical_residual_RMS_floor=numerical_floor,\n            policy=\'All raw k-index, raw permutation P, raw flags and raw BH fields are preserved; separate informative flags exclude RMS<=1e-12 numerical residuals only. No association or membership change.\',\n            permutation_resolution=\'Raw mgcv-compatible P=mean(null<observed); plus-one P=(count+1)/(500+1) and companion BH are separately archived. Diagnostics do not constitute a regulon discovery family.\',\n            families=summaries))\n        if \'fit_launch_code_sha256\' not in h.attrs:h.attrs[\'fit_launch_code_sha256\']=h.attrs[\'code_sha256\']\n        h.attrs[\'code_sha256\']=sha(Path(__file__))\n        h.attrs[\'qa_completed_UTC\']=time.strftime(\'%Y-%m-%d %H:%M:%S UTC\',time.gmtime())\n        # Set this storage path independently before running. cells.csv.\n        h.attrs[\'cells_input_sha256\']=sha(ROOT)\n        # Set this storage path independently before running. auc.tsv.gz.\n        h.attrs[\'auc_table_input_sha256\']=sha(ROOT)\n        strings(h,\'python_source\',Path(__file__).read_text(encoding=\'utf-8\').splitlines())\n    print(\'GLOBAL_NEW_NUMERICAL_QA \'+json.dumps(summaries),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.parse_args()\n    run()\n', 'early_fork': '"""Early CVG/VGN-candidate versus SGN-candidate fork on the registered Fig3 tree.\n\nAll 217 frozen AUCell signatures, common native-v support, Gaussian ML GAMs.\nThe unadjusted literature-style fork test is primary; source, depth and\ncategorical-stage conditioning are sensitivity analyses, not gene filters.\n"""\n\n# Configure each file/directory placeholder independently; no storage addresses are shipped.\nfrom pathlib import Path\nROOT = Path(".")\n\n#1. =========== Shared configuration\nfrom pathlib import Path\nimport os\nimport sys\nimport json\nimport pickle\nimport hashlib\nimport subprocess\nimport concurrent.futures\nimport time\n\nsys.dont_write_bytecode = True\n# Set this storage path independently before running.\nROOT = ROOT\n# Set this storage path independently before running.\nDESTINATION = ROOT\n# Set this storage path independently before running.\nWORK = ROOT\n# Set this storage path independently before running. Fig4A.h5.\nINPUT = ROOT\n# Set this storage path independently before running. tree_results-1.pkl. Accepted tree input produced by the released Figure 3 analysis.\nTREE = ROOT\n# Set this storage path independently before running. Fig4A_early_fork.h5.\nOUTPUT = ROOT\n# Set this storage path independently before running. Rscript.exe.\nRSCRIPT = ROOT\nSEED = 42\nK = 10\nMODES = ("none", "depth", "source", "source_depth", "stage", "stage_depth")\nWORKERS = 2\nfor key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n    os.environ[key] = "1"\n\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import f as f_distribution\n\nR_SOURCE = \'\\n#1. =========== Fixed Gaussian ML models and diagnostic settings\\nsuppressPackageStartupMessages(library(mgcv))\\nargs <- commandArgs(trailingOnly=TRUE)\\nstopifnot(length(args)==8L)\\nCELLS<-args[1]; AUC<-args[2]; PREFIX<-args[3]; KIND<-args[4]\\nMODE<-args[5]; K<-as.integer(args[6]); INDICES<-as.integer(strsplit(args[7],\\\',\\\',fixed=TRUE)[[1]])\\nVALIDATE<-identical(args[8],\\\'1\\\')\\nSEED<-42L; REPS<-500L; SUBSAMPLE<-2000L; GRID_N<-51L\\noptions(stringsAsFactors=FALSE,digits=17)\\nd<-read.csv(CELLS,row.names=1,check.names=FALSE)\\ncon<-gzfile(AUC,\\\'rt\\\'); Y<-tryCatch(as.matrix(read.delim(con,row.names=1,check.names=FALSE)),finally=close(con))\\nstorage.mode(Y)<-\\\'double\\\'\\nstopifnot(identical(rownames(d),rownames(Y)),ncol(Y)==217L,all(is.finite(Y)),\\n          all(is.finite(d$v)),all(is.finite(d$log_detected)),all(Y>=0 & Y<=1))\\nd$study<-factor(d$study); d$stage<-factor(d$stage); d$branch<-factor(as.character(d$branch))\\nstopifnot(all(table(d$branch)>=20L),all(vapply(split(d$v,d$branch),function(v)length(unique(v))>=K,logical(1))))\\nsource_term<-if(nlevels(d$study)>1L)\\\'study\\\'else character()\\nterms<-c(if(MODE %in% c(\\\'source\\\',\\\'source_depth\\\'))source_term,\\n         if(MODE %in% c(\\\'stage\\\',\\\'stage_depth\\\'))\\\'stage\\\',\\n         if(MODE %in% c(\\\'depth\\\',\\\'source_depth\\\',\\\'stage_depth\\\'))\\\'log_detected\\\')\\nC<-if(length(terms))paste(terms,collapse=\\\' + \\\')else\\\'1\\\'\\nif(KIND==\\\'early\\\') {\\n  f0<-as.formula(paste(\\\'y ~\\\',C))\\n  f1<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ s(v,bs=\\\'cr\\\',k=%d)",K)))\\n} else {\\n  stopifnot(nlevels(d$branch)==2L)\\n  f0<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ s(v,bs=\\\'cr\\\',k=%d)",K)))\\n  f1<-as.formula(paste(\\\'y ~\\\',C,sprintf("+ branch + s(v,by=branch,bs=\\\'cr\\\',k=%d)",K)))\\n}\\nd$y<-Y[,1L]\\ncache_qr<-function(G) {\\n  q<-qr(G$X,tol=0,LAPACK=TRUE); rp<-q$pivot; rp[rp]<-seq_len(ncol(G$X))\\n  G$fast_qr<-q; G$fast_R<-qr.R(q)[,rp,drop=FALSE]\\n  G$fast_sp<-mgcv:::initial.spg(G$X,G$y,G$w,G$family,G$S,G$rank,G$off,\\n                              offset=G$offset,L=G$L,lsp0=G$lsp0)\\n  stopifnot(max(abs(crossprod(G$fast_R)-crossprod(G$X)))<1e-7)\\n  G\\n}\\nG0<-cache_qr(gam(f0,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))\\nG1<-cache_qr(gam(f1,data=d,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))\\nX0<-G0$X; X1<-G1$X\\nsc<-lapply(G1$smooth,function(s)s$first.para:s$last.para)\\ntc<-sort(unique(c(which(startsWith(colnames(X1),\\\'branch\\\')),unlist(sc))))\\nXT<-X1[,tc,drop=FALSE]\\ngrid<-seq(min(d$v),max(d$v),length.out=GRID_N)\\n\\n#2. =========== Exact fixed-X sufficient-statistic fit and full-cell restoration\\nfit_response<-function(G,y) {\\n  X<-G$X; N<-length(y); p<-ncol(X)\\n  init<-list(sp=G$fast_sp,scale=mean((y-mean(y))^2)/10)\\n  qy<-qr.qty(G$fast_qr,y)[seq_len(p)]; extra<-sum(y*y)-sum(qy*qy)\\n  stopifnot(extra>=-1e-10)\\n  G$X<-G$fast_R;G$y<-qy;G$w<-rep(1,p);G$n<-p;G$offset<-rep(0,p)\\n  G$n.true<-N;G$dev.extra<-G$pearson.extra<-max(0,extra);G$mf$y<-y\\n  G$fast_qr<-G$fast_R<-G$fast_sp<-NULL\\n  m<-gam(G=G,method=\\\'ML\\\',gamma=1,in.out=init)\\n  mu<-as.numeric(X%*%coef(m));r<-y-mu\\n  m$y<-m$model$y<-y;m$fitted.values<-m$linear.predictors<-mu;m$residuals<-r\\n  m$prior.weights<-m$weights<-m$working.weights<-rep(1,N);m$offset<-rep(0,N);m$z<-y\\n  m$deviance<-sum(r*r);m$null.deviance<-sum((y-mean(y))^2)\\n  m$df.residual<-N-sum(m$edf);m$df.null<-N-as.integer(G$intercept)\\n  m$aic<-m$family$aic(y,1,mu,rep(1,N),m$deviance)+2*sum(m$edf)\\n  m$hat<-rowSums((X%*%m$Vp)*X)/m$sig2\\n  m\\n}\\nconverged<-function(m) {\\n  z<-m$outer.info$conv\\n  (is.null(m$converged)||isTRUE(m$converged)) &&\\n   (is.null(z)||identical(z,0L)||identical(z,0)||grepl(\\\'full convergence\\\',paste(z,collapse=\\\' \\\'),fixed=TRUE))\\n}\\ntie_expected_sse<-function(r,group,counts) {\\n  sums<-rowsum(cbind(r,r*r),group,reorder=TRUE)\\n  means<-sums[,1]/counts;ss<-pmax(0,sums[,2]-sums[,1]^2/counts)\\n  variances<-ss/counts\\n  total<-2*sum(ss)\\n  m<-length(counts)\\n  if(m>1L)total<-total+sum(variances[-m]+variances[-1L]+diff(means)^2)\\n  total\\n}\\n# The analytic expected SSE equals enumeration over every within-t tie order.\\ntoy_r<-c(.1,-.3,.2,.6,-.1,.9);toy_group<-c(1,1,2,2,2,3)\\ntoy_permutations<-function(x)if(length(x)==1L)list(x)else unlist(lapply(seq_along(x),function(i)\\n  lapply(toy_permutations(x[-i]),function(z)c(x[i],z))),recursive=FALSE)\\ntoy_orders<-lapply(toy_permutations(c(1L,2L)),function(a)lapply(toy_permutations(c(3L,4L,5L)),function(b)c(a,b,6L)))\\ntoy_orders<-unlist(toy_orders,recursive=FALSE)\\nstopifnot(abs(tie_expected_sse(toy_r,toy_group,c(2L,3L,1L))-\\n  mean(vapply(toy_orders,function(z)sum(diff(toy_r[z])^2),numeric(1))))<1e-12)\\nk_check<-function(v,r,seed) {\\n  set.seed(seed)\\n  if(length(r)>SUBSAMPLE) {idx<-sample.int(length(r),SUBSAMPLE);v<-v[idx];r<-r[idx]}\\n  n<-length(r);den<-mean(r*r)\\n  if(n<2L||den<=0)return(list(index=NA_real_,p=NA_real_,n=n,status=\\\'zero_residual_variance\\\'))\\n  group<-match(v,sort(unique(v)));counts<-tabulate(group)\\n  observed<-tie_expected_sse(r,group,counts)/(2*(n-1L))\\n  reference<-vapply(seq_len(REPS),function(i)tie_expected_sse(r[sample.int(n,n)],group,counts)/(2*(n-1L)),numeric(1))\\n  list(index=observed/den,p=mean(reference<observed),n=n,status=\\\'ok\\\',\\n    tied_cells=sum(counts[counts>1L]),tied_groups=sum(counts>1L),\\n    old_index=mean(diff(r[order(v)])^2)/2/den)\\n}\\ndiagnose<-function(m,G,r,j,role) {\\n  if(!length(G$smooth))return(NULL)\\n  rows<-list()\\n  for(si in seq_along(G$smooth)) {\\n    sm<-G$smooth[[si]]\\n    hit<-if(is.null(sm$by.level))seq_len(nrow(d))else which(as.character(d$branch)==as.character(sm$by.level))\\n    cols<-sm$first.para:sm$last.para;edf<-sum(m$edf[cols]);kp<-length(cols)\\n    seed<-SEED+j+si+if(role==\\\'null\\\')1000L else 0L\\n    z<-k_check(d$v[hit],r[hit],seed)\\n    check_k<-check_p<-NA_real_\\n    if(VALIDATE && j==1L && z$status==\\\'ok\\\' && !anyDuplicated(d$v[hit])) {\\n      fixture<-gam(rr~s(v,bs=\\\'cr\\\',k=K),data=data.frame(v=d$v[hit],rr=r[hit]),\\n                   family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE)\\n      fixture$y<-r[hit];fixture$fitted.values<-fixture$linear.predictors<-rep(0,length(hit))\\n      stopifnot(max(abs(residuals(fixture)-r[hit]))<1e-12)\\n      set.seed(seed);official<-k.check(fixture,subsample=SUBSAMPLE,n.rep=REPS)\\n      check_k<-abs(z$index-official[1,\\\'k-index\\\']);check_p<-abs(z$p-official[1,\\\'p-value\\\'])\\n      stopifnot(check_k<1e-12,check_p==0)\\n    }\\n    rows[[si]]<-data.frame(regulon=colnames(Y)[j],regulon_index=j,model_role=role,\\n      term=sm$label,branch=if(is.null(sm$by.level))\\\'all\\\'else as.character(sm$by.level),\\n      n_cells=length(hit),n_checked=z$n,unique_v=length(unique(d$v[hit])),k_prime=kp,\\n      edf=edf,edf_ratio=edf/kp,k_index=z$index,p_value=z$p,status=z$status,\\n      basis_flag=is.finite(z$p)&&z$p<.01&&edf/kp>.9,\\n      structure_flag=is.finite(z$p)&&z$p<.01&&z$index<.90,\\n      smoothing_parameter=as.numeric(m$sp[si]),seed=seed,permutations=REPS,\\n      official_k_difference=check_k,official_p_difference=check_p)\\n    rows[[si]]$tied_cells<-z$tied_cells\\n    rows[[si]]$tied_groups<-z$tied_groups\\n    rows[[si]]$old_ordered_k_index<-z$old_index\\n    rows[[si]]$diagnostic_method<-\\\'Tie-order-invariant expected adjacent SSE; fixed exact-t groups; permuted residuals\\\'\\n  }\\n  do.call(rbind,rows)\\n}\\n\\n#3. =========== Every response, local contrast and complete residual evidence\\nN<-nrow(d);J<-length(INDICES)\\nFULL<-RESID<-TRAJ<-matrix(NA_real_,N,J)\\nCONTRAST<-CONTRAST_SE<-matrix(NA_real_,GRID_N,J)\\nstatistics<-diagnostics<-validations<-list()\\nstarted<-proc.time()[[\\\'elapsed\\\']]\\nfor(ii in seq_along(INDICES)) {\\n  j<-INDICES[ii];y<-as.numeric(Y[,j]);tick<-proc.time()[[\\\'elapsed\\\']]\\n  row<-data.frame(regulon=colnames(Y)[j],regulon_index=j,status=\\\'failed\\\',test_status=\\\'not_run\\\',\\n    p_value=1,A=NA_real_,contrast_max_abs=NA_real_,contrast_mean=NA_real_,\\n    contrast_start=NA_real_,contrast_end=NA_real_,RSS0=NA_real_,RSS1=NA_real_,\\n    df0=NA_real_,df1=NA_real_,test_df0=NA_real_,test_df1=NA_real_,test_delta_df=NA_real_,\\n    F_statistic=NA_real_,partial_R2=NA_real_,converged0=FALSE,converged1=FALSE,\\n    full_rank0=FALSE,full_rank1=FALSE,full_basis_flags=NA_integer_,full_structure_flags=NA_integer_,\\n    null_basis_flags=NA_integer_,residual_mean=NA_real_,residual_sd=NA_real_,\\n    cache_prediction_difference=NA_real_,cache_F_relative_difference=NA_real_,\\n    warnings=\\\'\\\',error=\\\'\\\',fit_seconds=NA_real_)\\n  notices<-character()\\n  value<-tryCatch(withCallingHandlers({\\n    m0<-fit_response(G0,y);m1<-fit_response(G1,y)\\n    ok<-c(converged(m0),converged(m1),m0$rank==length(coef(m0)),m1$rank==length(coef(m1)))\\n    stopifnot(all(ok))\\n    full<-as.numeric(X1%*%coef(m1));null<-as.numeric(X0%*%coef(m0));r<-y-full\\n    tr<-as.numeric(XT%*%coef(m1)[tc]);tr<-tr-mean(tr)+mean(y)\\n    rss0<-sum((y-null)^2);rss1<-sum(r*r)\\n    stopifnot(max(abs(full-fitted(m1)))<1e-9,abs(rss1-m1$deviance)<1e-9)\\n    tab<-anova(m0,m1,test=\\\'F\\\');pcol<-grep(\\\'^Pr\\\\\\\\(\\\',names(tab),value=TRUE)\\n    p<-as.numeric(tab[2,pcol]);Fv<-as.numeric(tab[2,\\\'F\\\'])\\n    test_status<-if(is.finite(p)&&tab[2,\\\'Df\\\']>0&&is.finite(Fv)&&Fv>=0)\\\'ok\\\'else\\\'nonpositive_incremental_test\\\'\\n    if(test_status!=\\\'ok\\\')p<-1\\n    nd<-d[rep(1L,GRID_N),,drop=FALSE];nd$v<-grid\\n    if(KIND==\\\'early\\\') {\\n      L<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\');L<-sweep(L,2,L[1,],\\\'-\\\')\\n    } else {\\n      nd$branch<-factor(rep(levels(d$branch)[1],GRID_N),levels=levels(d$branch))\\n      LA<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\')\\n      nd$branch<-factor(rep(levels(d$branch)[2],GRID_N),levels=levels(d$branch))\\n      LB<-predict(m1,newdata=nd,type=\\\'lpmatrix\\\');L<-LB-LA\\n    }\\n    nuisance<-setdiff(seq_len(ncol(X1)),tc)\\n    if(length(nuisance))stopifnot(max(abs(L[,nuisance,drop=FALSE]))<1e-12)\\n    delta<-as.numeric(L%*%coef(m1));se<-sqrt(pmax(0,rowSums((L%*%m1$Vp)*L)))\\n    dq<-rbind(diagnose(m1,G1,r,j,\\\'full\\\'),diagnose(m0,G0,y-null,j,\\\'null\\\'))\\n    pred_error<-F_error<-NA_real_;validation<-NULL\\n    if(VALIDATE && j %in% c(1L,109L,217L)) {\\n      dd<-d;dd$y<-y\\n      fresh0<-gam(f0,data=dd,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      fresh1<-gam(f1,data=dd,family=gaussian(),method=\\\'ML\\\',gamma=1,select=FALSE,na.action=na.fail)\\n      ft<-anova(fresh0,fresh1,test=\\\'F\\\')\\n      pred_error<-max(abs(fitted(fresh0)-null),abs(fitted(fresh1)-full))\\n      F_error<-if(test_status==\\\'ok\\\')abs(Fv-as.numeric(ft[2,\\\'F\\\']))/(1+abs(Fv))else 0\\n      edf_error<-max(abs(m1$edf-fresh1$edf),abs(m0$edf-fresh0$edf))\\n      ml_error<-max(abs(as.numeric(m0$gcv.ubre)-as.numeric(fresh0$gcv.ubre)),\\n                    abs(as.numeric(m1$gcv.ubre)-as.numeric(fresh1$gcv.ubre)))\\n      stopifnot(pred_error<1e-7,F_error<1e-5,edf_error<1e-4,ml_error<1e-5)\\n      validation<-data.frame(regulon=colnames(Y)[j],prediction_difference=pred_error,\\n        F_relative_difference=F_error,EDF_difference=edf_error,ML_objective_difference=ml_error)\\n    }\\n    list(m0=m0,m1=m1,ok=ok,full=full,r=r,tr=tr,rss0=rss0,rss1=rss1,tab=tab,p=p,F=Fv,\\n      test_status=test_status,delta=delta,se=se,dq=dq,pred_error=pred_error,F_error=F_error,validation=validation)\\n  },warning=function(w){notices<<-c(notices,conditionMessage(w));invokeRestart(\\\'muffleWarning\\\')}),error=function(e)e)\\n  if(inherits(value,\\\'error\\\'))row$error<-conditionMessage(value) else {\\n    row$status<-\\\'ok\\\';row$test_status<-value$test_status;row$p_value<-value$p\\n    row$A<-diff(range(value$tr));row$contrast_max_abs<-max(abs(value$delta))\\n    row$contrast_mean<-mean(value$delta);row$contrast_start<-value$delta[1L];row$contrast_end<-tail(value$delta,1)\\n    row$RSS0<-value$rss0;row$RSS1<-value$rss1;row$df0<-value$m0$df.residual;row$df1<-value$m1$df.residual\\n    row$test_df0<-value$tab[1,\\\'Resid. Df\\\'];row$test_df1<-value$tab[2,\\\'Resid. Df\\\'];row$test_delta_df<-value$tab[2,\\\'Df\\\']\\n    row$F_statistic<-value$F;row$partial_R2<-(value$rss0-value$rss1)/value$rss0\\n    row$converged0<-value$ok[1];row$converged1<-value$ok[2];row$full_rank0<-value$ok[3];row$full_rank1<-value$ok[4]\\n    row$full_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role==\\\'full\\\'])\\n    row$full_structure_flags<-sum(value$dq$structure_flag[value$dq$model_role==\\\'full\\\'])\\n    row$null_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role==\\\'null\\\'])\\n    row$residual_mean<-mean(value$r);row$residual_sd<-sd(value$r)\\n    row$cache_prediction_difference<-value$pred_error;row$cache_F_relative_difference<-value$F_error\\n    FULL[,ii]<-value$full;RESID[,ii]<-value$r;TRAJ[,ii]<-value$tr\\n    CONTRAST[,ii]<-value$delta;CONTRAST_SE[,ii]<-value$se\\n    diagnostics[[length(diagnostics)+1L]]<-value$dq\\n    if(!is.null(value$validation))validations[[length(validations)+1L]]<-value$validation\\n  }\\n  row$warnings<-paste(unique(notices),collapse=\\\' | \\\');row$fit_seconds<-proc.time()[[\\\'elapsed\\\']]-tick\\n  statistics[[ii]]<-row\\n  if(ii%%20L==0L||ii==J)cat(sprintf(\\\'%s %s K%d %d/%d %.1fs\\\\n\\\',KIND,MODE,K,ii,J,proc.time()[[\\\'elapsed\\\']]-started))\\n}\\nwrite_table<-function(rows,suffix)if(length(rows))write.table(do.call(rbind,rows),paste0(PREFIX,suffix),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE,na=\\\'NA\\\')\\nwrite_matrix<-function(x,suffix){con<-file(paste0(PREFIX,suffix),\\\'wb\\\');writeBin(as.numeric(x),con,size=8,endian=\\\'little\\\');close(con)}\\nwrite_table(statistics,\\\'.stats.tsv\\\');write_table(diagnostics,\\\'.diagnostics.tsv\\\');write_table(validations,\\\'.validation.tsv\\\')\\nwrite_matrix(FULL,\\\'.full.bin\\\');write_matrix(RESID,\\\'.residual.bin\\\');write_matrix(TRAJ,\\\'.trajectory.bin\\\')\\nwrite_matrix(CONTRAST,\\\'.contrast.bin\\\');write_matrix(CONTRAST_SE,\\\'.contrast_se.bin\\\')\\nwrite.table(data.frame(metric=c(\\\'R_version\\\',\\\'mgcv_version\\\',\\\'null_formula\\\',\\\'full_formula\\\',\\\'mode\\\',\\\'K\\\',\\\'cells\\\',\\\'permutations\\\',\\\'subsample\\\',\\\'seed\\\',\\\'computation\\\'),\\n value=c(R.version.string,as.character(packageVersion(\\\'mgcv\\\')),paste(deparse(f0),collapse=\\\' \\\'),paste(deparse(f1),collapse=\\\' \\\'),MODE,K,N,REPS,SUBSAMPLE,SEED,\\n \\\'Exact thin-QR Gaussian ML with original full-X smoothing and full-y scale initialization; full-cell attributes restored\\\')),\\n paste0(PREFIX,\\\'.metadata.tsv\\\'),sep=\\\'\\\\t\\\',quote=FALSE,row.names=FALSE)\\n\'\nHELPERS = \'def write_frame(group, data):\\n    data = data.rename(columns={"index": "regulon_index"})\\n    group.create_dataset("index", data=np.asarray(data.index.astype(str), object), dtype=h5py.string_dtype())\\n    group.create_dataset("columns", data=np.asarray(data.columns, object), dtype=h5py.string_dtype())\\n    for column in data:\\n        values = data[column].to_numpy()\\n        if values.dtype.kind in "OSU":\\n            group.create_dataset(column, data=np.asarray(data[column].fillna("").astype(str), object), dtype=h5py.string_dtype(), compression="gzip")\\n        else:\\n            group.create_dataset(column, data=values, compression="gzip")\\n\\ndef read_frame(group):\\n    return pd.DataFrame({c: group[c].asstr()[()] if group[c].dtype.kind in "OSU" else group[c][()]\\n                         for c in group["columns"].asstr()[()]}, index=group["index"].asstr()[()])\\n\\ndef adjust(values):\\n    p = np.clip(np.nan_to_num(np.asarray(values, float), nan=1, posinf=1, neginf=1), 0, 1)\\n    order = np.argsort(p, kind="stable")\\n    q = np.empty(len(p))\\n    q[order] = np.minimum(1, np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1])\\n    return np.minimum(1, p * len(p)), q\\n\\ndef collect(group, prefixes, data, auc, names, shards):\\n    stats = pd.concat([pd.read_csv(str(p) + ".stats.tsv", sep="\\\\t", keep_default_na=False, na_values=["NA"]) for p in prefixes]).set_index("regulon").loc[names]\\n    assert len(stats) == 217 and stats.index.is_unique\\n    constant = auc.columns[np.ptp(auc.to_numpy(), axis=0) == 0]\\n    stats.loc[constant, "status"] = "constant_response"\\n    stats.loc[constant, "test_status"] = "not_estimable_constant_response"\\n    stats.loc[constant, "error"] = "Observed AUCell is exactly constant in the complete local cohort; fork association is not estimable"\\n    stats["p_value"] = stats.p_value.where(stats.status.eq("ok"), 1).fillna(1)\\n    stats["p_Bonferroni"], stats["q_BH"] = adjust(stats.p_value)\\n    write_frame(group.create_group("stats"), stats)\\n    diagnostics = pd.concat([pd.read_csv(str(p) + ".diagnostics.tsv", sep="\\\\t", keep_default_na=False, na_values=["NA"]) for p in prefixes], ignore_index=True)\\n    diagnostics["diagnostic_q_BH"] = adjust(diagnostics.p_value)[1]\\n    write_frame(group.create_group("diagnostics"), diagnostics)\\n    validation = [pd.read_csv(str(p) + ".validation.tsv", sep="\\\\t") for p in prefixes if Path(str(p) + ".validation.tsv").exists()]\\n    if validation:\\n        validation = pd.concat(validation, ignore_index=True)\\n        write_frame(group.create_group("validation"), validation)\\n        assert len(validation) == 3\\n        assert validation.prediction_difference.max() < 1e-7\\n    runtime = []\\n    for i, prefix in enumerate(prefixes):\\n        frame = pd.read_csv(str(prefix) + ".metadata.tsv", sep="\\\\t")\\n        frame["shard"] = i\\n        runtime.append(frame)\\n    write_frame(group.create_group("runtime_metadata"), pd.concat(runtime, ignore_index=True))\\n    valid = stats.status.eq("ok").to_numpy()\\n    arrays = {}\\n    for suffix, key, rows in ((".full.bin", "full_prediction", len(data)), (".residual.bin", "residuals", len(data)),\\n                               (".trajectory.bin", "trajectory", len(data)), (".contrast.bin", "contrast", 51),\\n                               (".contrast_se.bin", "contrast_se", 51)):\\n        array = np.full((rows, 217), np.nan)\\n        for prefix, indices in zip(prefixes, shards):\\n            values = np.fromfile(str(prefix) + suffix, dtype="<f8").reshape((rows, len(indices)), order="F")\\n            array[:, indices - 1] = values\\n        assert np.isfinite(array[:, valid]).all(), key\\n        array = np.where(np.isfinite(array), array, np.nan)\\n        group.create_dataset(key, data=array, compression="gzip", compression_opts=4)\\n        arrays[key] = array\\n    np.testing.assert_allclose(auc.to_numpy()[:, valid] - arrays["full_prediction"][:, valid], arrays["residuals"][:, valid], atol=1e-13)\\n    np.testing.assert_allclose((arrays["residuals"][:, valid] ** 2).sum(axis=0), stats.loc[valid, "RSS1"], atol=1e-10, rtol=1e-9)\\n    np.testing.assert_allclose(np.ptp(arrays["trajectory"][:, valid], axis=0), stats.loc[valid, "A"], atol=1e-12, rtol=1e-12)\\n    tested = stats.status.eq("ok") & stats.test_status.eq("ok")\\n    expected_F = ((stats.RSS0 - stats.RSS1) / stats.test_delta_df) / (stats.RSS1 / stats.df1)\\n    np.testing.assert_allclose(expected_F[tested], stats.loc[tested, "F_statistic"], atol=1e-8, rtol=1e-8)\\n    expected_p = f_distribution.sf(stats.F_statistic, stats.test_delta_df, stats.test_df1)\\n    np.testing.assert_allclose(expected_p[tested], stats.loc[tested, "p_value"], atol=1e-10, rtol=1e-6)\\n    return stats, diagnostics\'\n\n\ndef sha(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef vector(node):\n    return node.asstr()[()] if node.dtype.kind in "OSU" else node[()]\n\n\ndef write_json(group, key, value):\n    def safe(x):\n        if isinstance(x, dict): return {str(k): safe(v) for k, v in x.items()}\n        if isinstance(x, (list, tuple, np.ndarray)): return [safe(v) for v in x]\n        if isinstance(x, (bool, np.bool_)): return bool(x)\n        if isinstance(x, (int, np.integer)): return int(x)\n        if isinstance(x, (float, np.floating)): return float(x) if np.isfinite(x) else None\n        return x\n    group.create_dataset(key, data=json.dumps(safe(value), ensure_ascii=False, allow_nan=False), dtype=h5py.string_dtype())\n\n\ndef aggregate_pair(part, auc):\n    means = []\n    for branch in ("A", "B"):\n        subset = part[part.branch.eq(branch)]\n        if subset.empty: return np.full(217, np.nan)\n        values = [auc[rows.index].mean(axis=0) for _, rows in subset.groupby("library", sort=True)]\n        means.append(np.mean(values, axis=0))\n    return means[1] - means[0]\n\n\n#2. =========== Same-stage and paired-library descriptive evidence\ndef descriptive(obs, auc):\n    # These summaries do not fit separate curves in sparse same-stage subsets.\n    composition = obs.groupby(["study", "stage", "branch"], sort=True).agg(\n        cells=("cell_id", "size"), libraries=("library", "nunique"),\n        unique_v=("v", "nunique"), v_min=("v", "min"), v_max=("v", "max")).reset_index()\n    stage_rows, stage_effects, stage_lo, stage_hi, pair_rows, pair_effects = [], [], [], [], [], []\n    bin_rows = []\n    edges = np.linspace(obs.v.min(), obs.v.max(), 4)\n    for (study, stage), part in obs.groupby(["study", "stage"], sort=True):\n        record = {"study": study, "stage": stage}\n        for branch in ("A", "B"):\n            sub = part[part.branch.eq(branch)]\n            record.update({branch + "_cells": len(sub), branch + "_libraries": sub.library.nunique(),\n                           branch + "_unique_v": sub.v.nunique()})\n        record["both_branches_observed"] = record["A_cells"] > 0 and record["B_cells"] > 0\n        record["both_branches_ge5cells_ge2libraries"] = all(record[b + "_cells"] >= 5 and record[b + "_libraries"] >= 2 for b in ("A", "B"))\n        record["separate_K10_curves_supported"] = all(record[b + "_cells"] >= 20 and record[b + "_unique_v"] >= K for b in ("A", "B"))\n        record["interpretation"] = "Observed equal-library B-A only; no stage-specific curve test or causal claim"\n        stage_effects.append(aggregate_pair(part, auc))\n        deleted = [aggregate_pair(part[part.library.ne(library)], auc) for library in part.library.unique()]\n        deleted = np.asarray(deleted)\n        valid = np.isfinite(deleted).all(axis=1)\n        record["valid_library_deletion_folds"] = int(valid.sum())\n        if valid.sum() >= 2:\n            stage_lo.append(deleted[valid].min(axis=0)); stage_hi.append(deleted[valid].max(axis=0))\n        else:\n            stage_lo.append(np.full(217, np.nan)); stage_hi.append(np.full(217, np.nan))\n        stage_rows.append(record)\n        for library, block in part.groupby("library", sort=True):\n            counts = block.branch.value_counts()\n            if all(counts.get(b, 0) > 0 for b in ("A", "B")):\n                pair_rows.append({"study": study, "stage": stage, "library": library,\n                                  "A_cells": int(counts["A"]), "B_cells": int(counts["B"]),\n                                  "both_branches_ge5cells": min(counts["A"], counts["B"]) >= 5,\n                                  "scope": "Within global common-v domain; not identical within-library v distributions"})\n                pair_effects.append(aggregate_pair(block, auc))\n        binned = part.copy()\n        binned["v_bin"] = np.clip(np.searchsorted(edges, part.v, side="right") - 1, 0, 2)\n        for b in range(3):\n            group = binned[binned.v_bin.eq(b)]\n            row = {"study": study, "stage": stage, "v_bin": b,\n                   "v_lower": float(edges[b]), "v_upper": float(edges[b + 1])}\n            for branch in ("A", "B"):\n                g = group[group.branch.eq(branch)]\n                row.update({branch + "_cells": len(g), branch + "_libraries": g.library.nunique()})\n            row["supported_ge5cells_ge2libraries_each"] = all(row[a + "_cells"] >= 5 and row[a + "_libraries"] >= 2 for a in ("A", "B"))\n            bin_rows.append(row)\n    pairs = np.asarray(pair_effects) if pair_effects else np.empty((0, 217))\n    meta = pd.DataFrame(pair_rows)\n    pair_summary = []\n    if len(meta):\n        for (study, stage), rows in meta.groupby(["study", "stage"], sort=True):\n            a = pairs[rows.index]\n            loo = np.vstack([np.delete(a, j, axis=0).mean(axis=0) for j in range(len(a))]) if len(a) > 1 else np.full((1, 217), np.nan)\n            for j in range(217):\n                pair_summary.append({"study": study, "stage": stage, "regulon_index": j,\n                                     "paired_libraries": len(a), "mean_B_minus_A": float(a[:, j].mean()),\n                                     "loo_min": float(loo[:, j].min()), "loo_max": float(loo[:, j].max()),\n                                     "positive_libraries": int((a[:, j] > 1e-10).sum()),\n                                     "negative_libraries": int((a[:, j] < -1e-10).sum())})\n    return dict(composition=composition, same_stage_metadata=pd.DataFrame(stage_rows),\n                same_stage_B_minus_A=np.asarray(stage_effects), same_stage_loo_min=np.asarray(stage_lo),\n                same_stage_loo_max=np.asarray(stage_hi), paired_library_metadata=meta,\n                paired_library_B_minus_A=pairs, paired_library_summary=pd.DataFrame(pair_summary),\n                same_stage_v_bin_support=pd.DataFrame(bin_rows))\n\n\n#3. =========== Complete-family fixed-K Gaussian ML models\ndef fit_mode(mode):\n    directory = WORK / mode\n    directory.mkdir(parents=True, exist_ok=True)\n    # Set this storage path independently before running.\n    prefix = ROOT\n    # Set this storage path independently before running. cells.csv.\n    inputs = {str(p.name): sha(p) for p in (ROOT, ROOT, ROOT)}\n    fingerprint = {"files": inputs, "mode": mode, "K": K, "seed": SEED}\n    # Set this storage path independently before running. complete.json.\n    marker = ROOT\n    if marker.exists() and json.loads(marker.read_text()) == fingerprint:\n        return prefix\n    # Set this storage path independently before running. associate_early_fork.R.\n    command = [str(RSCRIPT), "--vanilla", str(ROOT),\n               # Set this storage path independently before running. cells.csv.\n               str(ROOT), str(ROOT), str(prefix), "fork", mode,\n               str(K), ",".join(str(i) for i in range(1, 218)), "1"]\n    environment = dict(os.environ, TMP=str(WORK), TEMP=str(WORK), LC_ALL="C", LANG="C")\n    # Set this storage path independently before running.\n    with (ROOT).open("w", encoding="utf-8") as log:\n        status = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, cwd=ROOT, env=environment)\n    if status.returncode:\n        # Set this storage path independently before running.\n        raise RuntimeError((ROOT).read_text(encoding="utf-8", errors="replace")[-5000:])\n    marker.write_text(json.dumps(fingerprint, sort_keys=True), encoding="utf-8")\n    return prefix\n\n\ndef main():\n    started = time.perf_counter()\n    WORK.mkdir(parents=True, exist_ok=True)\n    exec(HELPERS, globals())\n    with TREE.open("rb") as stream:\n        tree = pickle.load(stream)\n    with h5py.File(INPUT, "r") as h:\n        names = h["regulons"].asstr()[()].tolist()\n        cells = h["cells"].asstr()[()]\n        all_auc = h["auc"][()]\n        obs = pd.DataFrame({k: vector(h["obs"][k]) for k in ("study", "stage", "library", "log_detected")})\n    assert np.array_equal(cells, tree["obs"].index.astype(str).to_numpy())\n    obs["cell_id"] = cells\n    obs["seg"] = tree["segment_gradient_coloring"]["segment_ids"]\n    obs["t"] = tree["projection"]["t"]\n    obs["branch"] = np.where(obs.seg.eq(1), "A", "B")\n    chosen = obs.seg.isin([1, 2])\n    lower = max(obs.loc[obs.seg.eq(s), "t"].min() for s in (1, 2))\n    upper = min(obs.loc[obs.seg.eq(s), "t"].max() for s in (1, 2))\n    chosen &= obs.t.between(lower, upper)\n    positions = np.flatnonzero(chosen)\n    obs = obs.loc[chosen].reset_index(drop=True)\n    origin = float(tree["projection"]["root_distance"][tree["local_fork"]])\n    obs["v"] = obs.t - origin\n    auc = pd.DataFrame(all_auc[positions], index=pd.Index(obs.cell_id), columns=names)\n    assert len(obs) == 244 and obs.branch.value_counts().to_dict() == {"A": 175, "B": 69}\n    assert obs.groupby("branch").v.nunique().min() >= K and obs.groupby("branch").size().min() >= 20\n    # Source is determined by stage here. Stage sensitivities omit its redundant column.\n    assert obs.groupby("stage").study.nunique().eq(1).all()\n    desc = descriptive(obs, auc.to_numpy())\n    if len(desc["paired_library_summary"]):\n        desc["paired_library_summary"]["regulon"] = [names[j] for j in desc["paired_library_summary"].regulon_index]\n        desc["paired_library_summary"].rename(columns={"regulon_index": "regulon_index_zero_based"}, inplace=True)\n    # Set this storage path independently before running. cells.csv.\n    obs.set_index("cell_id").to_csv(ROOT, float_format="%.17g")\n    # Set this storage path independently before running. auc.tsv.gz.\n    auc.to_csv(ROOT, sep="\\t", float_format="%.17g", compression={"method": "gzip", "mtime": 0})\n    # Set this storage path independently before running. associate_early_fork.R.\n    (ROOT).write_text(R_SOURCE, encoding="utf-8", newline="\\n")\n    with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as pool:\n        prefixes = dict(zip(MODES, pool.map(fit_mode, MODES)))\n    # Set this storage path independently before running. Fig4A_early_fork.h5.\n    target = ROOT\n    summaries = {}\n    with h5py.File(target, "w") as h:\n        h.create_dataset("regulons", data=np.asarray(names, object), dtype=h5py.string_dtype())\n        h.create_dataset("cells", data=np.asarray(obs.cell_id, object), dtype=h5py.string_dtype())\n        write_frame(h.create_group("obs"), obs)\n        h.create_dataset("observed_auc", data=auc.to_numpy(), compression="gzip")\n        h.create_dataset("grid_v", data=np.linspace(obs.v.min(), obs.v.max(), 51))\n        modes = h.create_group("modes")\n        for mode in MODES:\n            g = modes.create_group(mode)\n            stats, diagnostics = collect(g, [prefixes[mode]], obs.set_index("cell_id"), auc, names, [np.arange(1, 218)])\n            runtime = read_frame(g["runtime_metadata"])\n            formulas = runtime.drop_duplicates("metric").set_index("metric")["value"]\n            write_json(g, "manifest_json", dict(mode=mode, K=K, primary=mode == "none",\n                null_formula=str(formulas["null_formula"]), full_formula=str(formulas["full_formula"]),\n                stage_adjustment="Categorical stage; redundant source omitted" if mode.startswith("stage") else "Not stage-adjusted",\n                inference="Approximate conditional cell-level Gaussian ML F comparison; no independent-animal or causal claim"))\n            g.create_dataset("contrast_lower_95", data=g["contrast"][()] - 1.96 * g["contrast_se"][()], compression="gzip")\n            g.create_dataset("contrast_upper_95", data=g["contrast"][()] + 1.96 * g["contrast_se"][()], compression="gzip")\n            summaries[mode] = dict(valid_models=int(stats.status.eq("ok").sum()), valid_tests=int(stats.test_status.eq("ok").sum()),\n                                   constant_responses=int(stats.status.eq("constant_response").sum()),\n                                   q_BH_lt_1e4=int(stats.q_BH.lt(1e-4).sum()),\n                                   full_basis_flags=int(stats.full_basis_flags.fillna(0).sum()),\n                                   full_structure_flags=int(stats.full_structure_flags.fillna(0).sum()))\n        h["primary"] = modes["none"]\n        g = h.create_group("descriptive_support")\n        for key, value in desc.items():\n            if isinstance(value, pd.DataFrame):\n                write_frame(g.create_group(key), value)\n            else:\n                g.create_dataset(key, data=value, compression="gzip")\n        write_json(h, "manifest_json", dict(schema="Fig4A_early_fork_v1", family=217, cells=len(obs), K=K, modes=list(MODES),\n            primary_mode="none", primary="Null common smooth versus branch + branch-specific smooth, Gaussian ML unit weights gamma1",\n            branches=dict(A="VGN candidate-enriched arm (native segment1)", B="SGN candidate-enriched arm (native segment2)"),\n            contrast="B-A: SGN candidate minus VGN candidate on common native-v interval",\n            native_t_lower=float(lower), native_t_upper=float(upper), native_fork_t=origin,\n            native_v_lower=float(obs.v.min()), native_v_upper=float(obs.v.max()),\n            branch_cells=obs.branch.value_counts().to_dict(), branch_unique_v=obs.groupby("branch").v.nunique().to_dict(),\n            K_rule="K10 fixed for every217 and every mode because minimum whole-domain branch unique v is41; no TF-specific tuning or escalation",\n            diagnostic="Tie-order-invariant expected adjacent residual SSE; 500 residual randomizations; all full/null smooth terms",\n            multiplicity="Complete217 Bonferroni and BH per mode; root will join primary raw P with three old forks for 868-test correction",\n            stage_source_collinearity="Source is exactly determined by stage; stage and stage_depth omit redundant source column",\n            same_stage_limit="E11.5 SGN arm has only5cells in3libraries; E10.5 SGN3; no separate stage-specific K10 curves fitted",\n            CI="Conditional pointwise Gaussian coefficient Vp95% intervals; not simultaneous, animal-level or causal intervals",\n            descriptive="Observed B-A uses equal library mean per branch; LOO deletes each library; paired-library comparisons are descriptive and not identical v matching",\n            source_tree_method=tree["provenance"]["method"], topology_scope=tree["provenance"]["caution"],\n            input_tree_sha256=sha(TREE), frozen_auc_content_sha256=hashlib.sha256(np.asarray(all_auc, dtype="<f8").tobytes()).hexdigest(),\n            # Set this storage path independently before running. associate_early_fork.R.\n            code_sha256=sha(Path(__file__)), R_source_sha256=sha(ROOT), summaries=summaries))\n    os.replace(target, OUTPUT)\n    print("EARLY_FORK_COMPLETE " + json.dumps(dict(output=str(OUTPUT), summaries=summaries,\n                                                   seconds=time.perf_counter() - started)), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'}
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='1'
for key in ('TMP','TEMP'):os.environ[key]=str(WORK)
# Set this storage path independently before running.
os.environ['MPLCONFIGDIR']=str(ROOT)
import h5py
import numpy as np
import pandas as pd
from scipy.stats import spearmanr,f as f_distribution
from scipy.cluster.hierarchy import linkage,optimal_leaf_ordering,leaves_list
from scipy.spatial.distance import pdist

#2. =========== Accepted tree, fixed AUCell, and native-coordinate bins
def read_component(name,key):
    path=DESTINATION/('Fig4A_'+name+'.h5')
    if path.exists():
        handle=h5py.File(path,'r');return handle,handle
    # Set this storage path independently before running. Fig4A.h5.
    handle=h5py.File(ROOT,'r');return handle,handle[key]

def load_data():
    with TREE_PATH.open('rb') as stream:tree=pickle.load(stream)
    # Set this storage path independently before running. Fig4A.h5.
    with h5py.File(ROOT,'r') as h:
        cells=pd.Index(h['cells'].asstr()[()],name='cell_id')
        names=pd.Index(h['regulons'].asstr()[()])
        auc=pd.DataFrame(h['auc'][()],index=cells,columns=names)
        old_obs=input_read_frame(h['obs'],index=cells)
        coverage=input_read_frame(h['coverage']).set_index('regulon').loc[names]
        info=json.loads(h['metadata_json'].asstr()[()])
        auc_hash=hashlib.sha256(np.asarray(auc,dtype='<f8').tobytes()).hexdigest()
        previous=json.loads(h['final/delivery_manifest_json'].asstr()[()])
        assert previous['auc_content_sha256']==auc_hash
        early=h['early_extension']
        early_meta=input_read_frame(early['group_metadata'])
        early_mean=pd.DataFrame(early['summaries/common_auc/mean'][()],columns=names)
        early_loo_min=early['summaries/common_auc/loo_min'][()]
        early_loo_max=early['summaries/common_auc/loo_max'][()]
        early_cells=pd.Index(early['cells'].asstr()[()])
        early_obs=input_read_frame(early['obs'],index=early_cells)
        early_rna=pd.DataFrame(early['context_RNA/log1p_CP10k/mean'][()],columns=early['context_RNA/genes'].asstr()[()])
    assert len(cells)==7355 and len(names)==217 and tree['obs'].index.equals(cells)
    assert np.isfinite(auc).all().all() and auc.ge(0).all().all()
    obs=old_obs.copy()
    for source,target in [('stage_standardized','stage'),('state','type'),('study_id','study')]:
        assert np.array_equal(tree['obs'][source].astype(str),old_obs[target].astype(str))
        obs[target]=tree['obs'][source].to_numpy()
    assert np.array_equal(tree['obs'].study_id+'__'+tree['obs'].library_id,old_obs.library)
    obs['seg']=np.asarray(tree['segment_gradient_coloring']['segment_ids'],int)
    obs['t']=np.asarray(tree['projection']['t'],float)
    obs['bin']=-1
    records=[];segments=[]
    gradient={s['id']:s for s in tree['segment_gradient_coloring']['segments']}
    for sid in BIO_SEGMENTS:
        s=next(s for s in tree['segments'] if s['id']==sid)
        rows=np.flatnonzero(obs.seg.to_numpy()==sid)
        left=tree['projection']['root_distance'][s['start']]
        right=tree['projection']['root_distance'][s['end']]
        boundaries=np.linspace(left,right,4)
        bins=np.clip(np.searchsorted(boundaries,obs.t.iloc[rows],side='right')-1,0,2)
        obs.iloc[rows,obs.columns.get_loc('bin')]=bins
        g=gradient[sid]
        segments.append(dict(s,id=sid,parent=PARENTS[sid],short_label=LABELS[sid],
            color_from=g['start_color'],color_to=g['end_color'],t_start=float(left),t_end=float(right),
            cells=len(rows),path=list(map(int,s['path']))))
        for b in range(3):
            z=obs.iloc[rows[bins==b]]
            assert len(z)>=20,'Insufficient display-bin support'
            records.append(dict(seg=sid,bin=b,t_left=boundaries[b],t_right=boundaries[b+1],
                cells=len(z),libraries=z.library.nunique(),studies=z.study.nunique(),
                stage_counts=json.dumps(z.stage.value_counts().to_dict(),sort_keys=True)))
    mask=obs.seg.ne(3)
    assert mask.sum()==7348 and (~mask).sum()==7
    obs['model_ok']=mask
    obs['included_in_association']=mask
    for name,ids in dict(Ia=[0,2,3,4,6,7],II=[0,2,3,4,6,8],
                         Ib=[0,2,3,4,5,9],Ic=[0,2,3,4,5,10],VGN_candidate=[0,1]).items():
        obs['path_'+name]=obs.seg.isin(ids)
    bin_meta=pd.DataFrame(records)
    index=pd.MultiIndex.from_frame(bin_meta[['seg','bin']])
    def aggregate(matrix):
        return matrix.loc[mask].groupby([obs.loc[mask,'seg'],obs.loc[mask,'bin']]).mean().reindex(index)
    model_handle,global_model=read_component('global_new','global_models')
    try:
        assert global_model['regulons'].asstr()[()].tolist()==names.tolist()
        model_cells=pd.Index(global_model['cells'].asstr()[()])
        assert model_cells.equals(cells[mask])
        model_obs=read_frame(global_model['obs'])
        np.testing.assert_allclose(model_obs.t,obs.loc[mask,'t'],rtol=0,atol=3e-15)
        for column in ('seg','stage','type','study','library'):
            np.testing.assert_array_equal(model_obs[column].astype(str),obs.loc[mask,column].astype(str))
        if 'log_detected' in model_obs:
            np.testing.assert_allclose(model_obs.log_detected,obs.loc[mask,'log_detected'],rtol=0,atol=1e-14)
        modes={}
        chosen=global_model['primary'].attrs.get('K',global_model.attrs.get('chosen_K'))
        for name,mg in global_model['modes'].items():
            available=sorted(int(k[1:]) for k in mg if k.startswith('k'))
            k=int(chosen) if chosen is not None else max(available)
            group=mg[f'k{k}'];st=read_frame(group['stats']).loc[names]
            fitted=pd.DataFrame(group['full_prediction'][()],index=model_cells,columns=names)
            residual=group['residuals'][()]
            np.testing.assert_allclose(auc.loc[model_cells]-fitted,residual,atol=1e-12)
            # Compare covariate-adjusted biological curves, excluding fitted
            # nuisance terms; full predictions are used only for residuals.
            trajectory=pd.DataFrame(group['trajectory'][()],index=model_cells,columns=names)
            extended=trajectory.reindex(cells)
            diagnostic=read_frame(group['diagnostics'])
            rms=pd.DataFrame({sid:np.sqrt(np.mean(residual[obs.loc[mask,'seg'].eq(sid)]**2,axis=0))
                              for sid in BIO_SEGMENTS},index=names).rename_axis('regulon').stack()
            rms.index.names=['regulon','seg']
            diagnostic=diagnostic.join(rms.rename('computed_residual_RMS'),on=['regulon','seg'])
            modes[name]=dict(stats=st,fitted=fitted,trajectory=trajectory,bins=aggregate(extended),diagnostic=diagnostic,K=k)
        primary=read_frame(global_model['primary/stats']).loc[names]
        assert 'none' in modes
        np.testing.assert_allclose(primary.p_value,modes['none']['stats'].p_value,rtol=0,atol=0)
    finally:model_handle.close()
    for row,record in early_meta.iterrows():
        ids=early_obs.index[(early_obs.stage==record.stage)&(early_obs.type==record.state)]
        assert len(ids)==record.cells
        x=obs.loc[ids,'t']
        for key,value in [('t_min',x.min()),('t_median',x.median()),('t_max',x.max()),('exact_root_cells',x.eq(0).sum())]:
            early_meta.loc[row,key]=value
    early_obs['t']=obs.loc[early_cells,'t'].to_numpy()
    early_obs['seg']=obs.loc[early_cells,'seg'].to_numpy()
    early_obs['bin']=obs.loc[early_cells,'bin'].to_numpy()
    return dict(tree=tree,cells=cells,names=names,auc=auc,obs=obs,mask=mask,segments=segments,
        bin_meta=bin_meta,observed=aggregate(auc),modes=modes,primary=primary,coverage=coverage,
        early_meta=early_meta,early_mean=early_mean,early_obs=early_obs,early_loo_min=early_loo_min,
        early_loo_max=early_loo_max,early_rna=early_rna,info=info,auc_hash=auc_hash)

#3. =========== Literature primary screen and separately corrected forks
def screen(data):
    names=data['names'];stats=data['primary'].copy()
    valid=stats.status.eq('ok')
    p=stats.p_value.where(valid,1).fillna(1)
    stats['q_global']=adjusted_pvalues(p)[1]
    stats['selected']=valid & stats.q_global.lt(Q_CUT) & stats.A.gt(A_CUT)
    stats=stats.join(data['coverage'],rsuffix='_coverage')
    fork_p=pd.DataFrame(index=names)
    handles=[]
    try:
        h,early=read_component('early_fork','early_fork');handles.append(h)
        e=read_frame(early['primary/stats']).loc[names]
        fork_p['early']=e.p_value.where(e.status.eq('ok'),1).fillna(1)
        early_stage=read_frame(early['modes/stage_depth/stats']).loc[names]
        extras=dict(early_stage_p=early_stage.p_value,early_unadjusted_p=e.p_value)
        # Set this storage path independently before running. Fig4A.h5.
        with h5py.File(ROOT,'r') as cached:
            for key in ('fork1','fork2','fork3'):
                base=cached['late_forks'][key] if 'late_forks' in cached else cached['local_association/analyses'][key]
                local=read_frame(base['modes/none/k10/stats']).loc[names]
                local_obs=read_frame(base['obs'])
                assert local_obs.index.isin(data['cells']).all()
                expected=data['obs'].loc[local_obs.index,'t'].to_numpy()
                expected-=expected.min()
                np.testing.assert_allclose(expected,local_obs.v.to_numpy()-local_obs.v.min(),rtol=0,atol=3e-15)
                np.testing.assert_array_equal(base['observed_auc'][()],data['auc'].loc[local_obs.index].to_numpy())
                fork_p[key]=local.p_value.where(local.status.eq('ok'),1).fillna(1)
                conditional=read_frame(base['modes/source_depth/k10/stats']).loc[names]
                extras[key+'_source_depth_p']=conditional.p_value
                extras[key+'_A']=local.A
    finally:
        for h in handles:h.close()
    assert fork_p.shape==(217,4)
    q=adjusted_pvalues(fork_p.to_numpy().ravel())[1].reshape(217,4)
    for j,key in enumerate(fork_p):extras['p_'+key]=fork_p[key];extras['q_'+key]=q[:,j]
    extras['q_early_stage']=adjusted_pvalues(extras['early_stage_p'].fillna(1))[1]
    reference=data['modes']['none']['bins']
    for key,mode in data['modes'].items():
        extras[key+'_p']=mode['stats'].p_value
        extras[key+'_A']=mode['stats'].A
        extras[key+'_q']=adjusted_pvalues(mode['stats'].p_value.where(mode['stats'].status.eq('ok'),1))[1]
        extras[key+'_threshold_pass']=(extras[key+'_q']<Q_CUT)&mode['stats'].A.gt(A_CUT)&mode['stats'].status.eq('ok')
        extras[key+'_profile_rho']=[spearmanr(reference[n],mode['bins'][n]).statistic for n in names]
    stats=pd.concat([stats,pd.DataFrame(extras,index=names)],axis=1)
    diag=data['modes']['none']['diagnostic'].copy()
    # Machine-precision residuals do not establish biological model misspecification.
    rms_col=next((c for c in ('computed_residual_RMS','residual_rms','residual_RMS','rms') if c in diag),None)
    if rms_col:
        diag['effective_structure_flag']=diag.structure_flag & diag[rms_col].gt(1e-10)
    else:diag['effective_structure_flag']=diag.structure_flag
    flag=diag.groupby('regulon').agg(residual_flags=('effective_structure_flag','sum'),basis_flags_check=('basis_flag','sum'))
    if 'basis_flags' in stats:np.testing.assert_array_equal(stats.basis_flags,flag.loc[names,'basis_flags_check'])
    stats['residual_flags']=flag.loc[names,'residual_flags']
    selected=stats.index[stats.selected]
    profiles=reference.loc[:,selected]
    order=[]
    peaks=np.argmax(profiles.to_numpy(),axis=0)//3
    for group in range(10):
        ids=np.flatnonzero(peaks==group)
        if len(ids)>2:
            distances=np.nan_to_num(pdist(profiles.iloc[:,ids].T,'correlation'),nan=1)
            z=linkage(distances,method='average')
            ids=ids[leaves_list(optimal_leaf_ordering(z,distances))]
        order.extend(selected[ids].tolist())
    assert len(order)>0 and len(order)==stats.selected.sum() and len(set(order))==len(order)
    stats['peak_band']=pd.Series(peaks,index=selected)
    return stats,fork_p,diag,order
def clean_json(value):
    if isinstance(value,dict):return {str(k):clean_json(v) for k,v in value.items()}
    if isinstance(value,(list,tuple,np.ndarray,pd.Index)):return [clean_json(v) for v in value]
    if value is None or value is pd.NA:return None
    if isinstance(value,(bool,np.bool_)):return bool(value)
    if isinstance(value,(int,np.integer)):return int(value)
    if isinstance(value,(float,np.floating)):return float(value) if np.isfinite(value) else None
    if isinstance(value,Path):return str(value)
    return value

def vector(node):
    return node.asstr()[()] if node.dtype.kind in 'OSU' else node[()]

def read_any_frame(group):
    if 'columns' in group and 'index' in group:return read_frame(group)
    return input_read_frame(group)

def read_column(item):
    if isinstance(item, h5py.Dataset):
        return item.asstr()[()] if item.dtype.kind in 'OS' else item[()]
    if 'codes' in item:
        categories = read_column(item['categories'])
        return np.array([categories[k] if k >= 0 else None for k in item['codes'][()]], dtype=object)
    values = np.asarray(read_column(item['values']), dtype=object)
    if 'mask' in item:
        values[item['mask'][()]] = None
    return values

def input_read_frame(group, index=None):
    key = group.attrs.get('_index')
    key = key.decode() if isinstance(key, bytes) else key
    frame = pd.DataFrame({k: read_column(v) for k, v in group.items() if k != key and (not k.endswith('__missing'))}, index=index)
    if key is not None:
        frame.index = pd.Index(read_column(group[key]), name='cell_id')
    for column in frame:
        if column + '__missing' in group:
            frame.loc[group[column + '__missing'][()], column] = np.nan
    return frame

def put_strings(group, key, values):
    group.create_dataset(key, data=np.asarray(values, dtype=object), dtype=h5py.string_dtype('utf8'), compression='gzip')

def put_frame(group, frame):
    for key in frame:
        series = frame[key]
        if series.dtype.kind in 'biuf':
            group.create_dataset(key, data=series.to_numpy(), compression='gzip')
        else:
            put_strings(group, key, series.fillna('').astype(str).to_numpy())
            if series.isna().any():
                group.create_dataset(key + '__missing', data=series.isna().to_numpy(), compression='gzip')

def put_json(group, key, value):
    group.create_dataset(key, data=json.dumps(value, ensure_ascii=False, allow_nan=False), dtype=h5py.string_dtype('utf8'))

def sha(path):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def adjusted_pvalues(values):
    p = np.nan_to_num(np.asarray(values,float),nan=1,posinf=1,neginf=1)
    p = np.clip(p,0,1)
    n = len(p)
    order = np.argsort(p,kind='stable')
    bh = np.empty(n)
    bh[order] = np.minimum(1,np.minimum.accumulate((p[order]*n/np.arange(1,n+1))[::-1])[::-1])
    return np.minimum(p*n,1),bh

def write_frame(group,frame):
    frame = frame.rename(columns={'index':'regulon_index'}).copy()
    group.create_dataset('index',data=np.asarray(frame.index.astype(str),object),dtype=h5py.string_dtype())
    group.create_dataset('columns',data=np.asarray(frame.columns,object),dtype=h5py.string_dtype())
    for col in frame.columns:
        values = frame[col].to_numpy()
        if values.dtype.kind in 'OSU':
            group.create_dataset(col,data=np.asarray(frame[col].fillna('').astype(str),object),dtype=h5py.string_dtype())
        else:
            group.create_dataset(col,data=values,compression='gzip')

def read_frame(group):
    return pd.DataFrame({c:(group[c].asstr()[()] if group[c].dtype.kind in 'OSU' else group[c][()])
        for c in group['columns'].asstr()[()]},index=group['index'].asstr()[()])

#4. =========== Charles-style tree strips and observed early context
def render(data,stats,order):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_rgb,LinearSegmentedColormap
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':6,'axes.linewidth':.45,
                         'savefig.dpi':600,'figure.dpi':120})
    selected=pd.Index(order)
    fitted=data['modes']['none']['bins'].loc[:,selected].to_numpy()
    maxima=fitted.max(axis=0)
    assert np.all(maxima>0)
    raw_relative=fitted/maxima
    # An unconstrained Gaussian GAM may make a tiny negative prediction. Show
    # an explicit grey cross; do not silently truncate the saved fitted values.
    negative=fitted < -1e-12
    plotted=np.where(negative,np.nan,np.maximum(raw_relative,0))
    assert np.nanmax(plotted)<=1+1e-12
    early=data['early_mean'].loc[:,selected].to_numpy()
    early_max=early.max(axis=0)
    early_relative=np.divide(early,early_max,out=np.zeros_like(early),where=early_max>0)
    red=stats.loc[selected,['q_early','q_fork1','q_fork2','q_fork3']].min(axis=1).lt(Q_CUT).to_numpy()
    flagged=stats.loc[selected,'residual_flags'].gt(0).to_numpy()
    cmap=LinearSegmentedColormap.from_list('white_red',['white','#ff0000'])
    dpi=600;col_pitch=.128;row_pitch=.100;gap=.045
    short={0:'CVG',1:'VGN*',2:'SGN*',4:'SGN',5:'Ib/Ic',9:'Ib',10:'Ic',6:'Ia/II',7:'Ia',8:'II'}
    output=[];checks={}
    def draw(indices,path,early_panel=False,part=None):
        n=len(indices);left=2.35 if early_panel else 1.02
        top=.62;bottom=1.24
        nrows=7 if early_panel else 30
        body=nrows*row_pitch+(0 if early_panel else 9*gap)
        width=left+n*col_pitch+.09;height=top+body+bottom
        fig=plt.figure(figsize=(width,height),facecolor='white')
        ax=fig.add_axes([left/width,bottom/height,n*col_pitch/width,body/height])
        ax.set_xlim(0,n);ax.set_ylim(body/row_pitch,0)
        ax.set_xticks(np.arange(n)+.5)
        labels=[selected[i]+(' †' if flagged[i] else '') for i in indices]
        ax.set_xticklabels(labels,rotation=90,ha='center',va='top',fontsize=6,fontstyle='italic')
        ax.tick_params(axis='x',length=0,pad=2)
        ax.set_yticks([])
        for spine in ax.spines.values():spine.set_visible(False)
        Y=np.arange(nrows)+.5
        if not early_panel:
            Y=np.concatenate([np.arange(3)+.5+j*(3+gap/row_pitch) for j in range(10)])
        vals=early_relative[:,indices] if early_panel else plotted[:,indices]
        x,y=np.meshgrid(np.arange(n)+.5,Y)
        diameter=.81*row_pitch*72
        size=diameter**2*np.maximum(np.nan_to_num(vals,nan=0),0)
        keep=np.isfinite(vals)
        ax.scatter(x[keep],y[keep],s=size[keep],c=vals[keep],cmap=cmap,vmin=0,vmax=1,
                   edgecolors='#4c4c4c',linewidths=.32,zorder=3)
        # Zero-activity locations are tiny centre points, as in the reference.
        zero=keep & (vals<=0)
        ax.scatter(x[zero],y[zero],s=.18,c='#555555',linewidths=0,zorder=4)
        missing=~keep
        if missing.any():ax.scatter(x[missing],y[missing],marker='x',s=7,c='#8a8a8a',linewidths=.55,zorder=5)
        for tick,i in zip(ax.get_xticklabels(),indices):tick.set_color('#dc2638' if red[i] else '#111111')
        title='observed early regulon activity' if early_panel else 'progression of regulons over the tree'
        suffix='' if part is None else f' ({part})'
        fig.text(.12/width,1-.23/height,('' if early_panel else 'a  ')+title+suffix,
                 fontsize=11 if not early_panel else 10,va='center')
        subtitle=('E9.5–E13.5; equal weight per library within each stage/state; observed AUC' if early_panel else
                  f'E9.5–E18.5; whole-tree F test, BH q < 10⁻⁴; fitted AUC range > 0.025; n = {len(selected)} regulons')
        fig.text(.12/width,1-.47/height,subtitle,fontsize=5.7,va='center')
        if early_panel:
            label_lines=['E9.5  CVG neuroblast','E10.5  delaminating otic/CVG','E10.5  developing CVG',
                         'E11.5  proliferative CVG','E11.5  differentiating SGN',
                         'E13.5  late CVG','E13.5  differentiating SGN']
            for r,label in enumerate(label_lines):
                yc=(bottom+body-Y[r]*row_pitch)/height
                m=data['early_meta'].iloc[r]
                fig.text((left-.09)/width,yc,label+f' ({int(m.cells)}; {int(m.libraries)})',
                         ha='right',va='center',fontsize=5.6)
            ax.add_patch(Rectangle((0,0),n,nrows,fill=False,edgecolor='#333333',lw=.45,clip_on=False))
        else:
            centre={}
            for j,s in enumerate(data['segments']):
                y0=j*(3+gap/row_pitch)
                ax.add_patch(Rectangle((0,y0),n,3,fill=False,edgecolor='#333333',lw=.45,clip_on=False))
                yb=(bottom+body-(y0+3)*row_pitch)/height
                strip=fig.add_axes([(left-.15)/width,yb,.065/width,3*row_pitch/height])
                colors=np.array([to_rgb(s['color_from']),to_rgb(s['color_to'])])
                gradient=np.linspace(0,1,128)[:,None,None]
                strip.imshow(colors[0][None,None,:]*(1-gradient)+colors[1][None,None,:]*gradient,
                             aspect='auto',origin='upper',extent=(0,1,0,1))
                strip.set_xticks([]);strip.set_yticks([])
                for spine in strip.spines.values():spine.set_linewidth(.4)
                centre[s['id']]=(bottom+body-(y0+1.5)*row_pitch)/height
                fig.text((left-.205)/width,centre[s['id']],short[s['id']],rotation=90,
                         ha='center',va='center',fontsize=5.6)
            end=(left-.26)/width
            for parent in (0,4,5,6):
                children=[s['id'] for s in data['segments'] if s['parent']==parent]
                lane=(.22 if parent in (0,4) else .53)/width
                ys=[centre[parent]]+[centre[c] for c in children]
                fig.add_artist(Line2D([lane,lane],[min(ys),max(ys)],transform=fig.transFigure,color='#444444',lw=.5))
                for sid in [parent]+children:
                    fig.add_artist(Line2D([lane,end],[centre[sid],centre[sid]],transform=fig.transFigure,
                                         color='#444444',lw=.5))
            # Segment 3 is a seven-cell zero-length computational bridge.
            lane=.66/width
            fig.add_artist(Line2D([end,lane,lane,end],[centre[2],centre[2],centre[4],centre[4]],
                                 transform=fig.transFigure,color='#666666',lw=.5,ls=(0,(2,2))))
        ly=.42
        legend_left=.14
        fig.text(legend_left/width,(ly+.17)/height,
                 'relative observed AUC' if early_panel else 'relative fitted AUC',fontsize=5.5,va='center')
        base=1.44
        for j,value in enumerate((0,.25,.5,.75,1)):
            xpos=base+j*.29
            fig.add_artist(Line2D([xpos/width],[(ly+.17)/height],marker='o',markersize=diameter*np.sqrt(value),
                                 markerfacecolor=cmap(float(value)),markeredgecolor='#4c4c4c',markeredgewidth=.32,
                                 lw=0,transform=fig.transFigure))
            fig.text(xpos/width,ly/height,f'{value:g}',fontsize=5,ha='center',va='top')
        note1=('Each regulon is max-normalized across the seven observed groups; row brackets give cells; libraries.' if early_panel else
               'Each regulon is max-normalized across 30 fitted bins. * Early candidate arm; dashed link: 7-cell bridge.')
        note2='Red labels: branch difference (separate BH across 868 tests); †: residual structure flag, retained for review.'
        note3=('Observed early values use the same fixed SCENIC network; early arm and stage effects remain partly confounded.' if early_panel else
               'Grey ×: negative Gaussian prediction (raw value retained). Early-arm effects remain partly stage-confounded.')
        for pos,note in zip((.23,.13,.03),(note1,note2,note3)):
            fig.text(.14/width,pos/height,note,fontsize=4.8,va='bottom')
        fig.canvas.draw()
        renderer=fig.canvas.get_renderer()
        texts=list(fig.texts)+list(ax.get_xticklabels())
        boxes=[(t,t.get_window_extent(renderer)) for t in texts if t.get_text()]
        canvas=fig.bbox
        outside=[t.get_text() for t,b in boxes if b.x0<-.5 or b.y0<-.5 or b.x1>canvas.x1+.5 or b.y1>canvas.y1+.5]
        overlaps=[]
        for i,(t,b) in enumerate(boxes):
            for u,c in boxes[i+1:]:
                if min(b.x1,c.x1)-max(b.x0,c.x0)>.5 and min(b.y1,c.y1)-max(b.y0,c.y0)>.5:
                    overlaps.append([t.get_text(),u.get_text()])
        assert not outside,(path.name,'text outside canvas',outside)
        assert not overlaps,(path.name,'text overlap',overlaps)
        fig.savefig(path,dpi=dpi,facecolor='white')
        checks[path.name]=dict(columns=n,rows=nrows,dpi=dpi,width_in=width,height_in=height,
                               text_outside=outside,text_overlap=overlaps,bytes=path.stat().st_size)
        output.append(path.name);plt.close(fig)
    # Set this storage path independently before running. Fig4A.png.
    draw(np.arange(len(selected)),ROOT)
    if len(selected)>72:
        for j,ids in enumerate(np.array_split(np.arange(len(selected)),int(np.ceil(len(selected)/72)))):
            draw(ids,DESTINATION/f'Fig4A_{j+1}.png',part=f'{j+1}/{int(np.ceil(len(selected)/72))}')
    # Set this storage path independently before running. Fig4A_early.png.
    draw(np.arange(len(selected)),ROOT,early_panel=True)
    return dict(files=output,figures=checks,negative_fitted_bin_count=int(negative.sum()),
                negative_bins=[dict(segment=int(data['bin_meta'].iloc[r].seg),bin=int(data['bin_meta'].iloc[r]['bin']),
                                    regulon=selected[c],raw_mean=float(fitted[r,c])) for r,c in np.argwhere(negative)],
                display_zero_tolerance=1e-12,normalization='per regulon maximum across fitted 30 bins; early observed independent maximum across 7 groups',
                label_red='any fork BH868 q<1e-4; not a primary gate',residual_mark='effective structure flag; not excluded')

#5. =========== Frozen analysis artifact
def assemble(data,stats,fork_p,diag,order,check):
    """Freeze validated current results atomically; never refit or reselect here."""
    from datetime import datetime,timezone

    schema='Fig4A_new_tree_v1'
    # Set this storage path independently before running. Fig4A.h5.
    output=ROOT
    # Set this storage path independently before running. Fig4A.h5.
    staged=ROOT
    cells=pd.Index(data['cells']);names=pd.Index(data['names'])
    mask=np.asarray(data['mask'],bool)
    selected=pd.Index(order)
    raw=np.asarray(data['auc'],dtype='<f8')
    auc_hash=hashlib.sha256(raw.tobytes()).hexdigest()
    assert auc_hash==data['auc_hash']
    assert len(cells)==7355 and len(names)==217 and mask.sum()==7348
    assert stats.index.equals(names) and fork_p.index.equals(names)
    assert fork_p.columns.tolist()==['early','fork1','fork2','fork3']
    assert len(selected)==len(set(selected))==int(stats.selected.sum())
    assert set(selected)==set(stats.index[stats.selected])
    assert data['obs'].index.equals(cells)
    assert len(data['bin_meta'])==30 and data['observed'].shape==(30,217)
    np.testing.assert_array_equal(mask,data['obs'].seg.ne(3))
    np.testing.assert_array_equal(mask,data['obs'].included_in_association)
    np.testing.assert_array_equal(mask,data['obs'].model_ok)
    np.testing.assert_array_equal(data['obs'].seg,data['tree']['segment_gradient_coloring']['segment_ids'])
    np.testing.assert_array_equal(data['obs'].t,data['tree']['projection']['t'])
    assert data['obs'].loc[~mask,'bin'].eq(-1).all()
    assert data['obs'].loc[mask,'bin'].isin([0,1,2]).all()

    fitted_all=data['modes']['none']['bins'].loc[:,names].to_numpy()
    fitted=data['modes']['none']['bins'].loc[:,selected].to_numpy()
    observed=data['observed'].loc[:,selected].to_numpy()
    maxima=fitted.max(axis=0)
    assert np.isfinite(fitted).all() and (maxima>0).all()
    raw_relative=fitted/maxima
    negative=fitted < -1e-12
    relative=np.where(negative,np.nan,np.maximum(raw_relative,0))
    early_all=data['early_mean'].loc[:,names].to_numpy()
    early=data['early_mean'].loc[:,selected].to_numpy()
    early_max=early.max(axis=0)
    early_relative=np.divide(early,early_max,out=np.zeros_like(early),where=early_max>0)
    assert int(negative.sum())==check['figure_qa']['negative_fitted_bin_count']
    assert np.nanmax(relative)<=1+1e-12 and np.nanmax(early_relative)<=1+1e-12

    figure_hash={}
    for name in check['figure_qa']['files']:
        path=(DESTINATION/name).resolve()
        assert path.parent==DESTINATION.resolve() and path.is_file()
        figure_hash[name]=sha(path)
    # Set this storage path independently before running. Fig4A.py. Use fig4A_regulon_dynamics_analysis.py extracted from the accepted source archive.
    master_path=ROOT
    code_hash=sha(master_path)
    tree_hash=sha(TREE_PATH);tree_check_hash=sha(TREE_CHECK)
    manifest=dict(schema=schema,auc_content_sha256=auc_hash,code_sha256=code_hash,
        tree_sha256=tree_hash,tree_validation_sha256=tree_check_hash,
        figure_sha256=figure_hash,cells_measured=len(cells),cells_modeled=int(mask.sum()),
        regulons_tested=len(names),regulons_displayed=len(order),
        primary_multiple_testing_family=217,fork_multiple_testing_family=868,
        primary_mode='none',chosen_K=int(data['modes']['none']['K']),
        created_utc=datetime.now(timezone.utc).isoformat())

    def array(group,key,value):
        values=np.asarray(value)
        group.create_dataset(key,data=values,compression='gzip' if values.ndim else None)

    def copy_group(source,destination,name):
        source.file.copy(source,destination,name=name)

    def equal_values(before,after):
        """Compare stored measurements exactly, including paired missing values."""
        if isinstance(before,h5py.Dataset):
            assert isinstance(after,h5py.Dataset) and before.shape==after.shape
            a=before.asstr()[()] if before.dtype.kind in 'OSU' else before[()]
            b=after.asstr()[()] if after.dtype.kind in 'OSU' else after[()]
            np.testing.assert_array_equal(a,b)
        else:
            assert set(before)==set(after)
            for key in before:equal_values(before[key],after[key])

    def assert_native(native):
        for key in ('F','B'):
            np.testing.assert_array_equal(native[key][()],data['tree']['model'][key])
        for key in ('edges','edge_id','alpha','t','root_distance','parent','residual'):
            np.testing.assert_array_equal(native[key][()],data['tree']['projection'][key])
        assert int(native.attrs['root'])==int(data['tree']['projection']['root'])

    fixed=('cells','regulons','auc','genes','coverage','original_regulons_json',
           'positive_genes','rank_gate','sensitivity_keep','tf_measured','tf_rna_counts',
           'total_counts_common_axis','total_counts_scoring_axis','native_sensitivity',
           'regulator_catalog_text')
    handles=[]
    WORK.mkdir(parents=True,exist_ok=True)
    try:
        gh,global_model=read_component('global_new','global_models');handles.append(gh)
        eh,early_fork=read_component('early_fork','early_fork');handles.append(eh)
        np.testing.assert_array_equal(global_model['regulons'].asstr()[()],names)
        np.testing.assert_array_equal(global_model['cells'].asstr()[()],cells[mask])
        np.testing.assert_array_equal(global_model['all_cells'].asstr()[()],cells)
        np.testing.assert_array_equal(global_model['analysis_mask'][()],mask)
        np.testing.assert_array_equal(global_model['all_observed_auc'][()],raw)
        np.testing.assert_array_equal(global_model['observed_auc'][()],raw[mask])
        assert global_model.attrs['input_auc_content_sha256']==auc_hash
        assert_native(global_model['native_tree'])
        np.testing.assert_array_equal(early_fork['regulons'].asstr()[()],names)
        early_fork_cells=pd.Index(early_fork['cells'].asstr()[()])
        assert len(early_fork_cells)==244 and early_fork_cells.isin(cells).all()
        np.testing.assert_array_equal(early_fork['observed_auc'][()],data['auc'].loc[early_fork_cells])
        early_fork_obs=read_frame(early_fork['obs'])
        np.testing.assert_array_equal(early_fork_obs.cell_id,early_fork_cells)
        early_fork_obs=early_fork_obs.set_index('cell_id')
        assert early_fork_obs.index.equals(early_fork_cells)
        np.testing.assert_array_equal(early_fork_obs.t,data['obs'].loc[early_fork_cells,'t'])
        assert data['obs'].loc[early_fork_cells,'seg'].isin([1,2]).all()
        with h5py.File(output,'r') as previous,h5py.File(staged,'w') as frozen:
            np.testing.assert_array_equal(previous['auc'][()],raw)
            frozen.attrs.update(schema=schema,seed=SEED,code_sha256=code_hash,
                auc_content_sha256=auc_hash,tree_sha256=tree_hash,
                primary_mode='none',chosen_K=int(data['modes']['none']['K']))
            for key in fixed:
                assert key in previous,'Missing fixed input: '+key
                copy_group(previous[key],frozen,key)
            put_frame(frozen.create_group('obs'),data['obs'])
            put_frame(frozen.create_group('bin_metadata'),data['bin_meta'])
            put_json(frozen,'segments_json',clean_json(data['segments']))

            info=json.loads(json.dumps(data['info']))
            old_tree=info.get('inputs',{}).get('tree',{})
            info['schema']=schema
            info.setdefault('legacy_tree_scoring_provenance',old_tree)
            info.setdefault('inputs',{})['tree']=dict(file=str(TREE_PATH.relative_to(ROOT)),sha256=tree_hash)
            info['inputs']['tree_validation']=dict(file=str(TREE_CHECK.relative_to(ROOT)),sha256=tree_check_hash)
            info['primary_cells']=int(mask.sum())
            info['measured_cells']=len(cells)
            info['family_size']=len(names)
            info['analysis_scope']=dict(biological_segments=BIO_SEGMENTS,connector_segment=3,
                connector_cells=7,modeling='10 branch-specific smooths; 30 equal-native-distance display bins',
                scoring='Unchanged fixed weighted AUCell, all 7355 cells and 217 networks',
                raw_sensitivity_flags='Historical score/native-axis input flags; not current association gates')
            info['current_tree']=dict(root=int(data['tree']['projection']['root']),
                nodes=int(data['tree']['model']['F'].shape[1]),edges=len(data['tree']['projection']['edges']),
                provenance=clean_json(data['tree'].get('provenance',{})),
                segments='native_tree/segments_json includes connector; root segments_json is the 10 biological display bands')
            put_json(frozen,'metadata_json',clean_json(info))

            native=frozen.create_group('native_tree')
            native.attrs['root']=int(data['tree']['projection']['root'])
            for key in ('F','B'):array(native,key,data['tree']['model'][key])
            for key,value in data['tree']['projection'].items():
                if key!='root':array(native,key,value)
            array(native,'segment_ids',data['obs'].seg)
            put_json(native,'segments_json',clean_json(data['tree']['segments']))
            put_json(native,'model_parameters_json',clean_json(data['tree']['model'].get('parameters',{})))
            put_json(native,'provenance_json',clean_json(data['tree'].get('provenance',{})))
            copy_group(global_model,frozen,'global_models')
            copy_group(early_fork,frozen,'early_fork')

            late=frozen.create_group('late_forks')
            for key in ('fork1','fork2','fork3'):
                source=previous['late_forks'][key] if 'late_forks' in previous else previous['local_association/analyses'][key]
                local_obs=read_frame(source['obs'])
                assert local_obs.index.isin(cells).all()
                t=data['obs'].loc[local_obs.index,'t'].to_numpy()
                np.testing.assert_allclose(t-t.min(),local_obs.v-local_obs.v.min(),rtol=0,atol=3e-15)
                np.testing.assert_array_equal(source['observed_auc'][()],data['auc'].loc[local_obs.index])
                copy_group(source,late,key)
                late[key].attrs['reuse_basis']='Exact cell/AUCell and within-fork relative-native-t equivalence; original local segment IDs are historical'
                late[key].attrs['verified_against_tree_sha256']=tree_hash
            support_key='late_branch_support' if 'late_branch_support' in previous else 'branch_support'
            copy_group(previous[support_key],frozen,'late_branch_support')
            frozen['late_branch_support'].attrs['scope']='Three retained late forks only; original fork-relative segment labels; descriptive library support'

            copy_group(previous['early_extension'],frozen,'early_extension')
            extension=frozen['early_extension']
            for key,frame in [('obs',data['early_obs']),('group_metadata',data['early_meta'])]:
                del extension[key]
                put_frame(extension.create_group(key),frame)
            early_info=json.loads(extension['metadata_json'].asstr()[()])
            early_info['coordinate_update']=dict(tree_sha256=tree_hash,
                fields=['obs/t','obs/seg','obs/bin','group_metadata/t_min','group_metadata/t_median',
                        'group_metadata/t_max','group_metadata/exact_root_cells'],
                measurements_unchanged=True,summary='Observed equal-library AUCell and native RNA values remain unchanged')
            del extension['metadata_json']
            put_json(extension,'metadata_json',clean_json(early_info))
            for key in ('cell_values','context_RNA','native_sensitivity','summaries','coverage'):
                if key in previous['early_extension']:equal_values(previous['early_extension'][key],extension[key])
            np.testing.assert_array_equal(extension['summaries/common_auc/mean'][()],early_all)

            code=frozen.create_group('analysis_code')
            old_code=previous['analysis_code']
            legacy=old_code['legacy'] if 'legacy' in old_code else old_code
            copy_group(legacy,code,'legacy')
            code.attrs['legacy_scope']='Original scoring and prior analysis source archive; not current fitted results or selection'
            raw_sources=code.create_group('raw_sources')
            # Set this storage path independently before running. Fig4A_global_new.py. Use fig4A_global_regulon_analysis.py, or the identical embedded source.
            sources={'master_source':master_path,'global_new_source':ROOT,
                     # Set this storage path independently before running. Fig4A_early_fork.py. Use fig4A_early_fork_analysis.py, or the identical embedded source.
                     'early_fork_source':ROOT}
            for key,path in sources.items():
                if path.is_file():content=path.read_bytes()
                else:
                    module={'global_new_source':'global_new','early_fork_source':'early_fork'}.get(key)
                    assert module in MODULE_SOURCES,'Missing exact source: '+str(path)
                    content=MODULE_SOURCES[module].encode('utf-8')
                code.create_dataset(key,data=content.decode('utf-8'),dtype=h5py.string_dtype('utf8'))
                code[key].attrs['sha256']=hashlib.sha256(content).hexdigest()
                array(raw_sources,path.name,np.frombuffer(content,dtype=np.uint8))

            final=frozen.create_group('final')
            put_strings(final,'regulons',order)
            write_frame(final.create_group('statistics'),stats)
            write_frame(final.create_group('fork_p_values'),fork_p)
            write_frame(final.create_group('diagnostics'),diag)
            put_frame(final.create_group('bin_metadata'),data['bin_meta'])
            arrays=dict(all_observed_bin_auc=data['observed'].loc[:,names],all_fitted_bin_auc=fitted_all,
                observed_bin_auc=observed,fitted_bin_auc=fitted,normalization_maxima=maxima,
                raw_relative_activity=raw_relative,relative_activity=relative,negative_fitted_bin_mask=negative,
                all_early_observed_auc=early_all,early_observed_auc=early,
                early_normalization_maxima=early_max,early_relative_activity=early_relative,
                early_loo_min=np.asarray(data['early_loo_min'])[:,names.get_indexer(selected)],
                early_loo_max=np.asarray(data['early_loo_max'])[:,names.get_indexer(selected)],analysis_mask=mask)
            for key,value in arrays.items():array(final,key,value)
            put_strings(final,'modeled_cells',cells[mask])
            put_strings(final,'all_regulons',names)
            put_json(final,'assessment_json',clean_json(check))
            put_json(final,'figure_qa_json',clean_json(check['figure_qa']))
            put_json(final,'delivery_manifest_json',clean_json(manifest))
            final.attrs['main_metric']='Whole-tree none-model fitted AUCell; max across 30 fitted bins per regulon'
            final.attrs['early_metric']='Observed equal-library AUCell; independent max across 7 stage/state groups per regulon'
            final.attrs['primary_selection']='status=ok, BH217 q<1e-4, fitted amplitude>0.025; no fork min-P or named-regulator gate'
            final.attrs['fork_labels']='Any fork BH868 q<1e-4; separate evidence family, not a primary screen'
            final.attrs['negative_fit_policy']='Raw values saved; main display masks values below -1e-12 and marks grey crosses; tiny negatives display as zero'
            frozen.flush()
            for key in fixed:equal_values(previous[key],frozen[key])
    finally:
        for handle in handles:handle.close()

    # Reopen the completed file before replacing the previous frozen input.
    with h5py.File(staged,'r') as frozen:
        assert frozen.attrs['schema']==schema
        stored_auc=np.asarray(frozen['auc'][()],dtype='<f8')
        assert hashlib.sha256(stored_auc.tobytes()).hexdigest()==auc_hash
        np.testing.assert_array_equal(stored_auc,raw)
        np.testing.assert_array_equal(frozen['regulons'].asstr()[()],names)
        np.testing.assert_array_equal(frozen['cells'].asstr()[()],cells)
        stored_obs=input_read_frame(frozen['obs'],index=cells)
        for column in ('seg','bin','t','model_ok','included_in_association'):
            np.testing.assert_array_equal(stored_obs[column],data['obs'][column])
        for column in data['obs'].columns[data['obs'].columns.str.startswith('path_')]:
            np.testing.assert_array_equal(stored_obs[column],data['obs'][column])
        assert_native(frozen['native_tree'])
        assert_native(frozen['global_models/native_tree'])
        np.testing.assert_array_equal(frozen['global_models/cells'].asstr()[()],cells[mask])
        np.testing.assert_array_equal(frozen['global_models/observed_auc'][()],raw[mask])
        np.testing.assert_array_equal(frozen['final/regulons'].asstr()[()],order)
        np.testing.assert_array_equal(frozen['final/fitted_bin_auc'][()],fitted)
        np.testing.assert_array_equal(frozen['final/relative_activity'][()],relative)
        np.testing.assert_array_equal(frozen['early_extension/summaries/common_auc/mean'][()],early_all)
        np.testing.assert_array_equal(frozen['final/early_relative_activity'][()],early_relative)
        early_cells=pd.Index(frozen['early_extension/cells'].asstr()[()])
        updated_early=input_read_frame(frozen['early_extension/obs'],index=early_cells)
        for column in ('t','seg','bin'):
            np.testing.assert_array_equal(updated_early[column],data['obs'].loc[early_cells,column])
        assert read_frame(frozen['final/statistics']).shape[0]==217
        assert read_frame(frozen['final/fork_p_values']).shape==(217,4)
        assert not any(key in frozen for key in ('models','local_association','matched_late_source_review','previous_displayed_regulons'))
        assert json.loads(frozen['final/delivery_manifest_json'].asstr()[()])['auc_content_sha256']==auc_hash
    assert sha(master_path)==code_hash and sha(TREE_PATH)==tree_hash and sha(TREE_CHECK)==tree_check_hash
    for name,digest in figure_hash.items():assert sha(DESTINATION/name)==digest
    os.replace(staged,output)
    return manifest

#6. =========== Acceptance checks, reproducibility and controlled cleanup
def acceptance(data,stats,fork_p,diag,order,figures):
    from scipy.stats import false_discovery_control
    valid=stats.status.eq('ok')
    np.testing.assert_allclose(stats.q_global,false_discovery_control(stats.p_value.where(valid,1)),atol=1e-15,rtol=1e-13)
    expected=valid & stats.q_global.lt(Q_CUT) & stats.A.gt(A_CUT)
    assert expected.equals(stats.selected)
    q=false_discovery_control(fork_p.to_numpy().ravel()).reshape(217,4)
    np.testing.assert_allclose(q,stats[['q_early','q_fork1','q_fork2','q_fork3']].to_numpy(),atol=1e-15,rtol=1e-13)
    assert data['modes']['none']['K']==20
    assert stats.basis_flags.eq(0).all(),'Basis-capacity flags require another model review before delivery'
    assert data['bin_meta'].cells.min()>=20 and len(data['bin_meta'])==30
    assert data['obs'].stage.nunique()==9 and 'E12.5' not in set(data['obs'].stage)
    assert len(data['early_obs'])==1234 and len(data['early_meta'])==7
    observed=data['early_mean']
    mrna=data['early_rna']
    biological={}
    # These are post-analysis sanity checks, never selection or tuning criteria.
    for gene in ('Mki67','Top2a','Dcx','Gap43'):
        biological[gene+'_early_observed_log1p_CP10k']=mrna[gene].tolist()
    for tf in ('E2f7(+)','Foxm1(+)'):
        i=data['names'].get_loc(tf)
        biological[tf]=dict(observed_AUC=observed[tf].tolist(),
            e11_CVG_greater_than_SGN=bool(observed.loc[3,tf]>observed.loc[4,tf]),
            e13_CVG_greater_than_SGN=bool(observed.loc[5,tf]>observed.loc[6,tf]),
            e11_leave_library_out_ranges_separated=bool(data['early_loo_min'][3,i]>data['early_loo_max'][4,i]),
            e13_leave_library_out_ranges_separated=bool(data['early_loo_min'][5,i]>data['early_loo_max'][6,i]),
            displayed=bool(stats.loc[tf,'selected']))
    mode_summary={}
    for mode,values in data['modes'].items():
        st=values['stats'];qv=adjusted_pvalues(st.p_value.where(st.status.eq('ok'),1))[1]
        mode_summary[mode]=dict(successful=int(st.status.eq('ok').sum()),K=values['K'],
            separate_screen_count=int(((qv<Q_CUT)&st.A.gt(A_CUT)&st.status.eq('ok')).sum()),
            primary_selected_still_passing=int(stats.loc[order,mode+'_threshold_pass'].sum()),
            primary_selected_q_passing=int(stats.loc[order,mode+'_q'].lt(Q_CUT).sum()),
            primary_selected_A_passing=int(stats.loc[order,mode+'_A'].gt(A_CUT).sum()),
            median_selected_profile_rho=float(stats.loc[order,mode+'_profile_rho'].median()))
    return dict(primary_method='whole-tree branch-specific cubic Gaussian ML smooths vs constant; F test; BH217; A>0.025',
        status='passed_with_documented_limitations',cells_measured=7355,cells_modeled=7348,connector=7,
        regulons_tested=217,regulons_displayed=len(order),fork_tests=868,
        biological_segments=10,display_bins=30,minimum_bin_cells=int(data['bin_meta'].cells.min()),
        selected_residual_flagged=stats.loc[order].query('residual_flags>0').index.tolist(),
        all_residual_structure_flags=int(diag.effective_structure_flag.sum()),
        raw_residual_structure_flags=int(diag.structure_flag.sum()),
        basis_flags=int(diag.basis_flag.sum()),model_sensitivity=mode_summary,
        sensitivity_profile_metric='Trajectory-only seg+smooth component centered to the observed mean; fitted nuisance terms excluded',
        fork_significant_regulons={k:int((q[:,j]<Q_CUT).sum()) for j,k in enumerate(fork_p)},
        biological_sanity=biological,figure_qa=figures,
        limitations=['Early candidate-arm effects are partially confounded with developmental stage; early same-age SGN support is sparse.',
          'Associations are conditional cell-level exploratory results on the accepted expression-derived tree; libraries are not assumed to be animals.',
          'The fixed SCENIC network was reused; missing early regulators cannot be inferred from this network.',
          'Gaussian curves are unconstrained: one negative fitted bin is marked explicitly and its raw value is retained.',
          'Local residual-structure flags remain in a few selected regulons; they are annotated and are not automatic exclusion gates.'])

def cleanup(keep,files):
    if keep:return dict(removed=[],bytes_freed=0,retained_for='requested intermediate inspection')
    names=['work','early_work','global_new_work','Fig4A_global_new.py','Fig4A_early_fork.py',
           'Fig4A_global_new.h5','Fig4A_early_fork.h5','assemble_snippet.py']
    names += [name for name in ('Fig4A_1.png','Fig4A_2.png','Fig4A_3.png') if name not in files]
    removed=[];size=0;base=_release_cleanup_target(DESTINATION).resolve()
    for name in names:
        target=(DESTINATION/name).resolve()
        assert target.parent==base and target!=base
        if not target.exists():continue
        count=sum(p.stat().st_size for p in _release_cleanup_target(target).rglob('*') if p.is_file()) if target.is_dir() else target.stat().st_size
        if target.is_dir():shutil.rmtree(_release_cleanup_target(target))
        else:_release_cleanup_target(target).unlink()
        removed.append(str(target));size+=count
    return dict(removed=removed,bytes_freed=size)

def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--assemble',action='store_true',help='Merge validated component fits into final frozen H5')
    parser.add_argument('--recompute',action='store_true',help='Refit global/early-fork models using embedded exact analysis sources')
    parser.add_argument('--keep-intermediates',action='store_true')
    args=parser.parse_args();WORK.mkdir(parents=True,exist_ok=True)
    if args.recompute:
        assert set(MODULE_SOURCES)=={'global_new','early_fork'}
        for name,source in MODULE_SOURCES.items():
            path=DESTINATION/f'Fig4A_{name}.py';path.write_bytes(source.encode('utf-8'))
            subprocess.run([sys.executable,str(path)],check=True,cwd=ROOT)
    data=load_data();stats,fork_p,diag,order=screen(data)
    figures=render(data,stats,order)
    check=acceptance(data,stats,fork_p,diag,order,figures)
    if args.assemble or args.recompute:
        assemble(data,stats,fork_p,diag,order,check)
    else:
        # Set this storage path independently before running. Fig4A.h5.
        with h5py.File(ROOT,'r') as frozen:
            assert frozen.attrs.get('schema')=='Fig4A_new_tree_v1','Run --assemble once to freeze the new model'
            assert frozen['final/regulons'].asstr()[()].tolist()==order
            previous=json.loads(frozen['final/delivery_manifest_json'].asstr()[()])
            for name in figures['files']:assert sha(DESTINATION/name)==previous['figure_sha256'][name]
    cleaned=cleanup(args.keep_intermediates,figures['files'])
    # Set this storage path independently before running. Fig4A.h5.
    with h5py.File(ROOT,'a') as final:
        history=json.loads(final['final/cleanup_history_json'].asstr()[()]) if 'final/cleanup_history_json' in final else []
        if 'final/cleanup_json' in final:
            history.append(json.loads(final['final/cleanup_json'].asstr()[()]))
            del final['final/cleanup_json']
        if 'final/cleanup_history_json' in final:del final['final/cleanup_history_json']
        put_json(final['final'],'cleanup_history_json',history)
        put_json(final['final'],'cleanup_json',cleaned)
    print(json.dumps(dict(status='complete',selected=len(order),files=figures['files'],
                         residual_flags=check['selected_residual_flagged'],cleanup=cleaned),ensure_ascii=False),flush=True)

if __name__=='__main__':main()


## Original standalone source: fig4A_fixed_network_scoring.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# Archived fixed-network native-axis AUCell/scoring support; not the accepted final global or fork fit.
# Original source label: analysis_code:legacy_early_source
# Original source SHA256: 00fc1ad9a85e105c29014b9bb1d7a337d6cae8cd68602a2632a6c7ca453b8803
# Only storage addresses and release comments were changed; analysis was not executed.

"""Early Fig4A observed summaries and fixed-network native-axis sensitivity.

Uses existing local raw source RNA and frozen 217 AUCell signatures only.
No GRN inference, cell exclusion, TF-expression gate or association test.
"""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

#1. =========== Shared configuration
from pathlib import Path
import os
import sys
import json
import hashlib
import time

sys.dont_write_bytecode = True
# Set this storage path independently before running.
ROOT = ROOT
# Set this storage path independently before running.
DESTINATION = ROOT
# Set this storage path independently before running. Fig4A.h5.
INPUT = ROOT
# Set this storage path independently before running. Fig4A_early.h5.
OUTPUT = ROOT
# Set this storage path independently before running.
SOURCE_DIR = ROOT
SEED = 42
AUC_THRESHOLD = 0.05
MIN_TARGETS = 20
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"

import h5py
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata

STUDIES = ("GSE178931", "GSE231513")
GROUPS = (
    ("E9.5", "CVG neuroblast", "E9.5 CVG neuroblast"),
    ("E10.5", "O-2: delaminating otic/CVG neuroblast", "E10.5 delaminating otic/CVG neuroblast"),
    ("E10.5", "O-3: developing CVG neuron (not SGN-specific)", "E10.5 developing CVG neuron"),
    ("E11.5", "Undifferentiated / proliferative CVG", "E11.5 proliferative CVG"),
    ("E11.5", "Differentiating SGN", "E11.5 differentiating SGN"),
    ("E13.5", "Late CVG", "E13.5 late CVG"),
    ("E13.5", "Differentiating SGN", "E13.5 differentiating SGN"),
)
CONTEXT_MARKERS = ("Mki67", "Top2a", "Neurog1", "Neurod1", "Eya1", "Dcx", "Gap43",
                   "Isl1", "Gata3", "Pou4f1", "Sox2", "Sox9")


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_vector(node):
    return node.asstr()[()] if node.dtype.kind in "OSU" else node[()]


def strings(group, key, values):
    group.create_dataset(key, data=np.asarray(values, dtype=object),
                         dtype=h5py.string_dtype("utf8"), compression="gzip")


def frame(group, values):
    for key in values:
        col = values[key]
        if col.dtype.kind in "biuf":
            group.create_dataset(key, data=col.to_numpy(), compression="gzip")
        else:
            strings(group, key, col.fillna("").astype(str))
            if col.isna().any():
                group.create_dataset(key + "__missing", data=col.isna().to_numpy(), compression="gzip")


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    return value


def json_dataset(group, key, value):
    group.create_dataset(key, data=json.dumps(json_safe(value), ensure_ascii=False, allow_nan=False),
                         dtype=h5py.string_dtype("utf8"))


def rho(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    a, b = rankdata(x[mask]), rankdata(y[mask])
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


#2. =========== Faithful weighted AUCell implementation
def create_rankings(counts, genes):
    """Same seeded column shuffle and zero-based first-tie ranks as pySCENIC."""
    expression = pd.DataFrame(counts.toarray(), columns=genes)
    return (expression.sample(frac=1.0, replace=False, axis=1, random_state=SEED)
            .rank(axis=1, ascending=False, method="first", na_option="bottom")
            .astype("uint32") - 1)


def weighted_auc1d(ranking, weights, cutoff, max_auc):
    """Arithmetic copied from local ctxcore.recovery.weighted_auc1d."""
    keep = ranking < cutoff
    x, y = ranking[keep], weights[keep]
    order = np.argsort(x)
    x = np.concatenate((x[order], np.full((1,), cutoff, dtype=np.int_)))
    y = y[order].cumsum()
    return np.sum(np.diff(x) * y) / max_auc


def score_rankings(rankings, signatures):
    cutoff = int(round(AUC_THRESHOLD * rankings.shape[1])) - 1
    assert cutoff > 0
    result = np.full((len(rankings), len(signatures)), np.nan)
    genes = rankings.columns.to_numpy()
    coverage = []
    for j, signature in enumerate(signatures):
        shared = np.array([g in signature for g in genes])
        matched = int(shared.sum())
        valid = matched >= MIN_TARGETS
        coverage.append((matched, matched / len(signature), valid))
        if not valid:
            continue
        weights = np.asarray([signature[g] for g in genes[shared]], dtype=np.float64)
        values = rankings.iloc[:, shared].to_numpy()
        maximum = float((cutoff + 1) * weights.sum())
        result[:, j] = [weighted_auc1d(row, weights, cutoff, maximum) for row in values]
    return result, coverage, cutoff


#3. =========== Equal-library observed summaries
def aggregate(values, obs, group_metadata):
    means, lower, upper, library_rows, library_values = [], [], [], [], []
    for row in group_metadata.to_dict("records"):
        local = obs[obs.group.eq(row["group"])]
        library_means = []
        for library, subset in local.groupby("library", sort=True):
            block = values[subset.index.to_numpy()]
            count = np.isfinite(block).sum(axis=0)
            mean = np.divide(np.nansum(block, axis=0), count,
                             out=np.full(values.shape[1], np.nan), where=count > 0)
            library_means.append(mean)
            library_values.append(mean)
            library_rows.append({"group": row["group"], "stage": row["stage"], "state": row["state"],
                                 "study": row["study"], "library": library, "cells": len(subset)})
        matrix = np.asarray(library_means)
        finite_count = np.isfinite(matrix).sum(axis=0)
        total = np.nansum(matrix, axis=0)
        mean = np.divide(total, finite_count, out=np.full(values.shape[1], np.nan), where=finite_count > 0)
        means.append(mean)
        lo, hi = np.full(values.shape[1], np.nan), np.full(values.shape[1], np.nan)
        if len(matrix) > 1:
            n = finite_count[None, :] - np.isfinite(matrix)
            deleted = np.divide(total[None, :] - np.nan_to_num(matrix), n,
                                out=np.full_like(matrix, np.nan), where=n > 0)
            supported = np.isfinite(deleted).any(axis=0)
            lo[supported] = np.min(np.where(np.isfinite(deleted[:, supported]), deleted[:, supported], np.inf), axis=0)
            hi[supported] = np.max(np.where(np.isfinite(deleted[:, supported]), deleted[:, supported], -np.inf), axis=0)
        lower.append(lo)
        upper.append(hi)
    return {"mean": np.asarray(means), "loo_min": np.asarray(lower), "loo_max": np.asarray(upper),
            "library_means": np.asarray(library_values), "library_metadata": pd.DataFrame(library_rows)}


def main():
    started = time.perf_counter()
    with h5py.File(INPUT, "r") as h:
        cells = h["cells"].asstr()[()]
        names = h["regulons"].asstr()[()]
        genes = h["genes"].asstr()[()]
        all_auc = h["auc"][()]
        obs = pd.DataFrame({k: read_vector(h["obs"][k]) for k in ("stage", "study", "library", "type", "t", "seg", "bin")})
        original = {r["name"]: r for r in json.loads(h["original_regulons_json"].asstr()[()])}
    obs["cell_id"] = cells
    early = obs[obs.stage.isin(["E9.5", "E10.5", "E11.5", "E13.5"])].copy()
    obs_positions = early.index.to_numpy()
    early.reset_index(drop=True, inplace=True)
    assert set(early.study) == set(STUDIES)
    early["group"] = -1
    rows = []
    for j, (stage, state, label) in enumerate(GROUPS):
        mask = early.stage.eq(stage) & early.type.eq(state)
        early.loc[mask, "group"] = j
        part = early[mask]
        assert len(part) and part.study.nunique() == 1
        rows.append({"group": j, "label": label, "stage": stage, "state": state,
                     "study": str(part.study.iloc[0]), "cells": len(part),
                     "libraries": part.library.nunique(), "t_min": part.t.min(),
                     "t_median": part.t.median(), "t_max": part.t.max(),
                     "exact_root_cells": int(part.t.eq(0).sum()),
                     "leave_one_library_out_available": part.library.nunique() > 1})
    assert (early.group >= 0).all() and len(early) == 1234 and len(names) == 217
    group_metadata = pd.DataFrame(rows)
    signatures = [original[str(name)]["gene2weight"] for name in names]
    common_gene_set = set(genes)
    common_signatures = [{g: w for g, w in sig.items() if g in common_gene_set} for sig in signatures]
    frozen = all_auc[obs_positions]
    native_full = np.full_like(frozen, np.nan)
    native_fixed = np.full_like(frozen, np.nan)
    tf_rna = np.full_like(frozen, np.nan)
    tf_detection = np.full_like(frozen, np.nan)
    target_rna = np.full_like(frozen, np.nan)
    target_detection = np.full_like(frozen, np.nan)
    coverage_rows, source_rows, agreement_rows = [], [], []
    hashes = {str(INPUT.relative_to(ROOT)): sha256(INPUT)}

    #4. =========== Fixed-network native-axis transferability
    for study in STUDIES:
        path_axes = SOURCE_DIR / f"{study}_axes.npz"
        path_counts = SOURCE_DIR / f"{study}_counts.npz"
        source = np.load(path_axes, allow_pickle=False)
        counts = sparse.load_npz(path_counts).tocsr()
        native_genes = source["genes"].astype(str)
        source_cells = pd.Index(source["cells"].astype(str))
        assert len(set(native_genes)) == len(native_genes)
        assert np.isfinite(counts.data).all() and (counts.data >= 0).all()
        chosen = np.flatnonzero(early.study.eq(study))
        positions = source_cells.get_indexer(early.iloc[chosen].cell_id)
        assert (positions >= 0).all() and source_cells.is_unique
        counts = counts[positions]
        assert counts.shape == (len(chosen), len(native_genes))
        gene_lookup = {g: i for i, g in enumerate(native_genes)}
        assert all(g in gene_lookup for g in genes)
        common_counts = counts[:, [gene_lookup[g] for g in genes]]
        ranked_common = create_rankings(common_counts, genes)
        common_scores, _, common_cutoff = score_rankings(ranked_common, common_signatures)
        error = float(np.max(np.abs(common_scores - frozen[chosen])))
        if error > 1e-12:
            raise ValueError(f"Common-axis AUCell reproduction failed for {study}: {error}")
        del ranked_common, common_counts
        expressed = np.asarray((counts > 0).sum(axis=0)).ravel() > 0
        ranking_genes = native_genes[expressed]
        ranking = create_rankings(counts[:, expressed], ranking_genes)
        full, full_coverage, cutoff = score_rankings(ranking, signatures)
        fixed, fixed_coverage, _ = score_rankings(ranking, common_signatures)
        native_full[chosen], native_fixed[chosen] = full, fixed
        total = np.asarray(counts.sum(axis=1)).ravel()
        normalized = counts.astype(np.float64).multiply(10000 / total[:, None]).tocsr()
        normalized.data = np.log1p(normalized.data)
        positive = np.asarray((counts > 0).sum(axis=1)).ravel()
        native_gene_set = set(native_genes)
        for j, name in enumerate(names):
            tf = str(name).rsplit("(", 1)[0]
            if tf in gene_lookup:
                tf_rna[chosen, j] = normalized[:, gene_lookup[tf]].toarray().ravel()
                tf_detection[chosen, j] = (counts[:, gene_lookup[tf]].toarray().ravel() > 0)
            matched = [g for g in signatures[j] if g in gene_lookup]
            ix = [gene_lookup[g] for g in matched]
            weights = np.asarray([signatures[j][g] for g in matched])
            if len(matched):
                target_rna[chosen, j] = np.asarray(normalized[:, ix] @ weights).ravel() / weights.sum()
                target_detection[chosen, j] = np.asarray((counts[:, ix] > 0) @ weights).ravel() / weights.sum()
            full_n, full_cov, full_valid = full_coverage[j]
            fixed_n, fixed_cov, fixed_valid = fixed_coverage[j]
            coverage_rows.append({"study": study, "regulon": str(name), "original_targets": len(signatures[j]),
                                  "common_targets": len(common_signatures[j]), "native_measured_targets": len(matched),
                                  "native_measured_coverage": len(matched) / len(signatures[j]),
                                  "native_expressed_targets": full_n, "native_expressed_coverage": full_cov,
                                  "native_full_scoreable": full_valid, "fixed_common_native_expressed_targets": fixed_n,
                                  "fixed_common_native_coverage": fixed_cov, "fixed_common_native_scoreable": fixed_valid,
                                  "TF_measured_native": tf in native_gene_set})
            agreement_rows.append({"study": study, "regulon": str(name), "cells": len(chosen),
                                   "common_vs_native_full_rho": rho(frozen[chosen, j], full[:, j]),
                                   "common_vs_native_fixed_targets_rho": rho(frozen[chosen, j], fixed[:, j]),
                                   "common_mean": float(frozen[chosen, j].mean()),
                                   "native_full_mean": float(np.mean(full[:, j])),
                                   "native_fixed_targets_mean": float(np.mean(fixed[:, j]))})
        source_rows.append({"study": study, "cells": len(chosen), "native_eligible_genes": len(native_genes),
                            "native_ranking_genes": len(ranking_genes), "native_rank_cutoff": cutoff,
                            "common_ranking_genes": len(genes), "common_rank_cutoff": common_cutoff,
                            "common_AUC_max_abs_reproduction_error": error,
                            "native_rank_gate_cells": int((positive > cutoff).sum()),
                            "minimum_positive_genes": int(positive.min()),
                            "native_ranking_gene_sha256": hashlib.sha256("\n".join(ranking_genes).encode()).hexdigest()})
        for p in (path_axes, path_counts):
            hashes[str(p.relative_to(ROOT))] = sha256(p)
        print(f"{study}: {len(chosen)} cells, {len(ranking_genes)} native ranking genes; frozen AUCell error {error:.3g}", flush=True)
        del ranking, counts, normalized

    #5. =========== Observed stage-state summaries and descriptive checks
    matrices = {"common_auc": frozen, "native_full_auc": native_full, "native_fixed_targets_auc": native_fixed,
                "native_TF_log1p_CP10k": tf_rna, "native_TF_detection_fraction": tf_detection,
                "native_target_weighted_log1p_CP10k": target_rna,
                "native_target_weighted_detection_fraction": target_detection}
    summaries = {name: aggregate(values, early, group_metadata) for name, values in matrices.items()}
    loo_check = summaries["common_auc"]
    assert np.isfinite(loo_check["mean"]).all()
    assert np.isnan(loo_check["loo_min"][0]).all() and np.isnan(loo_check["loo_max"][0]).all()
    assert np.isfinite(loo_check["loo_min"][1:]).all()
    group_agreement = []
    for j, name in enumerate(names):
        a = summaries["common_auc"]["mean"][:, j]
        for study in STUDIES:
            mask = group_metadata.study.eq(study).to_numpy()
            group_agreement.append({"regulon": str(name), "study": study, "groups": int(mask.sum()),
                                    "common_vs_native_full_group_rho": rho(a[mask], summaries["native_full_auc"]["mean"][mask, j]),
                                    "common_vs_native_fixed_group_rho": rho(a[mask], summaries["native_fixed_targets_auc"]["mean"][mask, j]),
                                    "reason": "descriptive_only" if mask.sum() >= 3 else "fewer_than_three_groups"})
    manifest = {"schema": "Fig4A_early_v1", "family": 217, "early_cells": len(early), "groups": len(GROUPS),
                "source_summaries": source_rows, "input_sha256": hashes,
                "weighted_AUCell": {"seed": SEED, "auc_threshold": AUC_THRESHOLD, "normalize": False,
                                    "weighted": True, "minimum_matched_targets": MIN_TARGETS,
                                    "rank_creation": "Seeded pandas column shuffle; descending first-tie ranks minus one",
                                    "cutoff": "int(round(0.05 * ranked_gene_count)) - 1",
                                    "arithmetic": "Local ctxcore.recovery.weighted_auc1d; sorted ranks and cumulative weights",
                                    "native_axis": "All source-native eligible genes expressed in at least one supplied source cell",
                                    "native_full": "Same original frozen network, subset only to available native ranking targets",
                                    "native_fixed_targets": "Frozen common-axis target subset, scored against native ranking background"},
                "summary_definition": "Cell means per stage-state-library, then equal arithmetic mean over available libraries",
                "LOO_definition": "Minimum and maximum equal-library means after deleting each library in turn; descriptive range, not CI",
                "RNA_definition": "Natural log1p(CP10k) with denominator all eligible native source features; no TF-expression gate",
                "target_RNA_definition": "Original positive target weights, weighted mean over measured source-native targets",
                "boundaries": ["E9.5 has 12 cells in one library; leave-one-library-out is unavailable.",
                               "GSE231513 provides E10.5 only; six plates are not assumed six independent animals.",
                               "Stage/state summaries are observed descriptions, not tree-ordered tests or causal effects.",
                               "Native full scores add available targets and change ranking background; native fixed-target scores isolate ranking-background sensitivity.",
                               "A missing source feature remains missing, not zero.",
                               "Native-axis scores are sensitivity analyses; common-axis frozen AUCell remains the displayed measurement.",
                               "No network re-inference, association P values, TF name selection or count target is used."],
                "seconds": time.perf_counter() - started}
    with h5py.File(OUTPUT, "w") as h:
        strings(h, "regulons", names)
        strings(h, "cells", early.cell_id)
        frame(h.create_group("obs"), early)
        frame(h.create_group("group_metadata"), group_metadata)
        frame(h.create_group("library_metadata"), loo_check["library_metadata"])
        for key, value in summaries.items():
            g = h.create_group("summaries/" + key)
            for field in ("mean", "loo_min", "loo_max", "library_means"):
                g.create_dataset(field, data=value[field], compression="gzip")
        for key, values in matrices.items():
            h.create_dataset("cell_values/" + key, data=values, compression="gzip")
        frame(h.create_group("coverage"), pd.DataFrame(coverage_rows))
        frame(h.create_group("cell_score_agreement"), pd.DataFrame(agreement_rows))
        frame(h.create_group("group_score_agreement"), pd.DataFrame(group_agreement))
        json_dataset(h, "metadata_json", manifest)
    with h5py.File(OUTPUT, "r") as h:
        assert h["summaries/common_auc/mean"].shape == (7, 217)
        np.testing.assert_array_equal(h["cell_values/common_auc"][()], frozen)
    print(json.dumps({"output": str(OUTPUT), "cells": len(early), "groups": len(GROUPS),
                      "libraries_per_group": group_metadata.libraries.tolist(),
                      "native_full_finite_columns": np.isfinite(native_full).all(axis=0).sum(),
                      "seconds": time.perf_counter() - started}, default=int), flush=True)


def add_context_to_output():
    """Predefined RNA context only; no change to candidate selection."""
    with h5py.File(OUTPUT, "r") as h:
        early = pd.DataFrame({k: read_vector(h["obs"][k]) for k in
                              ("cell_id", "stage", "study", "library", "type", "group")})
        groups = pd.DataFrame({k: read_vector(h["group_metadata"][k]) for k in
                               ("group", "label", "stage", "state", "study", "cells", "libraries")})
    rna = np.full((len(early), len(CONTEXT_MARKERS)), np.nan)
    detection = np.full_like(rna, np.nan)
    availability = []
    for study in STUDIES:
        source = np.load(SOURCE_DIR / f"{study}_axes.npz", allow_pickle=False)
        counts = sparse.load_npz(SOURCE_DIR / f"{study}_counts.npz").tocsr()
        genes = source["genes"].astype(str)
        positions = pd.Index(source["cells"].astype(str)).get_indexer(early.loc[early.study.eq(study), "cell_id"])
        chosen = np.flatnonzero(early.study.eq(study))
        assert (positions >= 0).all()
        counts = counts[positions]
        totals = np.asarray(counts.sum(axis=1)).ravel()
        lookup = {g: j for j, g in enumerate(genes)}
        for j, gene in enumerate(CONTEXT_MARKERS):
            measured = gene in lookup
            availability.append({"study": study, "gene": gene, "measured": measured,
                                 "source_cells": len(chosen)})
            if measured:
                values = counts[:, lookup[gene]].toarray().ravel()
                rna[chosen, j] = np.log1p(values * 10000 / totals)
                detection[chosen, j] = values > 0
    results = {"log1p_CP10k": aggregate(rna, early, groups),
               "detection_fraction": aggregate(detection, early, groups)}
    with h5py.File(OUTPUT, "a") as h:
        if "context_RNA" in h:
            del h["context_RNA"]
        context = h.create_group("context_RNA")
        strings(context, "genes", CONTEXT_MARKERS)
        frame(context.create_group("availability"), pd.DataFrame(availability))
        for name, values in results.items():
            g = context.create_group(name)
            for field in ("mean", "loo_min", "loo_max", "library_means"):
                g.create_dataset(field, data=values[field], compression="gzip")
        context.create_dataset("cell_log1p_CP10k", data=rna, compression="gzip")
        context.create_dataset("cell_detection", data=detection, compression="gzip")
        json_dataset(context, "metadata_json", {
            "purpose": "Predefined proliferation/neurogenic/CVG-state descriptive context, never a selector or TF quota",
            "normalization": "Natural log1p(count / all eligible source-native counts * 10000)",
            "aggregation": "Per-library cell means followed by equal library mean within the fixed seven stage-state groups",
            "missingness": "Absent source features remain NaN; measured zero remains zero",
            "LOO": "Descriptive leave-one-library-out range; E9.5 single library unavailable",
        })
        metadata = json.loads(h["metadata_json"].asstr()[()])
        metadata["code_sha256"] = sha256(Path(__file__))
        metadata["predefined_context_genes"] = list(CONTEXT_MARKERS)
        metadata["context_used_for_selection"] = False
        del h["metadata_json"]
        json_dataset(h, "metadata_json", metadata)
    print("Predefined early context RNA stored for 12 genes and seven groups", flush=True)


if __name__ == "__main__":
    if "--context-only" not in sys.argv:
        main()
    add_context_to_output()


## Original standalone source: fig4A_native_association_support.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# Archived native-axis local association and R-model support; retained as a supporting engine, not the accepted final fit.
# Original source label: analysis_code:legacy_local_source
# Original source SHA256: 218e314b7c14004f720df71bc67893cde04467cf1c2ac128e3c58653782d784e
# Only storage addresses and release comments were changed; analysis was not executed.

"""Local trajectory association for all 217 frozen Fig4A SCENIC signatures.

Early: GSE178931, native segment1, E9.5/E11.5/E13.5 only. Three forks:
direct daughter segments on their common native-pseudotime support. Gaussian
ML GAMs compare the same nuisance terms in null/full models. K10 is upgraded
to K20 for the complete analysis/all nuisance modes if any basis-saturation
diagnostic triggers. Source/age limitations and descriptive block deletion
are reported separately; cells and SS2 plates are not biological replicates.
"""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

#1. =========== Shared configuration and frozen input
from pathlib import Path
import argparse
import concurrent.futures
import hashlib
import json
import os
import subprocess
import sys
import time

sys.dont_write_bytecode = True
# Set this storage path independently before running.
ROOT = ROOT
# Set this storage path independently before running.
DIRECTORY = ROOT
# Set this storage path independently before running.
WORK = ROOT
# Set this storage path independently before running. Fig4A.h5.
INPUT = ROOT
# Set this storage path independently before running. Fig4A_local.h5.
OUTPUT = ROOT
# Set this storage path independently before running. Rscript.exe.
RSCRIPT = ROOT
SEED = 42
WORKERS = 3
K_INITIAL, K_EXTENSION = 10, 20
FAMILY_SIZE = 217
MODES = ("none", "depth", "source", "source_depth")
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
for key in ("TMP", "TEMP"):
    os.environ[key] = str(WORK)
os.environ["LC_ALL"] = os.environ["LANG"] = "C"

import h5py
import numpy as np
import pandas as pd
from scipy.stats import f as f_distribution, rankdata


R_SOURCE = r"""
#1. =========== Fixed Gaussian ML models and diagnostic settings
suppressPackageStartupMessages(library(mgcv))
args <- commandArgs(trailingOnly=TRUE)
stopifnot(length(args)==8L)
CELLS<-args[1]; AUC<-args[2]; PREFIX<-args[3]; KIND<-args[4]
MODE<-args[5]; K<-as.integer(args[6]); INDICES<-as.integer(strsplit(args[7],',',fixed=TRUE)[[1]])
VALIDATE<-identical(args[8],'1')
SEED<-42L; REPS<-500L; SUBSAMPLE<-2000L; GRID_N<-51L
options(stringsAsFactors=FALSE,digits=17)
d<-read.csv(CELLS,row.names=1,check.names=FALSE)
con<-gzfile(AUC,'rt'); Y<-tryCatch(as.matrix(read.delim(con,row.names=1,check.names=FALSE)),finally=close(con))
storage.mode(Y)<-'double'
stopifnot(identical(rownames(d),rownames(Y)),ncol(Y)==217L,all(is.finite(Y)),
          all(is.finite(d$v)),all(is.finite(d$log_detected)),all(Y>=0 & Y<=1))
d$study<-factor(d$study); d$branch<-factor(as.character(d$branch))
stopifnot(all(table(d$branch)>=20L),all(vapply(split(d$v,d$branch),function(v)length(unique(v))>=K,logical(1))))
source_term<-if(nlevels(d$study)>1L)'study'else character()
terms<-c(if(MODE %in% c('source','source_depth'))source_term,
         if(MODE %in% c('depth','source_depth'))'log_detected')
C<-if(length(terms))paste(terms,collapse=' + ')else'1'
if(KIND=='early') {
  f0<-as.formula(paste('y ~',C))
  f1<-as.formula(paste('y ~',C,sprintf("+ s(v,bs='cr',k=%d)",K)))
} else {
  stopifnot(nlevels(d$branch)==2L)
  f0<-as.formula(paste('y ~',C,sprintf("+ s(v,bs='cr',k=%d)",K)))
  f1<-as.formula(paste('y ~',C,sprintf("+ branch + s(v,by=branch,bs='cr',k=%d)",K)))
}
d$y<-Y[,1L]
cache_qr<-function(G) {
  q<-qr(G$X,tol=0,LAPACK=TRUE); rp<-q$pivot; rp[rp]<-seq_len(ncol(G$X))
  G$fast_qr<-q; G$fast_R<-qr.R(q)[,rp,drop=FALSE]
  G$fast_sp<-mgcv:::initial.spg(G$X,G$y,G$w,G$family,G$S,G$rank,G$off,
                              offset=G$offset,L=G$L,lsp0=G$lsp0)
  stopifnot(max(abs(crossprod(G$fast_R)-crossprod(G$X)))<1e-7)
  G
}
G0<-cache_qr(gam(f0,data=d,family=gaussian(),method='ML',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))
G1<-cache_qr(gam(f1,data=d,family=gaussian(),method='ML',gamma=1,select=FALSE,fit=FALSE,na.action=na.fail))
X0<-G0$X; X1<-G1$X
sc<-lapply(G1$smooth,function(s)s$first.para:s$last.para)
tc<-sort(unique(c(which(startsWith(colnames(X1),'branch')),unlist(sc))))
XT<-X1[,tc,drop=FALSE]
grid<-seq(min(d$v),max(d$v),length.out=GRID_N)

#2. =========== Exact fixed-X sufficient-statistic fit and full-cell restoration
fit_response<-function(G,y) {
  X<-G$X; N<-length(y); p<-ncol(X)
  init<-list(sp=G$fast_sp,scale=mean((y-mean(y))^2)/10)
  qy<-qr.qty(G$fast_qr,y)[seq_len(p)]; extra<-sum(y*y)-sum(qy*qy)
  stopifnot(extra>=-1e-10)
  G$X<-G$fast_R;G$y<-qy;G$w<-rep(1,p);G$n<-p;G$offset<-rep(0,p)
  G$n.true<-N;G$dev.extra<-G$pearson.extra<-max(0,extra);G$mf$y<-y
  G$fast_qr<-G$fast_R<-G$fast_sp<-NULL
  m<-gam(G=G,method='ML',gamma=1,in.out=init)
  mu<-as.numeric(X%*%coef(m));r<-y-mu
  m$y<-m$model$y<-y;m$fitted.values<-m$linear.predictors<-mu;m$residuals<-r
  m$prior.weights<-m$weights<-m$working.weights<-rep(1,N);m$offset<-rep(0,N);m$z<-y
  m$deviance<-sum(r*r);m$null.deviance<-sum((y-mean(y))^2)
  m$df.residual<-N-sum(m$edf);m$df.null<-N-as.integer(G$intercept)
  m$aic<-m$family$aic(y,1,mu,rep(1,N),m$deviance)+2*sum(m$edf)
  m$hat<-rowSums((X%*%m$Vp)*X)/m$sig2
  m
}
converged<-function(m) {
  z<-m$outer.info$conv
  (is.null(m$converged)||isTRUE(m$converged)) &&
   (is.null(z)||identical(z,0L)||identical(z,0)||grepl('full convergence',paste(z,collapse=' '),fixed=TRUE))
}
tie_expected_sse<-function(r,group,counts) {
  sums<-rowsum(cbind(r,r*r),group,reorder=TRUE)
  means<-sums[,1]/counts;ss<-pmax(0,sums[,2]-sums[,1]^2/counts)
  variances<-ss/counts
  total<-2*sum(ss)
  m<-length(counts)
  if(m>1L)total<-total+sum(variances[-m]+variances[-1L]+diff(means)^2)
  total
}
# The analytic expected SSE equals enumeration over every within-t tie order.
toy_r<-c(.1,-.3,.2,.6,-.1,.9);toy_group<-c(1,1,2,2,2,3)
toy_permutations<-function(x)if(length(x)==1L)list(x)else unlist(lapply(seq_along(x),function(i)
  lapply(toy_permutations(x[-i]),function(z)c(x[i],z))),recursive=FALSE)
toy_orders<-lapply(toy_permutations(c(1L,2L)),function(a)lapply(toy_permutations(c(3L,4L,5L)),function(b)c(a,b,6L)))
toy_orders<-unlist(toy_orders,recursive=FALSE)
stopifnot(abs(tie_expected_sse(toy_r,toy_group,c(2L,3L,1L))-
  mean(vapply(toy_orders,function(z)sum(diff(toy_r[z])^2),numeric(1))))<1e-12)
k_check<-function(v,r,seed) {
  set.seed(seed)
  if(length(r)>SUBSAMPLE) {idx<-sample.int(length(r),SUBSAMPLE);v<-v[idx];r<-r[idx]}
  n<-length(r);den<-mean(r*r)
  if(n<2L||den<=0)return(list(index=NA_real_,p=NA_real_,n=n,status='zero_residual_variance'))
  group<-match(v,sort(unique(v)));counts<-tabulate(group)
  observed<-tie_expected_sse(r,group,counts)/(2*(n-1L))
  reference<-vapply(seq_len(REPS),function(i)tie_expected_sse(r[sample.int(n,n)],group,counts)/(2*(n-1L)),numeric(1))
  list(index=observed/den,p=mean(reference<observed),n=n,status='ok',
    tied_cells=sum(counts[counts>1L]),tied_groups=sum(counts>1L),
    old_index=mean(diff(r[order(v)])^2)/2/den)
}
diagnose<-function(m,G,r,j,role) {
  if(!length(G$smooth))return(NULL)
  rows<-list()
  for(si in seq_along(G$smooth)) {
    sm<-G$smooth[[si]]
    hit<-if(is.null(sm$by.level))seq_len(nrow(d))else which(as.character(d$branch)==as.character(sm$by.level))
    cols<-sm$first.para:sm$last.para;edf<-sum(m$edf[cols]);kp<-length(cols)
    seed<-SEED+j+si+if(role=='null')1000L else 0L
    z<-k_check(d$v[hit],r[hit],seed)
    check_k<-check_p<-NA_real_
    if(VALIDATE && j==1L && z$status=='ok' && !anyDuplicated(d$v[hit])) {
      fixture<-gam(rr~s(v,bs='cr',k=K),data=data.frame(v=d$v[hit],rr=r[hit]),
                   family=gaussian(),method='ML',gamma=1,select=FALSE)
      fixture$y<-r[hit];fixture$fitted.values<-fixture$linear.predictors<-rep(0,length(hit))
      stopifnot(max(abs(residuals(fixture)-r[hit]))<1e-12)
      set.seed(seed);official<-k.check(fixture,subsample=SUBSAMPLE,n.rep=REPS)
      check_k<-abs(z$index-official[1,'k-index']);check_p<-abs(z$p-official[1,'p-value'])
      stopifnot(check_k<1e-12,check_p==0)
    }
    rows[[si]]<-data.frame(regulon=colnames(Y)[j],regulon_index=j,model_role=role,
      term=sm$label,branch=if(is.null(sm$by.level))'all'else as.character(sm$by.level),
      n_cells=length(hit),n_checked=z$n,unique_v=length(unique(d$v[hit])),k_prime=kp,
      edf=edf,edf_ratio=edf/kp,k_index=z$index,p_value=z$p,status=z$status,
      basis_flag=is.finite(z$p)&&z$p<.01&&edf/kp>.9,
      structure_flag=is.finite(z$p)&&z$p<.01&&z$index<.90,
      smoothing_parameter=as.numeric(m$sp[si]),seed=seed,permutations=REPS,
      official_k_difference=check_k,official_p_difference=check_p)
    rows[[si]]$tied_cells<-z$tied_cells
    rows[[si]]$tied_groups<-z$tied_groups
    rows[[si]]$old_ordered_k_index<-z$old_index
    rows[[si]]$diagnostic_method<-'Tie-order-invariant expected adjacent SSE; fixed exact-t groups; permuted residuals'
  }
  do.call(rbind,rows)
}

#3. =========== Every response, local contrast and complete residual evidence
N<-nrow(d);J<-length(INDICES)
FULL<-RESID<-TRAJ<-matrix(NA_real_,N,J)
CONTRAST<-CONTRAST_SE<-matrix(NA_real_,GRID_N,J)
statistics<-diagnostics<-validations<-list()
started<-proc.time()[['elapsed']]
for(ii in seq_along(INDICES)) {
  j<-INDICES[ii];y<-as.numeric(Y[,j]);tick<-proc.time()[['elapsed']]
  row<-data.frame(regulon=colnames(Y)[j],regulon_index=j,status='failed',test_status='not_run',
    p_value=1,A=NA_real_,contrast_max_abs=NA_real_,contrast_mean=NA_real_,
    contrast_start=NA_real_,contrast_end=NA_real_,RSS0=NA_real_,RSS1=NA_real_,
    df0=NA_real_,df1=NA_real_,test_df0=NA_real_,test_df1=NA_real_,test_delta_df=NA_real_,
    F_statistic=NA_real_,partial_R2=NA_real_,converged0=FALSE,converged1=FALSE,
    full_rank0=FALSE,full_rank1=FALSE,full_basis_flags=NA_integer_,full_structure_flags=NA_integer_,
    null_basis_flags=NA_integer_,residual_mean=NA_real_,residual_sd=NA_real_,
    cache_prediction_difference=NA_real_,cache_F_relative_difference=NA_real_,
    warnings='',error='',fit_seconds=NA_real_)
  notices<-character()
  value<-tryCatch(withCallingHandlers({
    m0<-fit_response(G0,y);m1<-fit_response(G1,y)
    ok<-c(converged(m0),converged(m1),m0$rank==length(coef(m0)),m1$rank==length(coef(m1)))
    stopifnot(all(ok))
    full<-as.numeric(X1%*%coef(m1));null<-as.numeric(X0%*%coef(m0));r<-y-full
    tr<-as.numeric(XT%*%coef(m1)[tc]);tr<-tr-mean(tr)+mean(y)
    rss0<-sum((y-null)^2);rss1<-sum(r*r)
    stopifnot(max(abs(full-fitted(m1)))<1e-9,abs(rss1-m1$deviance)<1e-9)
    tab<-anova(m0,m1,test='F');pcol<-grep('^Pr\\(',names(tab),value=TRUE)
    p<-as.numeric(tab[2,pcol]);Fv<-as.numeric(tab[2,'F'])
    test_status<-if(is.finite(p)&&tab[2,'Df']>0&&is.finite(Fv)&&Fv>=0)'ok'else'nonpositive_incremental_test'
    if(test_status!='ok')p<-1
    nd<-d[rep(1L,GRID_N),,drop=FALSE];nd$v<-grid
    if(KIND=='early') {
      L<-predict(m1,newdata=nd,type='lpmatrix');L<-sweep(L,2,L[1,],'-')
    } else {
      nd$branch<-factor(rep(levels(d$branch)[1],GRID_N),levels=levels(d$branch))
      LA<-predict(m1,newdata=nd,type='lpmatrix')
      nd$branch<-factor(rep(levels(d$branch)[2],GRID_N),levels=levels(d$branch))
      LB<-predict(m1,newdata=nd,type='lpmatrix');L<-LB-LA
    }
    nuisance<-setdiff(seq_len(ncol(X1)),tc)
    if(length(nuisance))stopifnot(max(abs(L[,nuisance,drop=FALSE]))<1e-12)
    delta<-as.numeric(L%*%coef(m1));se<-sqrt(pmax(0,rowSums((L%*%m1$Vp)*L)))
    dq<-rbind(diagnose(m1,G1,r,j,'full'),diagnose(m0,G0,y-null,j,'null'))
    pred_error<-F_error<-NA_real_;validation<-NULL
    if(VALIDATE && j %in% c(1L,109L,217L)) {
      dd<-d;dd$y<-y
      fresh0<-gam(f0,data=dd,family=gaussian(),method='ML',gamma=1,select=FALSE,na.action=na.fail)
      fresh1<-gam(f1,data=dd,family=gaussian(),method='ML',gamma=1,select=FALSE,na.action=na.fail)
      ft<-anova(fresh0,fresh1,test='F')
      pred_error<-max(abs(fitted(fresh0)-null),abs(fitted(fresh1)-full))
      F_error<-if(test_status=='ok')abs(Fv-as.numeric(ft[2,'F']))/(1+abs(Fv))else 0
      edf_error<-max(abs(m1$edf-fresh1$edf),abs(m0$edf-fresh0$edf))
      ml_error<-max(abs(as.numeric(m0$gcv.ubre)-as.numeric(fresh0$gcv.ubre)),
                    abs(as.numeric(m1$gcv.ubre)-as.numeric(fresh1$gcv.ubre)))
      stopifnot(pred_error<1e-7,F_error<1e-5,edf_error<1e-4,ml_error<1e-5)
      validation<-data.frame(regulon=colnames(Y)[j],prediction_difference=pred_error,
        F_relative_difference=F_error,EDF_difference=edf_error,ML_objective_difference=ml_error)
    }
    list(m0=m0,m1=m1,ok=ok,full=full,r=r,tr=tr,rss0=rss0,rss1=rss1,tab=tab,p=p,F=Fv,
      test_status=test_status,delta=delta,se=se,dq=dq,pred_error=pred_error,F_error=F_error,validation=validation)
  },warning=function(w){notices<<-c(notices,conditionMessage(w));invokeRestart('muffleWarning')}),error=function(e)e)
  if(inherits(value,'error'))row$error<-conditionMessage(value) else {
    row$status<-'ok';row$test_status<-value$test_status;row$p_value<-value$p
    row$A<-diff(range(value$tr));row$contrast_max_abs<-max(abs(value$delta))
    row$contrast_mean<-mean(value$delta);row$contrast_start<-value$delta[1L];row$contrast_end<-tail(value$delta,1)
    row$RSS0<-value$rss0;row$RSS1<-value$rss1;row$df0<-value$m0$df.residual;row$df1<-value$m1$df.residual
    row$test_df0<-value$tab[1,'Resid. Df'];row$test_df1<-value$tab[2,'Resid. Df'];row$test_delta_df<-value$tab[2,'Df']
    row$F_statistic<-value$F;row$partial_R2<-(value$rss0-value$rss1)/value$rss0
    row$converged0<-value$ok[1];row$converged1<-value$ok[2];row$full_rank0<-value$ok[3];row$full_rank1<-value$ok[4]
    row$full_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role=='full'])
    row$full_structure_flags<-sum(value$dq$structure_flag[value$dq$model_role=='full'])
    row$null_basis_flags<-sum(value$dq$basis_flag[value$dq$model_role=='null'])
    row$residual_mean<-mean(value$r);row$residual_sd<-sd(value$r)
    row$cache_prediction_difference<-value$pred_error;row$cache_F_relative_difference<-value$F_error
    FULL[,ii]<-value$full;RESID[,ii]<-value$r;TRAJ[,ii]<-value$tr
    CONTRAST[,ii]<-value$delta;CONTRAST_SE[,ii]<-value$se
    diagnostics[[length(diagnostics)+1L]]<-value$dq
    if(!is.null(value$validation))validations[[length(validations)+1L]]<-value$validation
  }
  row$warnings<-paste(unique(notices),collapse=' | ');row$fit_seconds<-proc.time()[['elapsed']]-tick
  statistics[[ii]]<-row
  if(ii%%20L==0L||ii==J)cat(sprintf('%s %s K%d %d/%d %.1fs\n',KIND,MODE,K,ii,J,proc.time()[['elapsed']]-started))
}
write_table<-function(rows,suffix)if(length(rows))write.table(do.call(rbind,rows),paste0(PREFIX,suffix),sep='\t',quote=FALSE,row.names=FALSE,na='NA')
write_matrix<-function(x,suffix){con<-file(paste0(PREFIX,suffix),'wb');writeBin(as.numeric(x),con,size=8,endian='little');close(con)}
write_table(statistics,'.stats.tsv');write_table(diagnostics,'.diagnostics.tsv');write_table(validations,'.validation.tsv')
write_matrix(FULL,'.full.bin');write_matrix(RESID,'.residual.bin');write_matrix(TRAJ,'.trajectory.bin')
write_matrix(CONTRAST,'.contrast.bin');write_matrix(CONTRAST_SE,'.contrast_se.bin')
write.table(data.frame(metric=c('R_version','mgcv_version','null_formula','full_formula','mode','K','cells','permutations','subsample','seed','computation'),
 value=c(R.version.string,as.character(packageVersion('mgcv')),paste(deparse(f0),collapse=' '),paste(deparse(f1),collapse=' '),MODE,K,N,REPS,SUBSAMPLE,SEED,
 'Exact thin-QR Gaussian ML with original full-X smoothing and full-y scale initialization; full-cell attributes restored')),
 paste0(PREFIX,'.metadata.tsv'),sep='\t',quote=FALSE,row.names=FALSE)
"""


def sha(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(group, name, value):
    def finite(x):
        if isinstance(x, dict):
            return {str(k): finite(v) for k, v in x.items()}
        if isinstance(x, (list, tuple, np.ndarray)):
            return [finite(v) for v in x]
        if isinstance(x, np.generic):
            return finite(x.item())
        if isinstance(x, float) and not np.isfinite(x):
            return None
        return x
    group.create_dataset(name, data=json.dumps(finite(value), ensure_ascii=False, allow_nan=False), dtype=h5py.string_dtype())


def write_frame(group, data):
    data = data.rename(columns={"index": "regulon_index"})
    group.create_dataset("index", data=np.asarray(data.index.astype(str), object), dtype=h5py.string_dtype())
    group.create_dataset("columns", data=np.asarray(data.columns, object), dtype=h5py.string_dtype())
    for column in data:
        values = data[column].to_numpy()
        if values.dtype.kind in "OSU":
            group.create_dataset(column, data=np.asarray(data[column].fillna("").astype(str), object), dtype=h5py.string_dtype(), compression="gzip")
        else:
            group.create_dataset(column, data=values, compression="gzip")


def read_frame(group):
    return pd.DataFrame({c: group[c].asstr()[()] if group[c].dtype.kind in "OSU" else group[c][()]
                         for c in group["columns"].asstr()[()]}, index=group["index"].asstr()[()])


def adjust(values):
    p = np.clip(np.nan_to_num(np.asarray(values, float), nan=1, posinf=1, neginf=1), 0, 1)
    order = np.argsort(p, kind="stable")
    q = np.empty(len(p))
    q[order] = np.minimum(1, np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1])
    return np.minimum(1, p * len(p)), q


def read_input():
    with h5py.File(INPUT, "r") as h:
        cells = pd.Index(h["cells"].asstr()[()], name="cell_id")
        names = h["regulons"].asstr()[()].tolist()
        fields = ["seg", "stage", "study", "library", "t", "log_detected"]
        obs = pd.DataFrame({k: h["obs"][k].asstr()[()] if h["obs"][k].dtype.kind in "OSU" else h["obs"][k][()] for k in fields}, index=cells)
        auc = pd.DataFrame(h["auc"][()], index=cells, columns=names)
        segments = json.loads(h["segments_json"].asstr()[()])
    assert len(names) == FAMILY_SIZE and auc.shape == (7355, 217)
    obs["seg"] = obs.seg.astype(str)
    assert obs.index.is_unique and np.isfinite(auc.to_numpy()).all()
    return obs, auc, {str(s["id"]): s for s in segments}


#2. =========== Fixed local estimands and observed-support manifests
def make_analyses(obs, auc, segments):
    definitions = {"early": None, "fork1": ("2", "5"), "fork2": ("3", "4"), "fork3": ("6", "7")}
    result = {}
    for label, pair in definitions.items():
        if label == "early":
            selected = obs.study.eq("GSE178931") & obs.seg.eq("1") & obs.stage.isin(["E9.5", "E11.5", "E13.5"])
            data = obs.loc[selected].copy()
            data["branch"] = "1"
            origin = float(segments["1"]["t_start"])
            lower, upper = float(data.t.min()), float(data.t.max())
        else:
            data = obs.loc[obs.seg.isin(pair)].copy()
            lower = max(float(data.loc[data.seg.eq(s), "t"].min()) for s in pair)
            upper = min(float(data.loc[data.seg.eq(s), "t"].max()) for s in pair)
            assert lower < upper
            data = data.loc[data.t.between(lower, upper)].copy()
            data["branch"] = data.seg
            origin = float(segments[pair[0]]["t_start"])
            assert origin == float(segments[pair[1]]["t_start"])
        data["v"] = data.t - origin
        assert data.groupby("branch").size().min() >= 20
        assert data.groupby("branch").v.nunique().min() >= K_EXTENSION
        directory = WORK / label
        directory.mkdir(parents=True, exist_ok=True)
        # Set this storage path independently before running. cells.csv.
        data.to_csv(ROOT, index_label="cell_id", float_format="%.17g")
        local_auc = auc.loc[data.index]
        # Set this storage path independently before running. auc.tsv.gz.
        local_auc.to_csv(ROOT, sep="\t", index_label="cell_id", float_format="%.17g", compression={"method": "gzip", "mtime": 0})
        manifest = dict(analysis=label, cells=len(data), branch_counts=data.branch.value_counts().to_dict(),
                        stage_counts=data.stage.value_counts().to_dict(), library_counts=data.groupby("branch").library.nunique().to_dict(),
                        study_branch_counts=pd.crosstab(data.study, data.branch).to_dict(),
                        native_t_common_lower=lower, native_t_common_upper=upper, native_fork_t=origin,
                        v_definition="native t minus common fork/root t; no daughter-specific rescaling",
                        zero_native_t_cells=int(data.t.eq(0).sum()), unique_v_by_branch=data.groupby("branch").v.nunique().to_dict(),
                        test_scope="Exploratory conditional cell-level Gaussian association, not animal-replicate inference",
                        early_scope="E9.5 single-library 12-cell boundary, tied root projections and E10.5 source mismatch are not resolved by this t test" if label == "early" else "Only direct child segments on their shared native t interval; later descendant segments are not included",
                        contrast="change from first grid location" if label == "early" else f"segment {pair[1]} minus segment {pair[0]}")
        result[label] = (data, local_auc, manifest)
    return result


#3. =========== Deterministic cached R shards and complete-family correction
def run_shard(label, mode, k, part, indices):
    directory = WORK / label
    run = directory / f"{mode}_k{k}"
    run.mkdir(parents=True, exist_ok=True)
    prefix = run / f"part{part}"
    validate = mode == ("depth" if label == "early" else "source_depth")
    fingerprint = dict(analysis=label, mode=mode, K=k, indices=indices.tolist(), validate=validate,
                       # Set this storage path independently before running. cells.csv.
                       files={str(p): sha(p) for p in (ROOT, ROOT, ROOT)})
    marker = Path(str(prefix) + ".complete.json")
    if marker.exists() and json.loads(marker.read_text(encoding="utf-8")) == fingerprint:
        return prefix
    # Set this storage path independently before running. associate_local.R.
    command = [str(RSCRIPT), "--vanilla", str(ROOT), str(ROOT),
               # Set this storage path independently before running. auc.tsv.gz.
               str(ROOT), str(prefix), "early" if label == "early" else "fork",
               mode, str(k), ",".join(map(str, indices)), "1" if validate else "0"]
    with open(str(prefix) + ".log", "w", encoding="utf-8") as log:
        result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(), cwd=ROOT)
    if result.returncode:
        raise RuntimeError(Path(str(prefix) + ".log").read_text(encoding="utf-8", errors="replace")[-5000:])
    for suffix in (".stats.tsv", ".diagnostics.tsv", ".full.bin", ".residual.bin", ".trajectory.bin", ".contrast.bin", ".contrast_se.bin"):
        assert Path(str(prefix) + suffix).is_file(), str(prefix) + suffix
    marker.write_text(json.dumps(fingerprint, sort_keys=True), encoding="utf-8")
    return prefix


def collect(group, prefixes, data, auc, names, shards):
    stats = pd.concat([pd.read_csv(str(p) + ".stats.tsv", sep="\t", keep_default_na=False, na_values=["NA"]) for p in prefixes]).set_index("regulon").loc[names]
    assert len(stats) == 217 and stats.index.is_unique
    stats["p_value"] = stats.p_value.where(stats.status.eq("ok"), 1).fillna(1)
    stats["p_Bonferroni"], stats["q_BH"] = adjust(stats.p_value)
    write_frame(group.create_group("stats"), stats)
    diagnostics = pd.concat([pd.read_csv(str(p) + ".diagnostics.tsv", sep="\t") for p in prefixes], ignore_index=True)
    diagnostics["diagnostic_q_BH"] = adjust(diagnostics.p_value)[1]
    write_frame(group.create_group("diagnostics"), diagnostics)
    validation = [pd.read_csv(str(p) + ".validation.tsv", sep="\t") for p in prefixes if Path(str(p) + ".validation.tsv").exists()]
    if validation:
        validation = pd.concat(validation, ignore_index=True)
        write_frame(group.create_group("validation"), validation)
        assert len(validation) == 3
        assert validation.prediction_difference.max() < 1e-7
    runtime = []
    for i, prefix in enumerate(prefixes):
        frame = pd.read_csv(str(prefix) + ".metadata.tsv", sep="\t")
        frame["shard"] = i
        runtime.append(frame)
    write_frame(group.create_group("runtime_metadata"), pd.concat(runtime, ignore_index=True))
    valid = stats.status.eq("ok").to_numpy()
    arrays = {}
    for suffix, key, rows in ((".full.bin", "full_prediction", len(data)), (".residual.bin", "residuals", len(data)),
                               (".trajectory.bin", "trajectory", len(data)), (".contrast.bin", "contrast", 51),
                               (".contrast_se.bin", "contrast_se", 51)):
        array = np.full((rows, 217), np.nan)
        for prefix, indices in zip(prefixes, shards):
            values = np.fromfile(str(prefix) + suffix, dtype="<f8").reshape((rows, len(indices)), order="F")
            array[:, indices - 1] = values
        assert np.isfinite(array[:, valid]).all(), key
        group.create_dataset(key, data=array, compression="gzip", compression_opts=4)
        arrays[key] = array
    np.testing.assert_allclose(auc.to_numpy()[:, valid] - arrays["full_prediction"][:, valid], arrays["residuals"][:, valid], atol=1e-13)
    np.testing.assert_allclose((arrays["residuals"][:, valid] ** 2).sum(axis=0), stats.loc[valid, "RSS1"], atol=1e-10, rtol=1e-9)
    np.testing.assert_allclose(np.ptp(arrays["trajectory"][:, valid], axis=0), stats.loc[valid, "A"], atol=1e-12, rtol=1e-12)
    tested = stats.status.eq("ok") & stats.test_status.eq("ok")
    expected_F = ((stats.RSS0 - stats.RSS1) / stats.test_delta_df) / (stats.RSS1 / stats.df1)
    np.testing.assert_allclose(expected_F[tested], stats.loc[tested, "F_statistic"], atol=1e-8, rtol=1e-8)
    expected_p = f_distribution.sf(stats.F_statistic, stats.test_delta_df, stats.test_df1)
    np.testing.assert_allclose(expected_p[tested], stats.loc[tested, "p_value"], atol=1e-10, rtol=1e-6)
    return stats, diagnostics


def rho(x, y):
    good = np.isfinite(x) & np.isfinite(y)
    if good.sum() < 3 or np.ptp(x[good]) == 0 or np.ptp(y[good]) == 0:
        return np.nan
    return float(np.corrcoef(rankdata(x[good]), rankdata(y[good]))[0, 1])


#4. =========== Descriptive library-deletion composition checks
def block_deletion(data, auc):
    edges = np.linspace(data.v.min(), data.v.max(), 7)
    bins = np.minimum(np.searchsorted(edges, data.v, side="right") - 1, 5)
    index = pd.MultiIndex.from_product([sorted(data.branch.unique()), range(6)])
    labels = pd.DataFrame(dict(branch=data.branch, bin=bins), index=data.index)
    baseline = auc.groupby([labels.branch, labels.bin]).mean().reindex(index)
    blocks = data.study.astype(str) + "::library::" + data.library.astype(str)
    rows = []
    for block in sorted(blocks.unique()):
        keep = blocks.ne(block)
        changed = auc.loc[keep].groupby([labels.loc[keep, "branch"], labels.loc[keep, "bin"]]).mean().reindex(index)
        for name in auc:
            x, y = baseline[name].to_numpy(), changed[name].to_numpy()
            good = np.isfinite(x) & np.isfinite(y)
            rows.append(dict(regulon=name, block=block, valid_bins=int(good.sum()),
                             profile_rho=rho(x, y), max_abs_bin_change=float(np.max(np.abs(x[good] - y[good]))) if good.any() else np.nan,
                             cells_removed=int((~keep).sum())))
    return pd.DataFrame(rows)


def run():
    WORK.mkdir(parents=True, exist_ok=True)
    # Set this storage path independently before running. associate_local.R.
    (ROOT).write_text(R_SOURCE, encoding="utf-8", newline="\n")
    obs, auc, segments = read_input()
    names = auc.columns.tolist()
    definitions = make_analyses(obs, auc, segments)
    shards = [np.arange(1, 218)[part::WORKERS] for part in range(WORKERS)]
    started = time.monotonic()
    # Set this storage path independently before running. Fig4A_local.h5.
    target = ROOT
    summaries = {}
    with h5py.File(target, "w") as h:
        h.attrs["schema"] = "Fig4A_local_v1"
        h.attrs["seed"] = SEED
        h.attrs["input_auc_sha256"] = hashlib.sha256(np.asarray(auc, dtype="<f8").tobytes()).hexdigest()
        h.attrs["code_sha256"] = sha(Path(__file__))
        # Set this storage path independently before running. associate_local.R.
        h.attrs["R_engine_sha256"] = sha(ROOT)
        h.create_dataset("regulons", data=np.asarray(names, object), dtype=h5py.string_dtype())
        h.create_dataset("cells", data=np.asarray(obs.index, object), dtype=h5py.string_dtype())
        write_json(h, "methods_json", dict(initial_K=10, extension_K=20, extension_rule="If any full/null smooth in any nuisance mode shows diagnostic p<.01 and EDF/kprime>.9, rerun every217 and every mode of that analysis atK20; K20 flags retained, no TF-specific adjustment", correction="Each local mode has a complete217 Bonferroni/BH family; root must correct joint prespecified global/local discovery hypotheses separately", contrasts="Fork B-A on common native t; early grid minus first supported grid location; pointwise conditional Vp uncertainty", block_stability="Observed-profile library deletion at fixed native-v bins; no model refit, no inferential P, no claim that plates are animals"))
        analyses = h.create_group("analyses")
        for label, (data, local_auc, manifest) in definitions.items():
            analysis = analyses.create_group(label)
            write_frame(analysis.create_group("obs"), data)
            analysis.create_dataset("observed_auc", data=local_auc.to_numpy(), compression="gzip")
            analysis.create_dataset("grid_v", data=np.linspace(data.v.min(), data.v.max(), 51))
            modes = analysis.create_group("modes")
            unique_modes = ("none", "depth") if label == "early" else MODES
            mode_groups = {mode: modes.create_group(mode) for mode in unique_modes}
            chosen = K_INITIAL
            trigger = []
            for k in (K_INITIAL, K_EXTENSION):
                flags = []
                for mode in unique_modes:
                    print(f"START_LOCAL {label} {mode} K{k} all217", flush=True)
                    with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as pool:
                        futures = [pool.submit(run_shard, label, mode, k, part, indices) for part, indices in enumerate(shards)]
                        prefixes = [future.result() for future in futures]
                    group = mode_groups[mode].create_group(f"k{k}")
                    group.attrs["K"] = k
                    group.attrs["mode"] = mode
                    stats, diagnostics = collect(group, prefixes, data, local_auc, names, shards)
                    flags.append(int(diagnostics.basis_flag.sum()))
                    print(json.dumps(dict(analysis=label, mode=mode, K=k, ok=int(stats.status.eq("ok").sum()), test_ok=int(stats.test_status.eq("ok").sum()), basis_flags=flags[-1], full_structure_flags=int(stats.full_structure_flags.sum()))), flush=True)
                    h.flush()
                if k == K_INITIAL:
                    trigger = flags
                    if not any(flags):
                        break
                    chosen = K_EXTENSION
            if label == "early":
                modes["source"] = modes["none"]
                modes["source_depth"] = modes["depth"]
            analysis["primary"] = modes["source_depth"][f"k{chosen}"]
            analysis.attrs["chosen_K"] = chosen
            analysis.attrs["primary_mode"] = "source_depth"
            manifest.update(chosen_K=chosen, K10_basis_trigger_counts=trigger, unique_mode_runs=list(unique_modes),
                            source_aliases="single-source early: source=none; source_depth=depth" if label == "early" else None)
            write_json(analysis, "manifest_json", manifest)
            write_frame(analysis.create_group("library_deletion"), block_deletion(data, local_auc))
            primary = read_frame(analysis["primary/stats"])
            summaries[label] = dict(cells=len(data), chosen_K=chosen, valid=int(primary.status.eq("ok").sum()),
                                    tests=int(primary.test_status.eq("ok").sum()), q_BH_lt_1e4=int(primary.q_BH.lt(1e-4).sum()),
                                    basis_flags=int(primary.full_basis_flags.sum()), structure_flags=int(primary.full_structure_flags.sum()))
            h.flush()
        write_json(h, "summary_json", summaries)
        h.attrs["elapsed_seconds"] = time.monotonic() - started
    os.replace(target, OUTPUT)
    print("LOCAL_COMPLETE " + json.dumps(dict(output=str(OUTPUT), summary=summaries, seconds=time.monotonic() - started)), flush=True)
    return summaries


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.parse_args()
    run()


## Original standalone source: fig4A_branch_support.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# Archived same-age/source branch-support engine retained by the accepted source archive.
# Original source label: analysis_code:legacy_branch_support_source
# Original source SHA256: ad5bbc73f14a1a20e50ccf900f1481ca40fb13ec9920454d93124b53f13d59f7
# Only storage addresses and release comments were changed; analysis was not executed.

"""Descriptive same-age/source support for the three fixed Fig4A fork tests.

No P values, cell exclusion in the source atlas, regulator selection or causal
interpretation. Libraries/plates are not treated as independent animals.
"""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

#1. =========== Shared configuration
from pathlib import Path
import sys
import json
import hashlib

sys.dont_write_bytecode = True
# Set this storage path independently before running.
ROOT = ROOT
# Set this storage path independently before running.
DESTINATION = ROOT
# Set this storage path independently before running. Fig4A.h5.
INPUT = ROOT
# Set this storage path independently before running. Fig4A_branch_support.h5.
OUTPUT = ROOT
# Set this storage path independently before running. E14_E16_E18_P1_GSE195500__all_5441_local_cells.h5ad.
BIO_METADATA = ROOT
N_BINS = 3
MIN_CELLS = 5
MIN_LIBRARIES = 2
NUMERIC_TIE = 1e-10
FORKS = {"fork1": ("2", "5"), "fork2": ("3", "4"), "fork3": ("6", "7")}

import h5py
import numpy as np
import pandas as pd


def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def vector(node):
    if isinstance(node, h5py.Dataset):
        return node.asstr()[()] if node.dtype.kind in "OSU" else node[()]
    categories = vector(node["categories"])
    return np.array([categories[i] if i >= 0 else None for i in node["codes"][()]], object)


def strings(group, key, values):
    group.create_dataset(key, data=np.asarray(values, object), dtype=h5py.string_dtype("utf8"), compression="gzip")


def frame(group, values):
    for key in values:
        col = values[key]
        if col.dtype.kind in "biuf":
            group.create_dataset(key, data=col.to_numpy(), compression="gzip")
        else:
            strings(group, key, col.fillna("").astype(str))


def json_dataset(group, key, value):
    group.create_dataset(key, data=json.dumps(value, ensure_ascii=False, allow_nan=False), dtype=h5py.string_dtype("utf8"))


def equal_library_mean(part, auc):
    values = [auc[rows.index.to_numpy()].mean(axis=0) for _, rows in part.groupby("library", sort=True)]
    return np.mean(values, axis=0)


def trim_pair(part, pair):
    groups = [part[part.seg.eq(s)] for s in pair]
    if any(g.empty for g in groups):
        return part.iloc[:0], np.nan, np.nan, "branch_absent"
    lo = max(g.v.min() for g in groups)
    hi = min(g.v.max() for g in groups)
    if lo >= hi:
        return part.iloc[:0], float(lo), float(hi), "no_shared_nonzero_v_interval"
    return part[part.v.between(lo, hi)], float(lo), float(hi), "shared_interval"


def effect_summary(effects, names, prefix):
    values = np.asarray(effects) if len(effects) else np.empty((0, len(names)))
    positive = (values > NUMERIC_TIE).sum(axis=0)
    negative = (values < -NUMERIC_TIE).sum(axis=0)
    informative = positive + negative
    consistency = np.divide(np.maximum(positive, negative), informative,
                            out=np.full(len(names), np.nan), where=informative > 0)
    return pd.DataFrame({"regulon": names, prefix + "rows": len(values),
                         prefix + "mean_B_minus_A": values.mean(axis=0) if len(values) else np.nan,
                         prefix + "median_B_minus_A": np.median(values, axis=0) if len(values) else np.nan,
                         prefix + "informative_rows": informative, prefix + "positive_rows": positive,
                         prefix + "negative_rows": negative, prefix + "dominant_sign": np.sign(positive - negative),
                         prefix + "sign_consistency": consistency})


#2. =========== Exact fixed fork domains and shared within-stratum support
def main():
    with h5py.File(INPUT, "r") as h:
        cells = h["cells"].asstr()[()]
        names = h["regulons"].asstr()[()]
        auc = h["auc"][()]
        obs = pd.DataFrame({k: vector(h["obs"][k]) for k in ("seg", "stage", "study", "library", "t")})
        segments = {str(s["id"]): s for s in json.loads(h["segments_json"].asstr()[()])}
    obs["cell_id"] = cells
    obs["seg"] = obs.seg.astype(str)
    assert len(obs) == 7355 and auc.shape == (7355, 217) and np.isfinite(auc).all()
    with h5py.File(BIO_METADATA, "r") as h:
        biocells = vector(h["obs/cell_id_global"])
        biorep = vector(h["obs/biological_replicate_id"])
    biomap = pd.Series(biorep, index=biocells)
    assert biomap.index.is_unique
    obs["known_biological_replicate"] = "unavailable"
    known = obs.study.eq("GSE195500")
    mapped = obs.loc[known, "cell_id"].map(biomap)
    assert mapped.notna().all()
    obs.loc[known, "known_biological_replicate"] = mapped.astype(str)
    outputs, summaries = {}, {}
    for label, pair in FORKS.items():
        part = obs[obs.seg.isin(pair)].copy()
        lower = max(float(part.loc[part.seg.eq(s), "t"].min()) for s in pair)
        upper = min(float(part.loc[part.seg.eq(s), "t"].max()) for s in pair)
        assert lower < upper
        origin = float(segments[pair[0]]["t_start"])
        assert origin == float(segments[pair[1]]["t_start"])
        part = part[part.t.between(lower, upper)].copy()
        part["v"] = part.t - origin
        edges = np.linspace(lower - origin, upper - origin, N_BINS + 1)
        part["v_bin"] = np.clip(np.searchsorted(edges, part.v, side="right") - 1, 0, N_BINS - 1)
        composition = part.groupby(["study", "stage", "seg"], sort=True).agg(
            cells=("cell_id", "size"), libraries=("library", "nunique"),
            known_biological_replicates=("known_biological_replicate", lambda x: x[x.ne("unavailable")].nunique()),
            v_min=("v", "min"), v_median=("v", "median"), v_max=("v", "max")).reset_index()
        coverage, supported_meta, matched_effects, matched_a, matched_b = [], [], [], [], []
        pair_meta, paired_effects, paired_a, paired_b = [], [], [], []
        for (study, stage, bi), stratum in part.groupby(["study", "stage", "v_bin"], sort=True):
            trimmed, lo, hi, reason = trim_pair(stratum, pair)
            gs = [trimmed[trimmed.seg.eq(s)] for s in pair]
            if reason == "shared_interval":
                if any(len(g) < MIN_CELLS for g in gs):
                    reason = "insufficient_cells"
                elif any(g.library.nunique() < MIN_LIBRARIES for g in gs):
                    reason = "insufficient_libraries"
                else:
                    reason = "eligible"
            record = {"study": study, "stage": stage, "v_bin": int(bi),
                      "bin_v_lower": float(edges[int(bi)]), "bin_v_upper": float(edges[int(bi) + 1]),
                      "shared_v_lower": lo, "shared_v_upper": hi, "eligible": reason == "eligible", "reason": reason}
            for side, seg, g in zip(("A", "B"), pair, gs):
                before = stratum[stratum.seg.eq(seg)]
                record.update({side + "_cells_before": len(before), side + "_libraries_before": before.library.nunique(),
                               side + "_cells": len(g), side + "_libraries": g.library.nunique(),
                               side + "_known_biological_replicates": g.loc[g.known_biological_replicate.ne("unavailable"), "known_biological_replicate"].nunique()})
            coverage.append(record)
            if reason == "eligible":
                a, b = [equal_library_mean(g, auc) for g in gs]
                supported_meta.append(record)
                matched_a.append(a); matched_b.append(b); matched_effects.append(b - a)
            # Paired library checks are retained independently of unpaired eligibility.
            # Each library must itself have both branches within common local v support.
            for library, block in stratum.groupby("library", sort=True):
                paired, blo, bhi, status = trim_pair(block, pair)
                if status != "shared_interval":
                    continue
                pg = [paired[paired.seg.eq(s)] for s in pair]
                if any(len(g) < MIN_CELLS for g in pg):
                    continue
                a, b = [auc[g.index.to_numpy()].mean(axis=0) for g in pg]
                pair_meta.append({"study": study, "stage": stage, "v_bin": int(bi), "library": library,
                                  "known_biological_replicate": str(block.known_biological_replicate.iloc[0]),
                                  "shared_v_lower": blo, "shared_v_upper": bhi,
                                  "A_cells": len(pg[0]), "B_cells": len(pg[1])})
                paired_a.append(a); paired_b.append(b); paired_effects.append(b - a)
        coverage = pd.DataFrame(coverage)
        supported = pd.DataFrame(supported_meta, columns=coverage.columns)
        paired = pd.DataFrame(pair_meta, columns=["study", "stage", "v_bin", "library", "known_biological_replicate",
                                                   "shared_v_lower", "shared_v_upper", "A_cells", "B_cells"])
        summary = effect_summary(matched_effects, names, "matched_")
        summary = summary.merge(effect_summary(paired_effects, names, "paired_stratum_"), on="regulon", validate="one_to_one")
        per_library, per_library_meta = [], []
        if len(paired):
            matrix = np.asarray(paired_effects)
            for (study, stage, library), g in paired.groupby(["study", "stage", "library"], sort=True):
                per_library.append(matrix[g.index].mean(axis=0))
                per_library_meta.append({"study": study, "stage": stage, "library": library, "supported_bins": len(g),
                                         "known_biological_replicate": str(g.known_biological_replicate.iloc[0])})
        summary = summary.merge(effect_summary(per_library, names, "paired_library_"), on="regulon", validate="one_to_one")
        per_library_meta = pd.DataFrame(per_library_meta, columns=["study", "stage", "library", "supported_bins", "known_biological_replicate"])
        manifest = {"analysis": label, "branch_A": pair[0], "branch_B": pair[1],
                    "branch_A_label": segments[pair[0]]["label"], "branch_B_label": segments[pair[1]]["label"],
                    "contrast": f"segment {pair[1]} minus segment {pair[0]}",
                    "native_t_lower": lower, "native_t_upper": upper, "native_fork_t": origin,
                    "v_bin_edges": edges.tolist(), "local_domain_cells": len(part),
                    "local_domain_branch_counts": {str(k): int(v) for k, v in part.seg.value_counts().items()},
                    "examined_strata": len(coverage), "supported_strata": len(supported),
                    "supported_studies": supported.study.unique().tolist(), "supported_ages": supported.stage.unique().tolist(),
                    "paired_library_strata": len(paired), "paired_libraries": len(per_library),
                    "excluded_stratum_reasons": {str(k): int(v) for k, v in coverage.loc[~coverage.eligible, "reason"].value_counts().items()},
                    "scope": "Descriptive only; no P values or animal-replication claim. Same library can occur in multiple v bins. Shared interval does not equate within-bin v distributions."}
        outputs[label] = {"manifest": manifest, "composition": composition, "coverage": coverage,
                          "matched_metadata": supported, "paired_metadata": paired,
                          "paired_library_metadata": per_library_meta, "summary": summary,
                          "matched_B_minus_A": matched_effects, "matched_A_mean": matched_a, "matched_B_mean": matched_b,
                          "paired_B_minus_A": paired_effects, "paired_A_mean": paired_a, "paired_B_mean": paired_b,
                          "paired_library_B_minus_A": per_library}
        summaries[label] = manifest

    #3. =========== Save complete descriptive family and reproducibility evidence
    with h5py.File(OUTPUT, "w") as h:
        strings(h, "regulons", names)
        json_dataset(h, "metadata_json", {"schema": "Fig4A_branch_support_v1", "family": 217,
            "n_native_v_bins": N_BINS, "minimum_cells_per_branch": MIN_CELLS,
            "minimum_libraries_per_branch_for_unpaired_strata": MIN_LIBRARIES,
            "paired_rule": "Each individual library has at least five cells per branch after its own shared-v restriction",
            "aggregation": "Unpaired strata: cell mean per library per branch then equal library mean. Paired libraries: mean B-A over supported bins per library, then equal library summaries.",
            "sign_consistency": "max(n_positive,n_negative)/(n_positive+n_negative); numeric absolute tie tolerance only",
            "numeric_tie_tolerance": NUMERIC_TIE, "used_for_selection": False,
            "known_biological_replicates": "Exact cell-ID join for GSE195500 only; libraries/plates elsewhere are not animals",
            "input_sha256": sha(INPUT), "biological_metadata_sha256": sha(BIO_METADATA),
            "code_sha256": sha(Path(__file__)), "analyses": summaries})
        for label, result in outputs.items():
            g = h.create_group("analyses/" + label)
            json_dataset(g, "manifest_json", result["manifest"])
            for key, value in result.items():
                if isinstance(value, pd.DataFrame):
                    frame(g.create_group(key), value)
                elif key != "manifest":
                    matrix = np.asarray(value) if len(value) else np.empty((0, len(names)))
                    g.create_dataset(key, data=matrix, compression="gzip")
    with h5py.File(OUTPUT, "r") as h:
        for label in FORKS:
            g = h["analyses/" + label]
            a, b, delta = [g[k][()] for k in ("matched_A_mean", "matched_B_mean", "matched_B_minus_A")]
            np.testing.assert_array_equal(b - a, delta)
            assert g["summary/regulon"].shape == (217,)
    print(json.dumps({label: {k: result["manifest"][k] for k in ("local_domain_cells", "supported_strata", "supported_studies", "supported_ages", "paired_libraries")}
                      for label, result in outputs.items()}), flush=True)


if __name__ == "__main__":
    main()


## Original standalone source: fig4_validate_figure_output.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
#!/usr/bin/env python3
"""Fail-closed validation for 600-DPI PNGs and editable figure PDFs.

The default PDF profile is intentionally strict. It is designed for figures that
must remain straightforward to edit after opening in Adobe Illustrator:

* no more than four explicit, nonzero-winding clipping paths per page;
* clipping paths are sibling, axis-aligned rectangles (never nested);
* every clipping path is explicitly scoped by q/Q and ends with W n;
* no raster images, soft/image masks, inline images, text clipping, shading,
  or uninspected recursive Form XObjects;
* live, embedded, Unicode-mapped Arial/SimSun text; and
* at least one painted vector path.

The program writes its report to stdout only. Poppler render files are created in
a task-local temporary directory and removed before exit unless the caller uses
``--render-dir``; that option always creates and retains a new unique child
directory, so existing files are never overwritten.
"""

from __future__ import annotations

import argparse
import math
import os
import re
import secrets
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Iterable, Sequence

try:
    from PIL import Image
    from pypdf import PdfReader
    from pypdf.generic import ArrayObject, ContentStream, DictionaryObject, IndirectObject
except ImportError as exc:  # pragma: no cover - exercised only in a broken runtime
    print(f"[NOT CERTIFIED] Required Python dependency is unavailable: {exc}")
    raise SystemExit(1)


EPSILON = 1e-7
PATH_BUILDING = {b"m", b"l", b"c", b"v", b"y", b"h", b"re"}
PATH_ENDING = {b"S", b"s", b"f", b"F", b"f*", b"B", b"B*", b"b", b"b*", b"n"}
TEXT_SHOWING = {b"Tj", b"TJ", b"'", b'"'}
FILLING = {b"f", b"F", b"f*"}
STROKING = {b"S", b"s"}
FILL_STROKING = {b"B", b"B*", b"b", b"b*"}
INLINE_IMAGE_OPS = {b"BI", b"ID", b"EI", b"INLINE IMAGE"}
SUBSET_PREFIX = re.compile(r"^[A-Z]{6}\+")
CJK_RE = re.compile(
    "[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff\U00020000-\U0002fa1f]"
)
ASCII_ALNUM_RE = re.compile(r"[A-Za-z0-9]")


@dataclass
class Issue:
    code: str
    message: str
    outcome: str = "FAIL"


@dataclass
class ClipEvent:
    number: int
    scope: str
    depth: int
    bbox: tuple[float, float, float, float] | None
    kinds: Counter[str] = field(default_factory=Counter)


@dataclass
class FontUse:
    scope: str
    resource_name: str
    base_font: str
    subtype: str
    embedded: bool
    to_unicode: bool


@dataclass
class GraphicsState:
    ctm: tuple[float, float, float, float, float, float] = (1, 0, 0, 1, 0, 0)
    clip_depth: int = 0
    active_clips: tuple[int, ...] = ()
    font_resource: str | None = None
    font_object: Any = None
    text_render_mode: int = 0


@dataclass
class PdfPageStats:
    page_number: int
    crop_size_points: tuple[float, float] = (0.0, 0.0)
    media_size_points: tuple[float, float] = (0.0, 0.0)
    rotation: int = 0
    clip_events: list[ClipEvent] = field(default_factory=list)
    max_clip_depth: int = 0
    q_count: int = 0
    Q_count: int = 0
    form_calls: int = 0
    image_count: int = 0
    inline_image_count: int = 0
    text_show_count: int = 0
    text_object_count: int = 0
    vector_paint_count: int = 0
    vector_ops: Counter[str] = field(default_factory=Counter)
    font_uses: dict[tuple[str, str, str], FontUse] = field(default_factory=dict)


@dataclass
class ValidationResult:
    path: Path
    kind: str
    issues: list[Issue] = field(default_factory=list)
    details: list[str] = field(default_factory=list)

    @property
    def passed(self) -> bool:
        return not self.issues

    @property
    def status(self) -> str:
        if any(issue.outcome == "FAIL" for issue in self.issues):
            return "FAIL"
        if self.issues:
            return "NOT CERTIFIED"
        return "PASS"

    def fail(self, code: str, message: str) -> None:
        self.issues.append(Issue(code, message, "FAIL"))

    def not_certified(self, code: str, message: str) -> None:
        self.issues.append(Issue(code, message, "NOT CERTIFIED"))


def dereference(value: Any) -> Any:
    try:
        return value.get_object()
    except AttributeError:
        return value


def dictionary(value: Any) -> DictionaryObject | dict:
    value = dereference(value)
    return value if isinstance(value, (DictionaryObject, dict)) else {}


def numeric(value: Any) -> float:
    return float(dereference(value))


def normalize_font_name(value: Any) -> str:
    name = str(value or "").lstrip("/")
    return SUBSET_PREFIX.sub("", name)


def font_family(base_font: str) -> str | None:
    compact = re.sub(r"[^a-z0-9]", "", normalize_font_name(base_font).lower())
    if compact.startswith("arial"):
        return "Arial"
    if compact.startswith("simsun") or compact.startswith("nsimsun"):
        return "SimSun"
    return None


def font_descriptor(font_obj: Any) -> dict:
    font = dictionary(font_obj)
    subtype = str(font.get("/Subtype", ""))
    if subtype == "/Type0":
        descendants = dereference(font.get("/DescendantFonts", []))
        if descendants:
            descendant = dictionary(descendants[0])
            return dictionary(descendant.get("/FontDescriptor", {}))
    return dictionary(font.get("/FontDescriptor", {}))


def font_is_embedded(font_obj: Any) -> bool:
    descriptor = font_descriptor(font_obj)
    return any(key in descriptor for key in ("/FontFile", "/FontFile2", "/FontFile3"))


def object_identity(value: Any) -> tuple[Any, ...]:
    if isinstance(value, IndirectObject):
        return ("indirect", value.idnum, value.generation)
    obj = dereference(value)
    ref = getattr(obj, "indirect_reference", None)
    if isinstance(ref, IndirectObject):
        return ("indirect", ref.idnum, ref.generation)
    return ("direct", id(obj))


def matrix_multiply(
    first: tuple[float, float, float, float, float, float],
    second: tuple[float, float, float, float, float, float],
) -> tuple[float, float, float, float, float, float]:
    """Compose affine transforms so result(point) == first(second(point))."""
    a1, b1, c1, d1, e1, f1 = first
    a2, b2, c2, d2, e2, f2 = second
    return (
        a1 * a2 + c1 * b2,
        b1 * a2 + d1 * b2,
        a1 * c2 + c1 * d2,
        b1 * c2 + d1 * d2,
        a1 * e2 + c1 * f2 + e1,
        b1 * e2 + d1 * f2 + f1,
    )


def transform_point(
    matrix: tuple[float, float, float, float, float, float], x: float, y: float
) -> tuple[float, float]:
    a, b, c, d, e, f = matrix
    return (a * x + c * y + e, b * x + d * y + f)


def normalized_bbox(points: Sequence[tuple[float, float]]) -> tuple[float, float, float, float]:
    xs = [point[0] for point in points]
    ys = [point[1] for point in points]
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    return (left, bottom, right - left, top - bottom)


def nearly_equal(left: float, right: float, tolerance: float) -> bool:
    return abs(left - right) <= tolerance


def boxes_match(
    observed: tuple[float, float, float, float],
    expected: tuple[float, float, float, float],
) -> bool:
    tolerance = max(0.5, max(abs(value) for value in expected[2:]) * 1e-4)
    return all(nearly_equal(a, b, tolerance) for a, b in zip(observed, expected))


def image_blank_reason(image: Image.Image) -> str | None:
    """Return a concrete blank-page reason without allocating a full RGBA copy."""
    extrema: Any = image.getextrema()
    if not isinstance(extrema, tuple):
        extrema = (extrema,)
    elif extrema and not isinstance(extrema[0], tuple):
        extrema = (extrema,)
    bands = image.getbands()
    if "A" in bands and extrema[bands.index("A")] == (0, 0):
        return "all pixels are fully transparent"
    if all(low == high for low, high in extrema):
        return "all pixels have one uniform color"
    return None


def parse_simple_rectangle(
    path_ops: Sequence[tuple[bytes, list[Any]]],
    ctm: tuple[float, float, float, float, float, float],
) -> tuple[tuple[float, float, float, float] | None, str | None]:
    if len(path_ops) == 1 and path_ops[0][0] == b"re":
        operands = path_ops[0][1]
        if len(operands) != 4:
            return None, "rectangle operator does not have four operands"
        x, y, width, height = (numeric(value) for value in operands)
        if abs(width) <= EPSILON or abs(height) <= EPSILON:
            return None, "rectangle has zero width or height"
        raw_points = [
            (x, y),
            (x + width, y),
            (x + width, y + height),
            (x, y + height),
        ]
    else:
        operators = [operator for operator, _ in path_ops]
        if not operators or operators[0] != b"m" or any(
            operator not in {b"m", b"l", b"h"} for operator in operators
        ):
            return None, "clip path is not one re rectangle or one m/l/h rectangle"
        if operators.count(b"m") != 1 or operators.count(b"h") > 1:
            return None, "clip path contains multiple subpaths"
        raw_points: list[tuple[float, float]] = []
        for operator, operands in path_ops:
            if operator in {b"m", b"l"}:
                if len(operands) != 2:
                    return None, "rectangle vertex does not have two coordinates"
                raw_points.append((numeric(operands[0]), numeric(operands[1])))
        if len(raw_points) == 5 and all(
            nearly_equal(a, b, 1e-6) for a, b in zip(raw_points[0], raw_points[-1])
        ):
            raw_points.pop()
        if len(raw_points) != 4 or (operators[-1] != b"h" and len(path_ops) != 5):
            return None, "m/l/h clip does not contain exactly four rectangle vertices"

    points = [transform_point(ctm, x, y) for x, y in raw_points]
    tolerance = max(1e-5, max(abs(value) for point in points for value in point) * 1e-7)
    for start, end in zip(points, points[1:] + points[:1]):
        dx, dy = abs(end[0] - start[0]), abs(end[1] - start[1])
        if not (dx <= tolerance < dy or dy <= tolerance < dx):
            return None, "clip rectangle is rotated, skewed, or self-degenerate"
    unique_x = []
    unique_y = []
    for x, y in points:
        if not any(nearly_equal(x, item, tolerance) for item in unique_x):
            unique_x.append(x)
        if not any(nearly_equal(y, item, tolerance) for item in unique_y):
            unique_y.append(y)
    if len(unique_x) != 2 or len(unique_y) != 2:
        return None, "clip path does not form one axis-aligned rectangle"
    return normalized_bbox(points), None


class PdfAuditor:
    def __init__(
        self,
        path: Path,
        result: ValidationResult,
        max_clips: int,
        expected_clips: int | None,
        declared_clip_boxes: list[tuple[float, float, float, float]],
        allow_raster_images: bool,
        allow_form_xobjects: bool,
        allow_no_text: bool,
        allow_no_vector: bool,
    ) -> None:
        self.path = path
        self.result = result
        self.max_clips = max_clips
        self.expected_clips = expected_clips
        self.declared_clip_boxes = declared_clip_boxes
        self.allow_raster_images = allow_raster_images
        self.allow_form_xobjects = allow_form_xobjects
        self.allow_no_text = allow_no_text
        self.allow_no_vector = allow_no_vector
        self.reader: PdfReader | None = None
        self._scanned_resources: set[tuple[Any, ...]] = set()
        self._reported_fonts: set[tuple[Any, ...]] = set()

    def audit(self) -> list[PdfPageStats]:
        try:
            self.reader = PdfReader(str(self.path), strict=True)
        except Exception as exc:
            self.result.not_certified("PDF_PARSE", f"PDF cannot be parsed strictly: {exc}")
            return []
        if self.reader.is_encrypted:
            self.result.not_certified("PDF_ENCRYPTED", "encrypted PDFs cannot be inspected")
            return []
        if not self.reader.pages:
            self.result.fail("PDF_EMPTY", "PDF contains no pages")
            return []

        page_stats: list[PdfPageStats] = []
        for page_number, page in enumerate(self.reader.pages, start=1):
            crop_box = page.cropbox
            media_box = page.mediabox
            user_unit = float(page.get("/UserUnit", 1) or 1)
            rotation = int(page.get("/Rotate", 0) or 0) % 360
            stats = PdfPageStats(
                page_number,
                crop_size_points=(
                    float(crop_box.width) * user_unit,
                    float(crop_box.height) * user_unit,
                ),
                media_size_points=(
                    float(media_box.width) * user_unit,
                    float(media_box.height) * user_unit,
                ),
                rotation=rotation,
            )
            page_stats.append(stats)
            resources = dictionary(page.get("/Resources", {}))
            page_boxes = [
                (
                    float(crop_box.left),
                    float(crop_box.bottom),
                    float(crop_box.width),
                    float(crop_box.height),
                ),
                (
                    float(media_box.left),
                    float(media_box.bottom),
                    float(media_box.width),
                    float(media_box.height),
                ),
            ]
            allowed_boxes = page_boxes + self.declared_clip_boxes
            contents = page.get_contents()
            if contents is None:
                self.result.fail("PDF_NO_CONTENT", f"page {page_number} has no content stream")
                continue
            try:
                content_stream = ContentStream(contents, self.reader)
            except Exception as exc:
                self.result.not_certified(
                    "CONTENT_PARSE", f"page {page_number} content stream cannot be parsed: {exc}"
                )
                continue
            self._scan_resources(resources, f"page {page_number}", stats)
            self._analyze_operations(
                content_stream.operations,
                resources,
                f"page {page_number}",
                stats,
                GraphicsState(),
                allowed_boxes,
                form_stack=set(),
                implicit_scope=False,
            )
            self._validate_page_text(page, stats)
            self._validate_page_summary(stats)
        return page_stats

    def _scan_resources(self, resources: dict, scope: str, stats: PdfPageStats) -> None:
        identity = object_identity(resources)
        if identity in self._scanned_resources:
            return
        self._scanned_resources.add(identity)

        ext_states = dictionary(resources.get("/ExtGState", {}))
        for name, reference in ext_states.items():
            ext_state = dictionary(reference)
            if "/SMask" in ext_state and str(dereference(ext_state["/SMask"])) != "/None":
                self.result.fail(
                    "SOFT_MASK", f"{scope} ExtGState {name} contains a non-None /SMask"
                )

        patterns = dictionary(resources.get("/Pattern", {}))
        if patterns:
            self.result.fail(
                "PATTERN_RESOURCE",
                f"{scope} contains Pattern resources; their Illustrator editability is not certified",
            )
        shadings = dictionary(resources.get("/Shading", {}))
        if shadings:
            self.result.fail(
                "SHADING_RESOURCE",
                f"{scope} contains Shading resources; their Illustrator editability is not certified",
            )

        xobjects = dictionary(resources.get("/XObject", {}))
        for name, reference in xobjects.items():
            xobject = dictionary(reference)
            subtype = str(xobject.get("/Subtype", ""))
            if "/SMask" in xobject:
                self.result.fail("SOFT_MASK", f"{scope} XObject {name} contains /SMask")
            if "/Mask" in xobject:
                self.result.fail("IMAGE_MASK", f"{scope} XObject {name} contains /Mask")
            if bool(xobject.get("/ImageMask", False)):
                self.result.fail("IMAGE_MASK", f"{scope} XObject {name} is an image mask")
            if subtype == "/Image":
                stats.image_count += 1
                if not self.allow_raster_images:
                    self.result.fail("RASTER_IMAGE", f"{scope} contains image XObject {name}")
            elif subtype == "/Form" and not self.allow_form_xobjects:
                self.result.fail(
                    "FORM_XOBJECT",
                    f"{scope} contains Form XObject {name}; use --allow-form-xobjects only for an intentionally reviewed Form workflow",
                )
            elif subtype not in {"/Form", ""}:
                self.result.fail(
                    "UNSUPPORTED_XOBJECT", f"{scope} contains unsupported XObject {name} ({subtype})"
                )

    def _record_clip_content(self, stats: PdfPageStats, state: GraphicsState, kind: str) -> None:
        for event_number in state.active_clips:
            stats.clip_events[event_number - 1].kinds[kind] += 1

    def _register_font(
        self, scope: str, resource_name: str, font_reference: Any, stats: PdfPageStats
    ) -> None:
        font = dictionary(font_reference)
        base_font = normalize_font_name(font.get("/BaseFont", ""))
        subtype = str(font.get("/Subtype", ""))
        embedded = font_is_embedded(font)
        to_unicode = "/ToUnicode" in font
        key = (scope, resource_name, base_font)
        stats.font_uses[key] = FontUse(
            scope, resource_name, base_font, subtype, embedded, to_unicode
        )
        identity = object_identity(font_reference)
        if identity in self._reported_fonts:
            return
        self._reported_fonts.add(identity)
        family = font_family(base_font)
        if family is None:
            self.result.fail(
                "FONT_FAMILY",
                f"{scope} font {resource_name} uses {base_font or '<unnamed>'}; only Arial and SimSun are allowed",
            )
        if not embedded:
            self.result.fail(
                "FONT_NOT_EMBEDDED",
                f"{scope} font {resource_name} ({base_font or '<unnamed>'}) is not embedded",
            )
        if not to_unicode:
            self.result.fail(
                "FONT_NO_TOUNICODE",
                f"{scope} font {resource_name} ({base_font or '<unnamed>'}) has no /ToUnicode map",
            )

    def _analyze_operations(
        self,
        operations: Sequence[tuple[list[Any], bytes]],
        resources: dict,
        scope: str,
        stats: PdfPageStats,
        initial_state: GraphicsState,
        allowed_boxes: list[tuple[float, float, float, float]],
        form_stack: set[tuple[Any, ...]],
        implicit_scope: bool,
    ) -> None:
        state = replace(initial_state)
        base_clip_depth = initial_state.clip_depth
        stack: list[GraphicsState] = []
        path_ops: list[tuple[bytes, list[Any]]] = []
        pending_clip = False
        in_text = False
        text_objects_at_start = stats.text_object_count

        for index, (operands, operator) in enumerate(operations):
            operands = list(operands)
            if operator == b"q":
                stats.q_count += 1
                stack.append(replace(state))
                continue
            if operator == b"Q":
                stats.Q_count += 1
                if not stack:
                    self.result.fail("Q_UNDERFLOW", f"{scope} has Q without a matching q at operation {index}")
                else:
                    state = stack.pop()
                path_ops = []
                pending_clip = False
                continue
            if operator == b"cm":
                if len(operands) != 6:
                    self.result.fail("BAD_CTM", f"{scope} has malformed cm at operation {index}")
                else:
                    matrix = tuple(numeric(value) for value in operands)
                    state.ctm = matrix_multiply(state.ctm, matrix)  # type: ignore[arg-type]
                continue

            if operator in PATH_BUILDING:
                path_ops.append((operator, operands))
                stats.vector_ops[operator.decode("ascii")] += 1
                continue
            if operator in {b"W", b"W*"}:
                if pending_clip:
                    self.result.fail(
                        "CLIP_PENDING", f"{scope} has consecutive clipping operators at operation {index}"
                    )
                if operator == b"W*":
                    self.result.fail(
                        "CLIP_EVEN_ODD",
                        f"{scope} uses W* at operation {index}; the strict profile requires nonzero-winding W",
                    )
                pending_clip = True
                continue
            if operator in PATH_ENDING:
                if pending_clip:
                    if operator != b"n":
                        self.result.fail(
                            "CLIP_PAINTED_PATH",
                            f"{scope} applies clipping with W/W* followed by {operator.decode('latin1')}; strict clipping must end with n",
                        )
                    bbox, reason = parse_simple_rectangle(path_ops, state.ctm)
                    event_number = len(stats.clip_events) + 1
                    if reason:
                        self.result.fail(
                            "CLIP_NOT_RECTANGLE",
                            f"{scope} clip {event_number} is invalid: {reason}",
                        )
                    elif bbox is not None and not any(
                        boxes_match(bbox, candidate) for candidate in allowed_boxes
                    ):
                        pretty = ", ".join(f"{value:.3f}" for value in bbox)
                        self.result.fail(
                            "CLIP_BOX_UNDECLARED",
                            f"{scope} clip {event_number} box ({pretty}) is neither the page box nor a declared --clip-box",
                        )
                    if not stack and not implicit_scope:
                        self.result.fail(
                            "CLIP_NOT_SCOPED",
                            f"{scope} clip {event_number} is not enclosed by an explicit q/Q scope",
                        )
                    state.clip_depth += 1
                    state.active_clips = state.active_clips + (event_number,)
                    stats.max_clip_depth = max(stats.max_clip_depth, state.clip_depth)
                    stats.clip_events.append(
                        ClipEvent(event_number, scope, state.clip_depth, bbox)
                    )
                    if state.clip_depth > 1:
                        self.result.fail(
                            "CLIP_NESTED",
                            f"{scope} clip {event_number} reaches effective clipping depth {state.clip_depth}",
                        )
                    pending_clip = False
                if operator in FILLING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-fill")
                elif operator in STROKING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-stroke")
                elif operator in FILL_STROKING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-fill+stroke")
                path_ops = []
                continue

            if operator == b"BT":
                if in_text:
                    self.result.fail("TEXT_NESTED", f"{scope} contains nested BT operators")
                in_text = True
                stats.text_object_count += 1
                continue
            if operator == b"ET":
                if not in_text:
                    self.result.fail("TEXT_ET_UNMATCHED", f"{scope} contains ET without BT")
                in_text = False
                continue
            if operator == b"Tr":
                try:
                    state.text_render_mode = int(numeric(operands[0]))
                except Exception:
                    self.result.fail("TEXT_RENDER_MODE", f"{scope} contains malformed Tr")
                    continue
                if state.text_render_mode in {4, 5, 6, 7}:
                    self.result.fail(
                        "TEXT_CLIPPING",
                        f"{scope} uses clipping text rendering mode {state.text_render_mode}",
                    )
                continue
            if operator == b"Tf":
                if len(operands) < 2:
                    self.result.fail("FONT_SELECT", f"{scope} contains malformed Tf")
                    continue
                resource_name = str(operands[0])
                fonts = dictionary(resources.get("/Font", {}))
                font_reference = fonts.get(resource_name)
                if font_reference is None:
                    self.result.fail(
                        "FONT_MISSING", f"{scope} selects missing font resource {resource_name}"
                    )
                    state.font_resource = None
                    state.font_object = None
                else:
                    state.font_resource = resource_name
                    state.font_object = font_reference
                continue
            if operator in TEXT_SHOWING:
                stats.text_show_count += 1
                if not in_text:
                    self.result.fail("TEXT_OUTSIDE_BT", f"{scope} shows text outside BT/ET")
                if state.text_render_mode in {3, 7}:
                    self.result.fail(
                        "INVISIBLE_TEXT", f"{scope} contains invisible text rendering mode {state.text_render_mode}"
                    )
                if state.text_render_mode in {4, 5, 6, 7}:
                    self.result.fail(
                        "TEXT_CLIPPING",
                        f"{scope} shows text in clipping rendering mode {state.text_render_mode}",
                    )
                if not state.font_resource or state.font_object is None:
                    self.result.fail("TEXT_NO_FONT", f"{scope} shows text without a resolved font")
                else:
                    self._register_font(scope, state.font_resource, state.font_object, stats)
                self._record_clip_content(stats, state, "text")
                continue

            if operator == b"Do":
                if not operands:
                    self.result.fail("XOBJECT_CALL", f"{scope} has malformed Do operator")
                    continue
                resource_name = str(operands[0])
                xobjects = dictionary(resources.get("/XObject", {}))
                reference = xobjects.get(resource_name)
                if reference is None:
                    self.result.fail(
                        "XOBJECT_MISSING", f"{scope} invokes missing XObject {resource_name}"
                    )
                    continue
                xobject = dictionary(reference)
                subtype = str(xobject.get("/Subtype", ""))
                if subtype == "/Image":
                    self._record_clip_content(stats, state, "image")
                    continue
                if subtype != "/Form":
                    self.result.fail(
                        "UNSUPPORTED_XOBJECT",
                        f"{scope} invokes unsupported XObject {resource_name} ({subtype})",
                    )
                    continue
                stats.form_calls += 1
                self._record_clip_content(stats, state, "form")
                identity = object_identity(reference)
                if identity in form_stack:
                    self.result.fail(
                        "FORM_RECURSION", f"{scope} has recursive Form XObject {resource_name}"
                    )
                    continue
                form_resources = dictionary(xobject.get("/Resources", resources))
                self._scan_resources(form_resources, f"{scope}/Form {resource_name}", stats)
                form_matrix_raw = dereference(xobject.get("/Matrix", [1, 0, 0, 1, 0, 0]))
                try:
                    form_matrix = tuple(numeric(value) for value in form_matrix_raw)
                    if len(form_matrix) != 6:
                        raise ValueError
                except Exception:
                    self.result.fail(
                        "FORM_MATRIX", f"{scope}/Form {resource_name} has malformed /Matrix"
                    )
                    form_matrix = (1, 0, 0, 1, 0, 0)
                form_state = replace(state)
                form_state.ctm = matrix_multiply(state.ctm, form_matrix)  # type: ignore[arg-type]
                try:
                    form_stream = ContentStream(dereference(reference), self.reader)
                except Exception as exc:
                    self.result.not_certified(
                        "FORM_PARSE", f"{scope}/Form {resource_name} cannot be parsed: {exc}"
                    )
                    continue
                self._analyze_operations(
                    form_stream.operations,
                    form_resources,
                    f"{scope}/Form {resource_name}",
                    stats,
                    form_state,
                    allowed_boxes,
                    form_stack | {identity},
                    implicit_scope=True,
                )
                continue

            if operator in INLINE_IMAGE_OPS:
                stats.inline_image_count += 1
                self.result.fail("INLINE_IMAGE", f"{scope} contains an inline image")
                self._record_clip_content(stats, state, "inline-image")
                continue
            if operator == b"sh":
                self.result.fail("SHADING", f"{scope} paints a shading object")
                self._record_clip_content(stats, state, "shading")
                continue

        if pending_clip:
            self.result.fail("CLIP_UNFINISHED", f"{scope} ends before its W/W* clip is applied")
        if path_ops:
            self.result.fail("PATH_UNFINISHED", f"{scope} ends with an unfinished path")
        if in_text:
            self.result.fail("TEXT_UNFINISHED", f"{scope} ends before ET")
        if stack:
            self.result.fail(
                "Q_UNBALANCED", f"{scope} has {len(stack)} q operator(s) without matching Q"
            )
        if state.clip_depth != base_clip_depth and not implicit_scope:
            self.result.fail(
                "CLIP_NOT_RESTORED",
                f"{scope} ends at clipping depth {state.clip_depth}, expected {base_clip_depth}",
            )
        if stats.text_object_count < text_objects_at_start:
            self.result.not_certified("INTERNAL_TEXT_COUNT", f"{scope} text-object accounting failed")

    def _validate_page_text(self, page: Any, stats: PdfPageStats) -> None:
        fragments: list[tuple[str, float, float, float, str]] = []
        parsed_font_names = {
            normalize_font_name(font.base_font) for font in stats.font_uses.values()
        }

        def visitor(text: str, cm: Sequence[float], tm: Sequence[float], font: Any, size: float) -> None:
            cleaned = text.strip()
            if not cleaned:
                return
            base_font = normalize_font_name(dictionary(font).get("/BaseFont", ""))
            try:
                x = float(cm[0]) * float(tm[4]) + float(cm[2]) * float(tm[5]) + float(cm[4])
                y = float(cm[1]) * float(tm[4]) + float(cm[3]) * float(tm[5]) + float(cm[5])
            except Exception:
                x = y = 0.0
            fragments.append((cleaned, x, y, float(size or 0), base_font))
            family = font_family(base_font)
            # pypdf can report a parent-page default font for text inside a Form
            # XObject. The content-stream pass above resolves the Form's actual
            # font. Apply script-specific checks only when the visitor font is a
            # font that the operation parser observed on a text-show operation.
            visitor_font_is_resolved = base_font in parsed_font_names
            if visitor_font_is_resolved and ASCII_ALNUM_RE.search(cleaned) and family != "Arial":
                self.result.fail(
                    "ASCII_FONT",
                    f"page {stats.page_number} text {cleaned!r} uses {base_font or '<unknown>'}, not Arial",
                )
            if visitor_font_is_resolved and CJK_RE.search(cleaned) and family != "SimSun":
                self.result.fail(
                    "CJK_FONT",
                    f"page {stats.page_number} Chinese text {cleaned!r} uses {base_font or '<unknown>'}, not SimSun",
                )

        try:
            page.extract_text(visitor_text=visitor)
        except Exception as exc:
            self.result.not_certified(
                "TEXT_EXTRACT", f"page {stats.page_number} live text cannot be decoded: {exc}"
            )
            return

        # Detect the common bad export in which a word or label becomes one text
        # object per character. Single tick labels are excluded by requiring a run.
        single_fragments = [
            item
            for item in fragments
            if len(item[0]) == 1 and (item[0].isalpha() or CJK_RE.search(item[0]))
        ]
        for index, current in enumerate(single_fragments):
            run = [current]
            for candidate in single_fragments[index + 1 :]:
                prior = run[-1]
                same_line = abs(candidate[2] - prior[2]) <= max(0.75, prior[3] * 0.35)
                same_font = candidate[4] == prior[4] and abs(candidate[3] - prior[3]) <= 0.25
                gap = candidate[1] - prior[1]
                nearby = -0.25 * max(prior[3], 1) <= gap <= 2.5 * max(prior[3], 1)
                if same_line and same_font and nearby:
                    run.append(candidate)
                    if len(run) >= 3:
                        word = "".join(item[0] for item in run)
                        self.result.fail(
                            "TEXT_FRAGMENTED",
                            f"page {stats.page_number} appears to split label {word!r} into individual character objects",
                        )
                        return
                elif candidate[2] < prior[2] - max(1.0, prior[3]):
                    break

    def _validate_page_summary(self, stats: PdfPageStats) -> None:
        clip_count = len(stats.clip_events)
        if clip_count > self.max_clips:
            self.result.fail(
                "TOO_MANY_CLIPS",
                f"page {stats.page_number} contains {clip_count} clipping paths; maximum is {self.max_clips}",
            )
        if self.expected_clips is not None and clip_count != self.expected_clips:
            self.result.fail(
                "CLIP_COUNT_EXPECTED",
                f"page {stats.page_number} contains {clip_count} clipping paths; expected exactly {self.expected_clips}",
            )
        if stats.max_clip_depth > 1:
            self.result.fail(
                "CLIP_DEPTH",
                f"page {stats.page_number} reaches clipping depth {stats.max_clip_depth}; maximum is 1",
            )
        for event in stats.clip_events:
            if not event.kinds:
                self.result.fail(
                    "EMPTY_CLIP",
                    f"page {stats.page_number} clip {event.number} contains no painted vector or live text content",
                )
        if stats.q_count != stats.Q_count:
            self.result.fail(
                "Q_COUNT",
                f"page {stats.page_number} recursively contains q={stats.q_count}, Q={stats.Q_count}",
            )
        if not self.allow_no_text and stats.text_show_count == 0:
            self.result.fail("NO_LIVE_TEXT", f"page {stats.page_number} contains no live text")
        if not self.allow_no_vector and stats.vector_paint_count == 0:
            self.result.fail("NO_VECTOR", f"page {stats.page_number} contains no painted vector path")


def validate_png(
    path: Path,
    target_dpi: float,
    dpi_tolerance: float,
    expected_size: tuple[float, float] | None,
) -> ValidationResult:
    result = ValidationResult(path, "PNG")
    if path.suffix.lower() != ".png":
        result.fail("PNG_EXTENSION", "filename does not end in .png")
    try:
        with Image.open(path) as image:
            if image.format != "PNG":
                result.fail("PNG_FORMAT", f"file content is {image.format}, not PNG")
            width, height = image.size
            dpi = image.info.get("dpi")
            image.load()
            blank_reason = image_blank_reason(image)
            if blank_reason:
                result.fail("PNG_BLANK", blank_reason)
    except Exception as exc:
        result.not_certified("PNG_DECODE", f"PNG cannot be decoded: {exc}")
        return result

    result.details.append(f"pixels={width}x{height}")
    if not dpi or len(dpi) < 2:
        result.fail("PNG_DPI_MISSING", "PNG has no two-axis physical-resolution metadata")
    else:
        x_dpi, y_dpi = float(dpi[0]), float(dpi[1])
        result.details.append(f"dpi={x_dpi:.3f}x{y_dpi:.3f}")
        if abs(x_dpi - target_dpi) > dpi_tolerance or abs(y_dpi - target_dpi) > dpi_tolerance:
            result.fail(
                "PNG_DPI",
                f"PNG resolution is {x_dpi:.3f}x{y_dpi:.3f} DPI; expected {target_dpi:g}+/-{dpi_tolerance:g}",
            )
        if expected_size:
            expected_width = expected_size[0] * target_dpi
            expected_height = expected_size[1] * target_dpi
            pixel_tolerance = max(1.0, target_dpi * 0.005)
            if abs(width - expected_width) > pixel_tolerance or abs(height - expected_height) > pixel_tolerance:
                result.fail(
                    "PNG_PIXEL_SIZE",
                    f"pixels {width}x{height} do not match {expected_size[0]:g}x{expected_size[1]:g} inches at {target_dpi:g} DPI",
                )
            result.details.append(
                f"expected-physical-size={expected_size[0]:g}x{expected_size[1]:g}in"
            )
        else:
            result.details.append(
                f"inferred-physical-size={width / x_dpi:.3f}x{height / y_dpi:.3f}in"
            )
    return result


def find_pdftoppm(explicit: str | None) -> str | None:
    if explicit:
        candidate = Path(explicit)
        return str(candidate) if candidate.is_file() else None
    return shutil.which("pdftoppm")


def _render_size_matches(
    observed: tuple[int, int], expected_points: tuple[float, float], dpi: float, rotation: int
) -> bool:
    width_points, height_points = expected_points
    if rotation in {90, 270}:
        width_points, height_points = height_points, width_points
    expected = (width_points * dpi / 72.0, height_points * dpi / 72.0)
    # Poppler normally rounds page boundaries outward. Accept at most one pixel
    # of normal raster-grid rounding in either dimension.
    return all(abs(actual - target) <= 1.05 for actual, target in zip(observed, expected))


def _prepare_render_directory(
    path: Path, result: ValidationResult, requested_base: Path | None
) -> tuple[Path | None, tempfile.TemporaryDirectory[str] | None]:
    if requested_base is not None:
        try:
            base = requested_base.expanduser().resolve(strict=False)
            if base.exists() and not base.is_dir():
                result.not_certified(
                    "RENDER_DIR_NOT_DIRECTORY", f"--render-dir is not a directory: {base}"
                )
                return None, None
            base.mkdir(parents=True, exist_ok=True)
            safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "-", path.stem).strip("-.") or "figure"
            # Always create a normal child directory so Windows applies the
            # parent's inherited ACL. Path.mkdir(exist_ok=False) is atomic and
            # cannot overwrite an existing entry; the retry loop handles the
            # extremely unlikely random-name collision.
            output = None
            for _ in range(20):
                candidate = base / f"{safe_stem[:60]}-render-{secrets.token_hex(4)}"
                try:
                    candidate.mkdir(exist_ok=False)
                except FileExistsError:
                    continue
                output = candidate
                break
            if output is None:
                result.not_certified(
                    "RENDER_DIR_COLLISION",
                    "could not allocate a unique render child after 20 attempts",
                )
                return None, None
        except OSError as exc:
            result.not_certified(
                "RENDER_DIR_CREATE", f"cannot create a unique child under --render-dir: {exc}"
            )
            return None, None
        result.details.append(f"render-output-dir={output} (retained; unique child, no overwrite)")
        return output, None

    task_tmp_root = Path.cwd() / "tmp"
    try:
        task_tmp_root.mkdir(parents=True, exist_ok=True)
        temporary = tempfile.TemporaryDirectory(prefix="validate-figure-", dir=task_tmp_root)
    except OSError:
        try:
            temporary = tempfile.TemporaryDirectory(prefix="validate-figure-")
        except OSError as exc:
            result.not_certified("RENDER_DIR_CREATE", f"cannot create a temporary render directory: {exc}")
            return None, None
    return Path(temporary.name), temporary


def render_pdf(
    path: Path,
    result: ValidationResult,
    page_stats: Sequence[PdfPageStats],
    executable: str | None,
    render_dpi: float,
    render_dir: Path | None,
) -> None:
    pdftoppm = find_pdftoppm(executable)
    if not pdftoppm:
        result.not_certified("POPPLER_MISSING", "pdftoppm was not found; visual rendering is mandatory")
        return
    output_dir, temporary = _prepare_render_directory(path, result, render_dir)
    if output_dir is None:
        return
    try:
        prefix = output_dir / "page"
        command = [
            pdftoppm,
            "-png",
            "-r",
            f"{render_dpi:g}",
            "-f",
            "1",
            "-l",
            str(len(page_stats)),
            str(path),
            str(prefix),
        ]
        try:
            completed = subprocess.run(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=max(60, len(page_stats) * 30),
                check=False,
            )
        except Exception as exc:
            result.not_certified("POPPLER_RUN", f"Poppler rendering could not run: {exc}")
            return
        if completed.returncode != 0:
            detail = completed.stderr.strip() or f"exit code {completed.returncode}"
            result.not_certified("POPPLER_RENDER", f"Poppler failed to render the PDF: {detail}")
            return
        rendered: dict[int, Path] = {}
        for candidate in output_dir.glob("page-*.png"):
            match = re.fullmatch(r"page-(\d+)\.png", candidate.name, flags=re.IGNORECASE)
            if match:
                rendered[int(match.group(1))] = candidate
        if len(rendered) != len(page_stats) or set(rendered) != set(
            range(1, len(page_stats) + 1)
        ):
            result.not_certified(
                "POPPLER_PAGE_COUNT",
                f"Poppler produced page numbers {sorted(rendered)}, expected 1..{len(page_stats)}",
            )
            return
        for stats in page_stats:
            output = rendered[stats.page_number]
            try:
                with Image.open(output) as image:
                    image.verify()
                with Image.open(output) as image:
                    image.load()
                    observed_size = image.size
                    blank_reason = image_blank_reason(image)
            except Exception as exc:
                result.not_certified("POPPLER_OUTPUT", f"rendered page {output.name} is invalid: {exc}")
                continue
            if stats.rotation not in {0, 90, 180, 270}:
                result.not_certified(
                    "PAGE_ROTATION",
                    f"page {stats.page_number} has unsupported /Rotate={stats.rotation}",
                )
                matched_box = None
            elif _render_size_matches(
                observed_size, stats.crop_size_points, render_dpi, stats.rotation
            ):
                matched_box = "CropBox"
            elif _render_size_matches(
                observed_size, stats.media_size_points, render_dpi, stats.rotation
            ):
                matched_box = "MediaBox"
            else:
                matched_box = None
                result.fail(
                    "RENDER_PIXEL_SIZE",
                    f"page {stats.page_number} rendered to {observed_size[0]}x{observed_size[1]} px at {render_dpi:g} DPI; "
                    f"this matches neither CropBox {stats.crop_size_points[0]:.3f}x{stats.crop_size_points[1]:.3f} pt "
                    f"nor MediaBox {stats.media_size_points[0]:.3f}x{stats.media_size_points[1]:.3f} pt",
                )
            if blank_reason:
                result.fail(
                    "RENDER_BLANK",
                    f"page {stats.page_number} rendered blank: {blank_reason}",
                )
            match_text = matched_box or "no-box-match"
            content_text = "blank" if blank_reason else "nonblank"
            result.details.append(
                f"render-page={stats.page_number} pixels={observed_size[0]}x{observed_size[1]} "
                f"dpi={render_dpi:g} box={match_text} content={content_text}"
            )
        result.details.append(f"poppler-rendered-pages={len(rendered)}")
    finally:
        if temporary is not None:
            temporary.cleanup()


def validate_pdf(path: Path, args: argparse.Namespace) -> ValidationResult:
    result = ValidationResult(path, "PDF")
    if path.suffix.lower() != ".pdf":
        result.fail("PDF_EXTENSION", "filename does not end in .pdf")
    auditor = PdfAuditor(
        path,
        result,
        max_clips=args.max_clips,
        expected_clips=args.expected_clips,
        declared_clip_boxes=args.clip_box,
        allow_raster_images=args.allow_raster_images,
        allow_form_xobjects=args.allow_form_xobjects,
        allow_no_text=args.allow_no_text,
        allow_no_vector=args.allow_no_vector,
    )
    page_stats = auditor.audit()
    if page_stats:
        for stats in page_stats:
            fonts = sorted({font.base_font for font in stats.font_uses.values()})
            clip_summary = "; ".join(
                f"#{event.number} depth={event.depth} kinds={'+'.join(sorted(event.kinds)) or 'empty'}"
                for event in stats.clip_events
            )
            result.details.append(
                f"page={stats.page_number} clips={len(stats.clip_events)} max-depth={stats.max_clip_depth} "
                f"q/Q={stats.q_count}/{stats.Q_count} text-shows={stats.text_show_count} "
                f"vector-paints={stats.vector_paint_count} images={stats.image_count + stats.inline_image_count} "
                f"forms={stats.form_calls} fonts={','.join(fonts) or 'none'}"
            )
            if clip_summary:
                result.details.append(f"page={stats.page_number} clip-groups: {clip_summary}")
        render_pdf(
            path,
            result,
            page_stats,
            args.pdftoppm,
            args.render_dpi,
            args.render_dir,
        )
    return result


def parse_box(value: str) -> tuple[float, float, float, float]:
    try:
        fields = tuple(float(part.strip()) for part in value.split(","))
    except ValueError as exc:
        raise argparse.ArgumentTypeError("clip box must be x,y,width,height") from exc
    if len(fields) != 4 or fields[2] <= 0 or fields[3] <= 0:
        raise argparse.ArgumentTypeError("clip box must be x,y,width,height with positive size")
    return fields  # type: ignore[return-value]


def parse_size(value: str) -> tuple[float, float]:
    match = re.fullmatch(r"\s*([0-9]*\.?[0-9]+)\s*[xX×]\s*([0-9]*\.?[0-9]+)\s*", value)
    if not match:
        raise argparse.ArgumentTypeError("expected size must be WIDTHxHEIGHT in inches")
    return (float(match.group(1)), float(match.group(2)))


def positive_float(value: str) -> float:
    try:
        parsed = float(value)
    except ValueError as exc:
        raise argparse.ArgumentTypeError("value must be a positive number") from exc
    if not math.isfinite(parsed) or parsed <= 0:
        raise argparse.ArgumentTypeError("value must be a finite positive number")
    return parsed


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Validate 600-DPI PNGs or Illustrator-editable vector figure PDFs."
    )
    parser.add_argument("files", nargs="+", type=Path)
    parser.add_argument("--target-dpi", type=float, default=600.0)
    parser.add_argument("--dpi-tolerance", type=float, default=1.0)
    parser.add_argument(
        "--expected-size-in",
        type=parse_size,
        metavar="WIDTHxHEIGHT",
        help="also verify PNG pixel dimensions against a physical size in inches",
    )
    parser.add_argument("--max-clips", type=int, default=4, choices=range(0, 5))
    parser.add_argument(
        "--expected-clips",
        type=int,
        choices=range(0, 5),
        help="require exactly this many clipping paths on every PDF page (use 3 for the full-period regression profile)",
    )
    parser.add_argument(
        "--clip-box",
        type=parse_box,
        action="append",
        default=[],
        metavar="X,Y,W,H",
        help="declare an allowed non-page rectangular PDF clipping box (repeatable)",
    )
    parser.add_argument(
        "--allow-raster-images",
        action="store_true",
        help="allow intentional image XObjects; masks and inline images remain forbidden",
    )
    parser.add_argument(
        "--allow-form-xobjects",
        action="store_true",
        help="allow recursively inspected Form XObjects; the strict default rejects all Forms",
    )
    parser.add_argument("--allow-no-text", action="store_true")
    parser.add_argument("--allow-no-vector", action="store_true")
    parser.add_argument("--pdftoppm", help="explicit path to Poppler pdftoppm")
    parser.add_argument(
        "--render-dpi",
        type=positive_float,
        default=72.0,
        help="Poppler verification resolution (default: 72 DPI)",
    )
    parser.add_argument(
        "--render-dir",
        type=Path,
        help="retain rendered pages in a newly created unique child of DIR; existing contents are never overwritten",
    )
    return parser


def print_result(result: ValidationResult) -> None:
    print(f"[{result.status}] {result.path}")
    for detail in result.details:
        print(f"  INFO {detail}")
    for issue in result.issues:
        label = "ERROR" if issue.outcome == "FAIL" else "BLOCKER"
        print(f"  {label} [{issue.code}] {issue.message}")


def main(argv: Sequence[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    if args.expected_clips is not None and args.expected_clips > args.max_clips:
        parser.error("--expected-clips cannot exceed --max-clips")
    results: list[ValidationResult] = []
    for raw_path in args.files:
        path = raw_path.resolve()
        if not path.is_file():
            result = ValidationResult(path, "UNKNOWN")
            result.not_certified("FILE_MISSING", "file does not exist or is not a regular file")
        elif path.suffix.lower() == ".png":
            result = validate_png(path, args.target_dpi, args.dpi_tolerance, args.expected_size_in)
        elif path.suffix.lower() == ".pdf":
            result = validate_pdf(path, args)
        else:
            result = ValidationResult(path, "UNKNOWN")
            result.not_certified("FILE_TYPE", "only .png and .pdf are supported")
        results.append(result)
        print_result(result)
    statuses = Counter(result.status for result in results)
    passed = statuses["PASS"]
    print(
        f"SUMMARY passed={passed} failed={statuses['FAIL']} "
        f"not-certified={statuses['NOT CERTIFIED']} total={len(results)}"
    )
    return 0 if passed == len(results) else 1


if __name__ == "__main__":
    raise SystemExit(main())


## Existing panel drawing: unchanged source and layout

The following existing notebook cells are copied unchanged, including original shared figure outputs. Use this section in its original context; similarly named globals in other sections are not a new shared execution API.


# Figure 4A: developmental regulon dynamics

Draw the accepted 61-regulon, nine-branch panel from the saved fitted activities. The 58 original representatives and three prespecified follow-ups use the original ordering, normalization and significance annotations.

Replace each file and directory placeholder before running.


In [ ]:
#1. =========== Saved result and output locations
from pathlib import Path
import os
import sys
import json
import hashlib
import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager

ROOT = Path(".")  # Set the working root and replace the individual placeholders below.
# Set Deve_regulon_dynamics.h5, produced by the frozen AUCell/GAM workflow in 05_Deve_regulon_dynamics.ipynb.
# The supplement_add3 group contains the accepted Nhlh1, Neurod1 and Tbx3 follow-up results.
RESULT_FILE = ROOT
PNG_FILE = ROOT  # Set the output PNG file path for Fig4A.
PDF_FILE = ROOT  # Set the output PDF file path for Fig4A.
CACHE_DIR = ROOT  # Set the temporary plotting-cache directory.
FONT_FILES = {
    "regular": ROOT,  # Set the Arial regular font file path.
    "italic": ROOT,  # Set the Arial italic font file path.
    "bold": ROOT,  # Set the Arial bold font file path.
    "bolditalic": ROOT,  # Set the Arial bold italic font file path.
}
DPI = 600
MAIN_TITLE = "Developmental regulon dynamics"
SEED = 42
Q_CUT = 1e-4
sys.dont_write_bytecode = True
CACHE_DIR.mkdir(parents=True, exist_ok=True)
for key in ("TMP", "TEMP", "MPLCONFIGDIR"):
    os.environ[key] = str(CACHE_DIR)
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
for font_path in FONT_FILES.values():
    font_manager.fontManager.addfont(str(font_path))
assert font_manager.findfont("Arial", fallback_to_default=False)
np.random.seed(SEED)


In [ ]:
#2. =========== Frozen-result readers
ADD3_GROUP = 'supplement_add3'
FOLLOWUP_NAMES = ['Nhlh1(+)', 'Neurod1(+)', 'Tbx3(+)']

def read_frame(group):
    return pd.DataFrame({c:(group[c].asstr()[()] if group[c].dtype.kind in 'OSU' else group[c][()])
        for c in group['columns'].asstr()[()]}, index=group['index'].asstr()[()])

def column_frame(group):
    return pd.DataFrame({k:(v.asstr()[()] if v.dtype.kind in 'OSU' else v[()])
        for k,v in group.items() if isinstance(v,h5py.Dataset) and not k.endswith('__missing')})

def load_plot_inputs(result_file):
    with h5py.File(result_file,'r') as h:
        g=h[ADD3_GROUP];assert g.attrs['complete']
        original_names=h['regulons'].asstr()[()].tolist()
        names=original_names+g['regulons'].asstr()[()].tolist()
        stats=read_frame(g['combined_statistics']).loc[names]
        data=dict(names=pd.Index(names),segments=json.loads(h['segments_json'].asstr()[()]),
            bin_meta=column_frame(h['bin_metadata']),
            modes={'none':{'bins':pd.DataFrame(np.column_stack([h['final/all_fitted_bin_auc'][()],g['fitted_bin_auc'][()]]),columns=names)}},
            early_mean=pd.DataFrame(np.column_stack([h['final/all_early_observed_auc'][()],g['early_group_mean_auc'][()]]),columns=names),
            early_meta=column_frame(h['early_extension/group_metadata']))
        original_order=h['display_selection/regulons'].asstr()[()].tolist()
        selected=original_order+FOLLOWUP_NAMES
        fitted=data['modes']['none']['bins'].loc[:,selected].to_numpy()
        peaks=np.argmax(fitted,axis=0)
        weights=np.maximum(fitted-fitted.min(axis=0),0)
        centres=(weights*np.arange(30)[:,None]).sum(axis=0)/weights.sum(axis=0)
        selected=[selected[k] for k in sorted(range(len(selected)),key=lambda k:(peaks[k],centres[k],selected[k]))]
        assert len(selected)==61 and len(set(selected))==61
        assert set(selected)==set(original_order)|set(FOLLOWUP_NAMES)
        assert np.all(np.diff(np.argmax(data['modes']['none']['bins'].loc[:,selected],axis=0))>=0)
        np.testing.assert_array_equal(data['modes']['none']['bins'].loc[:,original_order],
            h['final/all_fitted_bin_auc'][()][:,pd.Index(original_names).get_indexer(original_order)])
    return data,stats,selected


In [ ]:
#3. =========== Read the accepted 61-column display
data, stats, order = load_plot_inputs(RESULT_FILE)
assert len(order) == 61 and len(set(order)) == 61


In [ ]:
#5. =========== Appearance parameters and editable vector renderer
# These settings change presentation only; all primary results stay frozen.
MAX_COLUMNS_PER_PANEL=72
EXPORT_EARLY=False       # Optional observed early panel; independent normalization, not the fitted-tree metric.
SEGMENT_LABELS={0:'Undiff. CVG',1:'CVG to VGN',2:'CVG to Uns.',5:'Ib/Ic shared',
    9:'Ib-related end',10:'Ic-related end',6:'Ia/II shared',7:'Ia-related end',8:'II-related end'}
# Preserve every original bin. Only join consecutive pieces on a complete path.
DISPLAY_BRANCHES=[
    dict(id=0,parent=None,source_segments=[0],connector_segments=[]),
    dict(id=1,parent=0,source_segments=[1],connector_segments=[]),
    dict(id=2,parent=0,source_segments=[2,4],connector_segments=[3]),
    dict(id=5,parent=2,source_segments=[5],connector_segments=[]),
    dict(id=9,parent=5,source_segments=[9],connector_segments=[]),
    dict(id=10,parent=5,source_segments=[10],connector_segments=[]),
    dict(id=6,parent=2,source_segments=[6],connector_segments=[]),
    dict(id=7,parent=6,source_segments=[7],connector_segments=[]),
    dict(id=8,parent=6,source_segments=[8],connector_segments=[])]
PLOT_STYLE=dict(
    column_pitch_in=.128, row_pitch_in=.100, segment_gap_in=.045,
    dot_diameter_ratio=.81, dot_edge_width=.32,
    gene_fontsize=6.0, row_fontsize=5.6, title_fontsize=11.0,
    subtitle_fontsize=5.7, footer_fontsize=4.8, legend_fontsize=5.5, legend_tick_fontsize=5.0,
    main_left_in=1.27, early_left_in=2.35, top_in=.62, bottom_in=1.24,
    tree_end_offset_in=.77, tree_primary_lane_in=.22, tree_secondary_lane_in=.375,
    gradient_steps=128)

def render_add3(data,stats,order):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_rgb,LinearSegmentedColormap
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle,Circle
    plt.rcParams.update({'font.family':'Arial','font.size':6,'axes.linewidth':.45,
                         'savefig.dpi':DPI,'figure.dpi':120,'pdf.fonttype':42,
                         'pdf.use14corefonts':False,'pdf.compression':9})
    selected=pd.Index(order)
    fitted=data['modes']['none']['bins'].loc[:,selected].to_numpy()
    assert np.isfinite(fitted).all(), 'Missing fitted AUC must not be displayed as zero.'
    maxima=fitted.max(axis=0)
    assert np.all(maxima>0)
    raw_relative=fitted/maxima
    # Floor only the display of negative Gaussian predictions at the AUC lower bound.
    # Keep raw fitted values, tests, diagnostics and missing-data semantics unchanged.
    negative=fitted < -1e-12
    plotted=np.maximum(raw_relative,0)
    assert np.nanmax(plotted)<=1+1e-12
    early=data['early_mean'].loc[:,selected].to_numpy()
    assert np.isfinite(early).all(), 'Missing observed AUC must not be displayed as zero.'
    early_max=early.max(axis=0)
    early_relative=np.divide(early,early_max,out=np.zeros_like(early),where=early_max>0)
    red=stats.loc[selected,['q_early','q_fork1','q_fork2','q_fork3']].min(axis=1).lt(Q_CUT).to_numpy()
    references=stats.loc[selected,'charles_reference'].to_numpy(dtype=bool)
    cmap=LinearSegmentedColormap.from_list('white_red',['white','#ff0000'])
    dpi=DPI;col_pitch=PLOT_STYLE['column_pitch_in'];row_pitch=PLOT_STYLE['row_pitch_in'];gap=PLOT_STYLE['segment_gap_in']
    # A complete-branch display groups existing rows without resampling or refitting.
    branches=[];row_order=[];Y_main=np.empty(len(fitted),float);cursor=0.
    for specification in DISPLAY_BRANCHES:
        branch=dict(specification);rows=[]
        for sid in branch['source_segments']:
            rows.extend(np.flatnonzero(data['bin_meta'].seg.to_numpy()==sid).tolist())
        branch.update(label=SEGMENT_LABELS[branch['id']],rows=rows,
            row_count=len(rows),axis_top=float(cursor))
        Y_main[rows]=np.arange(len(rows))+.5+cursor
        row_order.extend(rows);branches.append(branch)
        cursor+=len(rows)+gap/row_pitch
    assert len(branches)==9 and [b['row_count'] for b in branches]==[3,3,6,3,3,3,3,3,3]
    np.testing.assert_array_equal(row_order,np.arange(len(fitted)))
    assert not data['bin_meta'].seg.eq(3).any(), 'Unmodeled connector cells must not acquire AUC points.'
    source_segments={s['id']:s for s in data['segments']}
    output=[];checks={}
    def draw(indices,path,early_panel=False,part=None):
        n=len(indices);left=PLOT_STYLE['early_left_in'] if early_panel else PLOT_STYLE['main_left_in']
        top=PLOT_STYLE['top_in'];bottom=PLOT_STYLE['bottom_in']
        nrows=7 if early_panel else 30
        body=nrows*row_pitch+(0 if early_panel else (len(branches)-1)*gap)
        width=left+n*col_pitch+.09;height=top+body+bottom
        fig=plt.figure(figsize=(width,height),facecolor='white')
        ax=fig.add_axes([left/width,bottom/height,n*col_pitch/width,body/height])
        ax.set_xlim(0,n);ax.set_ylim(body/row_pitch,0)
        ax.set_xticks(np.arange(n)+.5)
        labels=[selected[i] for i in indices]
        ax.set_xticklabels(labels,rotation=90,ha='center',va='top',fontsize=PLOT_STYLE['gene_fontsize'],fontstyle='italic')
        ax.tick_params(axis='x',length=0,pad=2)
        ax.set_yticks([])
        for spine in ax.spines.values():spine.set_visible(False)
        Y=np.arange(nrows)+.5
        if not early_panel:
            Y=Y_main.copy()
        vals=early_relative[:,indices] if early_panel else plotted[:,indices]
        x,y=np.meshgrid(np.arange(n)+.5,Y)
        diameter=PLOT_STYLE['dot_diameter_ratio']*row_pitch*72
        assert np.isfinite(vals).all()
        size=diameter**2*vals
        # Use independent closed paths in physical inches: no marker Forms.
        for r,c in np.ndindex(vals.shape):
            xpos=left+x[r,c]*col_pitch
            ypos=bottom+body-y[r,c]*row_pitch
            value=float(vals[r,c])
            radius=np.sqrt(float(size[r,c]))/(2*72)
            if value<=0:
                radius=np.sqrt(.18)/(2*72)
                face='#555555';edge='none';line=0
            else:
                face=cmap(value);edge='#4c4c4c';line=PLOT_STYLE['dot_edge_width']
            fig.add_artist(Circle((xpos,ypos),radius=radius,transform=fig.dpi_scale_trans,
                facecolor=face,edgecolor=edge,linewidth=line,clip_on=False,zorder=3))
        for tick,i in zip(ax.get_xticklabels(),indices):
            tick.set_color('#dc2638' if red[i] else '#111111')
            tick.set_fontweight('bold' if references[i] else 'normal')
        title='Observed early regulon activity' if early_panel else MAIN_TITLE
        suffix='' if part is None else f' ({part})'
        fig.text(.12/width,1-.23/height,('' if early_panel else 'a  ')+title+suffix,
                 fontsize=PLOT_STYLE['title_fontsize'],va='center')
        subtitle=('E9.5-E13.5; equal weight per library within each stage/state; observed AUC' if early_panel else
                  f'E9.5-E18.5; 58 original + 3 follow-up regulons across 9 complete branches')
        fig.text(.12/width,1-.47/height,subtitle,fontsize=PLOT_STYLE['subtitle_fontsize'],va='center')
        if early_panel:
            label_lines=['E9.5  CVG neuroblast','E10.5  delaminating otic/CVG','E10.5  developing CVG',
                         'E11.5  proliferative CVG','E11.5  differentiating SGN',
                         'E13.5  late CVG','E13.5  differentiating SGN']
            for r,label in enumerate(label_lines):
                yc=(bottom+body-Y[r]*row_pitch)/height
                m=data['early_meta'].iloc[r]
                fig.text((left-.09)/width,yc,label+f' ({int(m.cells)}; {int(m.libraries)})',
                         ha='right',va='center',fontsize=PLOT_STYLE['row_fontsize'])
            ax.add_patch(Rectangle((0,0),n,nrows,fill=False,edgecolor='#333333',lw=.45,clip_on=False))
        else:
            centre={}
            for branch in branches:
                y0=branch['axis_top'];nr=branch['row_count']
                ax.add_patch(Rectangle((0,y0),n,nr,fill=False,edgecolor='#333333',lw=.45,clip_on=False))
                yb=(bottom+body-(y0+nr)*row_pitch)/height
                strip=fig.add_axes([(left-.15)/width,yb,.065/width,nr*row_pitch/height])
                strip.set_xlim(0,1);strip.set_ylim(0,1)
                steps=PLOT_STYLE['gradient_steps'];piece_start=0
                # Preserve each native color transition inside the joined branch strip.
                for sid in branch['source_segments']:
                    source=source_segments[sid]
                    count=int(data['bin_meta'].seg.eq(sid).sum())
                    colors=np.array([to_rgb(source['color_from']),to_rgb(source['color_to'])])
                    for step in range(steps):
                        weight=step/(steps-1)
                        face=colors[0]*(1-weight)+colors[1]*weight
                        top_fraction=(piece_start+count*(step+1)/steps)/nr
                        strip.add_patch(Rectangle((0,1-top_fraction),1,count/(steps*nr),
                            transform=strip.transAxes,facecolor=face,edgecolor='none',
                            linewidth=0,antialiased=False,clip_on=False))
                    piece_start+=count
                strip.set_xticks([]);strip.set_yticks([])
                for spine in strip.spines.values():spine.set_linewidth(.4)
                centre[branch['id']]=(bottom+body-(y0+nr/2)*row_pitch)/height
                fig.text((left-.205)/width,centre[branch['id']],branch['label'],
                         ha='right',va='center',fontsize=PLOT_STYLE['row_fontsize'])
            end=(left-PLOT_STYLE['tree_end_offset_in'])/width
            for parent in (0,2,5,6):
                children=[b['id'] for b in branches if b['parent']==parent]
                lane=PLOT_STYLE['tree_primary_lane_in' if parent in (0,2) else 'tree_secondary_lane_in']/width
                ys=[centre[parent]]+[centre[c] for c in children]
                fig.add_artist(Line2D([lane,lane],[min(ys),max(ys)],transform=fig.transFigure,color='#444444',lw=.5))
                for sid in [parent]+children:
                    fig.add_artist(Line2D([lane,end],[centre[sid],centre[sid]],transform=fig.transFigure,
                                         color='#444444',lw=.5))
        ly=.42
        legend_left=.14
        fig.text(legend_left/width,(ly+.17)/height,
                 'relative observed AUC' if early_panel else 'relative fitted AUC',fontsize=PLOT_STYLE['legend_fontsize'],va='center')
        base=1.44
        for j,value in enumerate((0,.25,.5,.75,1)):
            xpos=base+j*.29
            fig.add_artist(Circle((xpos,ly+.17),radius=(np.sqrt(.18) if value==0 else diameter*np.sqrt(value))/(2*72),
                facecolor='#555555' if value==0 else cmap(float(value)),
                edgecolor='none' if value==0 else '#4c4c4c',linewidth=0 if value==0 else PLOT_STYLE['dot_edge_width'],
                transform=fig.dpi_scale_trans,clip_on=False,zorder=3))
            fig.text(xpos/width,ly/height,f'{value:g}',fontsize=PLOT_STYLE['legend_tick_fontsize'],ha='center',va='top')
        note1=('Each regulon is max-normalized across seven observed groups; row brackets give cells; libraries.' if early_panel else
               'Max-normalized across 30 fitted bins; CVG to Uns. joins two fitted pieces (six rows).')
        note2='Bold: Charles Fig3a TF match; red: fork BH880 q < 1e-4. Three preselected follow-up regulons; primary BH220.'
        for pos,note in zip((.18,.06),(note1,note2)):
            fig.text(.14/width,pos/height,note,fontsize=PLOT_STYLE['footer_fontsize'],va='bottom')
        # Clear default artist clipping; no clipping is needed for this layout.
        for axis in fig.axes:
            for axis_object in (axis.xaxis,axis.yaxis):
                for tick in axis_object.get_major_ticks()+axis_object.get_minor_ticks():
                    tick.tick1line.set_visible(False);tick.tick2line.set_visible(False)
        for artist in fig.findobj():
            if hasattr(artist,'set_clip_on'):artist.set_clip_on(False)
            if hasattr(artist,'set_clip_box'):artist.set_clip_box(None)
            if hasattr(artist,'set_clip_path'):artist.set_clip_path(None)
        fig.canvas.draw()
        renderer=fig.canvas.get_renderer()
        texts=list(fig.texts)+list(ax.get_xticklabels())
        boxes=[(t,t.get_window_extent(renderer)) for t in texts if t.get_text()]
        canvas=fig.bbox
        outside=[t.get_text() for t,b in boxes if b.x0<-.5 or b.y0<-.5 or b.x1>canvas.x1+.5 or b.y1>canvas.y1+.5]
        overlaps=[]
        for i,(t,b) in enumerate(boxes):
            for u,c in boxes[i+1:]:
                if min(b.x1,c.x1)-max(b.x0,c.x0)>.5 and min(b.y1,c.y1)-max(b.y0,c.y0)>.5:
                    overlaps.append([t.get_text(),u.get_text()])
        assert not outside,(path.name,'text outside canvas',outside)
        assert not overlaps,(path.name,'text overlap',overlaps)
        pdf_path=PDF_FILE
        fig.savefig(path,dpi=dpi,facecolor='white')
        fig.savefig(pdf_path,facecolor='white',metadata={'Title':MAIN_TITLE,'Creator':'05_Deve_regulon_dynamics.ipynb','CreationDate':None,'ModDate':None})
        checks[path.name]=dict(columns=n,rows=nrows,complete_branches=None if early_panel else len(branches),
            branch_row_counts=None if early_panel else [b['row_count'] for b in branches],
            dpi=dpi,width_in=width,height_in=height,
            text_outside=outside,text_overlap=overlaps,pdf=pdf_path.name,expected_clips=0,
            bytes=path.stat().st_size,pdf_bytes=pdf_path.stat().st_size)
        output.extend([path.name,pdf_path.name]);plt.close(fig)
    draw(np.arange(len(selected)),PNG_FILE)
    return dict(files=output,figures=checks,negative_fitted_bin_count=int(negative.sum()),
                negative_bins=[dict(segment=int(data['bin_meta'].iloc[r].seg),bin=int(data['bin_meta'].iloc[r]['bin']),
                                    regulon=selected[c],raw_mean=float(fitted[r,c])) for r,c in np.argwhere(negative)],
                display_zero_tolerance=1e-12,normalization='per regulon maximum across fitted 30 bins; early observed independent maximum across 7 groups',
                label_red='any fork BH880 q<1e-4; not a primary gate',
                reference_mark='bold italic: Charles Fig3a TF-name match',
                segment_labels=SEGMENT_LABELS,display_branches=branches,
                display_row_positions=Y_main.tolist(),original_row_order=row_order,
                diagnostic_symbols_shown=False,
                negative_display_policy='max(raw fitted AUC,0), then per-regulon maximum normalization; ordinary zero-activity dot',
                display_floor_applied_bin_count=int((fitted<0).sum()),
                diagnostics_location='final/statistics and display_selection/evidence_json; raw values retained')

In [ ]:
#5. =========== Export only the main developmental panel
figure_qa = render_add3(data, stats, order)
assert len(figure_qa["display_branches"]) == 9
assert figure_qa["original_row_order"] == list(range(30))
assert len(figure_qa["figures"]) == 1
from IPython.display import display, Image
display(Image(filename=str(PNG_FILE), width=1000))
